# ARC-AGI-2: Qwen3.8-Flash-Next (expert-pruned 384) on 4x L4, program-synthesis agent

Serving: Pennyroyal SGLang (Franzen's ARC-AGI-3 stack, Tufa Labs' Duck), Intel AutoRound W4A16, RadixArk FP8 PLE. Harness: Duck/Tufa-style python + run_tests tool loop (harness v2).


In [1]:
# [duck2:setup]  config, inputs, task selection, fallback submission.json
import os, sys, json, time, re, shutil, subprocess, signal, socket, threading, urllib.request, urllib.error, fnmatch
from pathlib import Path
NOTEBOOK_START_TIME = T0 = time.time()
CFG = json.loads('{"AGENT":{"ARGS":{"act-tokens":8192,"answer-reserve":0,"budget-note":"on","cap-stages":"150000,200000","context-mode":"pin","continue-cut":"on","continue-images":"chat","ctx-limit":110000,"effort":"xhigh","empty-retry":2,"est-session-rate":10,"exec-timeout":30,"hand-check-rule":"on","hide-budget":"on","images":"on","max-sessions-per-task":1,"max-tool-out":6000,"max-turns":120,"metrics-interval":30,"pin-keep":0.8,"propose":"on","resume":"continue","review-outputs":"on","salvage":"on","salvage-fix":"on","salvage-mode":"tool","salvage-tokens":6144,"salvage-tools":"session","salvage-turns":2,"sandbox-help":"on","second-opinion":"off","session-cap":100000,"submit-partial":"on","task-cap":100000,"temperature":0.7,"top-k":20,"top-p":0.95,"turn-max-tokens":32768,"workspace":"on","wrapup-s":240,"answer-mode":"direct","geo-kit":"on","two-attempts":"on","fresh-second":"on","fresh-min-tokens":80000,"fresh-cap":100000,"cap-action":"fresh","force-hypothesis":"on","pin-keep-reasoning":1500,"geo-v2":"on","geo-fits":"on","ws-import-fix":"on","task-kinds":"on","fresh-gate":"on","fresh-temperature":1.0},"CTX_PER_SESSION":44000,"HEALTH_FAILS_MAX":3,"LBPROBE":false,"LEAK_CHECK_S":60,"LOAD_STATE":true,"MAX_INFLIGHT":41,"MAX_RESTARTS":4,"MIN_SOLVE_S":600,"POLL_S":20,"RESTART_MIN_S":900,"SOLVE_MIN":22,"STALL_MIN":10,"STATUS_MIN":3},"BRINGUP":[{"KV":"fp8_e4m3","LAYOUT":"tp4","MAMBA":104,"MAMBA_STRATEGY":"extra_buffer_lazy","MAXREQ":32,"MEMFRAC":0.95,"NAME":"tp4_p384_mtp","SKIP_LOCK":true,"SPEC":true,"SWEEP":[16,32]},{"KV":"fp8_e4m3","LAYOUT":"tp4","MAMBA":152,"MAMBA_STRATEGY":"extra_buffer_lazy","MAXREQ":48,"MEMFRAC":0.95,"NAME":"tp4_p384_nomtp","SKIP_LOCK":true,"SPEC":false,"SWEEP":[16,32,48]}],"CHUNK":1024,"CTX":147456,"CTX_MAX":48000,"CTX_MIN":28000,"DP_GPU_ORDER":"0,2,1,3","DRIVER_TEST":true,"END_MARGIN_S":1200,"EP":4,"EST_TOK_S":150,"FATAL_GRACE_S":45,"FRSPEC":false,"FULL_BUDGET_S":43200,"GREEDY_TOKENS":512,"HARNESS":{"answer-reserve":0,"ctx-limit":24000,"effort":"xhigh","images":"on","max-sessions-per-task":2,"max-turns":40,"review-outputs":"on","session-cap":60000,"task-cap":150000,"temperature":0.7,"top-k":20,"top-p":0.95,"turn-max-tokens":12288},"HEALTH_FAILS_MAX":5,"JIT_EXPORT":{"DIRS":["triton","tilelang","torchinductor","flashinfer","cuda","sglang"],"MAX_GB":4,"OUT":"jit_cache"},"KV_USE":0.75,"LADDER":[{"KV":"fp8_e4m3","MAMBA":24,"MAXREQ":4,"MEMFRAC":0.97,"NAME":"fp8kv_mtp","SPEC":true},{"KV":"fp8_e4m3","MAMBA":36,"MAXREQ":6,"MEMFRAC":0.95,"NAME":"fp8kv_nomtp","SPEC":false},{"KV":"auto","MAMBA":24,"MAXREQ":4,"MEMFRAC":0.95,"NAME":"bf16kv_nomtp","SPEC":false},{"KV":"auto","MAMBA":24,"MAXREQ":4,"MEMFRAC":0.93,"NAME":"safe","NO_GRAPH":true,"QSA":"torch","SPEC":false}],"MAX_INFLIGHT":8,"MAX_RESTARTS":2,"MIN_INFLIGHT_MTP":2,"MIN_SOLVE_S":600,"MONITOR_MIN":3,"N_SMOKE":40,"PHASE0_BENCHES":[],"PHASE0_S":420,"PHASE0_TRUSTED":{},"PLE_LOCAL":{"DIR":"/tmp/ple_local","EARLY":true,"THREADS":32,"WAIT_MIN":150},"PLE_MODE":"auto","POLL_S":15,"PORT":8000,"PRECACHE_MODEL":true,"PROBE_TOKENS":1024,"PRUNE_N":384,"QSA_GPU_TEST":true,"QUICK_VERSION":true,"RESTART_MIN_S":4500,"RUNGS":[{"CRASH_NEXT":true,"ENV":{"SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION":"8192","SGLANG_EMBED_HOST":"1","SGLANG_HC_MIX_MAX_ROWS":"64","SGLANG_INIT_NEW_TOKEN_RATIO":"1.0","SGLANG_MAMBA_PREFILL_FINAL_ONLY_LAZY":"1","SGLANG_MAMBA_SLOTS_PER_REQ":"2","SGLANG_MIN_NEW_TOKEN_RATIO_FACTOR":"0.8","SGLANG_MM_PRECOMPUTE_HASH":"1","SGLANG_QSA_MQA_GROUPS_PER_CTA":"4","SGLANG_QSA_PAIR_OWN":"1","SGLANG_QSA_PAIR_OWN_GROUP":"pair","SGLANG_QSA_SM89_DECODE":"triton_splitk","SGLANG_SM120_ONLINE_MXFP8":"1","SGLANG_TOPK_LOGITS_EXCHANGE":"32","SGLANG_VLM_CACHE_SIZE_MB":"24","SGLANG_INPROJ_FP8":"1","SGLANG_FP8_FUSED_DEQUANT":"1"},"GRAPH_BS":[8,12,14,15,16,17,18,19,20,22,24,26,28,30,32,33,34,36,38,40,41],"KV":"fp8_e4m3","LAYOUT":"tp4","LMONLY":false,"MAMBA":104,"MAMBA_STRATEGY":"extra_buffer_lazy","MAXREQ":41,"MEMFRAC":0.935,"NAME":"tp4_p384_p41cap_hic","SERVE_ARGS":["--max-total-tokens","1850000","--mamba-max-states-per-path","1","--enable-hierarchical-cache","--hicache-ratio","2"],"SKIP_LOCK":true,"SPEC":false},{"ENV":{"SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION":"8192","SGLANG_EMBED_HOST":"1","SGLANG_HC_MIX_MAX_ROWS":"64","SGLANG_INIT_NEW_TOKEN_RATIO":"1.0","SGLANG_MAMBA_PREFILL_FINAL_ONLY_LAZY":"1","SGLANG_MAMBA_SLOTS_PER_REQ":"2","SGLANG_MIN_NEW_TOKEN_RATIO_FACTOR":"0.8","SGLANG_MM_PRECOMPUTE_HASH":"1","SGLANG_QSA_MQA_GROUPS_PER_CTA":"4","SGLANG_QSA_PAIR_OWN":"1","SGLANG_QSA_PAIR_OWN_GROUP":"pair","SGLANG_QSA_SM89_DECODE":"triton_splitk","SGLANG_SM120_ONLINE_MXFP8":"1","SGLANG_TOPK_LOGITS_EXCHANGE":"32","SGLANG_INPROJ_FP8":"1","SGLANG_FP8_FUSED_DEQUANT":"1"},"GRAPH_BS":[1,2,4,8,12,16,20,24,28,32,33,34,35,36,37,38,39,40,41],"KV":"fp8_e4m3","LAYOUT":"tp4","LMONLY":false,"MAMBA":96,"MAMBA_STRATEGY":"extra_buffer_lazy","MAXREQ":41,"MEMFRAC":0.94,"NAME":"tp4_p384_p41cap","SERVE_ARGS":["--max-total-tokens","1850000","--mamba-max-states-per-path","1"],"SKIP_LOCK":true,"SPEC":false},{"ENV":{"SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION":"8192","SGLANG_HC_MIX_MAX_ROWS":"64","SGLANG_INIT_NEW_TOKEN_RATIO":"1.0","SGLANG_MAMBA_PREFILL_FINAL_ONLY_LAZY":"1","SGLANG_MIN_NEW_TOKEN_RATIO_FACTOR":"0.8","SGLANG_MM_PRECOMPUTE_HASH":"1","SGLANG_QSA_MQA_GROUPS_PER_CTA":"4","SGLANG_QSA_PAIR_OWN":"1","SGLANG_QSA_PAIR_OWN_GROUP":"pair","SGLANG_QSA_SM89_DECODE":"triton_splitk","SGLANG_SM120_ONLINE_MXFP8":"1","SGLANG_TOPK_LOGITS_EXCHANGE":"32","SGLANG_INPROJ_FP8":"1","SGLANG_FP8_FUSED_DEQUANT":"1"},"GRAPH_BS":[1,2,4,8,12,16,18,20,22,24,26,28,30,32,34,36,40],"KV":"fp8_e4m3","LAYOUT":"tp4","LMONLY":false,"MAMBA":144,"MAMBA_STRATEGY":"extra_buffer_lazy","MAXREQ":48,"MEMFRAC":0.94,"NAME":"tp4_p384_pair48lm","SERVE_ARGS":["--mamba-max-states-per-path","1"],"SKIP_LOCK":true,"SPEC":false}],"RUNG_MAX_MIN":[45,30,30],"RUN_MODE":"smoke","SANITY_GATE":false,"SERVED_NAME":"qwen38","SERVE_BUDGET_MIN":300,"SESSION_CAP_MIN":20000,"SOLVE":{"answer_tokens":4096,"attempts":2,"b2_size":40000,"b_init":80000,"b_max":120000,"b_min":12000,"b_min2":24000,"cap_total":128000,"crash_halve":3,"crash_switch":1,"effort":"xhigh","est_tok_s":480,"ext_frac":1.0,"ext_min":8000,"ext_retries":2,"ext_time_frac":0.2,"extend":false,"force_think":false,"forced_suffix":"\\\\n\\\\n[[","grace_s":15,"grid_regex":false,"health_fails":3,"kv_admit_max":0.85,"kv_fill":1.0,"max_hits":3,"mean_frac":0.92,"prior_safety":0.95,"rate_warmup_s":300,"rate_window_s":600,"restart_min_s":900,"salvage":true,"salvage_temp":0.0,"stagger_s":2,"status_s":180,"tail_s":300,"temperature":1.0,"top_k":20,"top_p":0.95,"wave_spread":0.9},"SOLVE_MIN":30,"SPEC_STEPS":3,"STALL_MIN":20,"STATUS_MIN":3,"SWEEP_MAX_TOKENS":3000,"SWEEP_S":150,"TF_PROMPTS":8,"TF_TOKENS":2000,"TP":4}')
CFG.update(json.loads(os.environ.get("DUCK2_CFG_JSON", "{}") or "{}"))
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").lower() in ("1", "true")
RUN_MODE = "full" if TRUE_SUBMISSION else CFG["RUN_MODE"]
assert RUN_MODE in ("smoke", "full"), RUN_MODE
SIM = os.environ.get("KAGGLE_SIM") == "1"            # vast 4xL4 mirror of the Kaggle layout
INP = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
work = WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working")); work.mkdir(parents=True, exist_ok=True)
SUBMISSION = work / "submission.json"
print(f"duck2 | RUN_MODE {RUN_MODE} | TRUE_SUBMISSION {TRUE_SUBMISSION} | SIM {SIM} | start "
      f"{time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime(T0))}", flush=True)
print("CFG", json.dumps(CFG), flush=True)

def find(name, must=None, many=False, required=True):
    # os.walk(followlinks=True): Path.rglob (py3.12) does not descend into symlinked directories
    hits = sorted(Path(d) / f for d, _, fs in os.walk(INP, followlinks=True) for f in fs
                  if fnmatch.fnmatch(f, name) and (must is None or must in os.path.join(d, f)))
    if many:
        return hits
    if not hits:
        assert not required, f"missing input {name} ({must})"
        return None
    return hits[0]

def sh(cmd, timeout=30):
    try:
        r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=timeout)
        return (r.stdout + r.stderr).strip()
    except Exception as e:
        return f"({cmd.split()[0]}: {type(e).__name__})"

def write_json_atomic(path, obj):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj)); tmp.replace(path)

print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv; free -g; "
         "echo nproc $(nproc); df -h /tmp /kaggle/working 2>/dev/null"), flush=True)

TEST_CHAL = find("arc-agi_test_challenges.json")
EVAL_CHAL = find("arc-agi_evaluation_challenges.json", required=False)
EVAL_SOLS = find("arc-agi_evaluation_solutions.json", required=False)
test_tasks = json.loads(TEST_CHAL.read_text())
print(f"test challenges: {TEST_CHAL} ({len(test_tasks)} tasks, {sum(len(t['test']) for t in test_tasks.values())} outputs)")

def placeholder(tid):
    return [{"attempt_1": t["input"], "attempt_2": t["input"]} for t in test_tasks[tid]["test"]]
FALLBACK = {k: placeholder(k) for k in test_tasks}
write_json_atomic(SUBMISSION, FALLBACK)        # valid (zero-score) file from the first minute on
print("fallback submission.json written:", SUBMISSION, flush=True)

# shared state; later cells fill it. A failing stage is recorded, never raised (a raising rerun = "Notebook Threw Exception")
FAILED, server, SERVER_OK, SERVER_RUNG, KV_TOKENS, INFLIGHT, log_off, PROBE = [], None, False, None, 0, 0, 0, {}
CTX_LIMIT = CFG["HARNESS"]["ctx-limit"]
LOG = str(work / "serve.log")
def stage_failed(stage, ex, fatal=True):
    import traceback
    print(f"!!! stage {stage} failed: {type(ex).__name__}: {ex}\n{traceback.format_exc()[-4000:]}", flush=True)
    if fatal:
        FAILED.append(stage)

def log_text(off=0):
    try:
        with open(LOG, "rb") as f:
            f.seek(off); return f.read().decode(errors="replace")
    except OSError:
        return ""
def log_size():
    try:
        return Path(LOG).stat().st_size
    except OSError:
        return 0
KEY = ("max_total_num_tokens=", "KV Cache is allocated", "Mamba Cache is allocated", "Load weight end", "duck2", "mem usage",
       "GDN kernel dispatcher", "fired up", "Capture cuda graph end", "ple", "PLE", "out of memory", "Error", "error:")
def key_lines(off=0, n=40):
    ls = [l for l in re.split(r"[\r\n]+", log_text(off)) if any(k in l for k in KEY) and "server_args=" not in l]
    return "\n".join(l[:400] for l in ls[-n:])

if RUN_MODE == "smoke" and EVAL_CHAL is not None:
    RUN_CHAL, RUN_SOLS = EVAL_CHAL, EVAL_SOLS
    _ids = sorted(json.loads(EVAL_CHAL.read_text()))
    _sel = _ids[::5]                           # (past 24: the remaining ids in order, so the first 24 stay the same)
    RUN_IDS = (_sel + [x for x in _ids if x not in set(_sel)])[:CFG["N_SMOKE"]]
    OUT = work / "agent_eval"                  # eval results never go into submission.json
elif RUN_MODE == "smoke":
    RUN_CHAL, RUN_SOLS, RUN_IDS, OUT = TEST_CHAL, None, list(test_tasks)[:CFG["N_SMOKE"]], work / "agent"
else:
    RUN_CHAL, RUN_SOLS, RUN_IDS, OUT = TEST_CHAL, None, sorted(test_tasks), work / "agent"
MERGE_INTO_SUBMISSION = RUN_CHAL == TEST_CHAL
RUN_TASKS = test_tasks if RUN_CHAL == TEST_CHAL else json.loads(RUN_CHAL.read_text())
# smoke: solutions stay in the notebook (scoring + status); the harness never gets --solutions, so nothing under
# the working dir the model's python can reach carries per-task correctness
SOLS = json.loads(RUN_SOLS.read_text()) if RUN_SOLS else None
OUT_DIRS = [OUT]          # one per harness launch (a server restart relaunches the harness into a new dir)
print(f"harness tasks: {len(RUN_IDS)} from {RUN_CHAL.name} | solutions {'yes' if RUN_SOLS else 'no'} | out {OUT} | "
      f"merge into submission.json: {MERGE_INTO_SUBMISSION}", flush=True)

def as_grid(g):
    """A valid ARC grid (list of 1-30 equal rows of ints 0-9) with int-like values coerced, else None."""
    try:
        if not (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) for r in g)):
            return None
        w = len(g[0])
        if not (1 <= w <= 30 and all(len(r) == w for r in g)):
            return None
        out = []
        for r in g:
            row = []
            for v in r:
                if isinstance(v, bool) or not isinstance(v, (int, float)) or v != int(v) or not 0 <= int(v) <= 9:
                    return None
                row.append(int(v))
            out.append(row)
        return out
    except Exception:
        return None

def valid_grid(g):
    return as_grid(g) is not None and all(type(v) is int for r in g for v in r)

def agent_entries():
    """Every harness out dir overlaid in launch order, per test output: a valid attempt that differs from the
    input placeholder overrides earlier ones; an invalid attempt is replaced by its valid sibling."""
    merged, changed = {}, set()
    for d in OUT_DIRS:
        try:
            agent = json.loads((Path(d) / "submission.json").read_text())
        except Exception:
            continue
        for k, v in agent.items():
            if k not in RUN_TASKS or not isinstance(v, list):
                continue
            tests = RUN_TASKS[k]["test"]
            cur = merged.setdefault(k, [{"attempt_1": t["input"], "attempt_2": t["input"]} for t in tests])
            for j, e in enumerate(v[:len(tests)]):
                if not isinstance(e, dict):
                    continue
                a1, a2 = as_grid(e.get("attempt_1")), as_grid(e.get("attempt_2"))
                a1, a2 = a1 or a2, a2 or a1
                if a1 is None or (a1 == tests[j]["input"] and a2 == tests[j]["input"]):
                    continue
                cur[j] = {"attempt_1": a1, "attempt_2": a2}
                changed.add(k)
    return {k: v for k, v in merged.items() if k in changed}

def merged_submission():
    """Fallback overlaid with the merged harness entries; returns (submission, n_tasks_from_harness)."""
    sub = json.loads(json.dumps(FALLBACK)); n = 0
    if MERGE_INTO_SUBMISSION:
        for k, v in agent_entries().items():
            if k in sub:
                sub[k] = v
                n += v != FALLBACK[k]
    return sub, n

def nb_score(ids=None):
    """(score over ids, n tasks with a non-placeholder answer) against the notebook-held solutions."""
    if not SOLS:
        return None, 0
    ent = agent_entries(); ids = ids or RUN_IDS; sc = []
    for k in ids:
        e = ent.get(k)
        sc.append(sum(e is not None and j < len(e) and (e[j]["attempt_1"] == g or e[j]["attempt_2"] == g)
                      for j, g in enumerate(SOLS[k])) / len(SOLS[k]))
    return sum(sc) / max(1, len(ids)), len(ent)


duck2 | RUN_MODE smoke | TRUE_SUBMISSION False | SIM False | start 2026-11-02 00:00:56 UTC
CFG {"AGENT": {"ARGS": {"act-tokens": 8192, "answer-reserve": 0, "budget-note": "on", "cap-stages": "150000,200000", "context-mode": "pin", "continue-cut": "on", "continue-images": "chat", "ctx-limit": 110000, "effort": "xhigh", "empty-retry": 2, "est-session-rate": 10, "exec-timeout": 30, "hand-check-rule": "on", "hide-budget": "on", "images": "on", "max-sessions-per-task": 1, "max-tool-out": 6000, "max-turns": 120, "metrics-interval": 30, "pin-keep": 0.8, "propose": "on", "resume": "continue", "review-outputs": "on", "salvage": "on", "salvage-fix": "on", "salvage-mode": "tool", "salvage-tokens": 6144, "salvage-tools": "session", "salvage-turns": 2, "sandbox-help": "on", "second-opinion": "off", "session-cap": 100000, "submit-partial": "on", "task-cap": 100000, "temperature": 0.7, "top-k": 20, "top-p": 0.95, "turn-max-tokens": 32768, "workspace": "on", "wrapup-s": 240, "answer-mode": "direct", "

In [2]:
# [duck2:quick]  CFG["QUICK_VERSION"]: the interactive run of a submission build does no GPU work (~1 min of quota)
if CFG.get("QUICK_VERSION") and RUN_MODE != "full":
    write_json_atomic(SUBMISSION, FALLBACK)
    FAILED.append("quick_version")       # every gated stage (install, model, serve, phase0, boot, agent) skips
    print("QUICK_VERSION: interactive run -> placeholder submission.json; the competition rerun runs the full build",
          flush=True)


QUICK_VERSION: interactive run -> placeholder submission.json; the competition rerun runs the full build


In [3]:
# [duck2:monitor]  Franzen's precache + system census (verbatim apart from the paths and period)
from concurrent.futures import ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()
    CHUNK_SIZE = 32 * 1024 * 1024
    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0
    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))
    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass
    chunks.sort()
    _local = threading.local()
    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass
    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))
    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

def _sysline(paths=("/", "/tmp", str(work))):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)

def start_monitor(interval_minutes=10.0):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

_stop_monitor = start_monitor(10 if TRUE_SUBMISSION else CFG["MONITOR_MIN"])


In [4]:
# [duck2:install]  Franzen's offline Pennyroyal install, retargeted to sm_89, + duck2 patches + GPU unit test
if FAILED:
    print('skipping stage install: earlier stage failed:', FAILED, flush=True)
else:
  try:
      import copy, hashlib, shlex, zipfile
      PREFIX = "/tmp/sgl-intel"
      VENV = f"{PREFIX}/venv"; PYTHON = f"{VENV}/bin/python"; SGLANG = f"{VENV}/bin/sglang"
      PATCHES = json.loads('{"apply_patches.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch an installed Pennyroyal SGLang (0.5.19+gd00d88efc8d6) for 4x L4 (sm_89).\\n\\nUsage:\\n  apply_patches.py <site-packages dir> [--ple-scale] [--check]\\n\\n  qsa_sm89_decode (always): copies qsa_sm89_decode.py to\\n      sglang/srt/layers/attention/qsa/sm89_decode.py and routes the dense\\n      FA fallback of QwenSparseAttnBackend._forward_paged_attention to it.\\n      Same change as qsa_sm89_decode.patch (cd <site>; patch -p1 < it).\\n      Runtime switch: SGLANG_QSA_SM89_DECODE=auto|triton|torch|off.\\n  ple_fp8_scale (--ple-scale, only for BF16-PLE checkpoints served with\\n      text_config.ple_embedding_dtype=float8_e4m3fn): scaled+saturating\\n      BF16->FP8 PLE cast driven by SGLANG_PLE_FP8_SCALE. No-op when unset.\\n\\nAnchors are matched exactly once or the script fails without writing, so a\\ndifferent SGLang build is refused instead of being half-patched.\\n\\"\\"\\"\\nimport shutil\\nimport sys\\nfrom pathlib import Path\\n\\nHERE = Path(__file__).resolve().parent\\n\\nQSA_MARK = \\"# [duck2 sm89 patch]\\"\\nQSA_OLD = (\\n    \\"        flash_attn_varlen_func = _resolve_flash_attn_varlen_func()\\\\n\\"\\n    \\"        batch, topk = topk_indices.shape\\\\n\\"\\n)\\nQSA_NEW = (\\n    f\\"        {QSA_MARK} sm_8x: no FA2 in the wheelhouse, FA4-CuTe rejects\\\\n\\"\\n    \\"        # fp8 KV and hd256 SM80 tiles; read the top-k K/V straight from the pool.\\\\n\\"\\n    \\"        from sglang.srt.layers.attention.qsa.sm89_decode import (\\\\n\\"\\n    \\"            qsa_sparse_decode,\\\\n\\"\\n    \\"            select_sm89_decode_impl,\\\\n\\"\\n    \\"        )\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"        sm89_impl = select_sm89_decode_impl(q.device)\\\\n\\"\\n    \\"        if sm89_impl is not None:\\\\n\\"\\n    \\"            return qsa_sparse_decode(\\\\n\\"\\n    \\"                sm89_impl,\\\\n\\"\\n    \\"                q,\\\\n\\"\\n    \\"                k_buffer,\\\\n\\"\\n    \\"                v_buffer,\\\\n\\"\\n    \\"                self.req_to_token_pool.req_to_token,\\\\n\\"\\n    \\"                (\\\\n\\"\\n    \\"                    metadata.row_req_pool_indices\\\\n\\"\\n    \\"                    if metadata.row_req_pool_indices is not None\\\\n\\"\\n    \\"                    else forward_batch.req_pool_indices\\\\n\\"\\n    \\"                ),\\\\n\\"\\n    \\"                topk_indices,\\\\n\\"\\n    \\"                metadata.sequence_lengths,\\\\n\\"\\n    \\"                layer.scaling,\\\\n\\"\\n    \\"            ).reshape(q.shape[0], -1)\\\\n\\"\\n    \\"\\\\n\\" + QSA_OLD\\n)\\n\\nPLE_MARK = \\"# [duck2 ple-fp8-scale patch]\\"\\nPLE_OLD_1 = (\\n    \\"                emb.weight.data[local_start : local_start + n_rows].copy_(\\\\n\\"\\n    \\"                    loaded_weight[src_start : src_start + n_rows].to(\\\\n\\"\\n    \\"                        device=emb.weight.device, dtype=emb.weight.dtype\\\\n\\"\\n    \\"                    )\\\\n\\"\\n    \\"                )\\\\n\\"\\n)\\nPLE_NEW_1 = (\\n    \\"                _src = loaded_weight[src_start : src_start + n_rows]\\\\n\\"\\n    \\"                if (\\\\n\\"\\n    \\"                    emb.weight.dtype == torch.float8_e4m3fn\\\\n\\"\\n    \\"                    and _src.dtype != torch.float8_e4m3fn\\\\n\\"\\n    \\"                ):\\\\n\\"\\n    f\\"                    {PLE_MARK}\\\\n\\"\\n    \\"                    from sglang.srt.models.duck2_ple_fp8 import quantize_ple_rows\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"                    _src = quantize_ple_rows(emb, _src)\\\\n\\"\\n    \\"                emb.weight.data[local_start : local_start + n_rows].copy_(\\\\n\\"\\n    \\"                    _src.to(device=emb.weight.device, dtype=emb.weight.dtype)\\\\n\\"\\n    \\"                )\\\\n\\"\\n)\\nPLE_OLD_2 = (\\n    \\"        buffer.copy_(loaded_weight.to(device=buffer.device, dtype=buffer.dtype))\\\\n\\"\\n    \\"        loaded_buffers.add(name)\\\\n\\"\\n    \\"        return True\\\\n\\"\\n)\\nPLE_NEW_2 = (\\n    f\\"        {PLE_MARK}\\\\n\\"\\n    \\"        from sglang.srt.models.duck2_ple_fp8 import keep_quantised_scale\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"        if keep_quantised_scale(buffer_name, loaded_weight):\\\\n\\"\\n    \\"            loaded_buffers.add(name)\\\\n\\"\\n    \\"            return True\\\\n\\" + PLE_OLD_2\\n)\\n\\n\\ndef _patch_file(path: Path, mark: str, edits, check: bool, label: str) -> bool:\\n    text = path.read_text()\\n    if mark in text:\\n        print(f\\"{label}: already applied\\")\\n        return True\\n    if check:\\n        print(f\\"{label}: NOT applied\\")\\n        return False\\n    for old, _ in edits:\\n        if text.count(old) != 1:\\n            print(f\\"ERROR {label}: anchor found {text.count(old)}x in {path}; SGLang build differs\\")\\n            return False\\n    for old, new in edits:\\n        text = text.replace(old, new)\\n    path.write_text(text)\\n    for pyc in path.parent.glob(f\\"__pycache__/{path.stem}*.pyc\\"):\\n        pyc.unlink()\\n    print(f\\"{label}: applied\\")\\n    return True\\n\\n\\ndef main() -> int:\\n    if len(sys.argv) < 2:\\n        print(__doc__)\\n        return 2\\n    site = Path(sys.argv[1])\\n    check = \\"--check\\" in sys.argv\\n    ok = True\\n\\n    attn = site / \\"sglang/srt/layers/attention\\"\\n    if not check:\\n        shutil.copyfile(HERE / \\"qsa_sm89_decode.py\\", attn / \\"qsa/sm89_decode.py\\")\\n    ok &= (attn / \\"qsa/sm89_decode.py\\").is_file()\\n    ok &= _patch_file(\\n        attn / \\"qwen_sparse_attn_backend.py\\", QSA_MARK, [(QSA_OLD, QSA_NEW)], check,\\n        \\"qsa_sm89_decode\\",\\n    )\\n\\n    if \\"--ple-scale\\" in sys.argv:\\n        models = site / \\"sglang/srt/models\\"\\n        if not check:\\n            shutil.copyfile(HERE / \\"ple_fp8_scale.py\\", models / \\"duck2_ple_fp8.py\\")\\n        ok &= (models / \\"duck2_ple_fp8.py\\").is_file()\\n        ok &= _patch_file(\\n            models / \\"qwen4_exp.py\\", PLE_MARK,\\n            [(PLE_OLD_1, PLE_NEW_1), (PLE_OLD_2, PLE_NEW_2)], check, \\"ple_fp8_scale\\",\\n        )\\n    return 0 if ok else 1\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main())\\n", "bench_densefp8.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 GPU correctness check + microbenchmark for colab/perf_patches/densefp8_patch.py (key \\"densefp8\\").\\n\\nPer-rank (TP4) dense projection shapes of Qwen3.8-Flash-Next at decode, weight-only rowwise-FP8 (e4m3, per-row fp32 scale,\\nbf16 activations) vs the bf16 cuBLAS baseline, timed by replaying a CUDA graph over distinct weight copies (cold L2),\\nplus the HyperConnection mix (fused persistent kernel, 13/19/32 rows) with rowwise-FP8 vs bf16 weights.\\nSweeps the GEMV tile config per shape (BLOCK_N, BLOCK_K, stages, warps) and prints the best as a ready-to-paste table.\\nChecks: GEMV vs fp32 reference with the dequantised weights (kernel correctness), and FP8-vs-bf16 relative error.\\nusage: CUDA_VISIBLE_DEVICES=0 /tmp/sgl-intel/venv/bin/python bench_densefp8.py [<site-packages dir>] [--timeout 900]\\nStdout: ONE JSON line (fd 1 is redirected to stderr while running).\\n\\"\\"\\"\\nimport importlib.util\\nimport json\\nimport os\\nimport statistics\\nimport sys\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nKEY = \\"densefp8\\"\\nSHAPES = {  # name: (N, K, calls per decode step per GPU)\\n    \\"gdn_in_qkvz\\": (4096, 2560, 36),\\n    \\"gdn_out\\": (2560, 1536, 36),\\n    \\"qsa_qkv\\": (3584, 2560, 12),\\n    \\"qsa_o\\": (2560, 1536, 12),\\n    \\"shared_gu\\": (320, 2560, 48),\\n    \\"shared_down\\": (2560, 160, 48),\\n    \\"lm_head\\": (62080, 2560, 1),\\n}\\nROWS = [1, 8, 19, 32, 48, 64]\\nSWEEP_ROWS = {8: [1, 8, 16], 19: [24, 32], 48: [48, 64]}  # sweep rows -> buckets it decides\\nCFGS = [(16, 128, 4, 4), (16, 256, 3, 4), (32, 128, 4, 4), (32, 128, 3, 8), (32, 256, 3, 4), (64, 128, 3, 4), (64, 128, 3, 8)]\\nHC, HS, LOWRANK = 4, 2560, 320\\nRES = {\\"key\\": KEY, \\"ok\\": False, \\"failed\\": [], \\"baseline_us\\": None, \\"patched_us\\": None, \\"errors\\": []}\\n_REAL_STDOUT = os.dup(1)\\n\\n\\ndef log(m):\\n    print(f\\"[bench_densefp8 {time.time() - T0:7.1f}s] {m}\\", file=sys.stderr, flush=True)\\n\\n\\ndef emit(code=0):\\n    os.write(_REAL_STDOUT, (json.dumps(RES) + \\"\\\\n\\").encode())\\n    os._exit(code)\\n\\n\\ndef watchdog(sec):\\n    def run():\\n        time.sleep(sec)\\n        RES[\\"errors\\"].append(f\\"watchdog {sec}s\\")\\n        emit(3)\\n    threading.Thread(target=run, daemon=True).start()\\n\\n\\ndef main():\\n    global T0\\n    T0 = time.time()\\n    os.dup2(2, 1)\\n    args = [a for a in sys.argv[1:] if not a.startswith(\\"--\\")]\\n    timeout = 900.0\\n    if \\"--timeout\\" in sys.argv:\\n        timeout = float(sys.argv[sys.argv.index(\\"--timeout\\") + 1])\\n        args = [a for a in args if a != str(sys.argv[sys.argv.index(\\"--timeout\\") + 1])]\\n    if timeout > 0:\\n        watchdog(timeout)\\n    os.environ[\\"SGLANG_HC_MIX_MAX_ROWS\\"] = \\"32\\"  # read at import of hc_mix_triton\\n    import torch\\n    import torch.nn.functional as F\\n\\n    if not torch.cuda.is_available():\\n        RES[\\"errors\\"].append(\\"no CUDA\\")\\n        emit(2)\\n    RES[\\"device\\"] = torch.cuda.get_device_name(0)\\n    site = Path(args[0]) if args else None\\n    if site is None:\\n        import sglang\\n        site = Path(sglang.__file__).resolve().parent.parent\\n    path = site / \\"sglang/kernels/ops/gemm/sm120_online_fp8.py\\"\\n    spec = importlib.util.spec_from_file_location(\\"sm120_online_fp8_t\\", path)\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    if not hasattr(mod, \\"sm89_rowwise_fp8_gemv\\"):\\n        RES[\\"errors\\"].append(\\"densefp8 patch not applied to \\" + str(path))\\n        emit(2)\\n    dev = torch.device(\\"cuda\\")\\n    torch.manual_seed(0)\\n\\n    def time_graph(fns, iters=20, reps=5):\\n        for f in fns:\\n            f()\\n        torch.cuda.synchronize()\\n        g = torch.cuda.CUDAGraph()\\n        s = torch.cuda.Stream()\\n        with torch.cuda.stream(s):\\n            for f in fns:\\n                f()\\n        torch.cuda.synchronize()\\n        with torch.cuda.graph(g):\\n            for f in fns:\\n                f()\\n        for _ in range(3):\\n            g.replay()\\n        torch.cuda.synchronize()\\n        ts = []\\n        for _ in range(reps):\\n            a, b = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\\n            a.record()\\n            for _ in range(iters):\\n                g.replay()\\n            b.record()\\n            torch.cuda.synchronize()\\n            ts.append(a.elapsed_time(b) * 1000.0 / (iters * len(fns)))\\n        return statistics.median(ts)\\n\\n    per = {}\\n    best_cfg = {}\\n    bf16_us = {r: 0.0 for r in ROWS}\\n    fp8_us = {r: 0.0 for r in ROWS}\\n    max_ker_err, max_q_err = 0.0, 0.0\\n    for name, (N, K, count) in SHAPES.items():\\n        try:\\n            nbytes = N * K * 2\\n            copies = max(2, min(12, int(96e6 // nbytes) + 1))\\n            Ws = [(torch.randn(N, K, device=dev) * 0.02).to(torch.bfloat16) for _ in range(copies)]\\n            Qs = [mod.quantize_rowwise_fp8(w) for w in Ws]\\n            per[name] = {}\\n            best = {}  # sweep row -> (us, cfg)\\n            for sweep_row in SWEEP_ROWS:\\n                x = torch.randn(sweep_row, K, device=dev, dtype=torch.bfloat16)\\n                for cfg in CFGS:\\n                    try:\\n                        us = time_graph([(lambda q=q, cfg=cfg: mod.sm89_rowwise_fp8_gemv(x, q[0], q[1], cfg)) for q in Qs])\\n                    except Exception as exc:\\n                        log(f\\"{name} rows={sweep_row} cfg={cfg} unusable: {type(exc).__name__}: {str(exc)[:120]}\\")\\n                        continue\\n                    if sweep_row not in best or us < best[sweep_row][0]:\\n                        best[sweep_row] = (us, cfg)\\n                # correctness of the best cfg vs fp32 dequant reference and vs bf16\\n                us_b, cfg_b = best[sweep_row]\\n                out = mod.sm89_rowwise_fp8_gemv(x, Qs[0][0], Qs[0][1], cfg_b).float()\\n                deq = Qs[0][0].float() * Qs[0][1][:, None]\\n                ref = x.float() @ deq.T\\n                orig = x.float() @ Ws[0].float().T\\n                ker = ((out - ref).norm() / ref.norm()).item()\\n                q = ((out - orig).norm() / orig.norm()).item()\\n                max_ker_err, max_q_err = max(max_ker_err, ker), max(max_q_err, q)\\n                per[name][f\\"err_rows{sweep_row}\\"] = {\\"kernel_rel\\": round(ker, 5), \\"quant_rel\\": round(q, 5)}\\n            for r in ROWS:\\n                key = 8 if r <= 16 else (19 if r <= 32 else 48)\\n                x = torch.randn(r, K, device=dev, dtype=torch.bfloat16)\\n                us_b = time_graph([(lambda w=w: F.linear(x, w)) for w in Ws])\\n                cfg = best[key][1]\\n                us_f = time_graph([(lambda q=q: mod.sm89_rowwise_fp8_gemv(x, q[0], q[1], cfg)) for q in Qs])\\n                per[name][str(r)] = {\\"bf16_us\\": round(us_b, 2), \\"fp8_us\\": round(us_f, 2), \\"cfg\\": cfg}\\n                bf16_us[r] += us_b * count\\n                fp8_us[r] += us_f * count\\n            for sweep_row, (us, cfg) in best.items():\\n                best_cfg[f\\"{N},{K},{sweep_row}\\"] = list(cfg)\\n            log(f\\"{name}: \\" + \\" \\".join(f\\"r{r}:{per[name][str(r)][\'bf16_us\']}->{per[name][str(r)][\'fp8_us\']}\\" for r in ROWS))\\n            del Ws, Qs\\n            torch.cuda.empty_cache()\\n        except Exception as exc:\\n            RES[\\"failed\\"].append(f\\"{name}: {type(exc).__name__}: {str(exc)[:200]}\\")\\n            log(traceback.format_exc())\\n    RES[\\"per_shape\\"] = per\\n    RES[\\"best_cfg\\"] = best_cfg\\n    RES[\\"proj_step_us\\"] = {str(r): {\\"bf16\\": round(bf16_us[r], 1), \\"fp8\\": round(fp8_us[r], 1)} for r in ROWS}\\n    RES[\\"max_kernel_rel_err\\"] = round(max_ker_err, 5)\\n    RES[\\"max_quant_rel_err\\"] = round(max_q_err, 5)\\n    if max_ker_err > 0.02:\\n        RES[\\"failed\\"].append(f\\"gemv kernel error {max_ker_err:.4f} > 0.02\\")\\n    if max_q_err > 0.06:\\n        RES[\\"failed\\"].append(f\\"fp8 quantisation error {max_q_err:.4f} > 0.06\\")\\n\\n    # ---- HyperConnection mix: fused persistent kernel, bf16 vs rowwise-FP8 weights ---------------------------------\\n    hc_res = {}\\n    hc_bf16_step, hc_fp8_step = {}, {}\\n    try:\\n        sys.path.insert(0, str(site))\\n        hspec = importlib.util.spec_from_file_location(\\"hc_mix_triton_t\\", site / \\"sglang/srt/layers/hc_mix_triton.py\\")\\n        hm = importlib.util.module_from_spec(hspec)\\n        hspec.loader.exec_module(hm)\\n        hm._deterministic_inference_cached = False  # keep sglang.srt.server_args out of it\\n        RES[\\"hcmix32_active\\"] = hasattr(hm, \\"_hcmix32_launch\\") and hm._HCMIX32_MAX_ROWS > 0\\n\\n        K = HC * HS\\n        Wd = [(torch.randn(LOWRANK, K, device=dev) * 0.02).to(torch.bfloat16) for _ in range(4)]\\n        Wu = [(torch.randn(K, LOWRANK, device=dev) * 0.02).to(torch.bfloat16) for _ in range(4)]\\n        Qd = [mod.quantize_rowwise_fp8(w) for w in Wd]\\n        Qu = [mod.quantize_rowwise_fp8(w) for w in Wu]\\n        Dd, Du = [], []\\n        for (qd, sd), (qu, su) in zip(Qd, Qu):\\n            p_d, p_u = qd.clone(), qu.clone()\\n            setattr(p_d, mod._SCALE_ATTR, sd)\\n            setattr(p_u, mod._SCALE_ATTR, su)\\n            Dd.append(p_d)\\n            Du.append(p_u)\\n        for rows in (13, 19, 32):\\n            x = (torch.randn(rows, K, device=dev) * 1.0).to(torch.bfloat16)\\n            ok_b = hm.fused_hc_mix_supported(x, Wd[0], Wu[0])\\n            ok_f = hm.fused_hc_mix_supported(x, Dd[0], Du[0])\\n            entry = {\\"supported_bf16\\": bool(ok_b), \\"supported_fp8\\": bool(ok_f)}\\n            if ok_f:\\n                yf = hm.fused_hc_mix(x, Dd[0], Du[0], HC, HS).float()\\n                dd = (Dd[0].float() * Qd[0][1][:, None])\\n                du = (Du[0].float() * Qu[0][1][:, None])\\n                t = F.silu(F.linear(x.float(), dd) / HC)\\n                t = torch.sigmoid(F.linear(t, du)).unflatten(-1, (HC, HS))\\n                ref = (t * x.float().unflatten(-1, (HC, HS))).mean(dim=-2)\\n                entry[\\"fp8_vs_deq_ref_rel\\"] = round(((yf - ref).norm() / ref.norm()).item(), 5)\\n                entry[\\"fp8_us\\"] = round(time_graph([(lambda a=a, b=b: hm.fused_hc_mix(x, a, b, HC, HS)) for a, b in zip(Dd, Du)]), 2)\\n                hc_fp8_step[rows] = entry[\\"fp8_us\\"] * 97\\n                if entry[\\"fp8_vs_deq_ref_rel\\"] > 0.02:\\n                    RES[\\"failed\\"].append(f\\"hc fp8 rows={rows} err {entry[\'fp8_vs_deq_ref_rel\']}\\")\\n            else:\\n                RES[\\"failed\\"].append(f\\"hc fp8 rows={rows} not supported by fused kernel\\")\\n            if ok_b:\\n                entry[\\"bf16_us\\"] = round(time_graph([(lambda a=a, b=b: hm.fused_hc_mix(x, a, b, HC, HS)) for a, b in zip(Wd, Wu)]), 2)\\n                hc_bf16_step[rows] = entry[\\"bf16_us\\"] * 97\\n            hc_res[str(rows)] = entry\\n            log(f\\"hc rows={rows}: {entry}\\")\\n    except Exception as exc:\\n        RES[\\"failed\\"].append(f\\"hc: {type(exc).__name__}: {str(exc)[:200]}\\")\\n        log(traceback.format_exc())\\n    RES[\\"hc\\"] = hc_res\\n\\n    # ---- LinearMethod flow on a fake layer ------------------------------------------------------------------------\\n    try:\\n        sys.path.insert(0, str(site))\\n        method = mod.build_sm89_rowwise_method()\\n        lin = torch.nn.Module()\\n        lin.weight = torch.nn.Parameter((torch.randn(2560, 1536, device=dev) * 0.02).to(torch.bfloat16), requires_grad=False)\\n        w0 = lin.weight.data.clone()\\n        method.process_weights_after_loading(lin)\\n        x = torch.randn(19, 1536, device=dev, dtype=torch.bfloat16)\\n        y = method.apply(lin, x, None)\\n        x2 = torch.randn(300, 1536, device=dev, dtype=torch.bfloat16)  # prefill-size path (dequant + matmul)\\n        y2 = method.apply(lin, x2, None)\\n        r1 = ((y.float() - x.float() @ w0.float().T).norm() / (x.float() @ w0.float().T).norm()).item()\\n        r2 = ((y2.float() - x2.float() @ w0.float().T).norm() / (x2.float() @ w0.float().T).norm()).item()\\n        RES[\\"method_flow\\"] = {\\"dtype\\": str(lin.weight.dtype), \\"decode_rel\\": round(r1, 5), \\"prefill_rel\\": round(r2, 5)}\\n        if lin.weight.dtype != torch.float8_e4m3fn or r1 > 0.06 or r2 > 0.06:\\n            RES[\\"failed\\"].append(f\\"method flow {RES[\'method_flow\']}\\")\\n    except Exception as exc:\\n        RES[\\"failed\\"].append(f\\"method_flow: {type(exc).__name__}: {str(exc)[:200]}\\")\\n        log(traceback.format_exc())\\n\\n    # ---- totals: per-step dense \\u00b5s at 19 rows (the production operating point) ------------------------------------\\n    hc_b = hc_bf16_step.get(19, 0.0)\\n    hc_f = hc_fp8_step.get(19, 0.0)\\n    RES[\\"baseline_us\\"] = round(bf16_us[19] + hc_b, 1)\\n    RES[\\"patched_us\\"] = round(fp8_us[19] + hc_f, 1)\\n    RES[\\"saved_ms_per_step\\"] = {\\n        \\"19\\": round((bf16_us[19] + hc_bf16_step.get(19, 0) - fp8_us[19] - hc_fp8_step.get(19, 0)) / 1000, 2),\\n        \\"48\\": round((bf16_us[48] + hc_bf16_step.get(32, 0) - fp8_us[48] - hc_fp8_step.get(32, 0)) / 1000, 2),\\n    }\\n    RES[\\"ok\\"] = not RES[\\"failed\\"] and not RES[\\"errors\\"]\\n    emit(0 if RES[\\"ok\\"] else 1)\\n\\n\\nif __name__ == \\"__main__\\":\\n    try:\\n        main()\\n    except SystemExit:\\n        raise\\n    except BaseException as exc:\\n        RES[\\"errors\\"].append(f\\"{type(exc).__name__}: {exc}\\")\\n        log(traceback.format_exc())\\n        emit(2)\\n", "bench_hcmix32.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 GPU microbenchmark + correctness test for colab/perf_patches/hcmix32_patch.py (fused HC mix at 17-32 rows).\\n\\nRun on one free Kaggle L4 BEFORE the server starts, with the venv python, after the patch has been applied:\\n  CUDA_VISIBLE_DEVICES=0 timeout -k 10 600 /tmp/sgl-intel/venv/bin/python bench_hcmix32.py [<site-packages dir>]\\nThe site-packages dir defaults to wherever `sglang` is found; hc_mix_triton.py is loaded straight from its file\\n(no `import sglang`). Progress goes to stderr; the LAST stdout line is one JSON object:\\n  {\\"key\\": \\"hcmix32\\", \\"ok\\": bool, \\"baseline_us\\": ..., \\"patched_us\\": ..., \\"max_abs_err\\": ..., ...}\\n  baseline_us  97 mixes in one CUDA graph at --prod-rows (19) rows: stock fallback = torch.compile(_mix_compute) chain\\n               (ok requires chain_compiled: an eager-chain baseline would overstate the saving)\\n  patched_us   the same 97 mixes through the patched fused_hc_mix (ROWS=32 persistent kernel)\\n  max_abs_err  max |fused32 - stock chain| over all rows > 16 (vs the unpatched implementation at those rows);\\n               compare max_abs_err_stock16 = max |stock fused16 - chain| at 16 rows (today\'s bs16 -> bs17 jump)\\nok = patch applied and dormant without env + ROWS=32 kernel compiles, finishes (no grid-barrier hang) and leaves the\\nbarrier counters at 0 + outputs within ~1 bf16 ulp of an fp64 reference that mirrors the kernel\'s math (the same\\ncriterion the stock 16-row kernel is held to) + rows 0-15 agree with the stock 16-row kernel + faster than the chain\\nat every tested row count > 16.\\nExit code: 0 = finished (read \\"ok\\"), 1 = exception, 2 = GPU work did not finish (hang), 3 = watchdog time limit.\\nShapes: Qwen3.8-Flash-Next text_config hc_count 4, hidden_size 2560, hc_lowrank 320 -> K = 10240; 97 mixes per decode\\nstep (48 layers x attn/mlp + final mixer), each with its own weights (1.27 GB total, so every mix streams from DRAM\\nas in the server; L2 is 48 MB).\\n\\"\\"\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nimport os\\nimport statistics\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nimport torch\\n\\nKEY = \\"hcmix32\\"\\nHC, HS, LOWRANK = 4, 2560, 320\\nK = HC * HS\\nREL = \\"sglang/srt/layers/hc_mix_triton.py\\"\\nRES = {\\"key\\": KEY, \\"ok\\": False}\\n_STATE = {\\"stage\\": \\"init\\", \\"printed\\": False, \\"t0\\": time.time()}\\n_LOCK = threading.Lock()\\n# Per-element tolerance vs the fp64 reference: ~1 bf16 ulp relative plus an absolute floor that absorbs the rare\\n# bf16 flip of an intermediate t value caused by fp32 atomic summation order (effect ~1e-4 on the outputs of a row).\\nTOL_REL, TOL_ABS, BAD_FRAC = 2.0 ** -7, 2.0 ** -9, 1e-5\\n\\n\\ndef allowed_bad(n):\\n    return max(2, int(BAD_FRAC * n))\\n\\n\\ndef log(msg):\\n    print(f\\"[bench_hcmix32 +{time.time() - _STATE[\'t0\']:.1f}s] {msg}\\", file=sys.stderr, flush=True)\\n\\n\\ndef emit(**kw):\\n    with _LOCK:\\n        if _STATE[\\"printed\\"]:\\n            return\\n        _STATE[\\"printed\\"] = True\\n        RES.update(kw)\\n        RES[\\"elapsed_s\\"] = round(time.time() - _STATE[\\"t0\\"], 1)\\n        print(json.dumps(RES, default=str), flush=True)\\n\\n\\ndef die(error, code):\\n    emit(ok=False, error=error, failed_stage=_STATE[\\"stage\\"])\\n    sys.stderr.flush()\\n    os._exit(code)\\n\\n\\ndef stage(name):\\n    _STATE[\\"stage\\"] = name\\n    log(name)\\n\\n\\ndef watchdog(deadline):\\n    while time.time() < deadline:\\n        time.sleep(0.5)\\n    die(\\"watchdog: total time limit hit (grid-barrier deadlock or stalled compile?)\\", 3)\\n\\n\\ndef gpu_wait(what, timeout):\\n    \\"\\"\\"Wait for all queued work without blocking the thread, so a deadlocked persistent kernel is reported.\\"\\"\\"\\n    ev = torch.cuda.Event()\\n    ev.record()\\n    t = time.time()\\n    while not ev.query():\\n        if time.time() - t > timeout:\\n            die(f\\"GPU work \'{what}\' not finished after {timeout:.0f}s: likely grid-barrier deadlock\\", 2)\\n        time.sleep(0.0002)\\n\\n\\ndef module_path(site):\\n    if site:\\n        return Path(site) / REL\\n    spec = importlib.util.find_spec(\\"sglang\\")  # top-level find_spec does not import the package\\n    if spec is None or not spec.submodule_search_locations:\\n        raise RuntimeError(\\"sglang not found; pass the site-packages dir\\")\\n    return Path(list(spec.submodule_search_locations)[0]) / \\"srt/layers/hc_mix_triton.py\\"\\n\\n\\ndef load_module(path, name):\\n    spec = importlib.util.spec_from_file_location(name, str(path))\\n    mod = importlib.util.module_from_spec(spec)\\n    sys.modules[name] = mod\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef dormant_check(path):\\n    \\"\\"\\"Fresh process without the env vars: the patched module must keep the stock 16-row limit.\\"\\"\\"\\n    code = (\\n        \\"import importlib.util, sys\\\\n\\"\\n        \\"s = importlib.util.spec_from_file_location(\'_hcmix32_dormant\', sys.argv[1])\\\\n\\"\\n        \\"m = importlib.util.module_from_spec(s); sys.modules[\'_hcmix32_dormant\'] = m; s.loader.exec_module(m)\\\\n\\"\\n        \\"print(\'HCMIX32\', m._FUSED_MIX_MAX_ROWS, m._HCMIX32_MAX_ROWS, m._HCMIX32_STAGES)\\\\n\\"\\n    )\\n    env = {k: v for k, v in os.environ.items() if k not in (\\"SGLANG_HC_MIX_MAX_ROWS\\", \\"SGLANG_HC_MIX_NUM_STAGES\\")}\\n    try:\\n        p = subprocess.run([sys.executable, \\"-c\\", code, str(path)], env=env, capture_output=True, text=True,\\n                           timeout=240)\\n    except Exception as exc:  # noqa: BLE001\\n        log(f\\"dormancy subprocess failed: {exc}\\")\\n        return False\\n    lines = [ln for ln in p.stdout.splitlines() if ln.startswith(\\"HCMIX32 \\")]\\n    if not lines:\\n        log(f\\"dormancy subprocess output: rc={p.returncode} {p.stdout[-300:]!r} {p.stderr[-600:]!r}\\")\\n        return False\\n    return lines[-1].split()[1:] == [\\"16\\", \\"0\\", \\"None\\"]\\n\\n\\ndef kmeta(ck):\\n    d = {}\\n    for name in (\\"n_regs\\", \\"n_spills\\"):\\n        v = getattr(ck, name, None)\\n        if isinstance(v, int):\\n            d[name] = v\\n    md = getattr(ck, \\"metadata\\", None)\\n    sh = getattr(md, \\"shared\\", None) if md is not None else None\\n    if sh is None and isinstance(md, dict):\\n        sh = md.get(\\"shared\\")\\n    if isinstance(sh, int):\\n        d[\\"shared_bytes\\"] = sh\\n    return d\\n\\n\\n# Verbatim copy of hyperconnection.py GatedResidual.__init__._mix_compute (the stock path above 16 rows).\\ndef _mix_compute(\\n    hyper_input_normed: torch.Tensor,\\n    input_mix_weight_down: torch.Tensor,\\n    input_mix_weight_up: torch.Tensor,\\n    hc: int,\\n    hs: int,\\n) -> torch.Tensor:\\n    F = torch.nn.functional\\n    input_mix_weight = F.silu(\\n        F.linear(hyper_input_normed, input_mix_weight_down) / hc\\n    )\\n    input_mix_weight = F.linear(input_mix_weight, input_mix_weight_up)\\n    input_mix_weight = torch.sigmoid(input_mix_weight)\\n    input_mix_weight = input_mix_weight.unflatten(-1, (hc, hs))\\n    output = (\\n        input_mix_weight * hyper_input_normed.unflatten(-1, (hc, hs))\\n    ).mean(dim=-2)\\n    return output\\n\\n\\ndef run(args):\\n    os.environ[\\"SGLANG_HC_MIX_MAX_ROWS\\"] = \\"32\\"  # before the module import: enables the patched 17-32 row path\\n    import triton\\n\\n    stage(\\"cuda init\\")\\n    dev = torch.device(\\"cuda\\", torch.cuda.current_device())\\n    torch.backends.cuda.matmul.allow_tf32 = False\\n    props = torch.cuda.get_device_properties(dev)\\n    n_sm = props.multi_processor_count\\n    RES.update(gpu=props.name, cc=f\\"{props.major}.{props.minor}\\", sm_count=n_sm, torch=torch.__version__,\\n               triton=triton.__version__, mixes=args.mixes, prod_rows=args.prod_rows,\\n               smem_optin_bytes=getattr(props, \\"shared_memory_per_block_optin\\", None))\\n    hang = args.hang_timeout\\n\\n    stage(\\"load hc_mix_triton\\")\\n    path = module_path(args.site)\\n    RES[\\"module\\"] = str(path)\\n    mod = load_module(path, \\"_hcmix32_bench_hc_mix_triton\\")\\n    mod._deterministic_inference_cached = False  # keep sglang.srt.server_args (and `import sglang`) out of it\\n    patched = hasattr(mod, \\"_HCMIX32_MAX_ROWS\\")\\n    RES[\\"patched_module\\"] = patched\\n    checks = {}\\n    RES[\\"checks\\"] = checks\\n\\n    rows_list = sorted({int(r) for r in args.rows.split(\\",\\") if r.strip()} | {16, args.prod_rows})\\n    if not all(1 <= r <= 32 for r in rows_list):\\n        raise ValueError(f\\"--rows must be within 1..32: {rows_list}\\")\\n    big = [r for r in rows_list if r > 16]\\n    if not big:\\n        raise ValueError(\\"need at least one row count > 16\\")\\n    stages = [int(s) for s in args.stages.split(\\",\\") if s.strip()]\\n    M = args.mixes\\n\\n    stage(\\"allocate weights + fp64 reference\\")\\n    g = torch.Generator(device=dev)\\n    g.manual_seed(20261008)\\n    wd_all = torch.randn((M, LOWRANK, K), device=dev, dtype=torch.bfloat16, generator=g).mul_(HC * K ** -0.5)\\n    wu_all = torch.randn((M, K, LOWRANK), device=dev, dtype=torch.bfloat16, generator=g).mul_(2.0 * LOWRANK ** -0.5)\\n    x_all = torch.randn((32, K), device=dev, dtype=torch.bfloat16, generator=g)\\n    # Reference mirrors the kernel: fp32-accurate down GEMM, silu in high precision, t rounded to the activation\\n    # dtype (the kernel feeds bf16 t to tl.dot), fp32-accurate up GEMM, sigmoid gate, mean over hc. Rows independent.\\n    refs = []\\n    xd = x_all.double()\\n    for i in range(M):\\n        a = (xd @ wd_all[i].double().t()) * (1.0 / HC)\\n        t = (a * torch.sigmoid(a)).to(torch.bfloat16).double()\\n        gate = torch.sigmoid(t @ wu_all[i].double().t()).view(32, HC, HS)\\n        refs.append(((gate * xd.view(32, HC, HS)).sum(1) * (1.0 / HC)).float())\\n    del xd\\n    gpu_wait(\\"reference\\", 120)\\n\\n    def stats(outs, R):\\n        mx, tot, n, bad, finite = 0.0, 0.0, 0, 0, True\\n        for i, o in enumerate(outs):\\n            of = o.float()\\n            r = refs[i][:R]\\n            d = (of - r).abs()\\n            finite = finite and bool(torch.isfinite(of).all().item())\\n            mx = max(mx, d.max().item())\\n            tot += d.sum().item()\\n            n += d.numel()\\n            bad += int((d > r.abs() * TOL_REL + TOL_ABS).sum().item())\\n        return {\\"max_abs\\": mx, \\"mean_abs\\": tot / max(n, 1), \\"bad\\": bad, \\"n\\": n, \\"finite\\": finite}\\n\\n    def pair(a_list, b_list, rows=None):\\n        \\"\\"\\"(max |a-b|, #elements outside tolerance, #elements); b is the reference side.\\"\\"\\"\\n        mx, bad, n = 0.0, 0, 0\\n        for a, b in zip(a_list, b_list):\\n            af = a[:rows].float() if rows else a.float()\\n            bf = b[:rows].float() if rows else b.float()\\n            d = (af - bf).abs()\\n            mx = max(mx, d.max().item())\\n            bad += int((d > bf.abs() * TOL_REL + TOL_ABS).sum().item())\\n            n += d.numel()\\n        return mx, bad, n\\n\\n    def pair_ok(p):\\n        return p[1] <= allowed_bad(p[2])\\n\\n    def launch(x, wd, wu, rows_pad, st):\\n        \\"\\"\\"Direct launch mirroring fused_hc_mix (stock args) with an explicit num_stages (None = Triton default).\\"\\"\\"\\n        rows = x.shape[0]\\n        t_raw = torch.empty((rows_pad, LOWRANK), dtype=torch.float32, device=x.device)\\n        out = torch.empty((rows, HS), dtype=x.dtype, device=x.device)\\n        kw = dict(ROWS=rows_pad, HC=HC, BLOCK_N=32, BLOCK_K=256, BLOCK_J=32, BLOCK_R=64, FP8_WEIGHT=False,\\n                  num_warps=8)\\n        if st is not None:\\n            kw[\\"num_stages\\"] = st\\n        ck = mod._hc_mix_persistent_kernel[(n_sm,)](\\n            x, wd, wu, t_raw, out, mod._get_counters(x.device), wd, wu, K, LOWRANK, HS, rows, n_sm, 1.0 / HC, **kw)\\n        return out, ck\\n\\n    def counters_zero():\\n        gpu_wait(\\"counters\\", hang)\\n        return mod._get_counters(dev).tolist() == [0, 0, 0]\\n\\n    # ---- module-level checks (patched file) -----------------------------------------------------------------\\n    if patched:\\n        stage(\\"patched module: env + gate checks\\")\\n        checks[\\"env_enables_32\\"] = mod._HCMIX32_MAX_ROWS == 32\\n        x33 = torch.zeros((33, K), device=dev, dtype=torch.bfloat16)\\n        gate = {r: bool(mod.fused_hc_mix_supported(x33[:r], wd_all[0], wu_all[0])) for r in (1, 16, 17, 19, 20, 32, 33)}\\n        fp8d = torch.empty((LOWRANK, K), device=dev, dtype=torch.float8_e4m3fn)\\n        fp8u = torch.empty((K, LOWRANK), device=dev, dtype=torch.float8_e4m3fn)\\n        gate_fp8_17 = bool(mod.fused_hc_mix_supported(x33[:17], fp8d, fp8u))\\n        RES[\\"supported_gate\\"] = {**{str(r): v for r, v in gate.items()}, \\"fp8_17\\": gate_fp8_17}\\n        checks[\\"supported_gate\\"] = (all(gate[r] for r in (1, 16, 17, 19, 20, 32)) and not gate[33]\\n                                    and not gate_fp8_17)\\n        del x33, fp8d, fp8u\\n        stage(\\"dormancy check (fresh process, env unset)\\")\\n        checks[\\"dormant_without_env\\"] = dormant_check(path)\\n    else:\\n        log(\\"hc_mix_triton.py is NOT patched: testing the kernel by direct launches only\\")\\n\\n    # ---- stock 16-row kernel: first launch, metadata ----------------------------------------------------------\\n    stage(\\"stock ROWS=16 kernel: compile + first launch\\")\\n    o16, ck16 = launch(x_all[:16], wd_all[0], wu_all[0], 16, None)\\n    gpu_wait(\\"first ROWS=16 launch\\", hang)\\n    RES[\\"kernel16\\"] = kmeta(ck16)\\n    checks[\\"counters_zero_after_rows16\\"] = counters_zero()\\n\\n    # ---- ROWS=32 variants: compile, first launch (hang probe), counters, metadata -------------------------------\\n    stage_info = {}\\n    R0 = args.prod_rows\\n    for st in stages:\\n        stage(f\\"ROWS=32 kernel num_stages={st}: compile + first launch\\")\\n        try:\\n            o32, ck = launch(x_all[:R0], wd_all[0], wu_all[0], 32, st)\\n        except Exception as exc:  # noqa: BLE001  (compile / out-of-resources: nothing launched)\\n            stage_info[str(st)] = {\\"compiled\\": False, \\"error\\": f\\"{type(exc).__name__}: {str(exc)[:300]}\\"}\\n            log(f\\"  num_stages={st} unusable: {stage_info[str(st)][\'error\']}\\")\\n            continue\\n        gpu_wait(f\\"first ROWS=32 launch num_stages={st}\\", hang)\\n        info = {\\"compiled\\": True, **kmeta(ck), \\"counters_zero\\": counters_zero()}\\n        e = (o32.float() - refs[0][:R0]).abs()\\n        info[\\"max_abs_err_vs_fp64\\"] = e.max().item()\\n        info[\\"single_mix_ok\\"] = int((e > refs[0][:R0].abs() * TOL_REL + TOL_ABS).sum().item()) <= allowed_bad(e.numel())\\n        stage_info[str(st)] = info\\n        log(f\\"  {info}\\")\\n    RES[\\"stages\\"] = stage_info\\n    ok_stages = [st for st in stages if stage_info[str(st)].get(\\"compiled\\")]\\n    checks[\\"rows32_compiles\\"] = bool(ok_stages)\\n    if not ok_stages:\\n        emit(ok=False, correct=False, error=\\"no ROWS=32 kernel variant compiled\\")\\n        return\\n    checks[\\"counters_zero_after_rows32\\"] = all(stage_info[str(s)][\\"counters_zero\\"] for s in ok_stages)\\n    checks[\\"rows32_single_mix_ok\\"] = all(stage_info[str(s)][\\"single_mix_ok\\"] for s in ok_stages)\\n\\n    if patched:\\n        stage(f\\"patched fused_hc_mix at {R0} rows (module num_stages ladder)\\")\\n        o = mod.fused_hc_mix(x_all[:R0], wd_all[0], wu_all[0], HC, HS)\\n        gpu_wait(\\"patched fused_hc_mix\\", hang)\\n        e = (o.float() - refs[0][:R0]).abs()\\n        RES[\\"module_num_stages\\"] = mod._HCMIX32_STAGES\\n        n_bad = int((e > refs[0][:R0].abs() * TOL_REL + TOL_ABS).sum().item())\\n        checks[\\"module_rows32_path\\"] = (mod._HCMIX32_MAX_ROWS == 32 and mod._HCMIX32_STAGES is not None\\n                                        and n_bad <= allowed_bad(e.numel()))\\n        checks[\\"counters_zero_after_module\\"] = counters_zero()\\n    module_stage = RES.get(\\"module_num_stages\\") or ok_stages[0]\\n\\n    # ---- stock chain (torch.compile, as in GatedResidual) ------------------------------------------------------\\n    chain = {\\"fn\\": _mix_compute if args.no_compile else torch.compile(_mix_compute), \\"compiled\\": not args.no_compile}\\n\\n    def f_chain(x, wd, wu):\\n        return chain[\\"fn\\"](x, wd, wu, HC, HS)\\n\\n    stage(\\"chain warmup (torch.compile) for every row count\\")\\n    for R in rows_list:\\n        try:\\n            f_chain(x_all[:R], wd_all[0], wu_all[0])\\n        except Exception as exc:  # noqa: BLE001\\n            if not chain[\\"compiled\\"]:\\n                raise\\n            RES[\\"chain_compile_error\\"] = f\\"{type(exc).__name__}: {str(exc)[:300]}\\"\\n            log(f\\"torch.compile chain failed, timing the eager chain instead: {RES[\'chain_compile_error\']}\\")\\n            chain.update(fn=_mix_compute, compiled=False)\\n            f_chain(x_all[:R], wd_all[0], wu_all[0])\\n    gpu_wait(\\"chain warmup\\", 120)\\n    RES[\\"chain_compiled\\"] = chain[\\"compiled\\"]\\n\\n    def f_stock(x, wd, wu):  # module entry: <=16 rows = untouched stock launch; 17-32 rows = patched ROWS=32 path\\n        return mod.fused_hc_mix(x, wd, wu, HC, HS)\\n\\n    def f_direct32(st):\\n        return lambda x, wd, wu: launch(x, wd, wu, 32, st)[0]\\n\\n    def capture(name, fn, R):\\n        x = x_all[:R]\\n        s = torch.cuda.Stream(device=dev)\\n        s.wait_stream(torch.cuda.current_stream())\\n        with torch.cuda.stream(s):\\n            for i in range(min(2, M)):\\n                fn(x, wd_all[i], wu_all[i])\\n        torch.cuda.current_stream().wait_stream(s)\\n        gpu_wait(f\\"warmup {name}@{R}\\", hang)\\n        graph = torch.cuda.CUDAGraph()\\n        with torch.cuda.graph(graph):\\n            outs = [fn(x, wd_all[i], wu_all[i]) for i in range(M)]\\n        graph.replay()\\n        gpu_wait(f\\"first replay {name}@{R}\\", hang)\\n        return graph, outs\\n\\n    def time_graph(name, graph):\\n        st_ev = torch.cuda.Event(enable_timing=True)\\n        en_ev = torch.cuda.Event(enable_timing=True)\\n        st_ev.record()\\n        for _ in range(args.iters):\\n            graph.replay()\\n        en_ev.record()\\n        gpu_wait(f\\"timed replays {name}\\", hang * 2)\\n        return st_ev.elapsed_time(en_ev) * 1000.0 / args.iters  # us per graph = per decode step (M mixes)\\n\\n    def bench(R, variants):\\n        \\"\\"\\"variants: {name: fn}; returns ({name: median step us}, {name: outs}); interleaved reps.\\"\\"\\"\\n        graphs, outs = {}, {}\\n        for name, fn in variants.items():\\n            stage(f\\"capture {name} @ {R} rows ({M} mixes)\\")\\n            graphs[name], outs[name] = capture(name, fn, R)\\n        samples = {name: [] for name in variants}\\n        stage(f\\"timing @ {R} rows: {list(variants)}\\")\\n        for _ in range(args.reps):\\n            for name, graph in graphs.items():\\n                samples[name].append(time_graph(f\\"{name}@{R}\\", graph))\\n        med = {name: statistics.median(v) for name, v in samples.items()}\\n        return med, outs, graphs\\n\\n    # ---- num_stages sweep for the ROWS=32 kernel at the production row count ------------------------------------\\n    sweep, sweep_stats = {}, {}\\n    if len(ok_stages) > 1:\\n        med, _outs, _graphs = bench(R0, {f\\"s{st}\\": f_direct32(st) for st in ok_stages})\\n        sweep = {st: med[f\\"s{st}\\"] for st in ok_stages}\\n        for st in ok_stages:\\n            stage_info[str(st)][f\\"step_us_at_{R0}\\"] = round(sweep[st], 1)\\n            sweep_stats[st] = stats(_outs[f\\"s{st}\\"], R0)\\n            stage_info[str(st)][f\\"vs_fp64_at_{R0}\\"] = sweep_stats[st]\\n        del _outs, _graphs\\n        torch.cuda.empty_cache()\\n    best_stage = min(sweep, key=sweep.get) if sweep else ok_stages[0]\\n    RES[\\"best_num_stages\\"] = best_stage\\n\\n    # ---- per row count: stock fused16 / patched fused32 vs stock chain ----------------------------------------------\\n    per_rows = {}\\n    stock16_outs = None\\n    correct = all(checks.values())\\n    faster = True\\n    max_err_vs_chain = 0.0\\n    for R in rows_list:\\n        if R <= 16:\\n            variants = {\\"fused16\\": f_stock, \\"fused32\\": f_direct32(module_stage), \\"chain\\": f_chain}\\n        else:\\n            variants = {\\"fused32\\": f_stock if patched else f_direct32(module_stage), \\"chain\\": f_chain}\\n        med, outs, graphs = bench(R, variants)\\n        stage(f\\"checks @ {R} rows\\")\\n        row = {f\\"{n}_us\\": round(v, 1) for n, v in med.items()}\\n        row.update({f\\"{n}_per_mix_us\\": round(v / M, 2) for n, v in med.items()})\\n        for n in variants:\\n            row[f\\"{n}_vs_fp64\\"] = stats(outs[n], R)\\n        f_outs = outs[\\"fused16\\"] if R <= 16 else outs[\\"fused32\\"]\\n        row[\\"fused_vs_chain_max_abs\\"] = pair(f_outs, outs[\\"chain\\"])[0]\\n        if R <= 16:\\n            row[\\"fused32_vs_fused16\\"] = pair(outs[\\"fused32\\"], outs[\\"fused16\\"])\\n            if R == 16:\\n                stock16_outs = [o.clone() for o in outs[\\"fused16\\"]]\\n        else:\\n            if stock16_outs is not None:\\n                row[\\"first16_vs_stock16\\"] = pair(outs[\\"fused32\\"], stock16_outs, rows=16)\\n            max_err_vs_chain = max(max_err_vs_chain, row[\\"fused_vs_chain_max_abs\\"])\\n            faster = faster and med[\\"fused32\\"] < med[\\"chain\\"]\\n        # replay-to-replay: only atomic summation order may differ (and the barrier counters must reset)\\n        key = \\"fused16\\" if R <= 16 else \\"fused32\\"\\n        before = [o.clone() for o in outs[key]]\\n        graphs[key].replay()\\n        gpu_wait(f\\"consistency replay @ {R}\\", hang)\\n        row[\\"replay_diff\\"] = pair(outs[key], before)\\n        row[\\"counters_zero\\"] = counters_zero()\\n        per_rows[str(R)] = row\\n        log(f\\"  {R} rows: {row}\\")\\n        del graphs, outs, before, f_outs\\n        torch.cuda.empty_cache()\\n\\n    # ---- verdict --------------------------------------------------------------------------------------------------\\n    s16 = per_rows[\\"16\\"][\\"fused16_vs_fp64\\"]\\n\\n    def numerics_ok(s):\\n        # within tolerance of the fp64 reference (all but a handful of elements) AND no worse than the stock\\n        # 16-row kernel\'s own error profile (the same kernel serves <=16 rows in production today)\\n        return bool(s[\\"finite\\"] and s[\\"bad\\"] <= allowed_bad(s[\\"n\\"])\\n                    and s[\\"max_abs\\"] <= 2 * s16[\\"max_abs\\"] + 2.0 ** -8\\n                    and s[\\"mean_abs\\"] <= 1.5 * s16[\\"mean_abs\\"] + 1e-6)\\n\\n    num = {}\\n    for R in rows_list:\\n        row = per_rows[str(R)]\\n        key = \\"fused16\\" if R <= 16 else \\"fused32\\"\\n        num[f\\"{R}_{key}\\"] = numerics_ok(row[f\\"{key}_vs_fp64\\"])\\n        num[f\\"{R}_counters\\"] = row[\\"counters_zero\\"]\\n        num[f\\"{R}_replay\\"] = pair_ok(row[\\"replay_diff\\"])\\n        if R <= 16:\\n            num[f\\"{R}_fused32_pad\\"] = numerics_ok(row[\\"fused32_vs_fp64\\"]) and pair_ok(row[\\"fused32_vs_fused16\\"])\\n        elif \\"first16_vs_stock16\\" in row:\\n            num[f\\"{R}_first16_vs_stock16\\"] = pair_ok(row[\\"first16_vs_stock16\\"])\\n    for st, sst in sweep_stats.items():\\n        num[f\\"sweep_s{st}\\"] = numerics_ok(sst)\\n    stock_jump = per_rows[\\"16\\"][\\"fused_vs_chain_max_abs\\"]\\n    num[\\"vs_chain_like_stock_boundary\\"] = max_err_vs_chain <= 2 * stock_jump + 2.0 ** -7\\n    RES[\\"numerics\\"] = num\\n    correct = correct and all(num.values())\\n    if not patched:\\n        checks[\\"patch_applied\\"] = bool(args.allow_unpatched)\\n        correct = correct and bool(args.allow_unpatched)\\n\\n    prod = per_rows[str(R0)]\\n    baseline_us, patched_us = prod[\\"chain_us\\"], prod[\\"fused32_us\\"]\\n    # Enable exactly the row range that was measured faster (default --rows: 17..20 = every production decode graph\\n    # above 16). Larger decode graphs (24/32) need --rows ...,24,32 first.\\n    rec = f\\"SGLANG_HC_MIX_MAX_ROWS={max(big)}\\"\\n    if sweep and best_stage != module_stage and sweep[best_stage] < 0.98 * sweep.get(module_stage, float(\\"inf\\")):\\n        rec += f\\" SGLANG_HC_MIX_NUM_STAGES={best_stage}\\"\\n    # The server\'s fallback is torch.compile(_mix_compute); an eager-chain baseline is slower than production, so a\\n    # speed win measured against it is not evidence -> not ok.\\n    ok = bool(correct and faster and chain[\\"compiled\\"])\\n    failed = [f\\"check:{k}\\" for k, v in checks.items() if not v] + [f\\"numerics:{k}\\" for k, v in num.items() if not v]\\n    failed += [f\\"slower@{R}\\" for R in big if per_rows[str(R)][\\"fused32_us\\"] >= per_rows[str(R)][\\"chain_us\\"]]\\n    if not chain[\\"compiled\\"]:\\n        failed.append(\\"chain_not_compiled(baseline is the eager chain; speed verdict unverified)\\")\\n    emit(\\n        ok=ok,\\n        failed=failed,\\n        error=None if ok else (\\"patch not applied (kernel-level results only)\\" if not patched else \\"see failed\\"),\\n        correct=bool(correct),\\n        faster=bool(faster),\\n        baseline_us=round(baseline_us, 1),\\n        patched_us=round(patched_us, 1),\\n        saving_ms_per_step=round((baseline_us - patched_us) / 1000.0, 3),\\n        max_abs_err=max_err_vs_chain,\\n        max_abs_err_stock16=stock_jump,\\n        max_abs_err_vs_fp64={\\n            \\"fused32\\": max(per_rows[str(R)][\\"fused32_vs_fp64\\"][\\"max_abs\\"] for R in big),\\n            \\"chain\\": max(per_rows[str(R)][\\"chain_vs_fp64\\"][\\"max_abs\\"] for R in big),\\n            \\"stock_fused16@16\\": s16[\\"max_abs\\"],\\n        },\\n        cliff_16_to_17={\\"stock_fused16@16_us\\": per_rows[\\"16\\"][\\"fused16_us\\"],\\n                        \\"chain@17_us\\": per_rows.get(\\"17\\", {}).get(\\"chain_us\\"),\\n                        \\"fused32@17_us\\": per_rows.get(\\"17\\", {}).get(\\"fused32_us\\")},\\n        per_rows=per_rows,\\n        recommend_env=rec if ok else None,\\n    )\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    ap.add_argument(\\"site\\", nargs=\\"?\\", default=None, help=\\"site-packages dir (default: where sglang is found)\\")\\n    ap.add_argument(\\"--rows\\", default=\\"16,17,19,20\\", help=\\"row counts to time (16 and --prod-rows always added)\\")\\n    ap.add_argument(\\"--prod-rows\\", type=int, default=19, help=\\"row count for baseline_us / patched_us\\")\\n    ap.add_argument(\\"--mixes\\", type=int, default=97, help=\\"mixes per CUDA graph (97 = one decode step)\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=20, help=\\"graph replays per timing sample\\")\\n    ap.add_argument(\\"--reps\\", type=int, default=5, help=\\"interleaved timing samples per variant (median)\\")\\n    ap.add_argument(\\"--stages\\", default=\\"3,2,1\\", help=\\"num_stages values to try for the ROWS=32 kernel\\")\\n    ap.add_argument(\\"--timeout\\", type=float, default=540, help=\\"total watchdog seconds\\")\\n    ap.add_argument(\\"--hang-timeout\\", type=float, default=30, help=\\"seconds before queued GPU work counts as hung\\")\\n    ap.add_argument(\\"--no-compile\\", action=\\"store_true\\", help=\\"time the eager chain instead of torch.compile\\")\\n    ap.add_argument(\\"--allow-unpatched\\", action=\\"store_true\\", help=\\"ok may be true without the patch applied\\")\\n    args = ap.parse_args()\\n    threading.Thread(target=watchdog, args=(time.time() + args.timeout,), daemon=True).start()\\n    try:\\n        run(args)\\n    except Exception as exc:  # noqa: BLE001\\n        tb = traceback.format_exc()\\n        log(tb)\\n        die(f\\"{type(exc).__name__}: {str(exc)[:500]} | {tb[-1200:]}\\", 1)\\n    sys.stdout.flush()\\n    sys.stderr.flush()\\n    os._exit(0)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "bench_hisparse.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 4-GPU bench for the STOCK HiSparse swap-in kernel instantiated for QSA (K+V path, items = 4-token blocks), key \\"hisparse\\".\\n\\nNo sglang file is patched: the bench calls sglang.kernels.ops.kvcache.hisparse._jit_sparse_module(is_mla=False) directly\\n(the non-MLA K+V instantiation nobody ships). Item = one 4-token block of one KV head per GPU: 4 x 256 B fp8 = 1024 B for K and\\nagain for V. NUM_TOP_K = 512 block ids per row, HOT_BUFFER_SIZE = 1536 blocks per (layer, request), + 2 extra slots (newest block /\\nping-pong). Host KV is pinned memory read by the kernel through UVA (ld.global.nc) on a miss.\\n  * correctness (rank 0): the kernel, with RecordMissPlan on, against an exact python oracle of its LRU policy over a random\\n    chain of steps (hits / misses / evictions / lru order / newest-block binding / fast path nblk <= HOT / -1 padding beyond\\n    num_real_reqs / int32 + int64 index dtypes), plus the bytes of every selected slot (K and V) against the host pattern.\\n  * timing, per decode step = 12 layers (CUDA graph, 12 kernel launches reading one static top-k buffer per layer):\\n      io_us    full kernel, misses copied from pinned host memory\\n      plan_us  SkipIO instantiation (LRU planning only, no bytes)\\n    for rows {19,32,48,64} x churn {3,6,12,30} % of the 512 ids replaced per step (fresh random ids -> miss rate = churn),\\n    solo (rank 0 alone) and all4 (the four GPUs copying at once: the Kaggle host-link contention case).\\n    Headline: rows 32, churn 6 %: patched_us = io_us all4, baseline_us = None (nothing to gate; the bench reports numbers).\\nusage: python3 bench_hisparse.py <site-packages dir> [--rows 19,32,48,64] [--churn 0.03,0.06,0.12,0.30] [--iters 24] [--timeout 800]\\nstdout gets ONE JSON line.\\n\\"\\"\\"\\nimport argparse\\nimport importlib\\nimport json\\nimport os\\nimport socket\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nKEY = \\"hisparse\\"\\nITEM = 1024\\nTOPK = 512\\nHOT = 1536\\nPAD = HOT + 2\\nLAYERS = 12\\nMAX_BLK = 34816\\nHOST_BLOCKS = 1 << 18\\nRMAX = 64\\nWORLD = 4\\nTHREADS = 512   # the kernel needs BLOCK_SIZE >= NUM_TOP_K (v80 diag: 128 and 256 threads fault on the slow path)\\n\\n_OUT = sys.stdout\\n_EMITTED = []\\n_LOCK = threading.Lock()\\n\\n\\ndef log(*a):\\n    print(\\"[bench_hisparse]\\", *a, file=sys.stderr, flush=True)\\n\\n\\ndef claim_stdout():\\n    global _OUT\\n    try:\\n        sys.stdout.flush()\\n        _OUT = os.fdopen(os.dup(1), \\"w\\")\\n        os.dup2(2, 1)\\n    except Exception:\\n        _OUT = sys.stdout\\n\\n\\ndef emit(res):\\n    with _LOCK:\\n        if _EMITTED:\\n            return\\n        _EMITTED.append(1)\\n        _OUT.write(json.dumps(res, default=str) + \\"\\\\n\\")\\n        _OUT.flush()\\n\\n\\ndef pat_k(ids):\\n    ar = torch.arange(ITEM, device=ids.device, dtype=torch.int64)\\n    return ((ids.long()[:, None] * 131 + ar[None, :] * 7) % 251).to(torch.uint8)\\n\\n\\ndef pat_v(ids):\\n    ar = torch.arange(ITEM, device=ids.device, dtype=torch.int64)\\n    return ((ids.long()[:, None] * 137 + ar[None, :] * 11 + 5) % 251).to(torch.uint8)\\n\\n\\nclass Rig:\\n    \\"\\"\\"Buffers for one GPU: pinned host K/V (shared by all layers), per-layer device K/V + per-layer swap-in state.\\"\\"\\"\\n\\n    def __init__(self, hs, dev):\\n        self.hs, self.dev = hs, dev\\n        self.host_k = torch.empty((HOST_BLOCKS, ITEM), dtype=torch.uint8, pin_memory=True)\\n        self.host_v = torch.empty((HOST_BLOCKS, ITEM), dtype=torch.uint8, pin_memory=True)\\n        ch = 16384\\n        for s in range(0, HOST_BLOCKS, ch):\\n            ids = torch.arange(s, min(s + ch, HOST_BLOCKS), device=dev)\\n            self.host_k[s:s + len(ids)].copy_(pat_k(ids))\\n            self.host_v[s:s + len(ids)].copy_(pat_v(ids))\\n        r = torch.arange(RMAX, dtype=torch.int64)[:, None]\\n        b = torch.arange(MAX_BLK, dtype=torch.int64)[None, :]\\n        self.host_locs = ((b * 17 + r * 9973) % HOST_BLOCKS).to(dev)          # int64 [R, MAX_BLK]\\n        self.dev_k = [torch.zeros((RMAX * PAD, ITEM), dtype=torch.uint8, device=dev) for _ in range(LAYERS)]\\n        self.dev_v = [torch.zeros((RMAX * PAD, ITEM), dtype=torch.uint8, device=dev) for _ in range(LAYERS)]\\n        self.locs = (torch.arange(RMAX, dtype=torch.int32)[:, None] * PAD\\n                     + torch.arange(PAD, dtype=torch.int32)[None, :]).to(dev)   # int32 [R, PAD]\\n        self.tok = torch.full((LAYERS, RMAX, PAD), -1, dtype=torch.int32, device=dev)\\n        self.lru = torch.arange(HOT, dtype=torch.int16, device=dev).view(1, 1, -1).repeat(LAYERS, RMAX, 1).contiguous()\\n        self.empty = torch.empty(0)\\n        torch.cuda.synchronize()\\n\\n    def reset(self):\\n        self.tok.fill_(-1)\\n        self.lru.copy_(torch.arange(HOT, dtype=torch.int16, device=self.dev).view(1, 1, -1))\\n\\n    def mod(self, skip_io=False, plan=False, threads=THREADS):\\n        return self.hs._jit_sparse_module(ITEM, threads, TOPK, HOT, False, False, plan, skip_io)\\n\\n    def call(self, mod, layer, topk, out, rpi, seq, num_real, plan=None):\\n        ms, md, mc = plan if plan is not None else (self.empty, self.empty, self.empty)\\n        mod.load_cache_to_device_buffer(\\n            topk, self.tok[layer], self.host_locs, self.locs, self.host_k, self.host_v,\\n            self.dev_k[layer], self.dev_v[layer], out, rpi, seq, self.lru[layer], num_real, 1, ITEM, ms, md, mc)\\n\\n\\ndef oracle_step(tok, lru, topk, nblk):\\n    \\"\\"\\"Exact python model of load_cache_to_device_buffer_kernel for one request with nblk > HOT.\\"\\"\\"\\n    newest = nblk - 1\\n    want = {t for t in topk if t != newest and t >= 0}\\n    hits = [s for s in lru if tok[s] >= 0 and tok[s] in want]\\n    evict = [s for s in lru if not (tok[s] >= 0 and tok[s] in want)]\\n    cached = {tok[s] for s in hits}\\n    misses = [t for t in topk if t >= 0 and t != newest and t not in cached]\\n    loc = {tok[s]: s for s in hits}\\n    new_tok = list(tok)\\n    for k, t in enumerate(misses):\\n        new_tok[evict[k]] = t\\n        loc[t] = evict[k]\\n    m = len(misses)\\n    new_lru = evict[m:] + evict[:m] + hits\\n    out = [(HOT if t == newest else loc[t]) if t >= 0 else -1 for t in topk]\\n    return out, new_tok, new_lru, misses, evict\\n\\n\\ndef run_correctness(rig, dev, res):\\n    nblk = [20000, 5000, 1537, 3000, 34000, 1536, 1000, 100]\\n    B = len(nblk)\\n    rng = np.random.default_rng(3)\\n    mod = rig.mod(plan=True)\\n    rig.reset()\\n    # fast-path rows: blocks 0..nblk-1 sit in order in slots 0..nblk-1, like the coordinator\'s preload\\n    for r, n in enumerate(nblk):\\n        if n <= HOT:\\n            ids = rig.host_locs[r, :n]\\n            rig.dev_k[0][r * PAD:r * PAD + n] = pat_k(ids)\\n            rig.dev_v[0][r * PAD:r * PAD + n] = pat_v(ids)\\n    chains = [None] * B\\n    ms = torch.full((B, TOPK), -1, dtype=torch.int64, device=dev)\\n    md = torch.full((B, TOPK), -1, dtype=torch.int32, device=dev)\\n    mc = torch.zeros(B, dtype=torch.int32, device=dev)\\n    out = torch.full((B, TOPK), -2, dtype=torch.int32, device=dev)\\n    st = [(list(rig.tok[0, r, :HOT].tolist()), list(range(HOT))) for r in range(B)]\\n    bad = {\\"out\\": 0, \\"tok\\": 0, \\"lru\\": 0, \\"miss\\": 0, \\"plan\\": 0, \\"content\\": 0, \\"repeat\\": 0, \\"pad\\": 0}\\n    checked, misses_total = 0, 0\\n    notes = []\\n    steps = 36\\n    for step in range(steps):\\n        rows = []\\n        for r, n in enumerate(nblk):\\n            if n <= HOT:\\n                k = min(n, TOPK)\\n                ids = rng.choice(n, size=k, replace=False).astype(np.int32)\\n                row = np.full(TOPK, -1, np.int32)\\n                row[:k] = ids\\n            else:\\n                if chains[r] is None:\\n                    chains[r] = rng.choice(n - 1, size=TOPK, replace=False).astype(np.int32)\\n                else:\\n                    c = chains[r].copy()\\n                    pos = rng.choice(TOPK, size=int(TOPK * 0.12), replace=False)\\n                    s = set(c.tolist())\\n                    fresh = []\\n                    while len(fresh) < len(pos):\\n                        v = int(rng.integers(0, n - 1))\\n                        if v not in s:\\n                            s.add(v)\\n                            fresh.append(v)\\n                    c[pos] = fresh\\n                    chains[r] = c\\n                row = chains[r].copy()\\n                if step % 3 == 1:\\n                    row[int(rng.integers(0, TOPK))] = n - 1       # newest block selected: bound to the extra slot\\n                    if len(set(row.tolist())) != TOPK:\\n                        row = chains[r].copy()\\n            rows.append(row)\\n        topk = torch.from_numpy(np.stack(rows)).to(dev)\\n        i64 = step >= steps * 2 // 3\\n        rpi = torch.arange(B, dtype=torch.int64 if i64 else torch.int32, device=dev)\\n        seq = torch.tensor(nblk, dtype=torch.int64 if i64 else torch.int32, device=dev)\\n        for r, n in enumerate(nblk):          # newest block lives in the extra slot, written by the model each step\\n            if n > HOT:\\n                ids = rig.host_locs[r, n - 1:n]\\n                rig.dev_k[0][r * PAD + HOT] = pat_k(ids)[0]\\n                rig.dev_v[0][r * PAD + HOT] = pat_v(ids)[0]\\n        num_real = torch.tensor([B], dtype=torch.int32, device=dev)\\n        before = [(list(rig.tok[0, r, :HOT].tolist()), rig.lru[0, r].tolist()) for r in range(B)]\\n        rig.call(mod, 0, topk, out, rpi, seq, num_real, (ms, md, mc))\\n        torch.cuda.synchronize()\\n        out_c, tok_c, lru_c, mc_c = out.cpu(), rig.tok[0].cpu(), rig.lru[0].cpu(), mc.cpu()\\n        ms_c, md_c = ms.cpu(), md.cpu()\\n        for r, n in enumerate(nblk):\\n            checked += 1\\n            row = topk[r].tolist()\\n            got = out_c[r].tolist()\\n            if n <= HOT:\\n                exp = [r * PAD + t if t >= 0 else -1 for t in row]\\n                if got != exp:\\n                    bad[\\"out\\"] += 1\\n                if int(mc_c[r]) != 0:\\n                    bad[\\"miss\\"] += 1\\n            else:\\n                otok, olru = before[r]\\n                exp_loc, ntok, nlru, misses, evict = oracle_step(otok, olru, row, n)\\n                exp = [r * PAD + s for s in exp_loc]\\n                if got != exp:\\n                    bad[\\"out\\"] += 1\\n                if tok_c[r, :HOT].tolist() != ntok:\\n                    bad[\\"tok\\"] += 1\\n                if lru_c[r].tolist() != nlru:\\n                    bad[\\"lru\\"] += 1\\n                if int(mc_c[r]) != len(misses):\\n                    bad[\\"miss\\"] += 1\\n                else:\\n                    srcs = ms_c[r, :len(misses)].tolist()\\n                    dsts = md_c[r, :len(misses)].tolist()\\n                    hl = rig.host_locs[r].cpu()\\n                    if srcs != [int(hl[t]) for t in misses] or dsts != [r * PAD + evict[k] for k in range(len(misses))]:\\n                        bad[\\"plan\\"] += 1\\n                misses_total += len(misses)\\n            valid = out_c[r] >= 0\\n            if bool(valid.any()):\\n                locs_v = out_c[r][valid].to(dev).long()\\n                ids = topk[r][valid.to(dev)]\\n                hlv = rig.host_locs[r][ids.long()]\\n                if not (torch.equal(rig.dev_k[0][locs_v], pat_k(hlv)) and torch.equal(rig.dev_v[0][locs_v], pat_v(hlv))):\\n                    bad[\\"content\\"] += 1\\n                    if len(notes) < 4:\\n                        notes.append({\\"step\\": step, \\"row\\": r, \\"nblk\\": n, \\"what\\": \\"content\\"})\\n        # same top-k again: every id must now be a hit (0 misses), identical slots\\n        out2 = out.clone()\\n        mc.zero_()\\n        rig.call(mod, 0, topk, out2, rpi, seq, num_real, (ms, md, mc))\\n        torch.cuda.synchronize()\\n        if not torch.equal(out2, out) or int(mc.sum()) != 0:\\n            bad[\\"repeat\\"] += 1\\n    # padded graph rows: num_real_reqs < B must leave out = -1 for the padded rows\\n    num_real = torch.tensor([B - 2], dtype=torch.int32, device=dev)\\n    out3 = torch.zeros((B, TOPK), dtype=torch.int32, device=dev)\\n    rig.call(mod, 0, topk, out3, torch.arange(B, dtype=torch.int32, device=dev),\\n             torch.tensor(nblk, dtype=torch.int32, device=dev), num_real, (ms, md, mc))\\n    torch.cuda.synchronize()\\n    if not bool((out3[B - 2:] == -1).all()):\\n        bad[\\"pad\\"] += 1\\n    res[\\"correctness\\"] = {\\"rows_checked\\": checked, \\"misses_checked\\": misses_total, \\"bad\\": bad,\\n                          \\"rows_bad\\": int(sum(bad.values())), \\"notes\\": notes}\\n\\n\\ndef chain_trace(rng, nblk, steps, churn):\\n    cur = rng.choice(nblk - 1, size=TOPK, replace=False).astype(np.int32)\\n    out = np.empty((steps, TOPK), np.int32)\\n    out[0] = cur\\n    k = max(1, int(round(TOPK * churn)))\\n    for t in range(1, steps):\\n        pos = rng.choice(TOPK, size=k, replace=False)\\n        s = set(cur.tolist())\\n        fresh = []\\n        while len(fresh) < k:\\n            v = int(rng.integers(0, nblk - 1))\\n            if v not in s:\\n                s.add(v)\\n                fresh.append(v)\\n        cur = cur.copy()\\n        cur[pos] = fresh\\n        out[t] = cur\\n    return out\\n\\n\\ndef build_trace(rows, churn, steps, dev, mean_blk=20000):\\n    rng = np.random.default_rng(11)\\n    nblk = [int(min(MAX_BLK - 2, max(HOT + 64, mean_blk * (0.4 + 1.2 * r / max(1, rows - 1))))) for r in range(rows)]\\n    tr = np.empty((steps, LAYERS, rows, TOPK), np.int32)\\n    for l in range(LAYERS):\\n        for r in range(rows):\\n            tr[:, l, r] = chain_trace(rng, nblk[r], steps, churn)\\n    return torch.from_numpy(tr).to(dev), torch.tensor(nblk, dtype=torch.int32, device=dev)\\n\\n\\ndef measure(rig, dev, mods, trace, seq, rows, iters):\\n    \\"\\"\\"-> microseconds per decode step (12 layers) for each module in mods, measured on the current process only.\\"\\"\\"\\n    rpi = torch.arange(rows, dtype=torch.int32, device=dev)\\n    num_real = torch.tensor([rows], dtype=torch.int32, device=dev)\\n    topk_buf = trace[0].clone()\\n    out_buf = torch.empty((LAYERS, rows, TOPK), dtype=torch.int32, device=dev)\\n    res = {}\\n    for name, mod in mods.items():\\n        rig.reset()\\n\\n        def run():\\n            for l in range(LAYERS):\\n                rig.call(mod, l, topk_buf[l], out_buf[l], rpi, seq, num_real)\\n        s = torch.cuda.Stream()\\n        s.wait_stream(torch.cuda.current_stream())\\n        e0 = torch.cuda.Event(enable_timing=True)\\n        e1 = torch.cuda.Event(enable_timing=True)\\n        with torch.cuda.stream(s):\\n            e0.record(s)\\n            run()                                   # cold step: every id misses (also JIT-loads the module)\\n            e1.record(s)\\n        torch.cuda.current_stream().wait_stream(s)\\n        torch.cuda.synchronize()\\n        cold_ms = e0.elapsed_time(e1)\\n        g = torch.cuda.CUDAGraph()\\n        with torch.cuda.graph(g):\\n            run()\\n        torch.cuda.synchronize()\\n        res[name + \\"_cold_ms\\"] = round(cold_ms, 2)\\n        res[name] = g\\n    return res, topk_buf\\n\\n\\ndef time_steps(rig, dev, g, topk_buf, trace, iters):\\n    t0 = torch.cuda.Event(enable_timing=True)\\n    t1 = torch.cuda.Event(enable_timing=True)\\n    torch.cuda.synchronize()\\n    t0.record()\\n    for t in range(1, iters + 1):\\n        topk_buf.copy_(trace[t])\\n        g.replay()\\n    t1.record()\\n    torch.cuda.synchronize()\\n    return t0.elapsed_time(t1) * 1000.0 / iters\\n\\n\\ndef worker(rank, world, port, out_path, rows_list, churns, iters, site, deadline):\\n    import torch.distributed as dist\\n    res = {\\"errors\\": [], \\"correctness\\": None, \\"timing\\": [], \\"jit_s\\": None}\\n\\n    def save():\\n        if rank == 0:\\n            Path(out_path).write_text(json.dumps(res, default=str))\\n\\n    torch.cuda.set_device(rank)\\n    dev = torch.device(\\"cuda\\", rank)\\n    dist.init_process_group(\\"gloo\\", init_method=f\\"tcp://127.0.0.1:{port}\\", rank=rank, world_size=world)\\n    try:\\n        hs = importlib.import_module(\\"sglang.kernels.ops.kvcache.hisparse\\")\\n        rig = Rig(hs, dev)\\n        t0 = time.time()\\n        if rank == 0:       # JIT the three instantiations once; the other ranks then hit the on-disk cache\\n            rig.mod(plan=True); rig.mod(); rig.mod(skip_io=True)\\n        dist.barrier()\\n        if rank != 0:\\n            rig.mod(plan=True); rig.mod(); rig.mod(skip_io=True)\\n        dist.barrier()\\n        res[\\"jit_s\\"] = round(time.time() - t0, 1)\\n        if rank == 0:\\n            run_correctness(rig, dev, res)\\n            save()\\n        dist.barrier()\\n        steps = iters + 1\\n        for rows in rows_list:\\n            for churn in churns:\\n                if time.time() > deadline:\\n                    res[\\"errors\\"].append(\\"deadline: remaining scenarios skipped\\")\\n                    break\\n                trace, seq = build_trace(rows, churn, steps, dev)\\n                mods = {\\"io\\": rig.mod(), \\"plan\\": rig.mod(skip_io=True)}\\n                graphs, topk_buf = measure(rig, dev, mods, trace, seq, rows, iters)\\n                rec = {\\"rows\\": rows, \\"churn\\": churn}\\n                for name in (\\"io\\", \\"plan\\"):\\n                    for mode in (\\"solo\\", \\"all4\\"):\\n                        dist.barrier()\\n                        if mode == \\"all4\\" or rank == 0:\\n                            rig.reset()\\n                            topk_buf.copy_(trace[0]); graphs[name].replay(); torch.cuda.synchronize()\\n                            rec[f\\"{name}_{mode}_us\\"] = round(time_steps(rig, dev, graphs[name], topk_buf, trace, iters), 1)\\n                        dist.barrier()\\n                    rec[f\\"{name}_cold_ms\\"] = graphs.get(name + \\"_cold_ms\\")\\n                miss_bytes = rows * LAYERS * TOPK * churn * 2 * ITEM\\n                rec[\\"miss_MB_step\\"] = round(miss_bytes / 1e6, 1)\\n                if rec.get(\\"io_all4_us\\") and rec.get(\\"plan_all4_us\\") and rec[\\"io_all4_us\\"] > rec[\\"plan_all4_us\\"]:\\n                    rec[\\"eff_GBps_all4\\"] = round(miss_bytes / ((rec[\\"io_all4_us\\"] - rec[\\"plan_all4_us\\"]) * 1e3), 2)\\n                res[\\"timing\\"].append(rec)\\n                save()\\n                del graphs, trace\\n        # kernel launch shape: 512 threads at the headline scenario\\n        try:\\n            rows, churn = 32, 0.06\\n            trace, seq = build_trace(rows, churn, steps, dev)\\n            m512 = {\\"io512\\": rig.mod(threads=512)}\\n            graphs, topk_buf = measure(rig, dev, m512, trace, seq, rows, iters)\\n            dist.barrier()\\n            rig.reset()\\n            topk_buf.copy_(trace[0]); graphs[\\"io512\\"].replay(); torch.cuda.synchronize()\\n            res[\\"io512_all4_us_rows32_churn6\\"] = round(time_steps(rig, dev, graphs[\\"io512\\"], topk_buf, trace, iters), 1)\\n            dist.barrier()\\n        except Exception as exc:\\n            res[\\"errors\\"].append(f\\"threads512: {type(exc).__name__}: {str(exc)[:200]}\\")\\n        save()\\n    except Exception as exc:\\n        res[\\"errors\\"].append(f\\"rank {rank}: {type(exc).__name__}: {str(exc)[:400]}\\")\\n        res[\\"errors\\"].append(traceback.format_exc()[-1500:])\\n        save()\\n    finally:\\n        try:\\n            dist.barrier()\\n            dist.destroy_process_group()\\n        except Exception:\\n            pass\\n\\n\\nDIAG_CASES = {   # name: (nblk list, plan, skip_io, int64 seq/rpi, threads)\\n    \\"fast_B1\\": ([100], False, False, False, 256),\\n    \\"slow_B1_skipio\\": ([5000], False, True, False, 256),\\n    \\"slow_B1\\": ([5000], False, False, False, 256),\\n    \\"slow_B1_plan\\": ([5000], True, False, False, 256),\\n    \\"slow_B1_i64\\": ([5000], False, False, True, 256),\\n    \\"slow_B1_t128\\": ([5000], False, False, False, 128),\\n    \\"slow_B1_t512\\": ([5000], False, False, False, 512),\\n    \\"mixed2_plan\\": ([100, 5000], True, False, False, 256),\\n    \\"mixed8_plan\\": ([20000, 5000, 1537, 3000, 34000, 1536, 1000, 100], True, False, False, 256),\\n}\\n\\n\\ndef diag_case(name):\\n    \\"\\"\\"One correctness-style step in a fresh process (an illegal access kills the CUDA context, so one case per process).\\"\\"\\"\\n    nblk, plan, skip_io, i64, threads = DIAG_CASES[name]\\n    torch.cuda.set_device(0)\\n    dev = torch.device(\\"cuda\\", 0)\\n    hs = importlib.import_module(\\"sglang.kernels.ops.kvcache.hisparse\\")\\n    rig = Rig(hs, dev)\\n    mod = rig.mod(skip_io=skip_io, plan=plan, threads=threads)\\n    rig.reset()\\n    B = len(nblk)\\n    rng = np.random.default_rng(5)\\n    for step in range(2):\\n        rows = []\\n        for n in nblk:\\n            k = min(n, TOPK)\\n            row = np.full(TOPK, -1, np.int32)\\n            row[:k] = rng.choice(n if n <= HOT else n - 1, size=k, replace=False).astype(np.int32)\\n            rows.append(row)\\n        topk = torch.from_numpy(np.stack(rows)).to(dev)\\n        dt = torch.int64 if i64 else torch.int32\\n        rpi, seq = torch.arange(B, dtype=dt, device=dev), torch.tensor(nblk, dtype=dt, device=dev)\\n        out = torch.full((B, TOPK), -2, dtype=torch.int32, device=dev)\\n        pl = (torch.full((B, TOPK), -1, dtype=torch.int64, device=dev), torch.full((B, TOPK), -1, dtype=torch.int32, device=dev),\\n              torch.zeros(B, dtype=torch.int32, device=dev)) if plan else None\\n        rig.call(mod, 0, topk, out, rpi, seq, torch.tensor([B], dtype=torch.int32, device=dev), pl)\\n        torch.cuda.synchronize()\\n        log(f\\"diag {name} step {step} ok\\")\\n    print(json.dumps({\\"case\\": name, \\"ok\\": True}), flush=True)\\n\\n\\ndef run_diag(site, errors):\\n    import subprocess\\n    res = {}\\n    env = dict(os.environ, CUDA_LAUNCH_BLOCKING=\\"1\\")\\n    for name in DIAG_CASES:\\n        try:\\n            p = subprocess.run([sys.executable, str(Path(__file__).resolve()), site, \\"--diag-case\\", name], capture_output=True,\\n                               text=True, timeout=240, env=env)\\n            ok = p.returncode == 0 and \'\\"ok\\": true\' in p.stdout\\n            tail = \\"\\" if ok else (p.stderr or \\"\\")[-300:].replace(\\"\\\\n\\", \\" | \\")\\n            res[name] = \\"ok\\" if ok else f\\"FAIL rc={p.returncode} {tail}\\"\\n        except Exception as exc:\\n            res[name] = f\\"{type(exc).__name__}: {exc}\\"\\n        log(\\"diag\\", name, res[name][:120])\\n    return res\\n\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"site\\")\\n    ap.add_argument(\\"--rows\\", default=\\"19,32,48,64\\")\\n    ap.add_argument(\\"--churn\\", default=\\"0.03,0.06,0.12,0.30\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=24)\\n    ap.add_argument(\\"--timeout\\", type=int, default=800)\\n    ap.add_argument(\\"--diag-case\\", default=None)\\n    args = ap.parse_args()\\n    if args.diag_case:\\n        diag_case(args.diag_case)\\n        return\\n    claim_stdout()\\n    rows_list = [int(x) for x in args.rows.split(\\",\\") if x]\\n    churns = [float(x) for x in args.churn.split(\\",\\") if x]\\n    out = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"errors\\": [], \\"torch\\": torch.__version__,\\n           \\"gpu\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\\n\\n    def dog():\\n        time.sleep(args.timeout + 60)\\n        out[\\"errors\\"].append(\\"watchdog\\")\\n        emit(out)\\n        os._exit(3)\\n\\n    threading.Thread(target=dog, daemon=True).start()\\n    if torch.cuda.device_count() < WORLD:\\n        out[\\"errors\\"].append(f\\"needs {WORLD} GPUs, saw {torch.cuda.device_count()}\\")\\n        emit(out)\\n        return\\n    import torch.multiprocessing as mp\\n    with socket.socket() as s:\\n        s.bind((\\"127.0.0.1\\", 0))\\n        port = s.getsockname()[1]\\n    fd, out_path = tempfile.mkstemp(prefix=\\"bench_hisparse_\\", suffix=\\".json\\")\\n    os.close(fd)\\n    Path(out_path).write_text(\\"{}\\")\\n    t0 = time.time()\\n    err = None\\n    try:\\n        ctx = mp.spawn(worker, args=(WORLD, port, out_path, rows_list, churns, args.iters, args.site,\\n                                     t0 + args.timeout - 90), nprocs=WORLD, join=False)\\n        while True:\\n            try:\\n                if ctx.join(timeout=5):\\n                    break\\n            except Exception as exc:\\n                err = f\\"{type(exc).__name__}: {str(exc)[-400:]}\\"\\n                break\\n            if time.time() - t0 > args.timeout:\\n                err = f\\"timeout after {args.timeout}s\\"\\n                break\\n    except Exception as exc:\\n        err = f\\"spawn: {type(exc).__name__}: {exc}\\"\\n    try:\\n        import multiprocessing\\n        for p in multiprocessing.active_children():\\n            p.kill()\\n    except Exception:\\n        pass\\n    try:\\n        res = json.loads(Path(out_path).read_text() or \\"{}\\")\\n    except Exception:\\n        res = {}\\n    if err:\\n        out[\\"errors\\"].append(err)\\n    out[\\"errors\\"] += res.get(\\"errors\\", [])\\n    out[\\"correctness\\"] = res.get(\\"correctness\\")\\n    out[\\"timing\\"] = res.get(\\"timing\\")\\n    out[\\"jit_s\\"] = res.get(\\"jit_s\\")\\n    out[\\"io512_all4_us_rows32_churn6\\"] = res.get(\\"io512_all4_us_rows32_churn6\\")\\n    if any(\\"illegal\\" in str(e).lower() for e in out[\\"errors\\"]):     # localise the crash: one fresh process per case\\n        out[\\"diag\\"] = run_diag(args.site, out[\\"errors\\"])\\n    out[\\"seconds\\"] = round(time.time() - t0, 1)\\n    c = out[\\"correctness\\"] or {}\\n    head = next((t for t in (out[\\"timing\\"] or []) if t.get(\\"rows\\") == 32 and abs(t.get(\\"churn\\", 0) - 0.06) < 1e-9\\n                 and \\"io_all4_us\\" in t), None)\\n    if head:\\n        out[\\"patched_us\\"] = head[\\"io_all4_us\\"]\\n        out[\\"headline\\"] = head\\n    out[\\"ok\\"] = bool(c and c.get(\\"rows_bad\\") == 0 and c.get(\\"rows_checked\\", 0) > 0 and head)\\n    emit(out)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "bench_idxshard.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 4-GPU correctness check + microbenchmark for colab/perf_patches/idxshard_patch.py (key \\"idxshard\\").\\n\\nLoads the PATCHED sglang/srt/layers/attention/qsa/mqa.py by file path, spawns 4 ranks (one L4 each, real NCCL) and, at the\\nproduction decode shapes (8 kernel heads = 4 indexer heads + 4 zero pad, head_dim 128, compressed page 16, page-table\\nwidth 2048 = ceil(131072/4/16), 12 indexer layers per step on 12 distinct caches):\\n  * correctness: stock selection (full qsa_mqa_decode + fast_topk 512) vs the sharded selection with a real NCCL\\n    all-gather. Per row the selected set must be valid-first / -1-padded, unique, inside [0, len), the same size as stock,\\n    and its sorted stock logits must equal the stock selection\'s sorted logits (ties may swap equal-valued blocks).\\n    All four ranks must produce the identical block ids. Lengths include 0, 1, 15-17, 511-513, 2047-2049 and the full table.\\n  * timing, per decode step = 12 layers, rows {19,32,48} x mean context {24k,80k,130k} tokens (lengths spread 0.5x-1.5x):\\n      stock_us          12 x (qsa_mqa_decode + fast_topk), CUDA graph\\n      shard_compute_us  12 x sharded selection with a local stand-in for the gather, CUDA graph\\n      nccl_us           one all-gather of the [rows, 1024] fp32 payload, eager event-timed (upper bound, includes launch)\\n      nccl_graph_us     12 gathers inside one CUDA graph via torch.distributed (None if capture is not possible)\\n    patched_us = shard_compute_us + 12 * min(nccl_us, nccl_graph_us or inf). Headline scenario: rows 32, 80k tokens.\\nusage: python3 bench_idxshard.py <site-packages dir> [--rows 19,32,48] [--ctx 24576,81920,131072] [--iters 30] [--timeout 900]\\nstdout gets ONE JSON line.\\n\\"\\"\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport socket\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nimport torch\\n\\nKEY = \\"idxshard\\"\\nHEADS_REAL, HEADS, HEAD_DIM, PAGE = 4, 8, 128, 16\\nMAX_PAGES = 2048\\nMAX_MODEL_LEN = MAX_PAGES * PAGE\\nLAYERS = 12\\nNUM_PAGES = 12288\\nTOPK = 512\\nWORLD = 4\\n\\n_OUT = sys.stdout\\n_EMITTED = []\\n_LOCK = threading.Lock()\\n\\n\\ndef log(*a):\\n    print(\\"[bench_idxshard]\\", *a, file=sys.stderr, flush=True)\\n\\n\\ndef claim_stdout():\\n    global _OUT\\n    try:\\n        sys.stdout.flush()\\n        _OUT = os.fdopen(os.dup(1), \\"w\\")\\n        os.dup2(2, 1)\\n    except Exception:\\n        _OUT = sys.stdout\\n\\n\\ndef emit(res):\\n    with _LOCK:\\n        if _EMITTED:\\n            return\\n        _EMITTED.append(1)\\n        _OUT.write(json.dumps(res, default=str) + \\"\\\\n\\")\\n        _OUT.flush()\\n\\n\\ndef load_mqa(site):\\n    path = Path(site) / \\"sglang/srt/layers/attention/qsa/mqa.py\\"\\n    spec = importlib.util.spec_from_file_location(\\"_bench_idxshard_mqa\\", path)\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef make_inputs(rows, ctx_tokens, dev, gen):\\n    mean_blocks = max(1, ctx_tokens // 4)\\n    lens = (torch.rand(rows, generator=gen) + 0.5) * mean_blocks\\n    lens = lens.long().clamp_(1, MAX_MODEL_LEN)\\n    pt = torch.randint(0, NUM_PAGES, (rows, MAX_PAGES), generator=gen, dtype=torch.int32)\\n    q = torch.randn(rows, HEADS, HEAD_DIM, generator=gen).to(torch.bfloat16)\\n    q[:, HEADS_REAL:] = 0\\n    return q.to(dev), pt.to(dev), lens.to(torch.int32).to(dev)\\n\\n\\ndef edge_lengths(rows):\\n    base = [0, 1, 15, 16, 17, 63, 64, 65, 511, 512, 513, 1023, 2047, 2048, 2049, 4095, 32767, 32768]\\n    return [base[i % len(base)] for i in range(rows)]\\n\\n\\ndef stock_select(mod, fast_topk, q, kc, pt, cl):\\n    logits = mod.qsa_mqa_decode(q, kc, pt, cl, MAX_MODEL_LEN)\\n    return fast_topk(logits, cl, topk=TOPK, row_starts=None), logits\\n\\n\\ndef worker(rank, world, port, out_path, rows_list, ctx_list, iters, site, parent_pid):\\n    import torch.distributed as dist\\n    res = {\\"errors\\": [], \\"correctness\\": {}, \\"timing\\": []}\\n\\n    def save():\\n        if rank == 0:\\n            Path(out_path).write_text(json.dumps(res, default=str))\\n\\n    torch.cuda.set_device(rank)\\n    dev = torch.device(\\"cuda\\", rank)\\n    dist.init_process_group(\\"nccl\\", init_method=f\\"tcp://127.0.0.1:{port}\\", rank=rank, world_size=world,\\n                            device_id=dev)\\n    try:\\n        mod = load_mqa(site)\\n        if not hasattr(mod, \\"qsa_mqa_topk_sharded\\"):\\n            raise RuntimeError(\\"idxshard patch not applied to this tree\\")\\n        from sglang.kernels.ops.elementwise.fast_topk import fast_topk\\n        gen = torch.Generator().manual_seed(7)\\n        caches = [torch.randn(NUM_PAGES, PAGE, 1, HEAD_DIM, generator=gen).to(torch.bfloat16).to(dev) for _ in range(LAYERS)]\\n\\n        def warm():\\n            q0, pt0, cl0 = make_inputs(4, 4096, dev, gen)\\n            stock_select(mod, fast_topk, q0, caches[0], pt0, cl0)\\n            mod.qsa_mqa_topk_sharded(q0, caches[0], pt0, cl0, rank, world,\\n                                     lambda pay: pay.unsqueeze(0).expand(world, -1, -1).contiguous(), TOPK)\\n            torch.cuda.synchronize()\\n\\n        if rank == 0:        # JIT-compile the TileLang / fast_topk kernels once, then let the other ranks hit the cache\\n            warm()\\n        dist.barrier()\\n        if rank != 0:\\n            warm()\\n        dist.barrier()\\n\\n        def gather_nccl(pay):\\n            out = torch.empty((world * pay.shape[0], pay.shape[1]), dtype=pay.dtype, device=pay.device)\\n            dist.all_gather_into_tensor(out, pay)\\n            return out.view(world, pay.shape[0], pay.shape[1])\\n\\n        def gather_fake(pay):\\n            return pay.unsqueeze(0).expand(world, -1, -1).contiguous()\\n\\n        def sharded(q, kc, pt, cl, gather):\\n            return mod.qsa_mqa_topk_sharded(q, kc, pt, cl, rank, world, gather, TOPK)\\n\\n        # ---------------------------------------------------------------- correctness\\n        bad, checked, disagree = 0, 0, 0\\n        cases = []\\n        for rows, ctx in [(19, 24576), (32, 81920), (48, 131072)]:\\n            cases.append(make_inputs(rows, ctx, dev, gen))\\n        q, pt, cl = make_inputs(len(edge_lengths(18)), 4096, dev, gen)\\n        cases.append((q, pt, torch.tensor(edge_lengths(q.shape[0]), dtype=torch.int32, device=dev)))\\n        notes = []\\n        for ci, (q, pt, cl) in enumerate(cases):\\n            kc = caches[ci % LAYERS]\\n            ref, logits = stock_select(mod, fast_topk, q, kc, pt, cl)\\n            got = sharded(q, kc, pt, cl, gather_nccl)\\n            torch.cuda.synchronize()\\n            allg = torch.empty((world,) + tuple(got.shape), dtype=got.dtype, device=dev)\\n            dist.all_gather_into_tensor(allg.view(-1), got.contiguous().view(-1))\\n            if not all(torch.equal(allg[0], allg[r]) for r in range(1, world)):\\n                disagree += 1\\n            for b in range(q.shape[0]):\\n                n = int(cl[b]); g = got[b]; w = ref[b]\\n                gv, wv = g[g >= 0], w[w >= 0]\\n                nv = int((g >= 0).sum())\\n                ok = (len(gv) == len(wv) == min(n, TOPK) and len(set(gv.tolist())) == len(gv)\\n                      and bool((g[:nv] >= 0).all() and (g[nv:] == -1).all()) and (len(gv) == 0 or int(gv.max()) < n))\\n                if ok and len(gv):\\n                    a = torch.sort(logits[b, gv.long()], descending=True)[0]\\n                    c = torch.sort(logits[b, wv.long()], descending=True)[0]\\n                    ok = bool(torch.equal(a, c))\\n                checked += 1\\n                if not ok:\\n                    bad += 1\\n                    if len(notes) < 6:\\n                        notes.append({\\"case\\": ci, \\"row\\": b, \\"len\\": n, \\"got\\": len(gv), \\"want\\": len(wv)})\\n        res[\\"correctness\\"] = {\\"rows_checked\\": checked, \\"rows_bad\\": bad, \\"rank_disagree_cases\\": disagree, \\"notes\\": notes}\\n        save()\\n        dist.barrier()\\n\\n        # ---------------------------------------------------------------- timing\\n        def time_graph(fn, n_iter):\\n            fn(); torch.cuda.synchronize()\\n            g = torch.cuda.CUDAGraph()\\n            s = torch.cuda.Stream()\\n            s.wait_stream(torch.cuda.current_stream())\\n            with torch.cuda.stream(s):\\n                fn()\\n            torch.cuda.current_stream().wait_stream(s)\\n            torch.cuda.synchronize()\\n            with torch.cuda.graph(g):\\n                fn()\\n            for _ in range(3):\\n                g.replay()\\n            torch.cuda.synchronize()\\n            dist.barrier()\\n            t0 = torch.cuda.Event(enable_timing=True); t1 = torch.cuda.Event(enable_timing=True)\\n            t0.record()\\n            for _ in range(n_iter):\\n                g.replay()\\n            t1.record(); torch.cuda.synchronize()\\n            return t0.elapsed_time(t1) * 1000.0 / n_iter\\n\\n        for rows in rows_list:\\n            for ctx in ctx_list:\\n                q, pt, cl = make_inputs(rows, ctx, dev, gen)\\n\\n                def run_stock():\\n                    for kc in caches:\\n                        stock_select(mod, fast_topk, q, kc, pt, cl)\\n\\n                def run_shard():\\n                    for kc in caches:\\n                        sharded(q, kc, pt, cl, gather_fake)\\n\\n                rec = {\\"rows\\": rows, \\"ctx\\": ctx, \\"mean_blocks\\": int(cl.float().mean())}\\n                try:\\n                    rec[\\"stock_us\\"] = round(time_graph(run_stock, iters), 1)\\n                    rec[\\"shard_compute_us\\"] = round(time_graph(run_shard, iters), 1)\\n                except Exception as exc:\\n                    rec[\\"error\\"] = f\\"{type(exc).__name__}: {str(exc)[:300]}\\"\\n                pay = torch.randn(rows, 2 * TOPK, device=dev)\\n                outbuf = torch.empty(world * rows, 2 * TOPK, device=dev)\\n                for _ in range(5):\\n                    dist.all_gather_into_tensor(outbuf, pay)\\n                torch.cuda.synchronize(); dist.barrier()\\n                t0 = torch.cuda.Event(enable_timing=True); t1 = torch.cuda.Event(enable_timing=True)\\n                t0.record()\\n                for _ in range(100):\\n                    dist.all_gather_into_tensor(outbuf, pay)\\n                t1.record(); torch.cuda.synchronize()\\n                rec[\\"nccl_us\\"] = round(t0.elapsed_time(t1) * 10.0, 1)\\n                res[\\"timing\\"].append(rec)\\n                save()\\n        # CUDA-graph captured gathers (12 per graph), headline size only; may be unsupported -> None\\n        try:\\n            rows = 32\\n            pay = torch.randn(rows, 2 * TOPK, device=dev)\\n            outbuf = torch.empty(world * rows, 2 * TOPK, device=dev)\\n\\n            def run_ag():\\n                for _ in range(LAYERS):\\n                    dist.all_gather_into_tensor(outbuf, pay)\\n            res[\\"nccl_graph_us_12x_rows32\\"] = round(time_graph(run_ag, iters), 1)\\n        except Exception as exc:\\n            res[\\"nccl_graph_us_12x_rows32\\"] = None\\n            res[\\"errors\\"].append(f\\"nccl graph: {type(exc).__name__}: {str(exc)[:200]}\\")\\n        save()\\n    except Exception as exc:\\n        res[\\"errors\\"].append(f\\"rank {rank}: {type(exc).__name__}: {str(exc)[:400]}\\")\\n        res[\\"errors\\"].append(traceback.format_exc()[-1200:])\\n        save()\\n    finally:\\n        try:\\n            dist.barrier()\\n            dist.destroy_process_group()\\n        except Exception:\\n            pass\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"site\\")\\n    ap.add_argument(\\"--rows\\", default=\\"19,32,48\\")\\n    ap.add_argument(\\"--ctx\\", default=\\"24576,81920,131072\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=30)\\n    ap.add_argument(\\"--timeout\\", type=int, default=900)\\n    args = ap.parse_args()\\n    claim_stdout()\\n    rows_list = [int(x) for x in args.rows.split(\\",\\") if x]\\n    ctx_list = [int(x) for x in args.ctx.split(\\",\\") if x]\\n    out = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"errors\\": [], \\"torch\\": torch.__version__,\\n           \\"gpu\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\\n\\n    def dog():\\n        time.sleep(args.timeout + 60)\\n        out[\\"errors\\"].append(\\"watchdog\\")\\n        emit(out)\\n        os._exit(3)\\n\\n    threading.Thread(target=dog, daemon=True).start()\\n    if torch.cuda.device_count() < WORLD:\\n        out[\\"errors\\"].append(f\\"needs {WORLD} GPUs, saw {torch.cuda.device_count()}\\")\\n        emit(out)\\n        return\\n    import torch.multiprocessing as mp\\n    with socket.socket() as s:\\n        s.bind((\\"127.0.0.1\\", 0))\\n        port = s.getsockname()[1]\\n    fd, out_path = tempfile.mkstemp(prefix=\\"bench_idxshard_\\", suffix=\\".json\\")\\n    os.close(fd)\\n    Path(out_path).write_text(\\"{}\\")\\n    t0 = time.time()\\n    err = None\\n    try:\\n        ctx = mp.spawn(worker, args=(WORLD, port, out_path, rows_list, ctx_list, args.iters, args.site, os.getpid()),\\n                       nprocs=WORLD, join=False)\\n        while True:\\n            try:\\n                if ctx.join(timeout=5):\\n                    break\\n            except Exception as exc:\\n                err = f\\"{type(exc).__name__}: {str(exc)[-400:]}\\"\\n                break\\n            if time.time() - t0 > args.timeout:\\n                err = f\\"timeout after {args.timeout}s\\"\\n                break\\n    except Exception as exc:\\n        err = f\\"spawn: {type(exc).__name__}: {exc}\\"\\n    try:\\n        import multiprocessing\\n        for p in multiprocessing.active_children():\\n            p.kill()\\n    except Exception:\\n        pass\\n    try:\\n        res = json.loads(Path(out_path).read_text() or \\"{}\\")\\n    except Exception:\\n        res = {}\\n    if err:\\n        out[\\"errors\\"].append(err)\\n    out[\\"errors\\"] += res.get(\\"errors\\", [])\\n    out[\\"correctness\\"] = res.get(\\"correctness\\")\\n    out[\\"timing\\"] = res.get(\\"timing\\")\\n    out[\\"nccl_graph_us_12x_rows32\\"] = res.get(\\"nccl_graph_us_12x_rows32\\")\\n    out[\\"seconds\\"] = round(time.time() - t0, 1)\\n    c = out[\\"correctness\\"] or {}\\n    ok_corr = bool(c) and c.get(\\"rows_bad\\") == 0 and c.get(\\"rank_disagree_cases\\") == 0 and c.get(\\"rows_checked\\", 0) > 0\\n    head = next((t for t in (out[\\"timing\\"] or []) if t.get(\\"rows\\") == 32 and t.get(\\"ctx\\") == 81920 and \\"stock_us\\" in t), None)\\n    if head:\\n        gr = out[\\"nccl_graph_us_12x_rows32\\"]\\n        nccl_12 = min(12 * head[\\"nccl_us\\"], gr) if gr else 12 * head[\\"nccl_us\\"]\\n        out[\\"baseline_us\\"] = head[\\"stock_us\\"]\\n        out[\\"patched_us\\"] = round(head[\\"shard_compute_us\\"] + nccl_12, 1)\\n        for t in out[\\"timing\\"]:\\n            if \\"stock_us\\" in t:\\n                n12 = min(12 * t[\\"nccl_us\\"], gr) if (gr and t[\\"rows\\"] == 32) else 12 * t[\\"nccl_us\\"]\\n                t[\\"patched_us\\"] = round(t[\\"shard_compute_us\\"] + n12, 1)\\n                t[\\"saved_ms\\"] = round((t[\\"stock_us\\"] - t[\\"patched_us\\"]) / 1000.0, 2)\\n    out[\\"ok\\"] = bool(ok_corr and head and not [e for e in out[\\"errors\\"] if not str(e).startswith(\\"nccl graph\\")])\\n    emit(out)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "bench_marlinclean.py": "#!/usr/bin/env python3\\n\\"\\"\\"Single-GPU correctness + timing test for colab/perf_patches/marlinclean_patch.py (SGLANG_MARLIN_EP_CLEAN=1).\\n\\nEmulates EP4 on one L4: 384 random W4A16 (uint4b8, group 128) experts, K=2560, N=640, top-10, split into 4 \\"ranks\\" of\\n96. Each rank\'s call gets the dispatcher\'s exact global->local map (expert_map) and LOCAL ids with -1 for the other\\nranks\' experts, exactly like the standard dispatcher + marlin_ep_patch in the server. For every M (default\\n1,13,19,20,80 plus 2048 as a prefill-sized point) it compares the clean path against the unpatched path:\\n  - per rank, clean vs stock output (row-wise max error relative to the row\'s max |stock|)\\n  - sum over the 4 ranks (what the all-reduce produces) vs ONE stock call holding all 384 experts with no -1 ids\\n  - token 0 routed only to rank 0 and token 1 only to rank 3: the other ranks\' rows must be exactly 0\\n  - clean path again with every floating torch.empty NaN-filled (worst-case garbage in the now-empty cache13)\\n  - CUDA-graph replay of 4 ranks x reps calls (graph-pool memory is recycled garbage, as in serving)\\n  - moe_align_block_size with 97 vs 385 buckets on the same local ids: identical blocks / expert ids / padding\\n  - routed_scaling_factor=2.5 at M=19; optional fp16 (use_atomic_add=True Marlin path) with --fp16\\n  - ground truth: a small real-quantized config (marlin_quantize, 16 experts / 4 ranks) vs an fp32 torch MoE;\\n    the clean path must not be less accurate than stock (skip with --no-gt)\\nTiming: per-call GPU time from CUDA-graph replay (one call = one MoE layer on one rank), stock vs clean, interleaved.\\n\\n\\"stock\\" is always the TRUE unpatched source: the venv file if it is unpatched (the patch is then applied to a temp\\ncopy), or, if the venv is already patched, marlinclean_patch.revert_text() of it (exact inverse, verified by\\nround-trip) loaded as a temp module. Dormancy: the patched function with the flag OFF must be bit-identical to stock on\\nthe EP path (and, with --noep, with the flag ON on a non-EP call). marlinclean_patch.py must sit next to this file;\\nwithout it on a patched venv the reference degrades to the flag-off path (JSON: reference_is_unpatched_source=false).\\nusage: CUDA_VISIBLE_DEVICES=0 timeout 1800 /tmp/sgl-intel/venv/bin/python bench_marlinclean.py\\n       [--ms 1,13,19,20,80,2048] [--no-gt] [--fp16] [--noep] [--no-graph] [--reps 8] [--iters 20] [--rounds 5]\\n       [--timeout 1500]\\nPrints ONE JSON line on stdout (progress on stderr); an internal watchdog prints it (ok=false, failed+=timeout) and\\nexits 3 if the run has not finished after --timeout seconds.\\n\\"\\"\\"\\nimport argparse\\nimport contextlib\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport statistics\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nimport torch\\n\\nKEY = \\"marlinclean\\"\\nDEV = \\"cuda\\"\\nTOL = 2.0 ** -5  # row-wise relative error bound for \\"same result up to bf16 rounding\\" (real bugs give >= 0.1)\\nT0 = time.time()\\nRANKS = 4\\n\\n\\ndef log(*a):\\n    print(f\\"[bench_marlinclean +{time.time() - T0:7.1f}s]\\", *a, file=sys.stderr, flush=True)\\n\\n\\n# ----------------------------------------------------------------------------------------------- implementations\\nclass Impl:\\n    \\"\\"\\"run(mode, **kw) calls fused_marlin_moe in one of three modes:\\n    \\"stock\\": the TRUE unpatched source (venv file if unpatched, else the patch\'s exact inverse loaded as a temp module);\\n    \\"off\\":   the patched source with SGLANG_MARLIN_EP_CLEAN off (must be bit-identical to \\"stock\\": dormancy);\\n    \\"clean\\": the patched source with the flag on.\\n    stock_mod is None only if the unpatched source could not be rebuilt; \\"stock\\" then falls back to \\"off\\".\\"\\"\\"\\n\\n    def __init__(self, stock_mod, patched_mod, how):\\n        self.stock_mod, self.patched_mod, self.how = stock_mod, patched_mod, how\\n        self.has_ref = stock_mod is not None\\n\\n    def run(self, mode, **kw):\\n        assert mode in (\\"stock\\", \\"off\\", \\"clean\\"), mode\\n        if mode == \\"stock\\" and self.has_ref:\\n            assert not hasattr(self.stock_mod, \\"_MARLIN_EP_CLEAN\\")\\n            return self.stock_mod.fused_marlin_moe(**kw)\\n        mod = self.patched_mod\\n        old = mod._MARLIN_EP_CLEAN\\n        mod._MARLIN_EP_CLEAN = mode == \\"clean\\"\\n        try:\\n            return mod.fused_marlin_moe(**kw)\\n        finally:\\n            mod._MARLIN_EP_CLEAN = old\\n\\n\\ndef _patch_module():\\n    sys.dont_write_bytecode = True  # no __pycache__ next to the patch\\n    here = Path(__file__).resolve().parent\\n    if str(here) not in sys.path:\\n        sys.path.insert(0, str(here))\\n    import marlinclean_patch as P\\n\\n    return P\\n\\n\\ndef _load_temp_module(src, tag):\\n    \\"\\"\\"Load a fused_marlin_moe.py text as its own module (and its own custom-op name: the default\\n    torch.ops.sglang.fused_marlin_moe is already taken by the venv module).\\"\\"\\"\\n    deco = \'@register_custom_op(out_shape=\\"hidden_states\\")\\\\ndef fused_marlin_moe(\\\\n\'\\n    if src.count(deco) != 1:\\n        raise RuntimeError(\\"custom-op decorator anchor not found in fused_marlin_moe.py\\")\\n    name = f\\"fused_marlin_moe_mclean_{tag}\\"\\n    src = src.replace(deco, f\'@register_custom_op(op_name=\\"{name}\\", out_shape=\\"hidden_states\\")\\\\n\'\\n                            \\"def fused_marlin_moe(\\\\n\\")\\n    path = Path(tempfile.mkdtemp(prefix=\\"mclean_\\")) / f\\"{name}.py\\"\\n    path.write_text(src)  # a real file: triton.jit needs inspect.getsource\\n    spec = importlib.util.spec_from_file_location(name, path)\\n    mod = importlib.util.module_from_spec(spec)\\n    sys.modules[spec.name] = mod\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef load_impls():\\n    import sglang.srt.layers.moe.fused_moe_triton.fused_marlin_moe as fmm\\n\\n    text = Path(fmm.__file__).read_text()\\n    if hasattr(fmm, \\"_MARLIN_EP_CLEAN\\"):\\n        try:\\n            stock = _load_temp_module(_patch_module().revert_text(text), \\"ref\\")\\n        except Exception as exc:  # noqa: BLE001 - patch file missing or patched by another version\\n            log(f\\"WARNING: cannot rebuild the unpatched source ({type(exc).__name__}: {exc}); \\"\\n                \\"reference = flag-off path\\")\\n            return Impl(None, fmm, f\\"venv-patched; reference = flag-off path ({type(exc).__name__})\\")\\n        return Impl(stock, fmm, \\"venv-patched; reference = reverted unpatched source\\")\\n    return Impl(fmm, _load_temp_module(_patch_module().apply_text(text), \\"bench\\"),\\n                \\"venv unpatched; reference = venv source, patched = temp copy\\")\\n\\n\\n# ----------------------------------------------------------------------------------------------- data\\ndef rand_int32(shape, gen):\\n    n = math.prod(shape)\\n    b = torch.randint(0, 256, (n * 4,), dtype=torch.uint8, device=DEV, generator=gen)\\n    return b.view(torch.int32).view(*shape)\\n\\n\\ndef rand_weights(E, K, N, group, dtype, gen):\\n    \\"\\"\\"Random Marlin-layout W4A16 experts: any int32 is 8 valid uint4b8 codes; scales give O(1) activations.\\"\\"\\"\\n    w1 = rand_int32((E, K // 16, 2 * N * 16 // 8), gen)\\n    w2 = rand_int32((E, N // 16, K * 16 // 8), gen)\\n    s1 = (torch.rand(E, K // group, 2 * N, device=DEV, generator=gen) * 0.5 + 0.75) / (4.6 * math.sqrt(K))\\n    s2 = (torch.rand(E, N // group, K, device=DEV, generator=gen) * 0.5 + 0.75) / (4.6 * math.sqrt(N))\\n    return w1, w2, s1.to(dtype).contiguous(), s2.to(dtype).contiguous()\\n\\n\\ndef routing(M, E_glob, E_loc, topk, gen, force=True):\\n    logits = torch.randn(M, E_glob, device=DEV, generator=gen)\\n    w, ids = torch.topk(torch.softmax(logits, dim=-1), topk, dim=-1)\\n    ids = ids.to(torch.int32)\\n    if force and M >= 2:  # token 0 only on rank 0 -> rows of token 0 on ranks 1..3 must be exactly 0\\n        ids[0] = torch.randperm(E_loc, device=DEV, generator=gen)[:topk].to(torch.int32)\\n    if force and M >= 3:  # token 1 only on the last rank\\n        ids[1] = ((RANKS - 1) * E_loc + torch.randperm(E_loc, device=DEV, generator=gen)[:topk]).to(torch.int32)\\n    w = w / w.sum(dim=-1, keepdim=True)\\n    return logits, w.float().contiguous(), ids.contiguous()\\n\\n\\ndef local_map(rank, E_loc, E_glob):\\n    \\"\\"\\"standard.py dispatcher: local_expert_mapping (global id -> local id, -1 elsewhere), int32.\\"\\"\\"\\n    m = torch.full((E_glob,), -1, dtype=torch.int32, device=DEV)\\n    m[rank * E_loc:(rank + 1) * E_loc] = torch.arange(E_loc, dtype=torch.int32, device=DEV)\\n    return m\\n\\n\\ndef call_kwargs(x, logits, w, ids, W, sl, emap, n_glob, rsf):\\n    w1, w2, s1, s2 = W\\n    return dict(hidden_states=x, w1=w1[sl], w2=w2[sl], w1_scale=s1[sl], w2_scale=s2[sl], gating_output=logits,\\n                topk_weights=w, topk_ids=ids, global_num_experts=n_glob, expert_map=emap, workspace=None,\\n                num_bits=4, is_k_full=True, inplace=False, routed_scaling_factor=rsf, activation=\\"silu\\",\\n                is_gated=True)\\n\\n\\ndef ep_case(M, W, E_loc, topk, gen, dtype, rsf=None, force=True):\\n    \\"\\"\\"kwargs for the 4 EP ranks (local ids, -1 elsewhere) and for one stock all-expert call (no -1 ids).\\"\\"\\"\\n    K = W[0].shape[1] * 16\\n    E_glob = E_loc * RANKS\\n    logits, w, ids = routing(M, E_glob, E_loc, topk, gen, force)\\n    x = torch.randn(M, K, device=DEV, generator=gen).to(dtype)\\n    ranks = []\\n    for r in range(RANKS):\\n        emap = local_map(r, E_loc, E_glob)\\n        ids_loc = emap[ids.long()].contiguous()\\n        ranks.append(call_kwargs(x, logits, w, ids_loc, W, slice(r * E_loc, (r + 1) * E_loc), emap, E_glob, rsf))\\n    full_map = torch.arange(E_glob, dtype=torch.int32, device=DEV)\\n    full = call_kwargs(x, logits, w, ids, W, slice(0, E_glob), full_map, E_glob, rsf)\\n    return ranks, full, (x, w, ids)\\n\\n\\n# ----------------------------------------------------------------------------------------------- checks\\ndef rowrel(a, ref):\\n    \\"\\"\\"(max over rows of max|a - ref| / max|ref_row|, max abs err). All-zero ref rows are scaled by the global max.\\"\\"\\"\\n    a, ref = a.float(), ref.float()\\n    d = (a - ref).abs().amax(dim=-1)\\n    s = ref.abs().amax(dim=-1)\\n    g = max(float(ref.abs().max()) if ref.numel() else 0.0, 1e-30)\\n    rel = torch.where(s > 0, d / s.clamp_min(1e-30), d / g)\\n    return (float(rel.max()) if rel.numel() else 0.0), (float(d.max()) if d.numel() else 0.0)\\n\\n\\ndef finite(ts):\\n    return all(bool(torch.isfinite(t).all()) for t in ts)\\n\\n\\ndef nmax(vals):\\n    \\"\\"\\"max() that propagates NaN (the builtin silently drops it).\\"\\"\\"\\n    vals = list(vals)\\n    return float(\\"nan\\") if any(v != v for v in vals) else max(vals)\\n\\n\\n@contextlib.contextmanager\\ndef nan_empty():\\n    \\"\\"\\"Every floating torch.empty comes back NaN-filled (worst case for buffers the clean path no longer zeroes).\\"\\"\\"\\n    orig = torch.empty\\n\\n    def _empty(*a, **k):\\n        t = orig(*a, **k)\\n        if t.is_floating_point() and t.numel():\\n            t.fill_(float(\\"nan\\"))\\n        return t\\n\\n    torch.empty = _empty\\n    try:\\n        yield\\n    finally:\\n        torch.empty = orig\\n\\n\\ndef sync():\\n    if DEV == \\"cuda\\":\\n        torch.cuda.synchronize()\\n\\n\\ndef run_ranks(impl, mode, ranks):\\n    return [impl.run(mode, **kw) for kw in ranks]\\n\\n\\ndef dormancy(impl, ref_outs, run_ref, run_off, r, fails, tag, strict):\\n    \\"\\"\\"Flag off must reproduce the true unpatched source bit-for-bit. strict (decode sizes, M <= 256): bit-exact is\\n    required whenever stock itself is run-to-run deterministic. Not strict (prefill sizes, where an expert bucket can\\n    span several Marlin blocks whose row assignment follows the align kernel\'s atomic order and whose k-split\\n    differs per block, so stock is not guaranteed bit-reproducible): TOL only, bit-exactness reported.\\n    Skipped (reported) when no unpatched reference could be rebuilt.\\"\\"\\"\\n    if not impl.has_ref:\\n        r[f\\"{tag}_bitexact\\"] = \\"no-reference\\"\\n        return\\n    again, off = run_ref(), run_off()\\n    sync()\\n    det = all(torch.equal(a, b) for a, b in zip(ref_outs, again))\\n    exact = all(torch.equal(a, b) for a, b in zip(ref_outs, off))\\n    rel = nmax(rowrel(o, b)[0] for o, b in zip(off, ref_outs))\\n    r[f\\"{tag}_stock_deterministic\\"], r[f\\"{tag}_bitexact\\"], r[f\\"{tag}_rel_err\\"] = det, exact, rel\\n    if not (exact if (det and strict) else rel <= TOL):\\n        fails.append(tag)\\n\\n\\ndef ep_checks(impl, ranks, full, M, noep=False):\\n    \\"\\"\\"Eager checks for one case. Returns (result dict, failure list, eager stock outputs, eager clean outputs).\\"\\"\\"\\n    fails = []\\n    base = run_ranks(impl, \\"stock\\", ranks)\\n    clean = run_ranks(impl, \\"clean\\", ranks)\\n    ref_full = impl.run(\\"stock\\", **full)\\n    sync()\\n    r = {}\\n    r[\\"finite\\"] = finite(base) and finite(clean) and finite([ref_full])\\n    if not r[\\"finite\\"]:\\n        fails.append(\\"nonfinite\\")\\n    # dormancy on the EP path: flag off == unpatched source\\n    # fp16 takes Marlin\'s atomic-add path: never assume run-to-run bit-reproducibility there\\n    strict = M <= 256 and ranks[0][\\"hidden_states\\"].dtype != torch.float16\\n    dormancy(impl, base, lambda: run_ranks(impl, \\"stock\\", ranks), lambda: run_ranks(impl, \\"off\\", ranks), r, fails,\\n             \\"dormant\\", strict)\\n    if noep:  # flag ON but no expert_map (non-EP call, global ids): the clean path must not engage\\n        kw = dict(full, expert_map=None, global_num_experts=-1)\\n        ref_noep = [impl.run(\\"stock\\", **kw)]\\n        dormancy(impl, ref_noep, lambda: [impl.run(\\"stock\\", **kw)], lambda: [impl.run(\\"clean\\", **kw)], r, fails,\\n                 \\"noep_flag_on\\", strict)\\n    errs = [rowrel(c, b) for b, c in zip(base, clean)]\\n    rel_cb, abs_cb = nmax(e[0] for e in errs), nmax(e[1] for e in errs)\\n    r[\\"rel_err_vs_base\\"], r[\\"max_abs_err\\"] = rel_cb, abs_cb\\n    r[\\"absmax_out\\"] = max(float(b.float().abs().max()) for b in base)\\n    if not rel_cb <= TOL:\\n        fails.append(\\"clean_vs_base\\")\\n    s_base = sum(t.float() for t in base)\\n    s_clean = sum(t.float() for t in clean)\\n    r[\\"ep_rel_err_base\\"], r[\\"ep_abs_err_base\\"] = rowrel(s_base, ref_full)\\n    r[\\"ep_rel_err_clean\\"], r[\\"ep_abs_err_clean\\"] = rowrel(s_clean, ref_full)\\n    if not r[\\"ep_rel_err_base\\"] <= TOL:\\n        fails.append(\\"ep_sum_base\\")  # the harness itself disagrees with stock: investigate before trusting anything\\n    # TOL only: the reference is a STOCK call, so it shares stock\'s per-slot bf16 product rounding and stock always\\n    # looks closer to it than clean does (by design, not a bug). \\"Not less accurate\\" is the fp32 ground-truth check.\\n    if not r[\\"ep_rel_err_clean\\"] <= TOL:\\n        fails.append(\\"ep_sum_clean\\")\\n    zero_ok = True\\n    if M >= 2:\\n        for k in range(1, RANKS):\\n            zero_ok &= bool((base[k][0] == 0).all()) and bool((clean[k][0] == 0).all())\\n    if M >= 3:\\n        for k in range(RANKS - 1):\\n            zero_ok &= bool((base[k][1] == 0).all()) and bool((clean[k][1] == 0).all())\\n    r[\\"masked_rows_exact_zero\\"] = zero_ok\\n    if not zero_ok:\\n        fails.append(\\"masked_rows_nonzero\\")\\n    with nan_empty():\\n        clean_nan = run_ranks(impl, \\"clean\\", ranks)\\n        base_nan = run_ranks(impl, \\"stock\\", ranks)\\n        sync()\\n    r[\\"nan_empty_clean_finite\\"] = finite(clean_nan)\\n    r[\\"nan_empty_base_finite\\"] = finite(base_nan)\\n    r[\\"nan_empty_rel_err\\"] = nmax(rowrel(a, c)[0] for a, c in zip(clean_nan, clean))\\n    if not (r[\\"nan_empty_clean_finite\\"] and r[\\"nan_empty_rel_err\\"] <= TOL):\\n        fails.append(\\"nan_empty_clean\\")\\n    if not r[\\"nan_empty_base_finite\\"]:\\n        fails.append(\\"nan_empty_base\\")\\n    return r, fails, base, clean\\n\\n\\ndef capture(fn):\\n    s = torch.cuda.Stream()\\n    s.wait_stream(torch.cuda.current_stream())\\n    with torch.cuda.stream(s):\\n        fn()\\n    torch.cuda.current_stream().wait_stream(s)\\n    g = torch.cuda.CUDAGraph()\\n    with torch.cuda.graph(g):\\n        outs = fn()\\n    return g, outs\\n\\n\\ndef replay_us(g, iters):\\n    st, en = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\\n    st.record()\\n    for _ in range(iters):\\n        g.replay()\\n    en.record()\\n    en.synchronize()\\n    return st.elapsed_time(en) * 1000.0 / iters\\n\\n\\ndef graph_checks(impl, ranks, base, clean, reps, iters, rounds):\\n    \\"\\"\\"CUDA-graph replay correctness (recycled pool memory) + per-call timing, stock vs clean interleaved.\\"\\"\\"\\n    fails = []\\n\\n    def body(mode):\\n        def fn():\\n            outs = None\\n            for _ in range(reps):\\n                outs = run_ranks(impl, mode, ranks)\\n            return outs\\n        return fn\\n\\n    g_b, out_b = capture(body(\\"stock\\"))\\n    g_c, out_c = capture(body(\\"clean\\"))\\n    g_b.replay()\\n    g_c.replay()\\n    torch.cuda.synchronize()\\n    r = {\\"graph_finite\\": finite(out_b) and finite(out_c)}\\n    r[\\"graph_rel_err_clean\\"] = nmax(rowrel(a, c)[0] for a, c in zip(out_c, clean))\\n    r[\\"graph_rel_err_base\\"] = nmax(rowrel(a, b)[0] for a, b in zip(out_b, base))\\n    if not (r[\\"graph_finite\\"] and r[\\"graph_rel_err_clean\\"] <= TOL):\\n        fails.append(\\"graph_clean\\")\\n    if not r[\\"graph_rel_err_base\\"] <= TOL:\\n        fails.append(\\"graph_base\\")\\n    calls = reps * len(ranks)\\n    tb, tc = [], []\\n    for _ in range(rounds):\\n        tb.append(replay_us(g_b, iters) / calls)\\n        tc.append(replay_us(g_c, iters) / calls)\\n    r[\\"baseline_us\\"], r[\\"patched_us\\"] = statistics.median(tb), statistics.median(tc)\\n    r[\\"delta_us\\"] = r[\\"baseline_us\\"] - r[\\"patched_us\\"]\\n    del g_b, g_c, out_b, out_c\\n    torch.cuda.empty_cache()\\n    return r, fails\\n\\n\\ndef block_size_m(M, topk, E):\\n    for bs in [8, 16, 32, 48, 64]:  # fused_marlin_moe.py:240-242\\n        if M * topk / E / bs < 0.9:\\n            break\\n    return bs\\n\\n\\ndef align_equiv(ms, E_loc, topk, gen):\\n    \\"\\"\\"moe_align_block_size on LOCAL ids with E_glob + 1 vs E_loc + 1 buckets must give the same blocks.\\"\\"\\"\\n    from sglang.srt.layers.moe.fused_moe_triton import moe_align_block_size\\n\\n    E_glob = E_loc * RANKS\\n    out = {}\\n    for M in ms:\\n        _, _, ids = routing(M, E_glob, E_loc, topk, gen)\\n        ids_loc = local_map(1, E_loc, E_glob)[ids.long()].contiguous()\\n        bs = block_size_m(M, topk, E_loc)\\n        a = moe_align_block_size(ids_loc, bs, E_glob)\\n        b = moe_align_block_size(ids_loc, bs, E_loc)\\n        na, nb = int(a[2].item()), int(b[2].item())\\n        ok = na == nb and na % bs == 0\\n        if ok:\\n            nblk = na // bs\\n            ok = bool((a[1][:nblk] == b[1][:nblk]).all())\\n            sa = a[0][:na].view(nblk, bs).sort(dim=1).values\\n            sb = b[0][:nb].view(nblk, bs).sort(dim=1).values\\n            ok = ok and bool((sa == sb).all())\\n        out[str(M)] = {\\"ok\\": ok, \\"num_tokens_post_pad\\": [na, nb], \\"sorted_ids_len\\": [a[0].numel(), b[0].numel()]}\\n    return out\\n\\n\\ndef ground_truth(impl, ms, K, N, group, topk, dtype, gen):\\n    \\"\\"\\"Real quantized weights (marlin_quantize) vs an fp32 torch MoE (on the dequantized weights). Clean must not be\\n    less accurate than stock (RMS) and must stay within TOL (row-max relative).\\"\\"\\"\\n    from sgl_kernel.scalar_type import scalar_types\\n    from sglang.test.test_marlin_utils import marlin_quantize\\n\\n    E_loc, E_glob = 4, 4 * RANKS\\n    q1, s1, r1, q2, s2, r2 = [], [], [], [], [], []\\n    for _ in range(E_glob):\\n        w1 = (torch.randn(K, 2 * N, device=DEV, generator=gen) / math.sqrt(K)).to(dtype)  # [K, 2N] = gate | up\\n        w2 = (torch.randn(N, K, device=DEV, generator=gen) / math.sqrt(N)).to(dtype)  # [N, K]\\n        ref, q, s = marlin_quantize(w1, scalar_types.uint4b8, group, False)[:3]\\n        r1.append(ref.float()), q1.append(q), s1.append(s)\\n        ref, q, s = marlin_quantize(w2, scalar_types.uint4b8, group, False)[:3]\\n        r2.append(ref.float()), q2.append(q), s2.append(s)\\n    W = (torch.stack(q1).contiguous(), torch.stack(q2).contiguous(),\\n         torch.stack(s1).to(dtype).contiguous(), torch.stack(s2).to(dtype).contiguous())\\n    R1, R2 = torch.stack(r1), torch.stack(r2)\\n    res, fails = {}, []\\n    for M in ms:\\n        ranks, _, (x, w, ids) = ep_case(M, W, E_loc, topk, gen, dtype, force=False)\\n        base = sum(t.float() for t in run_ranks(impl, \\"stock\\", ranks))\\n        clean = sum(t.float() for t in run_ranks(impl, \\"clean\\", ranks))\\n        # fp64 reference (tiny GEMMs): independent of any TF32 setting sglang/torch may have enabled\\n        ref = torch.zeros(M, K, device=DEV, dtype=torch.float64)\\n        xf = x.double()\\n        for e in range(E_glob):\\n            tok, slot = (ids == e).nonzero(as_tuple=True)\\n            if tok.numel() == 0:\\n                continue\\n            h = xf[tok] @ R1[e].double()\\n            y = (torch.nn.functional.silu(h[:, :N]) * h[:, N:]) @ R2[e].double()\\n            ref.index_add_(0, tok, y * w[tok, slot][:, None].double())\\n        ref = ref.float()\\n        sync()\\n        eb, ec = rowrel(base, ref)[0], rowrel(clean, ref)[0]\\n        # gate on RMS error over all M*K elements (statistically stable; a single-element max is noisy). Clean drops\\n        # one bf16 rounding (the epilogue product), so its RMS error vs fp32 should not exceed stock\'s.\\n        den = float(ref.pow(2).mean().sqrt()) or 1e-30\\n        rb = float((base - ref).pow(2).mean().sqrt()) / den\\n        rc = float((clean - ref).pow(2).mean().sqrt()) / den\\n        ok = finite([base, clean]) and rc <= 1.10 * rb + 1e-6 and ec <= TOL\\n        res[str(M)] = {\\"rel_err_base\\": eb, \\"rel_err_clean\\": ec, \\"rms_rel_base\\": rb, \\"rms_rel_clean\\": rc, \\"ok\\": ok}\\n        if not ok:\\n            fails.append(f\\"gt_M{M}\\")\\n    return res, fails\\n\\n\\n# ----------------------------------------------------------------------------------------------- main\\n_EMIT_LOCK = threading.Lock()\\n_EMITTED = []\\n\\n\\ndef emit(res):\\n    \\"\\"\\"Print the ONE JSON line (first caller wins: main thread or watchdog).\\"\\"\\"\\n    with _EMIT_LOCK:\\n        if _EMITTED:\\n            return\\n        _EMITTED.append(1)\\n        res[\\"elapsed_s\\"] = round(time.time() - T0, 1)\\n        for _ in range(3):  # the watchdog may race the main thread mutating res\\n            try:\\n                line = json.dumps(res, separators=(\\",\\", \\":\\"), default=str)\\n                break\\n            except RuntimeError:\\n                time.sleep(0.05)\\n        else:\\n            line = json.dumps({k: res.get(k) for k in (\\"key\\", \\"ok\\", \\"error\\", \\"failed\\", \\"elapsed_s\\")}, default=str)\\n        print(line, flush=True)\\n\\n\\ndef start_watchdog(res, secs):\\n    \\"\\"\\"A hung kernel / graph replay (or a pathological cold JIT build) must still yield one JSON line, then exit.\\"\\"\\"\\n    if secs <= 0:\\n        return None\\n\\n    def fire():\\n        out = dict(res)\\n        out[\\"ok\\"] = False\\n        out[\\"error\\"] = f\\"watchdog: no result after {secs:.0f}s (hang, or a very cold JIT cache)\\"\\n        out[\\"failed\\"] = list(out.get(\\"failed\\") or []) + [\\"timeout\\"]\\n        emit(out)\\n        os._exit(3)\\n\\n    t = threading.Timer(secs, fire)\\n    t.daemon = True\\n    t.start()\\n    return t\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--ms\\", default=\\"1,13,19,20,80,2048\\")\\n    ap.add_argument(\\"--E-local\\", type=int, default=96)\\n    ap.add_argument(\\"--K\\", type=int, default=2560)\\n    ap.add_argument(\\"--N\\", type=int, default=640)\\n    ap.add_argument(\\"--topk\\", type=int, default=10)\\n    ap.add_argument(\\"--group\\", type=int, default=128)\\n    ap.add_argument(\\"--reps\\", type=int, default=8, help=\\"calls of 4 ranks per graph (M > 256 uses 1)\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=20)\\n    ap.add_argument(\\"--rounds\\", type=int, default=5)\\n    ap.add_argument(\\"--no-gt\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--no-graph\\", action=\\"store_true\\", help=\\"skip CUDA graphs (and timing)\\")\\n    ap.add_argument(\\"--fp16\\", action=\\"store_true\\", help=\\"also check fp16 activations (use_atomic_add Marlin path)\\")\\n    ap.add_argument(\\"--noep\\", action=\\"store_true\\",\\n                    help=\\"also check flag ON + expert_map=None (non-EP) is bit-identical to stock; builds the extra \\"\\n                         \\"is_ep=False Marlin JIT variant (not used by the EP server)\\")\\n    ap.add_argument(\\"--timeout\\", type=float, default=1500.0, help=\\"watchdog seconds (0 = off)\\")\\n    ap.add_argument(\\"--seed\\", type=int, default=1234)\\n    args = ap.parse_args()\\n    ms = [int(v) for v in args.ms.split(\\",\\") if v.strip()]\\n    res = {\\"key\\": KEY, \\"ok\\": False}\\n    failed = []\\n    per_m = {}\\n    res[\\"failed\\"], res[\\"per_m\\"] = failed, per_m  # same objects: a watchdog dump shows partial results\\n    wd = start_watchdog(res, args.timeout)\\n    try:\\n        dtype = torch.bfloat16\\n        gen = torch.Generator(device=DEV)\\n        gen.manual_seed(args.seed)\\n        if DEV == \\"cuda\\":\\n            res[\\"device\\"] = torch.cuda.get_device_name()\\n        impl = load_impls()\\n        res[\\"impl\\"] = impl.how\\n        res[\\"reference_is_unpatched_source\\"] = impl.has_ref\\n        E_loc, E_glob = args.E_local, args.E_local * RANKS\\n        res.update({\\"E_local\\": E_loc, \\"E_global\\": E_glob, \\"K\\": args.K, \\"N\\": args.N, \\"topk\\": args.topk,\\n                    \\"group\\": args.group})\\n        log(f\\"impl={impl.how}; random weights {E_glob} experts K={args.K} N={args.N}\\")\\n        W = rand_weights(E_glob, args.K, args.N, args.group, dtype, gen)\\n        log(\\"first calls JIT-compile Marlin (bf16, is_ep) + Triton kernels; can take minutes on a cold cache\\")\\n        for M in ms:\\n            ranks, full, _ = ep_case(M, W, E_loc, args.topk, gen, dtype)\\n            r, f, base, clean = ep_checks(impl, ranks, full, M, noep=args.noep)\\n            if not args.no_graph and DEV == \\"cuda\\":\\n                reps = args.reps if M <= 256 else 1\\n                rg, fg = graph_checks(impl, ranks, base, clean, reps, args.iters, args.rounds)\\n                r.update(rg)\\n                f += fg\\n            per_m[str(M)] = r\\n            failed += [f\\"M{M}:{x}\\" for x in f]\\n            log(f\\"M={M}: rel_vs_base={r[\'rel_err_vs_base\']:.2e} ep_rel base/clean={r[\'ep_rel_err_base\']:.2e}/\\"\\n                f\\"{r[\'ep_rel_err_clean\']:.2e} us base/clean={r.get(\'baseline_us\', float(\'nan\')):.1f}/\\"\\n                f\\"{r.get(\'patched_us\', float(\'nan\')):.1f} fails={f}\\")\\n            del ranks, full, base, clean\\n        # bit-exact flag-off == unpatched source at the decode sizes (M <= 256; larger M: see dormancy())\\n        res[\\"dormant_bitexact_decode\\"] = (all(v.get(\\"dormant_bitexact\\") is True for m, v in per_m.items()\\n                                              if int(m) <= 256) if impl.has_ref else \\"no-reference\\")\\n        res[\\"max_abs_err\\"] = nmax(v[\\"max_abs_err\\"] for v in per_m.values())\\n        res[\\"max_rel_err\\"] = nmax(v[\\"rel_err_vs_base\\"] for v in per_m.values())\\n        ref_m = \\"19\\" if \\"19\\" in per_m else str(ms[0])\\n        res[\\"baseline_us\\"] = per_m[ref_m].get(\\"baseline_us\\")\\n        res[\\"patched_us\\"] = per_m[ref_m].get(\\"patched_us\\")\\n        res[\\"timing_M\\"] = int(ref_m)\\n        if res[\\"baseline_us\\"] is not None:\\n            res[\\"est_step_saving_ms_48_layers\\"] = 48 * (res[\\"baseline_us\\"] - res[\\"patched_us\\"]) / 1000.0\\n\\n        # routed_scaling_factor != 1 (constexpr path of moe_fused_mul_sum)\\n        ranks, full, _ = ep_case(19, W, E_loc, args.topk, gen, dtype, rsf=2.5)\\n        r, f, _, _ = ep_checks(impl, ranks, full, 19)\\n        res[\\"rsf_2p5\\"] = {k: r.get(k) for k in (\\"rel_err_vs_base\\", \\"ep_rel_err_clean\\", \\"nan_empty_clean_finite\\",\\n                                                \\"dormant_bitexact\\")}\\n        failed += [f\\"rsf:{x}\\" for x in f]\\n        del ranks, full\\n\\n        al = align_equiv(ms, E_loc, args.topk, gen)\\n        res[\\"align_equiv\\"] = all(v[\\"ok\\"] for v in al.values())\\n        res[\\"align\\"] = al\\n        if not res[\\"align_equiv\\"]:\\n            failed.append(\\"align_equiv\\")\\n\\n        if args.fp16:\\n            W16 = tuple(t.to(torch.float16) if t.is_floating_point() else t for t in W)\\n            ranks, full, _ = ep_case(19, W16, E_loc, args.topk, gen, torch.float16)\\n            r, f, _, _ = ep_checks(impl, ranks, full, 19)\\n            res[\\"fp16\\"] = {k: r.get(k) for k in (\\"rel_err_vs_base\\", \\"ep_rel_err_base\\", \\"ep_rel_err_clean\\",\\n                                                 \\"nan_empty_clean_finite\\", \\"masked_rows_exact_zero\\",\\n                                                 \\"dormant_bitexact\\", \\"dormant_rel_err\\")}\\n            failed += [f\\"fp16:{x}\\" for x in f]\\n            del W16, ranks, full\\n\\n        del W\\n        if DEV == \\"cuda\\":\\n            torch.cuda.empty_cache()\\n        if args.no_gt:\\n            res[\\"gt\\"] = \\"skipped (--no-gt)\\"\\n        else:\\n            try:\\n                gt, f = ground_truth(impl, [m for m in (1, 19, 80) if m in ms] or [ms[0]], args.K, args.N,\\n                                     args.group, args.topk, dtype, gen)\\n                res[\\"gt\\"] = gt\\n                failed += f\\n            except ImportError as exc:  # helper missing in this wheel: report, do not fail the patch on it\\n                res[\\"gt\\"] = f\\"skipped ({type(exc).__name__}: {exc})\\"\\n        res[\\"failed\\"] = failed\\n        res[\\"ok\\"] = not failed\\n    except Exception as exc:  # noqa: BLE001 - one JSON line no matter what\\n        res[\\"error\\"] = f\\"{type(exc).__name__}: {str(exc)[:500]}\\"\\n        res[\\"traceback\\"] = traceback.format_exc()[-2000:]\\n        res[\\"failed\\"] = failed + [\\"exception\\"]\\n        res[\\"ok\\"] = False\\n    if wd is not None:\\n        wd.cancel()\\n    emit(res)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "bench_mqagrid.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 GPU correctness check + microbenchmark for colab/perf_patches/mqagrid_patch.py (key \\"mqagrid\\").\\n\\nRuns the shipped TileLang decode-MQA kernel from the venv\'s sglang/srt/layers/attention/qsa/mqa.py (loaded by file\\npath; the patch does not touch the kernel body, so patched and unpatched trees give the same kernel) with\\ngroups_per_cta G in {1,4,8,16} at the production decode shapes:\\n  8 kernel heads (4 indexer heads + 4 zero pad, as project_qk(q_heads_padded=8) writes them), head_dim 128,\\n  compressed page 16 (KV page 64 / compress ratio 4), CUDA-graph page-table width 2048 pages\\n  (ceil(131072 / 4 / 16)) -> max_model_len 32768, 12 indexer layers per decode step on 12 distinct caches.\\nChecks, all against G=1 (= the stock call, exact stock kwargs):\\n  * logits bit-identical (torch.equal, -inf mask included) at rows {13,19} x context {6k,24k} tokens, and on a\\n    boundary-length batch (0, 1, 15-17, 63-65, 64G-1..64G+1 for every G, ..., 32767, 32768); live rows are padded\\n    with zero-length rows up to the next decode-graph batch size (--graph-bs, 13 -> 16, 19 -> 20), exactly as the\\n    server\'s graph replay pads (seq_len fill 1 -> compressed length 0), since padded rows also launch CTAs;\\n  * each check re-run --repeat times (catches intermittent pipelining races), and again after CUDA-graph replay\\n    on all 12 layer caches (each variant\'s replayed outputs vs G1\'s replayed outputs; G1 layer 0 vs eager);\\n  * G=1 is deterministic and matches the torch fp32 reference torch_qsa_mqa_decode (setup sanity);\\n  * if the patch is applied to that tree: the module is dormant with the env unset, the env parser maps\\n    SGLANG_QSA_MQA_GROUPS_PER_CTA / SGLANG_QSA_MQA_NUM_STAGES as documented, and the patched wrapper\\n    tilelang_qsa_mqa_decode gives the same bits with _MQAGRID_KW set.\\nTiming: per-call us of (logits.fill_(-inf) + kernel), i.e. exactly what tilelang_qsa_mqa_decode records into the\\ndecode graph, from replaying a CUDA graph of 12 calls (one per layer cache, so L2 is cold per call). fill_us is the\\n-inf fill alone. Compiling each variant also warms the TileLang disk cache ($TILELANG_CACHE_DIR or ~/.tilelang/cache)\\nwith the same factory kwargs the patched server will request.\\nusage: CUDA_VISIBLE_DEVICES=0 /tmp/sgl-intel/venv/bin/python bench_mqagrid.py [<site-packages dir>]\\n         [--rows 13,19] [--ctx 6144,24576] [--groups 1,4,8,16] [--stages d] [--iters 50] [--reps 5] [--repeat 10]\\n         [--graph-bs 1,2,4,8,12,16,20] [--timeout 1200]\\n  --stages: comma list of num_stages for the G>1 variants; \\"d\\" = do not pass num_stages (kernel default 3, what\\n            SGLANG_QSA_MQA_GROUPS_PER_CTA alone selects); e.g. \\"d,0\\" also tries SGLANG_QSA_MQA_NUM_STAGES=0.\\n  --graph-bs: the server\'s cuda_graph_bs_decode list (\\"\\" = no padding); --timeout: watchdog seconds, after which\\n            the bench prints its JSON line (ok false) and exits 3 even if a kernel or compile hangs.\\nProgress goes to stderr; stdout gets ONE JSON line (fd 1 is redirected to stderr while running, so TileLang/nvcc\\nchatter cannot interleave with it).\\n\\"\\"\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport statistics\\nimport sys\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nKEY = \\"mqagrid\\"\\nHEADS_REAL, HEADS, HEAD_DIM = 4, 8, 128\\nPAGE = 16  # compressed page = KV page 64 / compress ratio 4\\nMAX_PAGES = 2048  # decode-graph page-table width: ceil(ceil(131072 / 4) / 16)\\nMAX_MODEL_LEN = MAX_PAGES * PAGE  # 32768 compressed positions\\nLAYERS = 12  # QSA (full-attention) layers per decode step\\nNUM_PAGES = 12288  # compressed pages per layer cache (48 MiB bf16), > L2\\nSCALE = float(math.sqrt(HEAD_DIM))  # tilelang_qsa_mqa_decode\'s default score_scale\\nENV_G, ENV_S = \\"SGLANG_QSA_MQA_GROUPS_PER_CTA\\", \\"SGLANG_QSA_MQA_NUM_STAGES\\"\\n\\n\\n_OUT = sys.stdout  # the real stdout once _claim_stdout() ran; only emit() writes to it\\n_EMIT_LOCK = threading.Lock()\\n_EMITTED = []\\n_STATE = {\\"res\\": None}  # the live result dict, for the watchdog\'s partial report\\n\\n\\ndef log(*a):\\n    print(\\"[bench_mqagrid]\\", *a, file=sys.stderr, flush=True)\\n\\n\\ndef _claim_stdout():\\n    # keep fd 1 for the single JSON line; everything else written to fd 1 (C-level prints too) goes to stderr\\n    global _OUT\\n    try:\\n        sys.stdout.flush()\\n        _OUT = os.fdopen(os.dup(1), \\"w\\")\\n        os.dup2(2, 1)\\n    except Exception:\\n        _OUT = sys.stdout\\n\\n\\ndef emit(res):\\n    with _EMIT_LOCK:\\n        if _EMITTED:\\n            return\\n        _EMITTED.append(True)\\n        _OUT.write(json.dumps(res, sort_keys=False, default=str) + \\"\\\\n\\")\\n        _OUT.flush()\\n\\n\\ndef _start_watchdog(seconds):\\n    def fire():\\n        if _EMITTED:  # main already reported; let it exit normally\\n            return\\n        try:\\n            out = json.loads(json.dumps(_STATE[\\"res\\"] or {}, default=str))\\n        except Exception:  # dict mutated mid-dump by the main thread\\n            out = {}\\n        out.update({\\"key\\": KEY, \\"ok\\": False})\\n        for k in (\\"baseline_us\\", \\"patched_us\\", \\"max_abs_err\\"):\\n            out.setdefault(k, None)\\n        out.setdefault(\\"errors\\", []).append(f\\"watchdog: no result after {seconds} s (hung kernel/compile?); exit 3\\")\\n        emit(out)\\n        os._exit(3)\\n\\n    t = threading.Timer(seconds, fire)\\n    t.daemon = True\\n    t.start()\\n    return t\\n\\n\\ndef load_mqa(site):\\n    if site:\\n        path = Path(site) / \\"sglang/srt/layers/attention/qsa/mqa.py\\"\\n    else:\\n        spec = importlib.util.find_spec(\\"sglang\\")\\n        path = Path(list(spec.submodule_search_locations)[0]) / \\"srt/layers/attention/qsa/mqa.py\\"\\n    spec = importlib.util.spec_from_file_location(\\"_bench_mqagrid_mqa\\", path)\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    return mod, path\\n\\n\\ndef vname(g, s):\\n    return f\\"G{g}\\" + (\\"\\" if s is None else f\\"_S{s}\\")\\n\\n\\ndef vkwargs(g, s):\\n    # exactly the extra kwargs _mqagrid_decode_kernel passes (none for G=1 = the stock call)\\n    if g == 1:\\n        return {}\\n    kw = {\\"groups_per_cta\\": g}\\n    if s is not None:\\n        kw[\\"num_stages\\"] = s\\n    return kw\\n\\n\\ndef row_lengths(rows, ctx):\\n    \\"\\"\\"Compressed lengths with mean ctx/4, spread 0.5x..1.5x, deliberately not page/group aligned.\\"\\"\\"\\n    base = ctx // 4\\n    out = []\\n    for r in range(rows):\\n        frac = 0.5 + (r / (rows - 1) if rows > 1 else 0.5)\\n        out.append(max(1, min(MAX_MODEL_LEN, int(base * frac) + (r * 7) % 13)))\\n    return out\\n\\n\\ndef edge_lengths(groups):\\n    s = {0, 1, 15, 16, 17, 63, 64, 65, 127, 128, 129, 1000, 4095, 8191, 16385, MAX_MODEL_LEN - 1, MAX_MODEL_LEN}\\n    for g in groups:\\n        s |= {64 * g - 1, 64 * g, 64 * g + 1, 2 * 64 * g + 17}\\n    return sorted(x for x in s if 0 <= x <= MAX_MODEL_LEN)\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"site\\", nargs=\\"?\\", default=None)\\n    ap.add_argument(\\"--rows\\", default=\\"13,19\\")\\n    ap.add_argument(\\"--ctx\\", default=\\"6144,24576\\")\\n    ap.add_argument(\\"--groups\\", default=\\"1,4,8,16\\")\\n    ap.add_argument(\\"--stages\\", default=\\"d\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=50)\\n    ap.add_argument(\\"--reps\\", type=int, default=5)\\n    ap.add_argument(\\"--repeat\\", type=int, default=10, help=\\"eager re-runs per variant in the bit-identity checks\\")\\n    ap.add_argument(\\"--graph-bs\\", default=\\"1,2,4,8,12,16,20\\", help=\\"decode CUDA-graph batch sizes (\'\' = no padding)\\")\\n    ap.add_argument(\\"--timeout\\", type=float, default=1200.0, help=\\"watchdog seconds (<= 0 disables)\\")\\n    a = ap.parse_args()\\n    rows_list = [int(x) for x in a.rows.split(\\",\\") if x]\\n    ctx_list = [int(x) for x in a.ctx.split(\\",\\") if x]\\n    groups = sorted({1} | {int(x) for x in a.groups.split(\\",\\") if x})\\n    stages = list(dict.fromkeys(None if x.strip() in (\\"d\\", \\"\\") else int(x) for x in a.stages.split(\\",\\"))) or [None]\\n    graph_bs = sorted({int(x) for x in a.graph_bs.split(\\",\\") if x.strip()})\\n    res = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"max_abs_err\\": None, \\"errors\\": []}\\n    _STATE[\\"res\\"] = res\\n    if a.timeout > 0:\\n        _start_watchdog(a.timeout)\\n\\n    import torch\\n\\n    if not torch.cuda.is_available():\\n        res[\\"errors\\"].append(\\"no CUDA\\")\\n        return emit(res)\\n    dev = torch.device(\\"cuda\\")\\n    res[\\"device\\"] = torch.cuda.get_device_name(dev)\\n    res[\\"torch\\"] = torch.__version__\\n\\n    # Load with the env vars removed, so module-level _MQAGRID_KW reflects the dormant default.\\n    saved = {k: os.environ.pop(k) for k in (ENV_G, ENV_S) if k in os.environ}\\n    try:\\n        mod, path = load_mqa(a.site)\\n    except Exception as exc:\\n        res[\\"errors\\"].append(f\\"import mqa.py: {type(exc).__name__}: {exc}\\")\\n        return emit(res)\\n    res[\\"mqa_path\\"] = str(path)\\n    if not getattr(mod, \\"HAS_TILELANG\\", False):\\n        res[\\"errors\\"].append(\\"tilelang unavailable in this venv (HAS_TILELANG False)\\")\\n        return emit(res)\\n    res[\\"tilelang\\"] = getattr(mod.tilelang, \\"__version__\\", None)\\n    patched = hasattr(mod, \\"_mqagrid_decode_kernel\\")\\n    res[\\"patch_applied\\"] = patched\\n    checks_ok = True\\n\\n    # ---- patch-only host checks: dormancy + env parsing ------------------------------------------------------\\n    if patched:\\n        res[\\"dormant_ok\\"] = mod._MQAGRID_KW == {}\\n        cases = [\\n            ({}, {}),\\n            ({ENV_G: \\"\\"}, {}),\\n            ({ENV_G: \\"1\\"}, {}),\\n            ({ENV_G: \\"1\\", ENV_S: \\"0\\"}, {}),\\n            ({ENV_S: \\"0\\"}, {}),\\n            ({ENV_G: \\"8\\"}, {\\"groups_per_cta\\": 8}),\\n            ({ENV_G: \\" 16 \\"}, {\\"groups_per_cta\\": 16}),\\n            ({ENV_G: \\"8\\", ENV_S: \\"0\\"}, {\\"groups_per_cta\\": 8, \\"num_stages\\": 0}),\\n            ({ENV_G: \\"8\\", ENV_S: \\"2\\"}, {\\"groups_per_cta\\": 8, \\"num_stages\\": 2}),\\n            ({ENV_G: \\"abc\\"}, {}),\\n            ({ENV_G: \\"0\\"}, {}),\\n            ({ENV_G: \\"-4\\"}, {}),\\n            ({ENV_G: \\"8\\", ENV_S: \\"9\\"}, {}),\\n            ({ENV_G: \\"8\\", ENV_S: \\"x\\"}, {}),\\n        ]\\n        bad = []\\n        for env, want in cases:\\n            for k in (ENV_G, ENV_S):\\n                os.environ.pop(k, None)\\n            os.environ.update(env)\\n            got = mod._mqagrid_env_kwargs()\\n            if got != want:\\n                bad.append({\\"env\\": env, \\"got\\": got, \\"want\\": want})\\n        for k in (ENV_G, ENV_S):\\n            os.environ.pop(k, None)\\n        res[\\"env_parse_ok\\"] = not bad\\n        if bad:\\n            res[\\"errors\\"].append(f\\"env parse mismatches: {bad}\\")\\n        checks_ok &= res[\\"dormant_ok\\"] and res[\\"env_parse_ok\\"]\\n    os.environ.update(saved)\\n\\n    factory = mod._tilelang_qsa_mqa_decode_kernel\\n    gen = torch.Generator(device=dev)\\n    gen.manual_seed(1234)\\n    caches = [\\n        torch.randn(NUM_PAGES, PAGE, 1, HEAD_DIM, generator=gen, device=dev, dtype=torch.bfloat16)\\n        for _ in range(LAYERS)\\n    ]\\n\\n    def make_inputs(lengths):\\n        rows = len(lengths)\\n        q = torch.zeros(rows, HEADS, HEAD_DIM, dtype=torch.bfloat16, device=dev)\\n        q[:, :HEADS_REAL] = torch.randn(rows, HEADS_REAL, HEAD_DIM, generator=gen, device=dev, dtype=torch.bfloat16)\\n        pt = torch.stack(\\n            [torch.randperm(NUM_PAGES, generator=gen, device=dev)[:MAX_PAGES] for _ in range(rows)]\\n        ).to(torch.int32).contiguous()\\n        cl = torch.tensor(lengths, dtype=torch.int32, device=dev)\\n        return q, q.unsqueeze(1).contiguous(), pt, cl\\n\\n    def run(kern, q4, cache, pt, cl, out=None):\\n        if out is None:\\n            out = torch.empty(q4.shape[0], MAX_MODEL_LEN, dtype=torch.float32, device=dev)\\n        out.fill_(-math.inf)\\n        kern(q4, cache, pt, cl, out, SCALE)\\n        return out\\n\\n    def compare(ref, out):\\n        fr, fo = torch.isfinite(ref), torch.isfinite(out)\\n        both = fr & fo\\n        err = (ref[both] - out[both]).abs().max().item() if bool(both.any()) else 0.0\\n        return {\\n            \\"bit_identical\\": bool(torch.equal(ref, out)),\\n            \\"mask_identical\\": bool(torch.equal(fr, fo)),\\n            \\"nan\\": bool(torch.isnan(out).any()),\\n            \\"max_abs_err\\": err,\\n        }\\n\\n    def reorder_ok(cmp, ref):\\n        # accept fp32 reduction reordering only (same -inf mask, no NaN, tiny error); bit identity is expected\\n        amax = ref[torch.isfinite(ref)].abs().max().item() if bool(torch.isfinite(ref).any()) else 1.0\\n        return cmp[\\"mask_identical\\"] and not cmp[\\"nan\\"] and cmp[\\"max_abs_err\\"] <= 1e-5 * max(1.0, amax)\\n\\n    # ---- compile every variant (first call also launches once) ------------------------------------------------\\n    variants = [(1, None)] + [(g, s) for g in groups if g > 1 for s in stages]\\n    kernels, compile_s = {}, {}\\n    tiny = make_inputs([64])\\n    for g, s in variants:\\n        name = vname(g, s)\\n        t0 = time.time()\\n        try:\\n            kern = factory(heads=HEADS, head_dim=HEAD_DIM, page_size=PAGE, **vkwargs(g, s))\\n            run(kern, tiny[1], caches[0], tiny[2], tiny[3])\\n            torch.cuda.synchronize()\\n            kernels[name] = kern\\n        except Exception as exc:\\n            res[\\"errors\\"].append(f\\"{name} compile/launch: {type(exc).__name__}: {str(exc)[:400]}\\")\\n            log(name, \\"FAILED\\", traceback.format_exc()[-1500:])\\n        compile_s[name] = round(time.time() - t0, 2)\\n        log(f\\"compiled {name} in {compile_s[name]} s\\")\\n    res[\\"compile_s\\"] = compile_s\\n    if \\"G1\\" not in kernels:\\n        res[\\"errors\\"].append(\\"stock G=1 kernel failed; nothing to compare against\\")\\n        return emit(res)\\n    passing = {n: True for n in kernels}\\n    for g, s in variants:\\n        if vname(g, s) not in kernels:\\n            checks_ok = False\\n    max_err, all_bit = 0.0, True\\n\\n    def check_variants(tag, q4, pt, cl, ref, repeat):\\n        # repeat > 1: re-run and re-compare, to catch intermittent (pipelining / race) differences\\n        nonlocal max_err, all_bit, checks_ok\\n        out = {}\\n        for name, kern in kernels.items():\\n            if name == \\"G1\\":\\n                continue\\n            cmp = None\\n            for _ in range(max(1, repeat)):\\n                c = compare(ref, run(kern, q4, caches[0], pt, cl))\\n                if cmp is None or (cmp[\\"bit_identical\\"] and not c[\\"bit_identical\\"]) or c[\\"max_abs_err\\"] > cmp[\\"max_abs_err\\"]:\\n                    cmp = c  # keep the worst run\\n                if not reorder_ok(c, ref):\\n                    cmp = c\\n                    break  # a real mismatch; no need to keep re-running\\n            out[name] = cmp\\n            max_err = max(max_err, cmp[\\"max_abs_err\\"])\\n            all_bit &= cmp[\\"bit_identical\\"]\\n            if not reorder_ok(cmp, ref):\\n                passing[name] = False\\n                checks_ok = False\\n                res[\\"errors\\"].append(f\\"{tag} {name} differs from G1: {cmp}\\")\\n        return out\\n\\n    def check_reference(tag, q3, q4, pt, cl):\\n        nonlocal checks_ok\\n        ref = run(kernels[\\"G1\\"], q4, caches[0], pt, cl)\\n        again = run(kernels[\\"G1\\"], q4, caches[0], pt, cl)\\n        det = bool(torch.equal(ref, again))\\n        tref = mod.torch_qsa_mqa_decode(q3, caches[0], pt, cl, MAX_MODEL_LEN)\\n        cmp = compare(tref, ref)\\n        expect_mask = torch.arange(MAX_MODEL_LEN, device=dev)[None, :] < cl[:, None].long()\\n        mask_ok = bool(torch.equal(torch.isfinite(ref), expect_mask))\\n        amax = tref[torch.isfinite(tref)].abs().max().item() if bool(torch.isfinite(tref).any()) else 1.0\\n        ok = det and mask_ok and cmp[\\"mask_identical\\"] and not cmp[\\"nan\\"] and cmp[\\"max_abs_err\\"] <= 1e-3 * max(1.0, amax)\\n        if not ok:\\n            checks_ok = False\\n            res[\\"errors\\"].append(f\\"{tag} G1 sanity failed: det={det} mask_ok={mask_ok} vs_torch={cmp} amax={amax}\\")\\n        del again, tref\\n        return ref, {\\"g1_deterministic\\": det, \\"g1_mask_ok\\": mask_ok, \\"ref_max_abs_err\\": cmp[\\"max_abs_err\\"],\\n                     \\"ref_amax\\": amax, \\"ref_ok\\": ok}\\n\\n    # ---- boundary-length batch ----------------------------------------------------------------------------------\\n    edges = edge_lengths(groups)\\n    q3, q4, pt, cl = make_inputs(edges)\\n    ref, sanity = check_reference(\\"edge\\", q3, q4, pt, cl)\\n    res[\\"edge\\"] = {\\"lengths\\": edges, **sanity, \\"variants\\": check_variants(\\"edge\\", q4, pt, cl, ref, a.repeat)}\\n    edge_inputs, edge_ref = (q3, q4, pt, cl), ref\\n    log(\\"edge batch:\\", json.dumps({k: v for k, v in res[\\"edge\\"].items() if k != \\"lengths\\"}))\\n\\n    # ---- production-shape configs: correctness + CUDA-graph timing --------------------------------------------\\n    results, ref_errs = [], [sanity[\\"ref_max_abs_err\\"]]\\n    graph_timing = True\\n    for rows in rows_list:\\n        for ctx in ctx_list:\\n            tag = f\\"rows{rows}_ctx{ctx}\\"\\n            # pad like the decode graph replay: next captured bs, extra rows have compressed length 0\\n            grows = next((b for b in graph_bs if b >= rows), rows)\\n            lengths = row_lengths(rows, ctx) + [0] * (grows - rows)\\n            q3, q4, pt, cl = make_inputs(lengths)\\n            ref, sanity = check_reference(tag, q3, q4, pt, cl)\\n            ref_errs.append(sanity[\\"ref_max_abs_err\\"])\\n            vcmp = check_variants(tag, q4, pt, cl, ref, a.repeat)\\n\\n            outs = {n: [torch.empty(grows, MAX_MODEL_LEN, dtype=torch.float32, device=dev) for _ in range(LAYERS)]\\n                    for n in kernels}\\n\\n            def make_step(name):\\n                kern, bufs = kernels[name], outs[name]\\n\\n                def step():\\n                    for layer in range(LAYERS):\\n                        bufs[layer].fill_(-math.inf)\\n                        kern(q4, caches[layer], pt, cl, bufs[layer], SCALE)\\n                return step\\n\\n            # own buffers: the replayed kernel outputs are re-checked after timing\\n            fill_bufs = [torch.empty(grows, MAX_MODEL_LEN, dtype=torch.float32, device=dev) for _ in range(LAYERS)]\\n\\n            def fill_step():\\n                for layer in range(LAYERS):\\n                    fill_bufs[layer].fill_(-math.inf)\\n\\n            fns = {n: make_step(n) for n in kernels}\\n            fns[\\"_fill\\"] = fill_step\\n            runners = {}\\n            for n, fn in fns.items():\\n                try:\\n                    side = torch.cuda.Stream()\\n                    side.wait_stream(torch.cuda.current_stream())\\n                    with torch.cuda.stream(side):\\n                        fn()\\n                        fn()\\n                    torch.cuda.current_stream().wait_stream(side)\\n                    torch.cuda.synchronize()\\n                    gr = torch.cuda.CUDAGraph()\\n                    with torch.cuda.graph(gr):\\n                        fn()\\n                    runners[n] = gr.replay\\n                    runners[n + \\"#graph\\"] = gr\\n                except Exception as exc:\\n                    # the server captures this kernel in every decode graph: a variant that cannot be captured is\\n                    # unusable (and its eager timing is not comparable), so it fails; G1/_fill failing fails the run\\n                    res[\\"errors\\"].append(f\\"{tag} CUDA graph capture failed ({n}), eager timing: {type(exc).__name__}: {str(exc)[:200]}\\")\\n                    graph_timing = False\\n                    checks_ok = False\\n                    if n in passing:\\n                        passing[n] = False\\n                    torch.cuda.synchronize()\\n                    runners[n] = fn\\n            for n in fns:\\n                for _ in range(5):\\n                    runners[n]()\\n            torch.cuda.synchronize()\\n            times = {n: [] for n in fns}\\n            st, en = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\\n            for _ in range(a.reps):\\n                for n in fns:\\n                    st.record()\\n                    for _ in range(a.iters):\\n                        runners[n]()\\n                    en.record()\\n                    en.synchronize()\\n                    times[n].append(st.elapsed_time(en) * 1000.0 / (a.iters * LAYERS))\\n            fill_us = statistics.median(times[\\"_fill\\"])\\n            # after all replays: G1 layer 0 must equal the eager reference, and every variant must equal G1 on all\\n            # 12 layer caches (different K data and page placement per layer)\\n            for n in kernels:\\n                for layer in range(LAYERS):\\n                    want = ref if (n == \\"G1\\" and layer == 0) else outs[\\"G1\\"][layer]\\n                    if n == \\"G1\\" and layer > 0:\\n                        continue\\n                    cmp = compare(want, outs[n][layer])\\n                    if not reorder_ok(cmp, want):\\n                        passing[n] = False\\n                        checks_ok = False\\n                        res[\\"errors\\"].append(f\\"{tag} {n} replayed layer {layer} differs from G1: {cmp}\\")\\n                    all_bit &= cmp[\\"bit_identical\\"]\\n                    max_err = max(max_err, cmp[\\"max_abs_err\\"])\\n                    if not passing.get(n, False):\\n                        break  # one error line per variant and config is enough\\n            for n in kernels:\\n                us = statistics.median(times[n])\\n                entry = {\\"rows\\": rows, \\"graph_rows\\": grows, \\"ctx\\": ctx, \\"variant\\": n, \\"us\\": round(us, 2),\\n                         \\"kernel_us\\": round(us - fill_us, 2),\\n                         \\"fill_us\\": round(fill_us, 2), \\"us_min\\": round(min(times[n]), 2), \\"us_max\\": round(max(times[n]), 2)}\\n                if n != \\"G1\\":\\n                    entry.update({k: vcmp[n][k] for k in (\\"bit_identical\\", \\"max_abs_err\\")})\\n                else:\\n                    entry.update({\\"ref_max_abs_err\\": sanity[\\"ref_max_abs_err\\"], \\"ref_ok\\": sanity[\\"ref_ok\\"]})\\n                results.append(entry)\\n                log(json.dumps(entry))\\n            runners = outs = fns = fill_bufs = ref = None  # free graphs + buffers before the next config\\n            torch.cuda.synchronize()\\n            torch.cuda.empty_cache()\\n    res[\\"graph_timing\\"] = graph_timing\\n    res[\\"results\\"] = results\\n    res[\\"bit_identical\\"] = all_bit\\n    res[\\"max_abs_err\\"] = max_err\\n    res[\\"ref_max_abs_err\\"] = max(ref_errs)\\n\\n    # ---- pick the best variant -----------------------------------------------------------------------------------\\n    configs = [(r, c) for r in rows_list for c in ctx_list]\\n    us = {(e[\\"variant\\"], e[\\"rows\\"], e[\\"ctx\\"]): e[\\"us\\"] for e in results}\\n\\n    def mean_us(n):\\n        return statistics.mean(us[(n, r, c)] for r, c in configs)\\n\\n    good = [n for n in kernels if passing[n]]\\n    best = min(good, key=mean_us) if good else \\"G1\\"\\n    # recommendation: fastest passing G>1 variant that is not >2% slower than G1 on any config and >=3% faster overall\\n    rec = None\\n    cands = [n for n in good if n != \\"G1\\" and all(us[(n, r, c)] <= 1.02 * us[(\\"G1\\", r, c)] for r, c in configs)]\\n    if cands:\\n        cand = min(cands, key=mean_us)\\n        if mean_us(cand) <= 0.97 * mean_us(\\"G1\\"):\\n            rec = cand\\n    chosen = rec or best  # headline numbers: the variant recommend_env would ship, else the fastest passing one\\n    res[\\"variant_mean_us\\"] = {n: round(mean_us(n), 2) for n in kernels}\\n    res[\\"best_variant\\"] = best\\n    res[\\"patched_variant\\"] = chosen\\n    res[\\"baseline_us\\"] = round(mean_us(\\"G1\\"), 2)\\n    res[\\"patched_us\\"] = round(mean_us(chosen), 2)\\n    res[\\"speedup\\"] = round(mean_us(\\"G1\\") / mean_us(chosen), 3)\\n    res[\\"per_step_saving_us\\"] = round((mean_us(\\"G1\\") - mean_us(chosen)) * LAYERS, 1)\\n    res[\\"fill_us_mean\\"] = round(statistics.mean(e[\\"fill_us\\"] for e in results if e[\\"variant\\"] == \\"G1\\"), 2)\\n    if rec:\\n        g, _, srest = rec[1:].partition(\\"_S\\")\\n        env = f\\"{ENV_G}={g}\\" + (f\\" {ENV_S}={srest}\\" if srest else \\"\\")\\n        res[\\"recommend_env\\"] = env\\n    else:\\n        res[\\"recommend_env\\"] = None\\n\\n    # ---- patched wrapper path (only if the patch is applied to this tree) --------------------------------------\\n    if patched:\\n        target = rec or next((n for n in good if n != \\"G1\\"), None)\\n        if target is None:\\n            res[\\"wrapper_ok\\"] = False\\n            res[\\"errors\\"].append(\\"no passing G>1 variant to exercise the patched wrapper\\")\\n            checks_ok = False\\n        else:\\n            g, _, srest = target[1:].partition(\\"_S\\")\\n            kw = vkwargs(int(g), int(srest) if srest else None)\\n            q3, q4, pt, cl = edge_inputs\\n            try:\\n                mod._MQAGRID_KW = {}\\n                stock = mod.tilelang_qsa_mqa_decode(q3, caches[0], pt, cl, MAX_MODEL_LEN)\\n                mod._MQAGRID_KW = dict(kw)\\n                gridded = mod.tilelang_qsa_mqa_decode(q3, caches[0], pt, cl, MAX_MODEL_LEN)\\n                still = mod._MQAGRID_KW == kw  # a compile failure would have reset it to {}\\n                # dormant wrapper must be bit-identical to the stock kernel; the gridded one is held to the same\\n                # standard as the variant checks (bit-identical expected, fp32 reorder tolerated)\\n                wok = bool(torch.equal(stock, edge_ref)) and reorder_ok(compare(edge_ref, gridded), edge_ref) and still\\n            except Exception as exc:\\n                wok = False\\n                res[\\"errors\\"].append(f\\"patched wrapper: {type(exc).__name__}: {str(exc)[:300]}\\")\\n            finally:\\n                mod._MQAGRID_KW = {}\\n            res[\\"wrapper_ok\\"] = wok\\n            res[\\"wrapper_variant\\"] = target\\n            checks_ok &= wok\\n\\n    res[\\"passing_variants\\"] = good\\n    res[\\"ok\\"] = bool(checks_ok and all(passing.values()) and len(kernels) == len(variants))\\n    emit(res)\\n\\n\\nif __name__ == \\"__main__\\":\\n    _claim_stdout()\\n    try:\\n        main()\\n    except BaseException as exc:  # always print one JSON line (partial results kept); bad args included\\n        if isinstance(exc, SystemExit) and exc.code in (0, None):\\n            raise  # --help\\n        try:\\n            out = json.loads(json.dumps(_STATE[\\"res\\"] or {}, default=str))\\n        except Exception:\\n            out = {}\\n        out.update({\\"key\\": KEY, \\"ok\\": False})\\n        for k in (\\"baseline_us\\", \\"patched_us\\", \\"max_abs_err\\"):\\n            out.setdefault(k, None)\\n        out.setdefault(\\"errors\\", []).extend(\\n            [f\\"crash: {type(exc).__name__}: {str(exc)[:500]}\\", traceback.format_exc()[-2000:]])\\n        emit(out)\\n        sys.exit(1)\\n", "bench_pairown.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 4-GPU bench for colab/perf_patches/pairown_patch.py (key \\"pairown\\"): the pair-exchange plumbing, no model.\\n\\nSpawns 4 ranks (one L4 each, real NCCL), brings up SGLang\'s own parallel state (TP4), loads the PATCHED qsa/pairown.py by\\nfile path and, at the production decode shapes (q of one rank = 6 heads x 256 bf16 = 3 KB/row, rows {19,32,48,64}, 12 QSA\\nlayers per step = 24 pair all-gathers), checks and times the two exchanges of pair_attend:\\n  * correctness: GroupCoordinator gather routing (slot i = pair member i); pair_attend with a stand-in attention\\n    (q_wide*2+1) on every layer in both ownership phases, eager and inside a CUDA graph captured the way CudaGraphRunner does\\n    (sglang graph_capture() context), replayed with changed inputs; compared bit-exact against q*2+1 per rank.\\n  * timing: 12-layer CUDA graph of pair_attend with a free stand-in attention vs the same graph with a local copy, for the\\n    dedicated 2-rank group (\\"pair\\") and for the TP group (\\"tp\\", 4-way gather, slots picked by pair). exchange_us = difference.\\n  * nccl_mem_mib: GPU memory consumed by creating the pair group and running its first collectives (communicator buffers).\\nIf SGLang\'s init fails the bench falls back to raw torch.distributed groups (timing only, mode \\"torch\\").\\npatched_us = exchange_us for the better mode at rows 32 (extra step time pair ownership adds); baseline_us stays None.\\nusage: python3 bench_pairown.py <site-packages dir> [--rows 19,32,48,64] [--iters 40] [--timeout 600]\\nstdout gets ONE JSON line.\\n\\"\\"\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nimport os\\nimport socket\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\nimport traceback\\nimport types\\nfrom pathlib import Path\\n\\nimport torch\\n\\nKEY = \\"pairown\\"\\nWORLD = 4\\nLAYERS = 12\\nHEADS_LOCAL, HEAD_DIM = 6, 256\\nC = HEADS_LOCAL * HEAD_DIM\\nLAYER_IDS = [3 + 4 * i for i in range(LAYERS)]\\nMIB = 1 << 20\\n\\n_OUT = sys.stdout\\n_EMITTED = []\\n_LOCK = threading.Lock()\\n\\n\\ndef log(*a):\\n    print(\\"[bench_pairown]\\", *a, file=sys.stderr, flush=True)\\n\\n\\ndef claim_stdout():\\n    global _OUT\\n    try:\\n        sys.stdout.flush()\\n        _OUT = os.fdopen(os.dup(1), \\"w\\")\\n        os.dup2(2, 1)\\n    except Exception:\\n        _OUT = sys.stdout\\n\\n\\ndef emit(res):\\n    with _LOCK:\\n        if _EMITTED:\\n            return\\n        _EMITTED.append(1)\\n        _OUT.write(json.dumps(res, default=str) + \\"\\\\n\\")\\n        _OUT.flush()\\n\\n\\ndef load_pairown(site):\\n    path = Path(site) / \\"sglang/srt/layers/attention/qsa/pairown.py\\"\\n    spec = importlib.util.spec_from_file_location(\\"_bench_pairown_mod\\", path)\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef layer_mod():\\n    attn = types.SimpleNamespace(tp_k_head_num=1, qk_head_dim=HEAD_DIM, tp_v_head_num=1, v_head_dim=HEAD_DIM)\\n    return types.SimpleNamespace(attn=attn)\\n\\n\\ndef make_plans(pm, cfg):\\n    plans = []\\n    for lid in LAYER_IDS:\\n        owner = cfg[\\"owner\\"][lid]\\n        shim = types.SimpleNamespace(tp_q_head_num=2 * HEADS_LOCAL) if owner == cfg[\\"pair_rank\\"] else None\\n        plans.append(pm.Plan(lid, owner, cfg[\\"pair_rank\\"], cfg[\\"R\\"], shim))\\n    return plans\\n\\n\\ndef run_layers(pm, plans, lm, qs, ks, vs, attend):\\n    outs = []\\n    for plan, q in zip(plans, qs):\\n        outs.append(pm.pair_attend(plan, lm, q, ks, vs, None, attend))\\n    return outs\\n\\n\\ndef stand_in(q_wide, k3, v3, shim):\\n    return q_wide * 2 + 1\\n\\n\\ndef free_attend(q_wide, k3, v3, shim):\\n    return q_wide\\n\\n\\ndef graph_run(dev, fn, iters, capture_ctx=None):\\n    \\"\\"\\"Capture fn() in a CUDA graph (inside SGLang\'s graph_capture() when given); returns (graph, outputs).\\"\\"\\"\\n    for _ in range(2):\\n        fn()\\n    torch.cuda.synchronize()\\n    g = torch.cuda.CUDAGraph()\\n    if capture_ctx is not None:\\n        with capture_ctx() as ctx:\\n            with torch.cuda.graph(g, stream=ctx.stream):\\n                outs = fn()\\n    else:\\n        with torch.cuda.graph(g):\\n            outs = fn()\\n    torch.cuda.synchronize()\\n    return g, outs\\n\\n\\ndef time_graph(dist, g, iters):\\n    dist.barrier()\\n    torch.cuda.synchronize()\\n    for _ in range(5):\\n        g.replay()\\n    torch.cuda.synchronize()\\n    dist.barrier()\\n    t0 = torch.cuda.Event(enable_timing=True)\\n    t1 = torch.cuda.Event(enable_timing=True)\\n    t0.record()\\n    for _ in range(iters):\\n        g.replay()\\n    t1.record()\\n    torch.cuda.synchronize()\\n    return t0.elapsed_time(t1) / iters * 1000.0\\n\\n\\ndef worker(rank, world, port, out_path, rows_list, iters, site):\\n    import torch.distributed as dist\\n\\n    res = {\\"errors\\": [], \\"correctness\\": {}, \\"timing\\": [], \\"modes\\": {}}\\n\\n    def save():\\n        if rank == 0:\\n            Path(out_path).write_text(json.dumps(res, default=str))\\n\\n    torch.cuda.set_device(rank)\\n    dev = torch.device(\\"cuda\\", rank)\\n    sgl_ok = False\\n    try:\\n        from sglang.srt.distributed import init_distributed_environment, initialize_model_parallel\\n        from sglang.srt.distributed.parallel_state import graph_capture, set_custom_all_reduce\\n\\n        set_custom_all_reduce(False)\\n        init_distributed_environment(world_size=world, rank=rank, distributed_init_method=f\\"tcp://127.0.0.1:{port}\\",\\n                                     local_rank=rank, backend=\\"nccl\\")\\n        initialize_model_parallel(tensor_model_parallel_size=world)\\n        sgl_ok = True\\n    except Exception as exc:\\n        res[\\"errors\\"].append(f\\"rank{rank} sglang init: {type(exc).__name__}: {str(exc)[-300:]}\\")\\n        if not dist.is_initialized():\\n            dist.init_process_group(\\"nccl\\", init_method=f\\"tcp://127.0.0.1:{port}\\", rank=rank, world_size=world, device_id=dev)\\n    res[\\"sglang_init\\"] = sgl_ok\\n    try:\\n        os.environ[\\"SGLANG_QSA_PAIR_OWN\\"] = \\"1\\"\\n        pm = load_pairown(site)\\n        cfg = pm.configure(world, rank, 2, LAYER_IDS)\\n        if cfg is None:\\n            raise RuntimeError(\\"pairown.configure returned None\\")\\n        res[\\"owned\\"] = cfg[\\"owned\\"]\\n        lm = layer_mod()\\n        cap = graph_capture if sgl_ok else None\\n        gen = torch.Generator(device=\\"cpu\\").manual_seed(1000 + rank)\\n\\n        if sgl_ok:\\n            for mode in (\\"pair\\", \\"tp\\"):\\n                pm._GROUP_MODE = mode\\n                pm._S.pop(\\"group\\", None)\\n                pm._S.pop(\\"gidx0\\", None)\\n                free0 = torch.cuda.mem_get_info()[0]\\n                dist.barrier()\\n                pm.ensure_group()\\n                warm = torch.zeros((8, 64), dtype=torch.bfloat16, device=dev)\\n                pm.gather(warm)\\n                torch.cuda.synchronize()\\n                dist.barrier()\\n                free1 = torch.cuda.mem_get_info()[0]\\n                mrec = res[\\"modes\\"].setdefault(mode, {})\\n                mrec[\\"group_mem_mib\\"] = round((free0 - free1) / MIB, 1)\\n                mrec[\\"pynccl\\"] = getattr(pm._S[\\"group\\"], \\"pynccl_comm\\", None) is not None\\n                # gather routing\\n                x = torch.full((4, 16), float(rank + 1), dtype=torch.bfloat16, device=dev)\\n                g = pm.gather(x)\\n                want = torch.tensor([cfg[\\"ranks\\"][0] + 1, cfg[\\"ranks\\"][1] + 1], dtype=torch.float32)\\n                got = g[:, 0, 0].float().cpu()\\n                ok_route = bool(torch.equal(got, want)) and g.shape[0] == 2\\n                mrec[\\"gather_routes\\"] = ok_route\\n                # pair_attend bit-exactness, eager then graph, two input draws\\n                for rows in (7, 32):\\n                    plans = make_plans(pm, cfg)\\n                    qs = [torch.randn(rows, HEADS_LOCAL, HEAD_DIM, generator=gen).to(torch.bfloat16).to(dev) for _ in plans]\\n                    ks = torch.randn(rows, HEAD_DIM, generator=gen).to(torch.bfloat16).to(dev)\\n                    outs = run_layers(pm, plans, lm, qs, ks, ks, stand_in)\\n                    eager_ok = all(torch.equal(o, q.reshape(rows, -1) * 2 + 1) for o, q in zip(outs, qs))\\n                    static_q = [q.clone() for q in qs]\\n                    g_, gouts = graph_run(dev, lambda: run_layers(pm, plans, lm, static_q, ks, ks, stand_in), 1, cap)\\n                    for it in range(2):\\n                        for s in static_q:\\n                            s.copy_(torch.randn(rows, HEADS_LOCAL, HEAD_DIM, generator=gen).to(torch.bfloat16))\\n                        g_.replay()\\n                        torch.cuda.synchronize()\\n                        graph_ok = all(torch.equal(o, q.reshape(rows, -1) * 2 + 1) for o, q in zip(gouts, static_q))\\n                        mrec[f\\"graph_ok_rows{rows}_it{it}\\"] = graph_ok\\n                    mrec[f\\"eager_ok_rows{rows}\\"] = eager_ok\\n                    dist.barrier()\\n                save()\\n                # timing\\n                for rows in rows_list:\\n                    plans = make_plans(pm, cfg)\\n                    qs = [torch.randn(rows, HEADS_LOCAL, HEAD_DIM, generator=gen).to(torch.bfloat16).to(dev) for _ in plans]\\n                    ks = torch.randn(rows, HEAD_DIM).to(torch.bfloat16).to(dev)\\n                    g_x, _ = graph_run(dev, lambda: run_layers(pm, plans, lm, qs, ks, ks, free_attend), 1, cap)\\n                    local_gather = lambda x: x.unsqueeze(0).expand(2, -1, -1).contiguous()\\n                    g_b, _ = graph_run(dev, lambda: [pm.pair_attend(p, lm, q, ks, ks, None, free_attend, gather_fn=local_gather)\\n                                                     for p, q in zip(plans, qs)], 1, cap)\\n                    t_x = time_graph(dist, g_x, iters)\\n                    t_b = time_graph(dist, g_b, iters)\\n                    res[\\"timing\\"].append({\\"mode\\": mode, \\"rows\\": rows, \\"pair_us\\": round(t_x, 1), \\"local_us\\": round(t_b, 1),\\n                                          \\"exchange_us\\": round(t_x - t_b, 1)})\\n                    save()\\n        else:\\n            # fallback: raw torch.distributed pair groups, timing only\\n            pgs = [dist.new_group([0, 1]), dist.new_group([2, 3])]\\n            pg = pgs[rank // 2]\\n            for rows in rows_list:\\n                qs = torch.randn(rows, C).to(torch.bfloat16).to(dev)\\n\\n                def fn():\\n                    for _ in range(2 * LAYERS):\\n                        out = torch.empty((2 * rows, C), dtype=qs.dtype, device=dev)\\n                        dist.all_gather_into_tensor(out, qs, group=pg)\\n                    return out\\n\\n                g_x, _ = graph_run(dev, fn, 1)\\n                res[\\"timing\\"].append({\\"mode\\": \\"torch\\", \\"rows\\": rows, \\"pair_us\\": round(time_graph(dist, g_x, iters), 1)})\\n            save()\\n    except Exception as exc:\\n        res[\\"errors\\"].append(f\\"rank{rank}: {type(exc).__name__}: {str(exc)[-400:]}\\\\n{traceback.format_exc()[-1500:]}\\")\\n    save()\\n    try:\\n        dist.barrier()\\n    except Exception:\\n        pass\\n    os._exit(0)\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"site\\")\\n    ap.add_argument(\\"--rows\\", default=\\"19,32,48,64\\")\\n    ap.add_argument(\\"--iters\\", type=int, default=40)\\n    ap.add_argument(\\"--timeout\\", type=int, default=600)\\n    args = ap.parse_args()\\n    claim_stdout()\\n    rows_list = [int(x) for x in args.rows.split(\\",\\") if x]\\n    out = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"errors\\": [], \\"torch\\": torch.__version__,\\n           \\"gpu\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\\n\\n    def dog():\\n        time.sleep(args.timeout + 60)\\n        out[\\"errors\\"].append(\\"watchdog\\")\\n        emit(out)\\n        os._exit(3)\\n\\n    threading.Thread(target=dog, daemon=True).start()\\n    if torch.cuda.device_count() < WORLD:\\n        out[\\"errors\\"].append(f\\"needs {WORLD} GPUs, saw {torch.cuda.device_count()}\\")\\n        emit(out)\\n        return\\n    import torch.multiprocessing as mp\\n\\n    with socket.socket() as s:\\n        s.bind((\\"127.0.0.1\\", 0))\\n        port = s.getsockname()[1]\\n    fd, out_path = tempfile.mkstemp(prefix=\\"bench_pairown_\\", suffix=\\".json\\")\\n    os.close(fd)\\n    Path(out_path).write_text(\\"{}\\")\\n    t0 = time.time()\\n    err = None\\n    try:\\n        ctx = mp.spawn(worker, args=(WORLD, port, out_path, rows_list, args.iters, args.site), nprocs=WORLD, join=False)\\n        while True:\\n            try:\\n                if ctx.join(timeout=5):\\n                    break\\n            except Exception as exc:\\n                err = f\\"{type(exc).__name__}: {str(exc)[-400:]}\\"\\n                break\\n            if time.time() - t0 > args.timeout:\\n                err = f\\"timeout after {args.timeout}s\\"\\n                break\\n    except Exception as exc:\\n        err = f\\"spawn: {type(exc).__name__}: {exc}\\"\\n    try:\\n        import multiprocessing\\n\\n        for p in multiprocessing.active_children():\\n            p.kill()\\n    except Exception:\\n        pass\\n    try:\\n        res = json.loads(Path(out_path).read_text() or \\"{}\\")\\n    except Exception:\\n        res = {}\\n    if err:\\n        out[\\"errors\\"].append(err)\\n    out[\\"errors\\"] += res.get(\\"errors\\", [])\\n    out[\\"sglang_init\\"] = res.get(\\"sglang_init\\")\\n    out[\\"modes\\"] = res.get(\\"modes\\")\\n    out[\\"timing\\"] = res.get(\\"timing\\")\\n    out[\\"owned_rank0\\"] = res.get(\\"owned\\")\\n    out[\\"seconds\\"] = round(time.time() - t0, 1)\\n    modes = out[\\"modes\\"] or {}\\n    correct = bool(modes) and all(\\n        m.get(\\"gather_routes\\") and all(v for k, v in m.items() if k.startswith((\\"eager_ok\\", \\"graph_ok\\"))) for m in modes.values())\\n    out[\\"correctness\\"] = {\\"all_modes_exact\\": correct}\\n    pm32 = {t[\\"mode\\"]: t for t in (out[\\"timing\\"] or []) if t.get(\\"rows\\") == 32 and \\"exchange_us\\" in t}\\n    if pm32:\\n        best = min(pm32.values(), key=lambda t: t[\\"exchange_us\\"])\\n        out[\\"headline\\"] = {\\"best_mode\\": best[\\"mode\\"], **{k: best[k] for k in (\\"pair_us\\", \\"local_us\\", \\"exchange_us\\")},\\n                           \\"by_mode\\": {k: v[\\"exchange_us\\"] for k, v in pm32.items()},\\n                           \\"group_mem_mib\\": {k: v.get(\\"group_mem_mib\\") for k, v in modes.items()}}\\n        out[\\"patched_us\\"] = best[\\"exchange_us\\"]\\n    out[\\"ok\\"] = bool(correct and pm32)\\n    emit(out)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "bench_pcie.py": "\\"\\"\\"PCIe host<->GPU microbenchmark for the KV-offload feasibility question (one process per GPU, GPUs visible via CUDA_VISIBLE_DEVICES).\\nargv[1] = wall-clock epoch to start at (all GPUs start together so the \'all4\' run measures root-complex contention).\\nPrints one JSON line: bulk H2D/D2H GB/s on 256 MB pinned buffers, and the rate of scattered page-sized (390 KB = 64 tokens x 6.1 KB\\nof K+V for the 12 QSA layers) pinned->GPU copies issued one by one (a lower bound for a gather kernel).\\"\\"\\"\\nimport json, sys, time\\nres = {\\"ok\\": False}\\ntry:\\n    import torch\\n    start = float(sys.argv[1]) if len(sys.argv) > 1 else 0.0\\n    dev = torch.device(\\"cuda:0\\")\\n    n = 256 * 1024 * 1024\\n    h = torch.empty(n, dtype=torch.uint8, pin_memory=True)\\n    g = torch.empty(n, dtype=torch.uint8, device=dev)\\n    h.fill_(1); g.fill_(2); torch.cuda.synchronize()\\n\\n    def bw(copy, reps):\\n        copy(); torch.cuda.synchronize()\\n        t = time.perf_counter()\\n        for _ in range(reps):\\n            copy()\\n        torch.cuda.synchronize()\\n        return n * reps / (time.perf_counter() - t) / 1e9\\n\\n    while time.time() < start:\\n        time.sleep(0.001)\\n    res[\\"h2d_GBps\\"] = round(bw(lambda: g.copy_(h, non_blocking=True), 60), 2)\\n    res[\\"d2h_GBps\\"] = round(bw(lambda: h.copy_(g, non_blocking=True), 60), 2)\\n    ps, pages = 390 * 1024, 640\\n    hp, gp = h[:pages * ps].view(pages, ps), g[:pages * ps].view(pages, ps)\\n    idx = torch.randperm(pages).tolist()[:512]\\n\\n    def scatter():\\n        for i in idx:\\n            gp[i].copy_(hp[i], non_blocking=True)\\n    scatter(); torch.cuda.synchronize()\\n    t = time.perf_counter()\\n    for _ in range(4):\\n        scatter()\\n    torch.cuda.synchronize()\\n    dt = (time.perf_counter() - t) / 4\\n    res[\\"page_copies_per_s\\"] = round(len(idx) / dt)\\n    res[\\"page_GBps\\"] = round(len(idx) * ps / dt / 1e9, 2)\\n    res[\\"ok\\"] = True\\nexcept Exception as ex:   # noqa: BLE001\\n    res[\\"error\\"] = repr(ex)\\nprint(json.dumps(res))\\n", "bench_qsasplitk.py": "#!/usr/bin/env python3\\n\\"\\"\\"GPU correctness + speed check for qsasplitk_patch.py (split-K QSA sparse decode) on one L4.\\n\\nPhase 0 (no server, GPU free), venv python, from the directory that holds the patch files:\\n  CUDA_VISIBLE_DEVICES=0 /tmp/sgl-intel/venv/bin/python bench_qsasplitk.py [<site-packages dir>] [--quick]\\n      [--prod-kv fp8|bf16]\\n\\nNeeds torch + triton only (no SGLang import). Modules are loaded by file path:\\n  * reference + mode \\"triton\\": the duck2 sm89_decode.py, from <site>/sglang/srt/layers/attention/qsa/sm89_decode.py\\n    if it is installed, else qsa_sm89_decode.py next to this script, else ../../duck2/patches/qsa_sm89_decode.py.\\n  * split-K: the installed sm89_decode_splitk.py when it equals qsasplitk_patch.MODULE_SRC (a stale install fails\\n    the run), else MODULE_SRC written to a temp dir. So the bench also works before the patch is applied.\\n\\nChecks (all must pass for \\"ok\\"):\\n  1. correctness vs the fp32 loop reference (qsa_sparse_decode_reference) for rows {1,13,19,20,48} x {bf16, fp8} KV,\\n     topk 2051, D 256, Hq 6 / Hkv 1 (TP4 shapes), with -1 holes, entries >= seq_len, a valid entry after the -1\\n     tail, a 3-token row and an empty row (must be exactly 0); \\"peaky\\" cases whose softmax max sits in different\\n     splits; topk 17 / 40 (one or two blocks, nsplit request > blocks); GQA 24/2 and 12/1; prefix layout.\\n     Tolerance as test_qsa_sm89_decode.py: max|out-ref| <= 2e-2*max|ref| + 1e-2. Every sweep config is checked too.\\n  2. CUDA graph: capture at 19 rows (fp8), mutate indices/lengths in place, replay == eager (bitwise).\\n  3. dormancy: splitk_route returns the very same (impl, dispatch) objects for every env value except\\n     triton_splitk; with triton_splitk the routed dispatch equals the direct split-K call.\\nTiming (does not affect \\"ok\\"): CUDA-graph replay of n back-to-back calls (n >= 12, enough that the K/V touched\\nexceeds 4x L2, like 12 QSA layers between MoE weight reads), all 2051 entries valid at 8k context, random slots\\nin a 2^20-slot pool. Reports us/call for mode triton (baseline) vs triton_splitk (default config) and a sweep of\\n(block_n, stages, target CTAs); \\"suggest_env\\" is the best sweep setting over rows {1,13,19,20} for the prod KV dtype.\\nHeadline baseline_us / patched_us = prod KV (default fp8), 19 rows.\\n\\nPrints progress on stderr and ONE JSON line on stdout.\\n\\"\\"\\"\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport statistics\\nimport sys\\nimport tempfile\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nHERE = Path(__file__).resolve().parent\\nKEY = \\"qsasplitk\\"\\nROWS = (1, 13, 19, 20, 48)\\nSUGGEST_ROWS = (1, 13, 19, 20)\\nTOPK = 2051\\nHQ, HKV, D = 6, 1, 256\\nCTX = 8192\\nPOOL_SLOTS = 1 << 20\\nENV = \\"SGLANG_QSA_SM89_DECODE\\"\\n\\n\\ndef log(*a):\\n    print(*a, file=sys.stderr, flush=True)\\n\\n\\ndef load_module(name, path):\\n    spec = importlib.util.spec_from_file_location(name, str(path))\\n    mod = importlib.util.module_from_spec(spec)\\n    sys.modules[name] = mod\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef find_base(site):\\n    cands = []\\n    if os.environ.get(\\"QSASPLITK_BASE\\"):\\n        cands.append(Path(os.environ[\\"QSASPLITK_BASE\\"]))\\n    if site is not None:\\n        cands.append(site / \\"sglang/srt/layers/attention/qsa/sm89_decode.py\\")\\n    cands += [HERE / \\"qsa_sm89_decode.py\\", HERE.parent.parent / \\"duck2\\" / \\"patches\\" / \\"qsa_sm89_decode.py\\"]\\n    for c in cands:\\n        if c.is_file():\\n            return c\\n    raise FileNotFoundError(f\\"duck2 sm89_decode.py not found in {[str(c) for c in cands]}\\")\\n\\n\\ndef main():\\n    t0 = time.time()\\n    argv = sys.argv[1:]\\n    pos, prod_kv, quick, i = [], \\"fp8\\", False, 0\\n    while i < len(argv):\\n        a = argv[i]\\n        if a == \\"--prod-kv\\" and i + 1 < len(argv):\\n            prod_kv, i = argv[i + 1], i + 1\\n        elif a.startswith(\\"--prod-kv=\\"):\\n            prod_kv = a.split(\\"=\\", 1)[1]\\n        elif a == \\"--quick\\":\\n            quick = True\\n        elif not a.startswith(\\"--\\"):\\n            pos.append(a)\\n        i += 1\\n    if prod_kv not in (\\"fp8\\", \\"bf16\\"):\\n        prod_kv = \\"fp8\\"\\n    site = Path(pos[0]) if pos else None\\n    res = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"max_abs_err\\": None,\\n           \\"prod_kv\\": prod_kv, \\"quick\\": quick}\\n    errors = []\\n    try:\\n        run(res, errors, site, quick, prod_kv)\\n    except Exception:\\n        errors.append(traceback.format_exc()[-1500:])\\n    res[\\"errors\\"] = errors\\n    res[\\"ok\\"] = bool(res.get(\\"_checks_ok\\")) and not errors\\n    res.pop(\\"_checks_ok\\", None)\\n    res[\\"elapsed_s\\"] = round(time.time() - t0, 1)\\n    print(json.dumps(res, default=str), flush=True)\\n    return 0 if res[\\"ok\\"] else 1\\n\\n\\ndef run(res, errors, site, quick, prod_kv):\\n    import torch\\n\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"needs CUDA\\")\\n    dev = torch.device(\\"cuda\\")\\n    props = torch.cuda.get_device_properties(0)\\n    sms = props.multi_processor_count\\n    l2 = int(getattr(props, \\"L2_cache_size\\", 48 << 20) or (48 << 20))\\n    import triton\\n\\n    res.update(device=props.name, sm_count=sms, l2_mb=round(l2 / 2**20, 1), torch=torch.__version__,\\n               triton=triton.__version__)\\n\\n    base_path = find_base(site)\\n    BASE = load_module(\\"qsasplitk_base\\", base_path)\\n    P = load_module(\\"qsasplitk_patch_src\\", HERE / \\"qsasplitk_patch.py\\")\\n    res[\\"base_module\\"] = str(base_path)\\n    mod_path = None\\n    if site is not None and (site / P.MODULE_REL).is_file():\\n        res[\\"installed\\"] = True\\n        res[\\"installed_matches\\"] = (site / P.MODULE_REL).read_text() == P.MODULE_SRC\\n        # the whole inserted block, verbatim and exactly once (not just the mark)\\n        res[\\"callsite_patched\\"] = (site / P.BACKEND_REL).read_text().count(P.NEW) == 1\\n        if not res[\\"installed_matches\\"]:\\n            errors.append(\\"installed sm89_decode_splitk.py differs from qsasplitk_patch.MODULE_SRC; re-run the patch\\")\\n        elif not res[\\"callsite_patched\\"]:\\n            errors.append(\\"module installed but call site not patched (qsasplitk_patch.NEW not found exactly once)\\")\\n        else:\\n            mod_path = site / P.MODULE_REL\\n    else:\\n        res[\\"installed\\"] = False\\n    if mod_path is None:\\n        mod_path = Path(tempfile.mkdtemp(prefix=\\"qsasplitk_\\")) / \\"sm89_decode_splitk.py\\"\\n        mod_path.write_text(P.MODULE_SRC)\\n    SK = load_module(\\"qsasplitk_mod\\", mod_path)\\n    res[\\"splitk_module\\"] = str(mod_path)\\n    res[\\"default_cfg\\"] = dict(SK.splitk_config())\\n    res[\\"default_plan\\"] = {r: SK.plan_splits(r, HKV, TOPK, res[\\"default_cfg\\"][\\"block_n\\"],\\n                                             res[\\"default_cfg\\"][\\"nsplit\\"], res[\\"default_cfg\\"][\\"target_ctas\\"], sms)[0]\\n                           for r in ROWS}\\n    REF = BASE.qsa_sparse_decode_reference\\n    TRI = BASE.qsa_sparse_decode_triton\\n    SPK = SK.qsa_sparse_decode_triton_splitk\\n    KVS = {\\"bf16\\": torch.bfloat16, \\"fp8\\": torch.float8_e4m3fn}\\n\\n    # sweep configs: (block_n, num_stages, num_warps, target_ctas); target 1 = never split (FINAL path)\\n    sweep = [(32, 2, 4, 1)]\\n    if not quick:\\n        for bn, st in ((32, 2), (32, 1), (64, 2), (16, 2)):\\n            for mult in (1, 2, 4, 8):\\n                if bn == 64 and mult == 8:\\n                    continue\\n                sweep.append((bn, st, 4, mult * sms))\\n    res[\\"sweep_cfgs\\"] = [list(c) for c in sweep]\\n\\n    def cfg_kw(c):\\n        return dict(block_n=c[0], num_stages=c[1], num_warps=c[2], target_ctas=c[3], nsplit=0)\\n\\n    # ---------------------------------------------------------------- correctness\\n    def make_case(rows, topk, kv_dtype, hq=HQ, hkv=HKV, d=D, ctx=6000, pool_reqs=8, seed=0,\\n                  prefix_layout=False, peaky=False):\\n        g = torch.Generator().manual_seed(seed)\\n        slots_n = pool_reqs * ctx + 64\\n        q = torch.randn(rows, hq, d, generator=g).to(torch.bfloat16)\\n        k = torch.randn(slots_n, hkv, d, generator=g) * 0.5\\n        v = torch.randn(slots_n, hkv, d, generator=g) * 2.0\\n        r2t = (torch.randperm(slots_n - 64, generator=g)[: pool_reqs * ctx] + 64).view(pool_reqs, ctx)\\n        r2t = r2t.to(torch.int32)\\n        req = torch.randint(0, pool_reqs, (rows,), generator=g).to(torch.int64)\\n        seq = torch.randint(1, ctx, (rows,), generator=g).to(torch.int32)\\n        if rows > 2:\\n            seq[1] = 3\\n            seq[2] = 0\\n        idx = torch.full((rows, topk), -1, dtype=torch.int32)\\n        grp = hq // hkv\\n        for r in range(rows):\\n            n = min(int(seq[r]), topk - 3)\\n            if n:\\n                idx[r, :n] = torch.randperm(int(seq[r]), generator=g)[:n].to(torch.int32)\\n            if not prefix_layout and n > 8:\\n                idx[r, n // 2] = -1\\n                idx[r, topk - 1] = int(seq[r]) - 1\\n                idx[r, topk - 2] = int(seq[r]) + 5\\n            if peaky and n > 8:  # big scores at the start / middle / end of the row -> maxima in different splits\\n                for frac, amp in ((0.05, 0.6), (0.55, 0.8), (0.95, 1.0)):\\n                    c = min(n - 1, max(0, int(n * frac)))\\n                    if c == n // 2:\\n                        c += 1\\n                    slot = int(r2t[int(req[r]), int(idx[r, c])])\\n                    for h in range(hkv):\\n                        k[slot, h] = q[r, h * grp].float() * amp\\n        zero_rows = [2] if rows > 2 else []\\n        args = [q.to(dev), k.to(kv_dtype).to(dev), v.to(kv_dtype).to(dev), r2t.to(dev), req.to(dev),\\n                idx.to(dev), seq.to(dev), d ** -0.5]\\n        return args, zero_rows\\n\\n    def measure(out, ref, zero_rows):\\n        err = (out.float() - ref.float()).abs().max().item()\\n        tol = 2e-2 * ref.float().abs().max().item() + 1e-2\\n        zeros_ok = all(out[r].abs().max().item() == 0 for r in zero_rows)\\n        finite = bool(torch.isfinite(out.float()).all().item())\\n        return err, tol, zeros_ok and finite and err <= tol\\n\\n    cases = []\\n    for kvn in (\\"bf16\\", \\"fp8\\"):\\n        for rows in ROWS:\\n            cases.append((f\\"{kvn}/r{rows}\\", dict(rows=rows, topk=TOPK, kv_dtype=KVS[kvn], seed=rows), {}, True))\\n        for rows in (1, 19):\\n            cases.append((f\\"{kvn}/r{rows}/peaky\\",\\n                          dict(rows=rows, topk=TOPK, kv_dtype=KVS[kvn], seed=100 + rows, peaky=True), {}, True))\\n    cases += [\\n        (\\"bf16/r5/topk17\\", dict(rows=5, topk=17, kv_dtype=torch.bfloat16, ctx=1000, seed=7), {}, False),\\n        (\\"fp8/r3/topk40/nsplit64\\", dict(rows=3, topk=40, kv_dtype=torch.float8_e4m3fn, ctx=1000, seed=8),\\n         {\\"nsplit\\": 64}, False),\\n        (\\"bf16/gqa24x2\\", dict(rows=4, topk=TOPK, kv_dtype=torch.bfloat16, hq=24, hkv=2, ctx=4000, seed=9), {}, False),\\n        (\\"bf16/gqa12x1\\", dict(rows=4, topk=TOPK, kv_dtype=torch.bfloat16, hq=12, hkv=1, ctx=4000, seed=10), {}, False),\\n        (\\"fp8/r7/prefix\\", dict(rows=7, topk=TOPK, kv_dtype=torch.float8_e4m3fn, seed=11, prefix_layout=True),\\n         {}, False),\\n    ]\\n    corr = {}\\n    checks_ok = True\\n    max_err = max_err_tri = max_err_vs_tri = max_ratio = 0.0\\n    final_bitexact = True\\n    sweep_fail, sweep_errs, sweep_worst = [], {}, 0.0\\n    for name, kw, extra, do_sweep in cases:\\n        args, zr = make_case(**kw)\\n        ref = REF(*args)\\n        tri = TRI(*args)\\n        spk = SPK(*args, **extra)\\n        e_s, tol, ok_s = measure(spk, ref, zr)\\n        e_t, _, ok_t = measure(tri, ref, zr)\\n        e_st = (spk.float() - tri.float()).abs().max().item()\\n        fin = SPK(*args, target_ctas=1, nsplit=0, block_n=32, num_warps=4, num_stages=2)\\n        bitexact = torch.equal(fin, tri)\\n        final_bitexact &= bitexact\\n        corr[name] = dict(err=round(e_s, 6), err_triton=round(e_t, 6), err_vs_triton=round(e_st, 6),\\n                          tol=round(tol, 5), ok=ok_s, triton_ok=ok_t, final_bitexact=bitexact)\\n        log(f\\"[corr] {\'ok \' if ok_s else \'BAD\'} {name}: splitk err {e_s:.3e} triton err {e_t:.3e} \\"\\n            f\\"splitk-triton {e_st:.3e} tol {tol:.3e} final==triton {bitexact}\\")\\n        checks_ok &= ok_s\\n        if not ok_s:\\n            errors.append(f\\"correctness {name}: err {e_s:.3e} > tol {tol:.3e} or zero/finite check failed\\")\\n        max_err, max_err_tri = max(max_err, e_s), max(max_err_tri, e_t)\\n        max_err_vs_tri, max_ratio = max(max_err_vs_tri, e_st), max(max_ratio, e_s / tol)\\n        if do_sweep:\\n            for c in sweep:\\n                try:\\n                    o = SPK(*args, **cfg_kw(c))\\n                    e_c, tol_c, ok_c = measure(o, ref, zr)\\n                    sweep_worst = max(sweep_worst, e_c / tol_c)\\n                    if not ok_c:\\n                        sweep_fail.append(f\\"{name} cfg={c} err={e_c:.3e}\\")\\n                except Exception as ex:  # e.g. out of shared memory for a large block_n\\n                    sweep_errs[str(c)] = f\\"{type(ex).__name__}: {str(ex)[:160]}\\"\\n        del args, ref, tri, spk, fin\\n    if sweep_fail:\\n        checks_ok = False\\n        errors.append(\\"sweep correctness failures: \\" + \\"; \\".join(sweep_fail[:8]))\\n    res.update(max_abs_err=round(max_err, 6), max_abs_err_triton=round(max_err_tri, 6),\\n               max_abs_err_vs_triton=round(max_err_vs_tri, 6), max_err_over_tol=round(max_ratio, 4),\\n               sweep_max_err_over_tol=round(sweep_worst, 4), sweep_cfg_errors=sweep_errs,\\n               final_path_bitexact_vs_triton=final_bitexact, correctness=corr)\\n\\n    # ---------------------------------------------------------------- CUDA graph\\n    args, zr = make_case(rows=19, topk=TOPK, kv_dtype=torch.float8_e4m3fn, ctx=30000, pool_reqs=12, seed=3)\\n    SPK(*args)\\n    s = torch.cuda.Stream()\\n    s.wait_stream(torch.cuda.current_stream())\\n    with torch.cuda.stream(s):\\n        for _ in range(2):\\n            SPK(*args)\\n    torch.cuda.current_stream().wait_stream(s)\\n    graph = torch.cuda.CUDAGraph()\\n    with torch.cuda.graph(graph):\\n        static_out = SPK(*args)\\n    args[5].copy_(torch.roll(args[5], 1, dims=1))\\n    args[6].sub_(1).clamp_(min=0)\\n    graph.replay()\\n    torch.cuda.synchronize()\\n    fresh = SPK(*args)\\n    graph_ok = torch.equal(static_out, fresh)\\n    e_g, tol_g, ok_g = measure(static_out, REF(*args), zr)\\n    graph_ok &= ok_g\\n    res[\\"graph_ok\\"] = graph_ok\\n    log(f\\"[graph] {\'ok \' if graph_ok else \'BAD\'} replay==eager after input update; vs ref err {e_g:.3e}\\")\\n    if not graph_ok:\\n        errors.append(\\"CUDA graph replay mismatch\\")\\n    checks_ok &= graph_ok\\n    del graph, static_out, fresh\\n\\n    # ---------------------------------------------------------------- dormancy / routing\\n    saved = os.environ.get(ENV)\\n    sentinel = object()\\n    dormant_ok = True\\n    try:\\n        for val in (None, \\"\\", \\"auto\\", \\"triton\\", \\"torch\\", \\"off\\", \\"1\\", \\"triton-splitk\\", \\"splitk\\"):\\n            if val is None:\\n                os.environ.pop(ENV, None)\\n            else:\\n                os.environ[ENV] = val\\n            for impl in (\\"triton\\", \\"torch\\", None):\\n                r = SK.splitk_route(impl, sentinel)\\n                dormant_ok &= r[0] is impl and r[1] is sentinel\\n        os.environ[ENV] = \\" Triton_SplitK \\"\\n        impl, fn = SK.splitk_route(\\"triton\\", sentinel)\\n        routed_ok = impl == \\"triton_splitk\\" and fn is not sentinel and torch.equal(fn(impl, *args), SPK(*args))\\n        BASE._IMPL_CACHE.clear()\\n        res[\\"duck2_select_with_splitk_env\\"] = BASE.select_sm89_decode_impl(dev)\\n    finally:\\n        if saved is None:\\n            os.environ.pop(ENV, None)\\n        else:\\n            os.environ[ENV] = saved\\n        BASE._IMPL_CACHE.clear()\\n    res[\\"dormant_ok\\"], res[\\"routed_ok\\"] = dormant_ok, routed_ok\\n    log(f\\"[route] dormant_ok={dormant_ok} routed_ok={routed_ok}\\")\\n    if not (dormant_ok and routed_ok):\\n        errors.append(\\"splitk_route dormancy/routing check failed\\")\\n    checks_ok &= dormant_ok and routed_ok\\n    del args\\n    res[\\"_checks_ok\\"] = checks_ok\\n\\n    # ---------------------------------------------------------------- timing\\n    def graph_time_us(fn, calls):\\n        for c in calls[:2]:\\n            fn(*c)\\n        torch.cuda.synchronize()\\n        st = torch.cuda.Stream()\\n        st.wait_stream(torch.cuda.current_stream())\\n        with torch.cuda.stream(st):\\n            for c in calls:\\n                fn(*c)\\n        torch.cuda.current_stream().wait_stream(st)\\n        torch.cuda.synchronize()\\n        g = torch.cuda.CUDAGraph()\\n        with torch.cuda.graph(g):\\n            for c in calls:\\n                fn(*c)\\n        g.replay()\\n        torch.cuda.synchronize()\\n        e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\\n        e0.record()\\n        g.replay()\\n        e1.record()\\n        e1.synchronize()\\n        inner = max(1, min(50, math.ceil(5.0 / max(e0.elapsed_time(e1), 1e-3))))\\n        samples = []\\n        for _ in range(5):\\n            e0.record()\\n            for _ in range(inner):\\n                g.replay()\\n            e1.record()\\n            e1.synchronize()\\n            samples.append(e0.elapsed_time(e1) * 1000.0 / (inner * len(calls)))\\n        del g\\n        return statistics.median(samples)\\n\\n    timing = {}\\n    for kvn in (\\"fp8\\", \\"bf16\\"):\\n        kv = KVS[kvn]\\n        # separate allocations: K and V must not alias (bf16 .to(bf16) would return the same tensor)\\n        K = torch.empty(POOL_SLOTS, HKV, D, dtype=torch.bfloat16, device=dev).normal_(0, 0.5).to(kv)\\n        V = torch.empty(POOL_SLOTS, HKV, D, dtype=torch.bfloat16, device=dev).normal_(0, 2.0).to(kv)\\n        assert K.data_ptr() != V.data_ptr()\\n        timing[kvn] = {}\\n        for rows in ROWS:\\n            bytes_call = rows * TOPK * HKV * D * K.element_size() * 2\\n            n_calls = int(min(256, max(12, math.ceil(4 * l2 / bytes_call))))\\n            calls = []\\n            for _ in range(n_calls):\\n                q = torch.randn(rows, HQ, D, device=dev, dtype=torch.bfloat16)\\n                r2t = torch.randint(0, POOL_SLOTS, (rows, CTX), dtype=torch.int32, device=dev)\\n                req = torch.arange(rows, dtype=torch.int64, device=dev)\\n                seq = torch.full((rows,), CTX, dtype=torch.int32, device=dev)\\n                idx = torch.rand(rows, CTX, device=dev).argsort(dim=1)[:, :TOPK].to(torch.int32).contiguous()\\n                calls.append((q, K, V, r2t, req, idx, seq, D ** -0.5))\\n            ent = {\\"n_calls\\": n_calls, \\"mb_per_call\\": round(bytes_call / 2**20, 2)}\\n            ent[\\"triton_us\\"] = round(graph_time_us(TRI, calls), 2)\\n            ent[\\"splitk_us\\"] = round(graph_time_us(SPK, calls), 2)\\n            ent[\\"nsplit\\"] = res[\\"default_plan\\"][rows]\\n            ent[\\"speedup\\"] = round(ent[\\"triton_us\\"] / ent[\\"splitk_us\\"], 3)\\n            ent[\\"splitk_gbps\\"] = round(bytes_call / ent[\\"splitk_us\\"] / 1e3, 1)\\n            ent[\\"triton_gbps\\"] = round(bytes_call / ent[\\"triton_us\\"] / 1e3, 1)\\n            seen, sw = {}, {}\\n            for c in sweep:\\n                ns = SK.plan_splits(rows, HKV, TOPK, c[0], 0, c[3], sms)[0]\\n                pk = (c[0], c[1], c[2], ns)\\n                if pk not in seen:\\n                    try:\\n                        seen[pk] = round(graph_time_us(lambda *a, _c=c: SPK(*a, **cfg_kw(_c)), calls), 2)\\n                    except Exception as ex:\\n                        seen[pk] = None\\n                        sweep_errs.setdefault(str(c), f\\"{type(ex).__name__}: {str(ex)[:160]}\\")\\n                sw[str(list(c))] = {\\"nsplit\\": ns, \\"us\\": seen[pk]}\\n            ent[\\"sweep\\"] = sw\\n            okv = [(v[\\"us\\"], k) for k, v in sw.items() if v[\\"us\\"] is not None]\\n            if okv:\\n                ent[\\"best_us\\"], ent[\\"best_cfg\\"] = min(okv)\\n            timing[kvn][rows] = ent\\n            log(f\\"[time] {kvn} rows={rows:2d} calls={n_calls:3d} triton {ent[\'triton_us\']:8.2f} us \\"\\n                f\\"splitk {ent[\'splitk_us\']:8.2f} us (nsplit {ent[\'nsplit\']}, x{ent[\'speedup\']:.2f}) \\"\\n                f\\"best {ent.get(\'best_us\')} {ent.get(\'best_cfg\')}\\")\\n            del calls\\n        del K, V\\n        torch.cuda.empty_cache()\\n    res[\\"timing\\"] = timing\\n    pt = timing.get(prod_kv, timing[\\"fp8\\"])\\n    res[\\"baseline_us\\"] = pt[19][\\"triton_us\\"]\\n    res[\\"patched_us\\"] = pt[19][\\"splitk_us\\"]\\n    res[\\"speedup_r19\\"] = pt[19][\\"speedup\\"]\\n    res[\\"speedup_by_rows\\"] = {r: pt[r][\\"speedup\\"] for r in ROWS}\\n    res[\\"min_speedup_rows_le20\\"] = min(pt[r][\\"speedup\\"] for r in SUGGEST_ROWS)\\n    res[\\"gate_30pct_rows_le20\\"] = res[\\"min_speedup_rows_le20\\"] >= 1.30\\n    res[\\"step_saving_ms_12_layers\\"] = {r: round(12 * (pt[r][\\"triton_us\\"] - pt[r][\\"splitk_us\\"]) / 1000, 3)\\n                                       for r in ROWS}\\n    if not quick:\\n        default_total = sum(pt[r][\\"splitk_us\\"] for r in SUGGEST_ROWS)\\n        best = None\\n        for c in sweep:\\n            ts = [pt[r][\\"sweep\\"][str(list(c))][\\"us\\"] for r in SUGGEST_ROWS]\\n            if None in ts:\\n                continue\\n            if best is None or sum(ts) < best[0]:\\n                best = (sum(ts), c)\\n        sug = {}\\n        if best is not None and best[0] < 0.97 * default_total:\\n            c = best[1]\\n            d = res[\\"default_cfg\\"]\\n            if c[0] != d[\\"block_n\\"]:\\n                sug[\\"SGLANG_QSA_SPLITK_BLOCK_N\\"] = str(c[0])\\n            if c[1] != d[\\"num_stages\\"]:\\n                sug[\\"SGLANG_QSA_SPLITK_STAGES\\"] = str(c[1])\\n            if c[2] != d[\\"num_warps\\"]:\\n                sug[\\"SGLANG_QSA_SPLITK_WARPS\\"] = str(c[2])\\n            sug[\\"SGLANG_QSA_SPLITK_CTAS\\"] = str(c[3])\\n        res[\\"suggest_env\\"] = sug\\n        res[\\"suggest_total_us_rows_1_13_19_20\\"] = {\\"default\\": round(default_total, 2),\\n                                                   \\"best\\": round(best[0], 2) if best else None,\\n                                                   \\"best_cfg\\": list(best[1]) if best else None}\\n        res[\\"suggest_min_speedup_rows_le20\\"] = (\\n            round(min(pt[r][\\"triton_us\\"] / pt[r][\\"sweep\\"][str(list(best[1]))][\\"us\\"] for r in SUGGEST_ROWS), 3)\\n            if best else None)\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main())\\n", "bench_topklogits.py": "#!/usr/bin/env python3\\n\\"\\"\\"Phase-0 GPU correctness check + microbenchmark for colab/perf_patches/topklogits_patch.py (key \\"topklogits\\").\\n\\nTests the exact code the patch injects: PURE_SRC / GLUE_SRC / LP_HELPER are exec\'d from topklogits_patch.py (with a\\n<site-packages dir> it also checks that the installed sampler.py / logits_processor.py carry that same text). The\\nreference is the unpatched path: full-vocab fp32 logits -> div_(T) -> softmax -> the stock sampler (flashinfer\\ntop_k_top_p_sampling_from_probs(filter_apply_order=\\"joint\\"), or sglang\'s pytorch fallback if flashinfer is missing).\\n\\nSingle GPU, TP sharding simulated (world 4, V 248320, W 62080, plus a padded layout vocab < world*W whose pad\\ncolumns hold junk logits larger than any real one):\\n  * carrier: _tkx_shard_carrier writes the exact fp32 cast of the bf16 shard into the shared logits buffer;\\n  * greedy: tkx_greedy_* == torch.argmax over the full fp32 row, bit-exact, with planted cross-shard / in-shard ties;\\n  * weights: tkx_sample_weights vs the stock rule evaluated on the full vocab with the stock numerics (fp32\\n    softmax(logits/T); \\"joint\\" = strictly-greater count < top_k and strictly-greater mass < top_p; \\"sorted\\" = the\\n    pytorch backend\'s position rule): allowed sets equal, max_abs_err of the allowed probabilities, logsumexp\\n    error. Rows whose top-p cut lands within 2e-6 of a cumulative mass are counted as \\"boundary\\" and skipped.\\n    An adversarial row family (~300 near-equal tokens -> many exact bf16 ties) is reported separately (not gated);\\n  * distribution: N draws per case from tkx_sample_merge (exponential race) and N draws from the stock sampler on\\n    the full-vocab probs, each vs the exact reference distribution (total variation under a 2.5x-noise threshold,\\n    zero draws outside the allowed set). This also confirms the installed flashinfer implements the \\"joint\\" rule;\\n  * glue: topk_logits_exchange_sample with a simulated 4-rank group, for every rank: compact ids allowed, greedy\\n    exact, every ineligibility reason falls back with bit-identical stock full-vocab logits, full-width (prefill)\\n    logits untouched.\\nMulti-GPU (>= 2 visible GPUs, world = min(4, n), real NCCL, the same glue on every rank):\\n  * fallback logits bit-identical to the stock all-gather; compact ids identical on all ranks and allowed; greedy\\n    exact; CUDA RNG state identical on all ranks afterwards (the ranks\' generators stay in lockstep);\\n  * timing per B: stock = all-gather [B, W] bf16 + movedim + [:, :vocab] + fp32 copy into the logits buffer (eager,\\n    and inside a CUDA graph as the decode graph runs it) + stock sampling (div_, softmax, copy, sampler);\\n    patched = shard carrier copy + topk_logits_exchange_sample (eager, as the server runs it). GPU time only:\\n    every sample is enqueued behind a ~4 ms torch.cuda._sleep so CPU launch latency is hidden, as it is behind\\n    the decode graph replay in the server. Raw all-gather times (full logits vs payload) are reported too.\\nJSON (one line on stdout): baseline_us / patched_us = per-step GPU us at B=19 for the first K (multi-GPU numbers when\\navailable, else the 1-GPU local parts without NCCL); max_abs_err = max allowed-probability error, non-adversarial.\\nusage: /tmp/sgl-intel/venv/bin/python bench_topklogits.py [<site-packages dir>] [--k 32,64] [--bs 1,13,19,20]\\n         [--samples 40000] [--iters 40] [--world 4] [--no-dist] [--quick] [--device cuda|cpu] [--vocab N]\\n         [--timeout 900]\\n\\"\\"\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nimport logging\\nimport math\\nimport os\\nimport signal\\nimport socket\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\nimport torch\\n\\nKEY = \\"topklogits\\"\\nHERE = Path(__file__).resolve().parent\\nV_REAL = 248320\\nKINDS = {  # head tokens, head logit from, to, jitter, head only in shard 0; background N(0, 1.5^2); bf16-rounded\\n    \\"peaked\\": (64, 16.0, 4.0, 0.25, False),\\n    \\"medium\\": (64, 9.0, 3.0, 0.25, False),\\n    \\"long\\": (400, 7.0, 3.5, 0.25, False),  # head inside the background tail: mass spread over thousands of tokens\\n    \\"flat\\": (300, 8.0, 8.0, 0.3, False),  # ~10 distinct bf16 values -> ~30-way exact ties in the head\\n    \\"flat1\\": (300, 8.0, 8.0, 0.3, True),  # same, all in shard 0: tie groups straddle that shard\'s top-K cut\\n}\\nCONFIGS = [  # (temperature, top_k, top_p); \\"K\\" = the exchange K\\n    (1.0, 20, 0.95), (1.0, 20, 1.0), (0.7, 20, 0.95), (1.3, 20, 0.8),\\n    (1.0, 1, 1.0), (1.0, 5, 0.9), (1.0, \\"K\\", 0.95), (0.5, 10, 0.99),\\n]\\nSTAT_CASES = [(\\"peaked\\", 0), (\\"medium\\", 0), (\\"long\\", 3), (\\"medium\\", 5), (\\"long\\", 6)]  # (kind, config index)\\n\\n\\ndef log(*a):\\n    print(\\"[bench_topklogits]\\", *a, file=sys.stderr, flush=True)\\n\\n\\n# ---------------------------------------------------------------------------------------------- shipped code\\ndef load_patch():\\n    spec = importlib.util.spec_from_file_location(\\"topklogits_patch\\", HERE / \\"topklogits_patch.py\\")\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef build_ns(patch):\\n    ns = {\\"torch\\": torch}\\n    exec(compile(patch.PURE_SRC, \\"topklogits_patch:PURE_SRC\\", \\"exec\\"), ns)\\n    exec(compile(patch.GLUE_SRC, \\"topklogits_patch:GLUE_SRC\\", \\"exec\\"), ns)\\n    lp = {\\"torch\\": torch, \\"logger\\": logging.getLogger(\\"bench_topklogits.lp\\")}\\n    exec(compile(patch.LP_HELPER, \\"topklogits_patch:LP_HELPER\\", \\"exec\\"), lp)\\n    ns[\\"_tkx_shard_carrier\\"] = lp[\\"_tkx_shard_carrier\\"]\\n    ns[\\"_LP_STATE\\"] = lp[\\"_TKX_STATE\\"]\\n    return ns\\n\\n\\nclass _FakeSampler:  # stands in for sglang\'s Sampler class (type check in the glue)\\n    rl_on_policy_target = None\\n    use_ascend_backend = False\\n\\n    def _sync_token_ids_across_tp(self, ids, si):\\n        pass\\n\\n\\ndef install_stubs(ns, group, k, backend):\\n    ns[\\"Sampler\\"] = _FakeSampler\\n    ns[\\"get_exec\\"] = lambda: SimpleNamespace(kernel=SimpleNamespace(sampling_backend=backend))\\n    ns[\\"get_tp_group\\"] = lambda: group\\n    ns[\\"sanitize_nan_logits\\"] = lambda x, msg=\\"\\": None\\n    ns[\\"logger\\"] = logging.getLogger(\\"bench_topklogits.glue\\")\\n    ns[\\"_tkx_lp\\"] = lambda: SimpleNamespace(TKX_K=k, _TKX_STATE=ns[\\"_LP_STATE\\"])\\n    return SimpleNamespace(sampler=_FakeSampler())  # fake model_runner\\n\\n\\ndef fake_batch(T, topk, topp, greedy, return_logprob=False, **over):\\n    si = SimpleNamespace(\\n        temperatures=T, top_ks=topk, top_ps=topp, is_all_greedy=greedy, need_min_p_sampling=False, grammars=None,\\n        logit_bias=None, has_custom_logit_processor=False, sampling_seed=None, acc_additive_penalties=None,\\n        acc_scaling_penalties=None, penalizer_orchestrator=None, return_sampling_masks=None,\\n        tkx_top_ks_cpu=[int(v) for v in topk.tolist()], grammar_mask=\\"stale\\",\\n    )\\n    for name, val in over.items():\\n        setattr(si, name, val)\\n    return SimpleNamespace(sampling_info=si, return_logprob=return_logprob,\\n                           forward_mode=SimpleNamespace(is_decode=lambda: True))\\n\\n\\ndef fallback_overrides(dev, rows, k):\\n    t = torch.zeros(rows, 1, device=dev)\\n    return [\\n        (\\"logprobs\\", dict(return_logprob=True)),\\n        (\\"top_k > K\\", dict(tkx_top_ks_cpu=[k + 1] + [1] * (rows - 1))),\\n        (\\"top_k unset\\", dict(tkx_top_ks_cpu=[1 << 30] * rows)),\\n        (\\"no list\\", dict(tkx_top_ks_cpu=None)),\\n        (\\"list length\\", dict(tkx_top_ks_cpu=[1] * (rows + 1))),\\n        (\\"min_p\\", dict(need_min_p_sampling=True)),\\n        (\\"grammar\\", dict(grammars=[object()] * rows)),\\n        (\\"logit_bias\\", dict(logit_bias=t)),\\n        (\\"custom lp\\", dict(has_custom_logit_processor=True)),\\n        (\\"seed\\", dict(sampling_seed=t)),\\n        (\\"additive penalty\\", dict(acc_additive_penalties=t)),\\n        (\\"scaling penalty\\", dict(acc_scaling_penalties=t)),\\n        (\\"orchestrator\\", dict(penalizer_orchestrator=SimpleNamespace(is_required=True))),\\n        (\\"sampling mask\\", dict(return_sampling_masks=[True] + [False] * (rows - 1))),\\n    ]\\n\\n\\n# ---------------------------------------------------------------------------------------------- data + refs\\ndef padded_vocab(vocab, world):\\n    q = world * 64\\n    return (vocab + q - 1) // q * q\\n\\n\\ndef make_logits(kinds, Vp, vocab, seed, dev):\\n    \\"\\"\\"fp32 values that are exactly bf16 (as the lm_head emits them); pad columns >= vocab hold junk (99).\\"\\"\\"\\n    g = torch.Generator().manual_seed(seed)\\n    out = torch.empty(len(kinds), Vp)\\n    for i, kind in enumerate(kinds):\\n        n, hi, lo, jit, shard0 = KINDS[kind]\\n        x = torch.randn(Vp, generator=g) * 1.5\\n        idx = torch.randperm(Vp // 4 if shard0 else vocab, generator=g)[:n]\\n        x[idx] = torch.linspace(hi, lo, n) + jit * torch.randn(n, generator=g)\\n        out[i] = x\\n    out = out.to(torch.bfloat16).float()\\n    if Vp > vocab:\\n        out[:, vocab:] = 99.0\\n    return out.to(dev)\\n\\n\\ndef row_params(n_rows, k, dev, cfg_idx=None):\\n    cfgs = [CONFIGS[(i if cfg_idx is None else cfg_idx) % len(CONFIGS)] for i in range(n_rows)]\\n    T = torch.tensor([[c[0]] for c in cfgs], dtype=torch.float32, device=dev)\\n    tk = torch.tensor([k if c[1] == \\"K\\" else c[1] for c in cfgs], dtype=torch.int32, device=dev)\\n    tp = torch.tensor([c[2] for c in cfgs], dtype=torch.float32, device=dev)\\n    return T, tk, tp\\n\\n\\ndef ref_rule(xv, T, topk, topp, joint, depth=2048):\\n    \\"\\"\\"Stock numerics on the full vocab: p = softmax(fp32 logits / T); allowed set by the stock filter rule.\\n    Returns (dense allowed probs [B, vocab], boundary rows [B] bool).\\"\\"\\"\\n    probs = torch.softmax(xv / T, dim=-1)\\n    depth = min(depth, probs.shape[-1])\\n    ps, idx = torch.topk(probs, depth, dim=-1, sorted=True)\\n    pos = torch.arange(depth, device=ps.device).view(1, -1).expand_as(ps)\\n    excl = torch.cumsum(ps.double(), dim=-1) - ps.double()\\n    tpd = topp.double().view(-1, 1)\\n    if joint:\\n        new = torch.ones_like(ps, dtype=torch.bool)\\n        new[:, 1:] = ps[:, 1:] != ps[:, :-1]\\n        first = torch.where(new, pos, torch.zeros_like(pos)).cummax(dim=-1).values  # start of the tie group\\n        n_gt, m_gt = first, excl.gather(1, first)\\n    else:\\n        n_gt, m_gt = pos, excl\\n    in_k = n_gt < topk.view(-1, 1)\\n    allow = in_k & (m_gt < tpd) if joint else in_k & ~(m_gt > tpd)\\n    boundary = (in_k & ((m_gt - tpd).abs() < 2e-6)).any(dim=-1)\\n    dense = torch.zeros_like(probs).scatter_(1, idx, torch.where(allow, ps, torch.zeros_like(ps)))\\n    return dense, boundary\\n\\n\\ndef sim_gather(ns, x, T, k, world, W, vocab, greedy=False):\\n    \\"\\"\\"The tensor every rank receives from the payload all-gather, built on one device ([world, B, P]).\\"\\"\\"\\n    pays = []\\n    for r in range(world):\\n        off = r * W\\n        valid = max(0, min(W, vocab - off))\\n        shard = x[:, off:off + W]\\n        if greedy:\\n            pays.append(ns[\\"tkx_greedy_payload\\"](shard, off, valid))\\n        else:\\n            pays.append(ns[\\"tkx_sample_payload\\"](shard, T, k, world * k, off, valid))\\n    return torch.stack(pays)\\n\\n\\ndef get_stock_sampler():\\n    \\"\\"\\"The unpatched sampler call: flashinfer joint top-k/top-p, else sglang\'s pytorch implementation.\\"\\"\\"\\n    try:\\n        from flashinfer.sampling import top_k_top_p_sampling_from_probs as fi\\n\\n        def flashinfer_sample(probs, tk, tp):\\n            return fi(probs.contiguous(), tk, tp, filter_apply_order=\\"joint\\")\\n\\n        return \\"flashinfer\\", True, flashinfer_sample\\n    except Exception as exc:  # noqa: BLE001\\n        log(\\"flashinfer unavailable (%s): stock reference = sglang pytorch sampler\\" % type(exc).__name__)\\n\\n        def torch_sample(probs, top_ks, top_ps):  # sampler.py top_k_top_p_min_p_sampling_from_probs_torch, no min_p\\n            probs_sort, probs_idx = probs.sort(dim=-1, descending=True)\\n            probs_sum = torch.cumsum(probs_sort, dim=-1)\\n            probs_sort[torch.arange(0, probs.shape[-1], device=probs.device).view(1, -1) >= top_ks.view(-1, 1)] = 0.0\\n            probs_sort[(probs_sum - probs_sort) > top_ps.view(-1, 1)] = 0.0\\n            sampled = torch.multinomial(probs_sort, num_samples=1)\\n            return torch.gather(probs_idx.to(torch.int32), dim=1, index=sampled).view(-1)\\n\\n        return \\"pytorch\\", False, torch_sample\\n\\n\\n# ---------------------------------------------------------------------------------------------- timing\\ndef gpu_us(fn, iters, pre=None, barrier=None, sleep_cycles=8_000_000):\\n    \\"\\"\\"Median GPU time of fn() in us, with fn enqueued behind a ~4 ms GPU sleep (CPU launch latency hidden).\\"\\"\\"\\n    ts = []\\n    for i in range(iters + 3):\\n        if pre is not None:\\n            pre()\\n        torch.cuda.synchronize()\\n        if barrier is not None:\\n            barrier()\\n            torch.cuda.synchronize()\\n        torch.cuda._sleep(sleep_cycles)\\n        e0 = torch.cuda.Event(enable_timing=True)\\n        e1 = torch.cuda.Event(enable_timing=True)\\n        e0.record()\\n        fn()\\n        e1.record()\\n        torch.cuda.synchronize()\\n        if i >= 3:\\n            ts.append(e0.elapsed_time(e1) * 1000.0)\\n    ts.sort()\\n    return ts[len(ts) // 2]\\n\\n\\ndef graph_us(fn, iters, barrier=None):\\n    s = torch.cuda.Stream()\\n    s.wait_stream(torch.cuda.current_stream())\\n    with torch.cuda.stream(s):\\n        for _ in range(3):\\n            fn()\\n    torch.cuda.current_stream().wait_stream(s)\\n    torch.cuda.synchronize()\\n    g = torch.cuda.CUDAGraph()\\n    with torch.cuda.graph(g):\\n        fn()\\n    torch.cuda.synchronize()\\n    return gpu_us(g.replay, iters, barrier=barrier)\\n\\n\\n# ---------------------------------------------------------------------------------------------- 1-device tests\\ndef test_carrier(ns, dev, W, vocab):\\n    lo = torch.randn(5, W, device=dev).to(torch.bfloat16)\\n    buf = torch.full((7, vocab), float(\\"nan\\"), device=dev)\\n    out = ns[\\"_tkx_shard_carrier\\"](lo, buf[:5], vocab)\\n    st = ns[\\"_LP_STATE\\"]\\n    ok = (out.data_ptr() == buf.data_ptr() and out.shape == (5, W) and torch.equal(out, lo.float())\\n          and st.get(\\"W\\") == W and st.get(\\"vocab\\") == vocab and st.get(\\"dtype\\") == torch.bfloat16)\\n    out2 = ns[\\"_tkx_shard_carrier\\"](lo, None, vocab)\\n    out3 = ns[\\"_tkx_shard_carrier\\"](lo, buf[:4], vocab)  # row mismatch -> no buffer\\n    ok = ok and torch.equal(out2, lo.float()) and torch.equal(out3, lo.float()) and out3.data_ptr() != buf.data_ptr()\\n    return bool(ok)\\n\\n\\ndef test_greedy(ns, dev, world, W, Vp, vocab):\\n    kinds = [\\"peaked\\", \\"medium\\", \\"long\\", \\"flat\\"] * 4\\n    x = make_logits(kinds, Vp, vocab, seed=11, dev=dev)\\n    top = x[:, :vocab].max(dim=-1).values\\n    a, b = 1 * W + 17, min(3, world - 1) * W + 5  # cross-shard tie: expect the lower column\\n    x[0, a] = x[0, b] = top[0] + 1.0\\n    c, d = (world - 2) * W + 100, (world - 2) * W + 40  # in-shard tie: expect the lower column (d)\\n    x[1, c] = x[1, d] = top[1] + 1.0\\n    x[2, 0] = x[2, vocab - 1] = top[2] + 2.0  # first and last real columns\\n    x[3, :vocab] = 0.5  # all equal -> column 0\\n    ids = ns[\\"tkx_greedy_merge\\"](sim_gather(ns, x, None, 1, world, W, vocab, greedy=True))\\n    ref = torch.argmax(x[:, :vocab], dim=-1)\\n    return bool(torch.equal(ids.cpu(), ref.cpu()) and ids.dtype == torch.int64)\\n\\n\\ndef value_classes(xrow):\\n    \\"\\"\\"Class id per token: tokens with the same logit are interchangeable under every stock rule (the pytorch\\n    backend\'s position rule breaks ties in arbitrary sort order, flashinfer keeps or drops a tie group whole).\\"\\"\\"\\n    return torch.unique(xrow, return_inverse=True)[1]\\n\\n\\ndef allowed_by_value(xv, ref, ids):\\n    \\"\\"\\"[B] bool: ids[b] has the logit of some allowed token of row b.\\"\\"\\"\\n    out = []\\n    for b in range(xv.shape[0]):\\n        out.append(bool(torch.isin(xv[b, ids[b].long()].view(1), xv[b][ref[b] > 0]).item()))\\n    return torch.tensor(out, device=xv.device)\\n\\n\\ndef test_weights(ns, dev, k, world, W, Vp, vocab, joint, kinds, seed):\\n    x = make_logits(kinds, Vp, vocab, seed=seed, dev=dev)\\n    xv = x[:, :vocab]\\n    T, tk, tp = row_params(len(kinds), k, dev)\\n    g = sim_gather(ns, x, T, k, world, W, vocab)\\n    ids, w, inexact = ns[\\"tkx_sample_weights\\"](g, T, tk, tp, k, joint)\\n    dense = torch.zeros(len(kinds), vocab, device=dev).scatter_add_(1, ids.long(), w)\\n    ref, boundary = ref_rule(xv, T, tk, tp, joint)\\n    p64 = torch.softmax(xv.double() / T.double(), dim=-1)  # truth, to compare patched vs stock accuracy\\n    sel = ref > 0\\n    if joint:  # allowed sets must be identical (ties are kept or dropped whole)\\n        mism = (((dense > 0) != sel) & (torch.maximum(dense, ref) > 1e-6)).any(dim=-1)\\n        err = (dense - ref).abs().amax(dim=-1)\\n        err64_p = torch.where(sel, (dense.double() - p64).abs(), torch.zeros_like(p64)).amax(dim=-1)\\n        err64_s = torch.where(sel, (ref.double() - p64).abs(), torch.zeros_like(p64)).amax(dim=-1)\\n    else:  # position rule: tie order is arbitrary in both, compare the allowed count and mass\\n        cnt = lambda t: (t > 1e-30).sum(dim=-1)  # noqa: E731\\n        err = (dense.sum(-1) - ref.sum(-1)).abs()\\n        mism = (err > 5e-5) | (cnt(dense) != cnt(ref))\\n        m64 = torch.where(sel, p64, torch.zeros_like(p64)).sum(-1)\\n        err64_p, err64_s = (dense.double().sum(-1) - m64).abs(), (ref.double().sum(-1) - m64).abs()\\n    lse = torch.logsumexp(g[:, :, 2 * k], dim=0)\\n    lse_ref = torch.logsumexp(xv / T, dim=-1)\\n    keep = ~boundary\\n    mx = lambda t: float(t[keep].max()) if bool(keep.any()) else 0.0  # noqa: E731\\n    return {\\n        \\"rows\\": len(kinds),\\n        \\"boundary_rows\\": int(boundary.sum()),\\n        \\"mismatch_rows\\": int((mism & keep).sum()),\\n        \\"inexact_flagged_rows\\": int(inexact.sum()),\\n        \\"unflagged_mismatch_rows\\": int((mism & keep & ~inexact).sum()),\\n        \\"max_abs_err\\": mx(err),\\n        \\"err_vs_fp64_patched\\": mx(err64_p),\\n        \\"err_vs_fp64_stock\\": mx(err64_s),\\n        \\"lse_max_abs_err\\": float((lse - lse_ref).abs().max()),\\n    }\\n\\n\\ndef tv_case(counts, r, n):\\n    emp = counts / n\\n    tv = 0.5 * float((emp - r).abs().sum())\\n    expect = 0.5 * float(torch.sqrt(2.0 * r * (1.0 - r) / (math.pi * n)).sum())\\n    viol = int(counts[r <= 0].sum())\\n    return tv, 2.5 * expect + 0.003, viol\\n\\n\\ndef test_distribution(ns, dev, k, world, W, Vp, vocab, n_samples, joint, stock):\\n    R = 1000 if dev.type == \\"cuda\\" else 250\\n    reps = max(1, n_samples // R)\\n    n = reps * R\\n    out = []\\n    for ci, (kind, cfg) in enumerate(STAT_CASES):\\n        x = make_logits([kind], Vp, vocab, seed=100 + ci, dev=dev)\\n        T, tk, tp = row_params(1, k, dev, cfg_idx=cfg)\\n        ref, boundary = ref_rule(x[:, :vocab], T, tk, tp, joint)\\n        cls = value_classes(x[0, :vocab]).cpu()\\n        ncls = int(cls.max()) + 1\\n        by_cls = lambda t: torch.zeros(ncls, dtype=torch.float64).index_add_(0, cls, t)  # noqa: E731\\n        r = by_cls((ref[0] / ref[0].sum()).double().cpu())\\n        Xr, Tr, tkr, tpr = x.expand(R, -1), T.repeat(R, 1), tk.repeat(R), tp.repeat(R)\\n        cc = torch.zeros(vocab, dtype=torch.float64)\\n        for _ in range(reps):\\n            ids, _inexact = ns[\\"tkx_sample_merge\\"](sim_gather(ns, Xr, Tr, k, world, W, vocab), Tr, tkr, tpr, k, joint)\\n            cc += torch.bincount(ids.long().cpu(), minlength=vocab).double()\\n        probs = torch.softmax(x[:, :vocab] / T, dim=-1).expand(R, -1).contiguous()\\n        cs = torch.zeros(vocab, dtype=torch.float64)\\n        for _ in range(reps):\\n            ids = stock(probs, tkr, tpr)\\n            cs += torch.bincount(ids.long().cpu(), minlength=vocab).double()\\n        cc, cs = by_cls(cc), by_cls(cs)\\n        tv_c, thr, viol_c = tv_case(cc, r, n)\\n        tv_s, _, viol_s = tv_case(cs, r, n)\\n        out.append({\\"case\\": f\\"{kind}/T{CONFIGS[cfg][0]}/k{tk.item()}/p{CONFIGS[cfg][2]}\\",\\n                    \\"allowed\\": int((ref[0] > 0).sum()), \\"boundary\\": bool(boundary[0]), \\"tv_patched\\": round(tv_c, 5),\\n                    \\"tv_stock\\": round(tv_s, 5), \\"tv_thr\\": round(thr, 5),\\n                    \\"tv_patched_vs_stock\\": round(0.5 * float((cc / n - cs / n).abs().sum()), 5),\\n                    \\"viol_patched\\": viol_c, \\"viol_stock\\": viol_s})\\n    return n, out\\n\\n\\nclass SimGroup:\\n    \\"\\"\\"A TP group simulated on one device: this rank\'s payload / shard + the other ranks\' ones computed with the\\n    same shipped functions from their own shards.\\"\\"\\"\\n\\n    def __init__(self, ns, carriers, rank, W, vocab, T):\\n        self.ns, self.carriers, self.rank_in_group, self.world_size = ns, carriers, rank, len(carriers)\\n        self.W, self.vocab, self.T = W, vocab, T\\n\\n    def all_gather_into_tensor(self, out, inp):\\n        world, rows, P = self.world_size, inp.shape[0], inp.shape[1]\\n        parts = []\\n        for q in range(world):\\n            if q == self.rank_in_group:\\n                parts.append(inp)\\n                continue\\n            off = q * self.W\\n            valid = max(0, min(self.W, self.vocab - off))\\n            if P == 2:\\n                parts.append(self.ns[\\"tkx_greedy_payload\\"](self.carriers[q], off, valid))\\n            else:\\n                k = (P - 1) // (2 + world)\\n                parts.append(self.ns[\\"tkx_sample_payload\\"](self.carriers[q], self.T, k, world * k, off, valid))\\n        out.copy_(torch.cat(parts, dim=0))\\n\\n    def all_gather(self, x, dim=-1):\\n        parts = [x if q == self.rank_in_group else self.carriers[q].to(x.dtype) for q in range(self.world_size)]\\n        return torch.cat(parts, dim=dim)\\n\\n\\ndef test_glue_sim(ns, dev, k, world, W, Vp, vocab, backend):\\n    rows = 19\\n    kinds = ([\\"peaked\\", \\"medium\\", \\"long\\"] * 7)[:rows]\\n    x = make_logits(kinds, Vp, vocab, seed=21, dev=dev)\\n    T, tk, tp = row_params(rows, k, dev)\\n    joint = backend == \\"flashinfer\\"\\n    ref, boundary = ref_rule(x[:, :vocab], T, tk, tp, joint)\\n    stock_full = x[:, :vocab].contiguous()\\n    res = {\\"compact_allowed\\": True, \\"greedy_exact\\": True, \\"fallback_exact\\": True, \\"prefill_untouched\\": True,\\n           \\"fallbacks\\": 0, \\"grammar_mask_cleared\\": True}\\n    for rank in range(world):\\n        bufs = [torch.zeros(rows, vocab, device=dev) for _ in range(world)]\\n        carriers = [ns[\\"_tkx_shard_carrier\\"](x[:, q * W:(q + 1) * W].to(torch.bfloat16), bufs[q], vocab)\\n                    for q in range(world)]\\n        group = SimGroup(ns, carriers, rank, W, vocab, T)\\n        mr = install_stubs(ns, group, k, backend)\\n        for _ in range(4):\\n            lo = SimpleNamespace(next_token_logits=carriers[rank])\\n            fb = fake_batch(T, tk, tp, greedy=False)\\n            ids = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fb, None)\\n            okr = ids is not None and ids.dtype == torch.int32\\n            if okr:\\n                okr = bool((allowed_by_value(x[:, :vocab], ref, ids) | boundary).all())\\n            res[\\"compact_allowed\\"] &= okr\\n            res[\\"grammar_mask_cleared\\"] &= fb.sampling_info.grammar_mask is None\\n        g1 = torch.ones(rows, dtype=torch.int32, device=dev)\\n        lo = SimpleNamespace(next_token_logits=carriers[rank])\\n        ids = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fake_batch(T, g1, tp, greedy=True), None)\\n        res[\\"greedy_exact\\"] &= ids is not None and torch.equal(ids, torch.argmax(stock_full, dim=-1))\\n        for name, over in fallback_overrides(dev, rows, k) + [(\\"observer\\", {})]:\\n            lo = SimpleNamespace(next_token_logits=carriers[rank])\\n            observer = SimpleNamespace(is_active=lambda si: True) if name == \\"observer\\" else None\\n            fb = fake_batch(T, tk, tp, greedy=False, **over)\\n            r = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fb, observer)\\n            good = r is None and lo.next_token_logits.shape == (rows, vocab) and torch.equal(lo.next_token_logits,\\n                                                                                            stock_full)\\n            if not good:\\n                log(f\\"glue fallback check failed: rank {rank} case {name}\\")\\n            res[\\"fallback_exact\\"] &= bool(good)\\n            res[\\"fallbacks\\"] += int(r is None)\\n        full = SimpleNamespace(next_token_logits=stock_full.clone())\\n        r = ns[\\"topk_logits_exchange_sample\\"](mr, full, fake_batch(T, tk, tp, greedy=False), None)\\n        res[\\"prefill_untouched\\"] &= r is None and torch.equal(full.next_token_logits, stock_full)\\n    res[\\"ok\\"] = all(res[k_] for k_ in (\\"compact_allowed\\", \\"greedy_exact\\", \\"fallback_exact\\", \\"prefill_untouched\\",\\n                                       \\"grammar_mask_cleared\\"))\\n    return res\\n\\n\\ndef local_timing(ns, dev, k, world, W, Vp, vocab, Bs, iters, stock, joint):\\n    out = {}\\n    for B in Bs:\\n        kinds = ([\\"peaked\\", \\"medium\\", \\"long\\"] * 8)[:B]\\n        x = make_logits(kinds, Vp, vocab, seed=31, dev=dev)\\n        T, tk, tp = row_params(B, k, dev)\\n        shard = x[:, :W].to(torch.bfloat16).contiguous()\\n        ag_out = torch.cat([x[:, q * W:(q + 1) * W] for q in range(world)], 0).to(torch.bfloat16)  # NCCL output\\n        buf = torch.zeros(B, vocab, device=dev)\\n\\n        def stock_local():\\n            full = ag_out.reshape(world, B, W).movedim(0, 1).reshape(B, world * W)\\n            buf.copy_(full[:, :vocab])\\n            buf.div_(T)\\n            buf[:] = torch.softmax(buf, dim=-1)\\n            stock(buf, tk, tp)\\n\\n        g_sim = sim_gather(ns, x, T, k, world, W, vocab)\\n\\n        def tkx_local():\\n            c = ns[\\"_tkx_shard_carrier\\"](shard, buf, vocab)\\n            pay = ns[\\"tkx_sample_payload\\"](c, T, k, world * k, 0, min(W, vocab))\\n            g_sim[0].copy_(pay)\\n            ns[\\"tkx_sample_merge\\"](g_sim, T, tk, tp, k, joint)\\n\\n        out[B] = {\\"stock_local_us\\": round(gpu_us(stock_local, iters), 1),\\n                  \\"patched_local_us\\": round(gpu_us(tkx_local, iters), 1)}\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------------------------- multi-GPU\\nclass DistGroup:\\n    \\"\\"\\"Real NCCL; all_gather replicates sglang GroupCoordinator.all_gather (parallel_state.py:1318-1351).\\"\\"\\"\\n\\n    def __init__(self, world, rank):\\n        self.world_size, self.rank_in_group = world, rank\\n\\n    def all_gather_into_tensor(self, out, inp):\\n        import torch.distributed as dist\\n\\n        dist.all_gather_into_tensor(out, inp)\\n\\n    def all_gather(self, x, dim=-1):\\n        import torch.distributed as dist\\n\\n        if dim < 0:\\n            dim += x.dim()\\n        size = x.size()\\n        out = torch.empty((size[0] * self.world_size,) + size[1:], dtype=x.dtype, device=x.device)\\n        dist.all_gather_into_tensor(out, x)\\n        out = out.reshape((self.world_size,) + size).movedim(0, dim)\\n        return out.reshape(size[:dim] + (self.world_size * size[dim],) + size[dim + 1:])\\n\\n\\ndef _die_with_parent(parent_pid):\\n    \\"\\"\\"Linux: SIGKILL this rank when the bench\'s main process dies (watchdog os._exit, an outer `timeout` SIGTERM /\\n    SIGKILL), so no rank can outlive the bench spinning in an NCCL kernel on a GPU the server is about to use.\\"\\"\\"\\n    try:\\n        import ctypes\\n        import signal\\n\\n        ctypes.CDLL(\\"libc.so.6\\", use_errno=True).prctl(1, int(signal.SIGKILL), 0, 0, 0)  # PR_SET_PDEATHSIG\\n    except Exception:  # noqa: BLE001  (non-Linux: best effort only)\\n        return\\n    if parent_pid is not None and os.getppid() != parent_pid:  # parent already gone before prctl took effect\\n        os._exit(5)\\n\\n\\ndef kill_children(grace=10.0):\\n    \\"\\"\\"Terminate, then SIGKILL, every child process of this one (the spawned ranks) and reap them.\\"\\"\\"\\n    import multiprocessing\\n\\n    try:\\n        kids = multiprocessing.active_children()\\n    except Exception:  # noqa: BLE001\\n        return\\n    for p in kids:\\n        try:\\n            p.terminate()\\n        except Exception:  # noqa: BLE001\\n            pass\\n    t_end = time.time() + grace\\n    for p in kids:\\n        try:\\n            p.join(max(0.1, t_end - time.time()))\\n        except Exception:  # noqa: BLE001\\n            pass\\n    for p in kids:\\n        try:\\n            if p.is_alive():\\n                p.kill()\\n                p.join(5)\\n        except Exception:  # noqa: BLE001\\n            pass\\n\\n\\ndef dist_worker(rank, world, port, out_path, ks, Bs, iters, vocab, cpu=False, parent_pid=None):\\n    \\"\\"\\"cpu=True: gloo on CPU processes, correctness only (a local logic test of this worker, no timing).\\"\\"\\"\\n    _die_with_parent(parent_pid)\\n    import torch.distributed as dist\\n\\n    logging.basicConfig(level=logging.ERROR)\\n    if cpu:\\n        dev = torch.device(\\"cpu\\")\\n        torch.set_num_threads(1)\\n    else:\\n        torch.cuda.set_device(rank)\\n        dev = torch.device(\\"cuda\\", rank)\\n    dist.init_process_group(\\"gloo\\" if cpu else \\"nccl\\", init_method=f\\"tcp://127.0.0.1:{port}\\", rank=rank,\\n                            world_size=world)\\n    torch.manual_seed(1234)\\n    res = {\\"world\\": world, \\"errors\\": [], \\"backend\\": \\"gloo/cpu\\" if cpu else \\"nccl\\"}\\n\\n    def save():\\n        if rank == 0:\\n            Path(out_path).write_text(json.dumps(res))\\n\\n    def barrier():\\n        dist.barrier()\\n\\n    def rmax(vals):\\n        t = torch.tensor(vals, dtype=torch.float64, device=dev)\\n        dist.all_reduce(t, op=dist.ReduceOp.MAX)\\n        return t.tolist()\\n\\n    def same_on_all(t):\\n        t = t.contiguous()\\n        g = torch.empty((world,) + tuple(t.shape), dtype=t.dtype, device=dev)\\n        dist.all_gather_into_tensor(g.view(-1), t.view(-1))\\n        return all(torch.equal(g[0], g[q]) for q in range(1, world))\\n\\n    try:\\n        patch = load_patch()\\n        ns = build_ns(patch)\\n        stock_name, stock_joint, stock = get_stock_sampler()\\n        backend = \\"flashinfer\\" if stock_joint else \\"pytorch\\"\\n        Vp = padded_vocab(vocab, world)\\n        W = Vp // world\\n        Bmax = max(Bs + [19])\\n        kinds = ([\\"peaked\\", \\"medium\\", \\"long\\"] * 16)[:Bmax]\\n        x = make_logits(kinds, Vp, vocab, seed=41, dev=dev)\\n        stock_full_all = x[:, :vocab].contiguous()\\n        shard_all = x[:, rank * W:(rank + 1) * W].to(torch.bfloat16).contiguous()  # this rank\'s lm_head output\\n        buf_all = torch.zeros(Bmax, vocab, device=dev)  # shared next_token_logits_buffer\\n        group = DistGroup(world, rank)\\n        res.update(stock=stock_name, W=W, padded_vocab=Vp)\\n        # ---------------- correctness at B = 19\\n        B = 19\\n        corr = {}\\n        for k in ks:\\n            mr = install_stubs(ns, group, k, backend)\\n            T, tk, tp = row_params(B, k, dev)\\n            ref, boundary = ref_rule(stock_full_all[:B], T, tk, tp, stock_joint)\\n            buf = buf_all[:B]\\n            sh = shard_all[:B]\\n            c = {}\\n            lo = SimpleNamespace(next_token_logits=ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab))\\n            r = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fake_batch(T, tk, tp, False, return_logprob=True), None)\\n            stock_gather = group.all_gather(sh, dim=-1)[:, :vocab].float()\\n            c[\\"fallback_bit_exact\\"] = bool(r is None and torch.equal(lo.next_token_logits, stock_gather)\\n                                           and torch.equal(stock_gather, stock_full_all[:B]))\\n            allowed, agree = True, True\\n            for _ in range(16):\\n                lo = SimpleNamespace(next_token_logits=ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab))\\n                ids = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fake_batch(T, tk, tp, False), None)\\n                allowed &= bool((allowed_by_value(stock_full_all[:B], ref, ids) | boundary).all())\\n                agree &= same_on_all(ids)\\n            c[\\"compact_allowed\\"], c[\\"ids_identical_on_all_ranks\\"] = allowed, agree\\n            g1 = torch.ones(B, dtype=torch.int32, device=dev)\\n            lo = SimpleNamespace(next_token_logits=ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab))\\n            ids = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fake_batch(T, g1, tp, True), None)\\n            c[\\"greedy_exact\\"] = bool(torch.equal(ids, torch.argmax(stock_full_all[:B], dim=-1)) and same_on_all(ids))\\n            tk_big = tk.clone()\\n            tk_big[0] = k + 1\\n            lo = SimpleNamespace(next_token_logits=ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab))\\n            r = ns[\\"topk_logits_exchange_sample\\"](mr, lo, fake_batch(T, tk_big, tp, False), None)\\n            c[\\"top_k_gt_K_falls_back\\"] = bool(r is None and torch.equal(lo.next_token_logits, stock_full_all[:B]))\\n            st = (torch.get_rng_state() if cpu else torch.cuda.get_rng_state()).to(dev)\\n            c[\\"rng_lockstep\\"] = same_on_all(st)\\n            corr[str(k)] = c\\n        res[\\"correctness\\"] = corr\\n        res[\\"ok\\"] = all(all(v for v in c.values()) for c in corr.values())\\n        save()\\n        if cpu:\\n            return\\n        # ---------------- timing\\n        timing = {}\\n        flat = []\\n        for B in Bs:\\n            T, tk, tp = row_params(B, ks[0], dev)\\n            sh, buf, sfull = shard_all[:B].contiguous(), buf_all[:B], stock_full_all[:B]\\n            P_full = torch.empty(world * B, W, dtype=torch.bfloat16, device=dev)\\n\\n            def stock_ag():\\n                full = group.all_gather(sh, dim=-1)\\n                buf.copy_(full[:, :vocab])\\n\\n            def stock_sample():\\n                buf.div_(T)\\n                buf[:] = torch.softmax(buf, dim=-1)\\n                stock(buf, tk, tp)\\n\\n            row = {\\"stock_allgather_eager_us\\": gpu_us(stock_ag, iters, barrier=barrier),\\n                   \\"stock_sample_us\\": gpu_us(stock_sample, iters, pre=lambda: buf.copy_(sfull), barrier=barrier),\\n                   \\"raw_allgather_full_us\\": gpu_us(lambda: dist.all_gather_into_tensor(P_full, sh), iters,\\n                                                   barrier=barrier)}\\n            for k in ks:\\n                mr = install_stubs(ns, group, k, backend)\\n                T, tk, tp = row_params(B, k, dev)\\n                fb = fake_batch(T, tk, tp, False)\\n                pay = torch.zeros(B, 2 * k + 1 + world * k, device=dev)\\n                P_pay = torch.empty(world * B, pay.shape[1], device=dev)\\n\\n                carrier_t = ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab)\\n\\n                def carrier():\\n                    ns[\\"_tkx_shard_carrier\\"](sh, buf, vocab)\\n\\n                def glue():\\n                    ns[\\"topk_logits_exchange_sample\\"](mr, SimpleNamespace(next_token_logits=carrier_t), fb, None)\\n\\n                row[f\\"K{k}\\"] = {\\"carrier_us\\": gpu_us(carrier, iters, barrier=barrier),\\n                                \\"exchange_sample_us\\": gpu_us(glue, iters, pre=carrier, barrier=barrier),\\n                                \\"raw_allgather_payload_us\\": gpu_us(\\n                                    lambda: dist.all_gather_into_tensor(P_pay, pay), iters, barrier=barrier)}\\n            timing[B] = row\\n        for B in Bs:  # max over ranks\\n            row = timing[B]\\n            keys = [kk for kk in row if not kk.startswith(\\"K\\")]\\n            subs = [(f\\"K{k}\\", kk) for k in ks for kk in sorted(row[f\\"K{k}\\"])]\\n            vals = rmax([row[kk] for kk in keys] + [row[a][b] for a, b in subs])\\n            for kk, v in zip(keys, vals[:len(keys)]):\\n                row[kk] = round(v, 1)\\n            for (a, b), v in zip(subs, vals[len(keys):]):\\n                row[a][b] = round(v, 1)\\n        res[\\"timing\\"] = {str(B): v for B, v in timing.items()}\\n        save()\\n        # ---------------- the stock all-gather inside a CUDA graph (as the decode graph runs it); last, may fail\\n        for B in Bs:\\n            sh, buf = shard_all[:B].contiguous(), buf_all[:B]\\n\\n            def stock_ag_g():\\n                full = group.all_gather(sh, dim=-1)\\n                buf.copy_(full[:, :vocab])\\n\\n            try:\\n                v = rmax([graph_us(stock_ag_g, iters, barrier=barrier)])[0]\\n                res[\\"timing\\"][str(B)][\\"stock_allgather_graph_us\\"] = round(v, 1)\\n            except Exception as exc:  # noqa: BLE001\\n                res[\\"errors\\"].append(f\\"graph capture B={B}: {type(exc).__name__}: {str(exc)[:200]}\\")\\n                break\\n            save()\\n    except Exception as exc:  # noqa: BLE001\\n        res[\\"ok\\"] = False\\n        res[\\"errors\\"].append(f\\"rank {rank}: {type(exc).__name__}: {str(exc)[:300]}\\")\\n        if rank == 0:\\n            res[\\"traceback\\"] = traceback.format_exc()[-2000:]\\n        save()\\n        raise\\n    finally:\\n        save()\\n        try:\\n            dist.destroy_process_group()\\n        except Exception:  # noqa: BLE001\\n            pass\\n\\n\\ndef run_dist(world, ks, Bs, iters, vocab, timeout, cpu=False):\\n    import torch.multiprocessing as mp\\n\\n    with socket.socket() as s:\\n        s.bind((\\"127.0.0.1\\", 0))\\n        port = s.getsockname()[1]\\n    fd, out_path = tempfile.mkstemp(prefix=\\"bench_topklogits_\\", suffix=\\".json\\")\\n    os.close(fd)\\n    Path(out_path).write_text(\\"{}\\")\\n    t0 = time.time()\\n    err = None\\n    try:\\n        ctx = mp.spawn(dist_worker, args=(world, port, out_path, ks, Bs, iters, vocab, cpu, os.getpid()),\\n                       nprocs=world, join=False)\\n        timing_seen = None\\n        while True:\\n            try:\\n                if ctx.join(timeout=5):\\n                    break\\n            except Exception as exc:  # noqa: BLE001\\n                err = f\\"{type(exc).__name__}: {str(exc)[-400:]}\\"\\n                break\\n            if timing_seen is None and \'\\"timing\\"\' in Path(out_path).read_text():\\n                timing_seen = time.time()\\n            graph_stuck = timing_seen is not None and time.time() - timing_seen > 120\\n            if time.time() - t0 > timeout or graph_stuck:\\n                err = \\"CUDA-graph all-gather phase did not finish in 120 s\\" if graph_stuck else f\\"timeout after {timeout}s\\"\\n                break\\n    except Exception as exc:  # noqa: BLE001\\n        err = f\\"spawn: {type(exc).__name__}: {exc}\\"\\n    kill_children()  # never leave a rank alive (a rank stuck in NCCL spins its GPU forever)\\n    try:\\n        res = json.loads(Path(out_path).read_text() or \\"{}\\")\\n    except Exception:  # noqa: BLE001\\n        res = {}\\n    if err:\\n        res.setdefault(\\"errors\\", []).append(err)\\n        if \\"correctness\\" not in res:\\n            res[\\"ok\\"] = False\\n    res[\\"seconds\\"] = round(time.time() - t0, 1)\\n    return res\\n\\n\\n# ---------------------------------------------------------------------------------------------- main\\ndef installed_check(site, patch):\\n    try:\\n        sp = (Path(site) / \\"sglang/srt/layers/sampler.py\\").read_text()\\n        lp = (Path(site) / \\"sglang/srt/layers/logits_processor.py\\").read_text()\\n        return bool(patch.PURE_SRC in sp and patch.GLUE_SRC in sp and patch.LP_HELPER in lp)\\n    except Exception as exc:  # noqa: BLE001\\n        return f\\"unreadable: {type(exc).__name__}\\"\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"site\\", nargs=\\"?\\", default=None)\\n    ap.add_argument(\\"--k\\", default=\\"32,64\\")\\n    ap.add_argument(\\"--bs\\", default=\\"1,13,19,20\\")\\n    ap.add_argument(\\"--samples\\", type=int, default=40000)\\n    ap.add_argument(\\"--iters\\", type=int, default=40)\\n    ap.add_argument(\\"--world\\", type=int, default=4)\\n    ap.add_argument(\\"--no-dist\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--quick\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--device\\", default=\\"cuda\\")\\n    ap.add_argument(\\"--vocab\\", type=int, default=0)\\n    ap.add_argument(\\"--timeout\\", type=int, default=900)\\n    ap.add_argument(\\"--cpu-dist\\", action=\\"store_true\\", help=\\"--device cpu only: run the multi-rank part on gloo\\")\\n    args = ap.parse_args()\\n    ks = [int(v) for v in args.k.split(\\",\\") if v]\\n    Bs = [int(v) for v in args.bs.split(\\",\\") if v]\\n    if args.quick:\\n        args.samples, args.iters = min(args.samples, 10000), min(args.iters, 15)\\n    dev = torch.device(args.device if (args.device != \\"cuda\\" or torch.cuda.is_available()) else \\"cpu\\")\\n    vocab = args.vocab or (V_REAL if dev.type == \\"cuda\\" else V_REAL // 10)\\n    out = {\\"key\\": KEY, \\"ok\\": False, \\"baseline_us\\": None, \\"patched_us\\": None, \\"max_abs_err\\": None,\\n           \\"device\\": str(dev), \\"gpu\\": torch.cuda.get_device_name(0) if dev.type == \\"cuda\\" else None,\\n           \\"torch\\": torch.__version__, \\"vocab\\": vocab, \\"K\\": ks}\\n    done = threading.Event()\\n    real_stdout = os.dup(1)\\n\\n    def watchdog():\\n        if not done.wait(args.timeout + 120):\\n            out[\\"ok\\"] = False\\n            out[\\"errors\\"] = out.get(\\"errors\\", []) + [f\\"watchdog: bench exceeded {args.timeout + 120}s\\"]\\n            kill_children(grace=5.0)\\n            os.write(real_stdout, (json.dumps(out, default=str) + \\"\\\\n\\").encode())\\n            os._exit(3)\\n\\n    def on_sigterm(signum, frame):  # an outer `timeout`: reap the ranks, still print the JSON line\\n        out[\\"ok\\"] = False\\n        out[\\"errors\\"] = out.get(\\"errors\\", []) + [f\\"killed by signal {signum}\\"]\\n        kill_children(grace=5.0)\\n        os.write(real_stdout, (json.dumps(out, default=str) + \\"\\\\n\\").encode())\\n        os._exit(4)\\n\\n    t_start = time.time()\\n    signal.signal(signal.SIGTERM, on_sigterm)\\n    threading.Thread(target=watchdog, daemon=True).start()\\n    os.dup2(2, 1)  # keep stdout clean for the JSON line (library chatter goes to stderr)\\n    try:\\n        patch = load_patch()\\n        ns = build_ns(patch)\\n        if args.site:\\n            out[\\"installed_patch_matches\\"] = installed_check(args.site, patch)\\n        stock_name, joint, stock = get_stock_sampler()\\n        out[\\"stock_sampler\\"] = stock_name\\n        world = 4\\n        Vp = padded_vocab(vocab, world)\\n        W = Vp // world\\n        layouts = {\\"exact\\": (Vp, vocab), \\"padded\\": (Vp, vocab - 1000)}  # padded: last shard has 1000 junk columns\\n        checks = {\\"carrier_exact\\": test_carrier(ns, dev, W, vocab)}\\n        checks[\\"greedy_exact\\"] = all(test_greedy(ns, dev, world, W, vp, vb) for vp, vb in layouts.values())\\n        weights, worst_err, lse_err, mismatch, adv, acc_ok = {}, 0.0, 0.0, 0, {}, True\\n        main_kinds = ([\\"peaked\\", \\"medium\\", \\"long\\"] * 22)[:64]\\n        for k in ks:\\n            for lname, (vp, vb) in layouts.items():\\n                for rule in (\\"joint\\", \\"sorted\\"):\\n                    r = test_weights(ns, dev, k, world, W, vp, vb, rule == \\"joint\\", main_kinds, seed=1 + k)\\n                    weights[f\\"K{k}/{lname}/{rule}\\"] = r\\n                    worst_err = max(worst_err, r[\\"max_abs_err\\"])\\n                    lse_err = max(lse_err, r[\\"lse_max_abs_err\\"])\\n                    mismatch += r[\\"mismatch_rows\\"]\\n                    acc_ok &= r[\\"err_vs_fp64_patched\\"] <= r[\\"err_vs_fp64_stock\\"] + 1e-6\\n                adv[f\\"K{k}/{lname}\\"] = test_weights(ns, dev, k, world, W, vp, vb, True, [\\"flat\\"] * 8 + [\\"flat1\\"] * 8,\\n                                                    seed=5 + k)\\n        log(\\"weights\\", json.dumps(weights))\\n        checks[\\"weights_sets_equal\\"] = mismatch == 0\\n        checks[\\"weights_err_ok\\"] = worst_err < 5e-5 and lse_err < 1e-4 and acc_ok\\n        checks[\\"inexact_flag_sound\\"] = all(r[\\"unflagged_mismatch_rows\\"] == 0\\n                                           for r in list(weights.values()) + list(adv.values()))\\n        out.update(max_abs_err=worst_err, lse_max_abs_err=lse_err, weights=weights, adversarial_ties=adv)\\n        n, dist_cases = test_distribution(ns, dev, ks[0], world, W, Vp, vocab, args.samples, joint, stock)\\n        out[\\"distribution\\"] = {\\"samples_per_case\\": n, \\"cases\\": dist_cases}\\n        checks[\\"patched_distribution_ok\\"] = all(c[\\"tv_patched\\"] <= c[\\"tv_thr\\"] and c[\\"viol_patched\\"] == 0\\n                                                for c in dist_cases if not c[\\"boundary\\"])\\n        checks[\\"stock_matches_rule\\"] = all(c[\\"tv_stock\\"] <= c[\\"tv_thr\\"] and c[\\"viol_stock\\"] == 0\\n                                           for c in dist_cases if not c[\\"boundary\\"])\\n        out[\\"glue_sim\\"] = {f\\"K{k}/{lname}\\": test_glue_sim(ns, dev, k, world, W, vp, vb, \\"flashinfer\\" if joint else \\"pytorch\\")\\n                           for k in ks for lname, (vp, vb) in layouts.items()}\\n        checks[\\"glue_sim_ok\\"] = all(v[\\"ok\\"] for v in out[\\"glue_sim\\"].values())\\n        out[\\"checks\\"] = checks\\n        log(\\"1-device checks\\", json.dumps(checks))\\n        if dev.type == \\"cuda\\":\\n            out[\\"local_timing\\"] = {f\\"K{k}\\": local_timing(ns, dev, k, world, W, Vp, vocab, Bs, args.iters, stock, joint)\\n                                   for k in ks}\\n            ref_b = 19 if 19 in Bs else Bs[-1]\\n            lt = out[\\"local_timing\\"][f\\"K{ks[0]}\\"][ref_b]\\n            out.update(baseline_us=lt[\\"stock_local_us\\"], patched_us=lt[\\"patched_local_us\\"],\\n                       timing_scope=f\\"local 1 GPU, no NCCL, B={ref_b}\\")\\n        ngpu = torch.cuda.device_count() if dev.type == \\"cuda\\" else (args.world if args.cpu_dist else 0)\\n        dist_world = min(args.world, ngpu)\\n        ok = all(checks.values())\\n        if dist_world >= 2 and not args.no_dist:\\n            log(f\\"multi-GPU part: world {dist_world}\\")\\n            ns.clear()\\n            if dev.type == \\"cuda\\":\\n                torch.cuda.empty_cache()\\n            budget = min(args.timeout, t_start + args.timeout + 120 - 45 - time.time())  # finish before the watchdog\\n            d = run_dist(dist_world, ks, Bs, args.iters, vocab, max(60.0, budget), cpu=dev.type == \\"cpu\\")\\n            out[\\"dist\\"] = d\\n            ok = ok and bool(d.get(\\"ok\\"))\\n            tb = d.get(\\"timing\\", {}).get(\\"19\\") or d.get(\\"timing\\", {}).get(str(Bs[-1]))\\n            if tb:\\n                ag = tb.get(\\"stock_allgather_graph_us\\", tb[\\"stock_allgather_eager_us\\"])\\n                kk = tb[f\\"K{ks[0]}\\"]\\n                out.update(baseline_us=round(ag + tb[\\"stock_sample_us\\"], 1),\\n                           patched_us=round(kk[\\"carrier_us\\"] + kk[\\"exchange_sample_us\\"], 1),\\n                           timing_scope=f\\"{dist_world}-GPU NCCL, per step, B=19, K={ks[0]}\\"\\n                                        + (\\"\\" if \\"stock_allgather_graph_us\\" in tb else \\" (stock gather eager)\\"))\\n        out[\\"ok\\"] = bool(ok)\\n    except Exception as exc:  # noqa: BLE001\\n        out[\\"ok\\"] = False\\n        out[\\"errors\\"] = out.get(\\"errors\\", []) + [f\\"{type(exc).__name__}: {str(exc)[:300]}\\"]\\n        out[\\"traceback\\"] = traceback.format_exc()[-2000:]\\n    finally:\\n        done.set()\\n        sys.stdout.flush()\\n        os.dup2(real_stdout, 1)\\n    print(json.dumps(out, default=str), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "densefp8_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently enable Pennyroyal\'s online rowwise-FP8 dense weights on SM89 (L4). Env-gated: nothing changes unless\\nSGLANG_SM120_ONLINE_MXFP8=1 (the stock switch; its name is historical) is set.\\n\\nWhy: decode on 4x L4 is HBM-bound and reads ~3.2 GB/step of bf16 dense weights per GPU (HC mix 1.27, GDN proj 1.04,\\nlm_head 0.32, QSA proj 0.30, shared expert 0.28). Stock qwen4_exp already has an online-FP8 path (HyperConnection mix\\nweights + lm_head as rowwise-FP8 weight-only with a Triton kernel; large projections via MXFP8) but locks it to exact\\nSM120. This patch opens it for SM89:\\n  * sm120_online_fp8.configure_online_fp8 accepts capability (8, 9);\\n  * _MAX_KERNEL_ROWS 32 -> 64 (lm_head / HC / projection GEMV kernels serve up to 64 decode rows);\\n  * adds sm89_rowwise_fp8_linear() (weight-only W8A16, per-output-row fp32 scale, fp32 accumulate, bf16 out) and\\n    build_sm89_rowwise_method() -> a LinearMethod that quantises the loaded bf16 weight to rowwise e4m3 in\\n    process_weights_after_loading (and releases the bf16 copy + empty_cache) and runs the GEMV for <= 64 rows,\\n    chunked dequant + bf16 matmul above that (prefill);\\n  * qwen4_exp._maybe_convert_linears_to_mxfp8 uses that method instead of MXFP8 when the device is SM89.\\nHC mix on 17-32 rows needs hcmix32_patch.py (it accepts the rowwise-FP8 pair since this revision).\\nNumerics: e4m3 weights, rowwise absmax/448 scale, bf16 activations, fp32 accumulate. Eligible linears are the stock\\nset (BF16, >=128x128, K%32==0, not \\"gate\\" routers, not FusedMoE, not the PLE layer).\\nusage: densefp8_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [densefp8 patch]\\"\\nF_FP8 = \\"sglang/kernels/ops/gemm/sm120_online_fp8.py\\"\\nF_QWEN = \\"sglang/srt/models/qwen4_exp.py\\"\\n\\nCFG_LITERAL = \'{(320, 2560, 1): (32, 256, 3, 4), (320, 2560, 8): (32, 256, 3, 4), (320, 2560, 16): (16, 256, 3, 4), (320, 2560, 24): (16, 256, 3, 4), (320, 2560, 32): (16, 256, 3, 4), (320, 2560, 48): (16, 128, 4, 4), (320, 2560, 64): (16, 128, 4, 4), (2560, 160, 1): (64, 128, 3, 8), (2560, 160, 8): (64, 128, 3, 8), (2560, 160, 16): (64, 128, 3, 4), (2560, 160, 24): (64, 128, 3, 4), (2560, 160, 32): (64, 128, 3, 4), (2560, 160, 48): (64, 128, 3, 8), (2560, 160, 64): (64, 128, 3, 8), (2560, 1536, 1): (64, 128, 3, 8), (2560, 1536, 8): (64, 128, 3, 8), (2560, 1536, 16): (64, 128, 3, 4), (2560, 1536, 24): (64, 128, 3, 4), (2560, 1536, 32): (64, 128, 3, 4), (2560, 1536, 48): (64, 128, 3, 8), (2560, 1536, 64): (64, 128, 3, 8), (3584, 2560, 1): (32, 256, 3, 4), (3584, 2560, 8): (32, 256, 3, 4), (3584, 2560, 16): (64, 128, 3, 8), (3584, 2560, 24): (64, 128, 3, 8), (3584, 2560, 32): (64, 128, 3, 8), (3584, 2560, 48): (32, 128, 4, 4), (3584, 2560, 64): (32, 128, 4, 4), (4096, 2560, 1): (32, 256, 3, 4), (4096, 2560, 8): (32, 256, 3, 4), (4096, 2560, 16): (64, 128, 3, 4), (4096, 2560, 24): (64, 128, 3, 4), (4096, 2560, 32): (64, 128, 3, 4), (4096, 2560, 48): (64, 128, 3, 4), (4096, 2560, 64): (64, 128, 3, 4), (62080, 2560, 1): (32, 256, 3, 4), (62080, 2560, 8): (32, 256, 3, 4), (62080, 2560, 16): (16, 128, 4, 4), (62080, 2560, 24): (16, 128, 4, 4), (62080, 2560, 32): (16, 128, 4, 4), (62080, 2560, 48): (32, 256, 3, 4), (62080, 2560, 64): (32, 256, 3, 4)}\'\\n\\nAPPEND = f\'\'\'\\n\\n{MARK} colab/perf_patches/densefp8_patch.py: SM89 weight-only rowwise-FP8 projections.\\n_SM89_CFG = {CFG_LITERAL}  # (N, K, rows_bucket) -> (BLOCK_N, BLOCK_K, num_stages, num_warps); tuned on L4 by bench_densefp8.py (v76)\\n_SM89_DEFAULT = (32, 128, 4, 4)\\n\\n\\n@triton.jit\\ndef _sm89_w8a16_kernel(\\n    x_ptr, weight_ptr, scale_ptr, output_ptr, M, N, K,\\n    stride_xm, stride_xk, stride_wn, stride_wk, stride_om, stride_on,\\n    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,\\n):\\n    output_block = tl.program_id(0)\\n    rows_n = output_block * BLOCK_N + tl.arange(0, BLOCK_N)\\n    rows_m = tl.arange(0, BLOCK_M)\\n    mask_n = rows_n < N\\n    mask_m = rows_m < M\\n    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)\\n    for start_k in range(0, K, BLOCK_K):\\n        columns_k = start_k + tl.arange(0, BLOCK_K)\\n        mask_k = columns_k < K\\n        x = tl.load(\\n            x_ptr + rows_m[:, None] * stride_xm + columns_k[None, :] * stride_xk,\\n            mask=mask_m[:, None] & mask_k[None, :],\\n            other=0.0,\\n        )\\n        weight = tl.load(\\n            weight_ptr + rows_n[:, None] * stride_wn + columns_k[None, :] * stride_wk,\\n            mask=mask_n[:, None] & mask_k[None, :],\\n            other=0.0,\\n        ).to(x_ptr.dtype.element_ty)\\n        accumulator += tl.dot(x, tl.trans(weight), out_dtype=tl.float32)\\n    scale = tl.load(scale_ptr + rows_n, mask=mask_n, other=0.0)\\n    result = accumulator * scale[None, :]\\n    tl.store(\\n        output_ptr + rows_m[:, None] * stride_om + rows_n[None, :] * stride_on,\\n        result.to(output_ptr.dtype.element_ty),\\n        mask=mask_m[:, None] & mask_n[None, :],\\n    )\\n\\n\\ndef _sm89_rows_bucket(rows: int) -> int:\\n    for b in (1, 8, 16, 24, 32, 48, 64):\\n        if rows <= b:\\n            return b\\n    return 64\\n\\n\\ndef sm89_rowwise_fp8_gemv(hidden_2d, weight, scale, cfg=None):\\n    rows, k = hidden_2d.shape\\n    n = weight.shape[0]\\n    output = torch.empty((rows, n), dtype=torch.bfloat16, device=hidden_2d.device)\\n    bn, bk, st, nw = cfg or _SM89_CFG.get((n, k, _sm89_rows_bucket(rows))) or _SM89_DEFAULT\\n    _sm89_w8a16_kernel[(triton.cdiv(n, bn),)](\\n        hidden_2d, weight, scale, output, rows, n, k,\\n        hidden_2d.stride(0), hidden_2d.stride(1), weight.stride(0), weight.stride(1),\\n        output.stride(0), output.stride(1),\\n        BLOCK_M=max(16, triton.next_power_of_2(rows)), BLOCK_N=bn, BLOCK_K=bk,\\n        num_warps=nw, num_stages=st,\\n    )\\n    return output\\n\\n\\ndef sm89_rowwise_fp8_linear(x, weight, bias=None):\\n    \\"\\"\\"x[..., K] bf16 @ rowwise-FP8 weight[N, K]^T (+ bias) -> bf16 [..., N].\\"\\"\\"\\n    scale = _require_rowwise_scale(weight)\\n    if x.dtype != torch.bfloat16:\\n        x = x.bfloat16()\\n    shape = x.shape[:-1]\\n    x2 = x.reshape(-1, x.shape[-1])\\n    rows, n = x2.shape[0], weight.shape[0]\\n    if rows == 0:\\n        out = torch.empty((0, n), dtype=torch.bfloat16, device=x.device)\\n    elif rows <= _MAX_KERNEL_ROWS:\\n        out = sm89_rowwise_fp8_gemv(x2, weight, scale)\\n    else:\\n        out = torch.empty((rows, n), dtype=torch.bfloat16, device=x.device)\\n        block_rows = max(1, min(8192, _DEQUANT_TARGET_BYTES // max(1, weight.shape[1] * 4)))\\n        for start in range(0, n, block_rows):\\n            stop = min(start + block_rows, n)\\n            dense = (weight[start:stop].float() * scale[start:stop, None]).to(torch.bfloat16)\\n            out[:, start:stop] = x2 @ dense.T\\n    if bias is not None:\\n        out = out + bias\\n    return out.reshape(*shape, n)\\n\\n\\ndef build_sm89_rowwise_method():\\n    from sglang.srt.layers.quantization.unquant import UnquantizedLinearMethod\\n\\n    class Sm89RowwiseFp8LinearMethod(UnquantizedLinearMethod):\\n        \\"\\"\\"bf16 weights are loaded by the stock loader, then converted in place to rowwise FP8.\\"\\"\\"\\n\\n        def process_weights_after_loading(self, layer):\\n            replace_linear_weight_rowwise_fp8(layer)\\n            torch.cuda.empty_cache()  # the freed bf16 copy must go back to the driver so KV sizing sees it\\n\\n        def apply(self, layer, x, bias=None):\\n            return sm89_rowwise_fp8_linear(x, layer.weight, bias)\\n\\n    return Sm89RowwiseFp8LinearMethod()\\n\'\'\'\\n\\n# (file, anchor, replacement-or-None for append); anchors must occur exactly once in the STOCK file.\\nEDITS = [\\n    (F_FP8, \\"    if capability != (12, 0):\\\\n\\",\\n     f\\"    if capability not in ((12, 0), (8, 9)):  {MARK} SM89 (L4) allowed\\\\n\\"),\\n    (F_FP8, \\"_MAX_KERNEL_ROWS = 32\\\\n\\", f\\"_MAX_KERNEL_ROWS = 64  {MARK} was 32\\\\n\\"),\\n    (F_QWEN,\\n     \\"        def build_method():\\\\n            class CheckedOnlineMxfp8LinearMethod(Fp8LinearMethod):\\\\n\\",\\n     \\"        def build_method():\\\\n\\"\\n     f\\"            if torch.cuda.get_device_capability() == (8, 9):  {MARK}\\\\n\\"\\n     \\"                from sglang.kernels.ops.gemm.sm120_online_fp8 import build_sm89_rowwise_method\\\\n\\\\n\\"\\n     \\"                return build_sm89_rowwise_method()\\\\n\\"\\n     \\"            class CheckedOnlineMxfp8LinearMethod(Fp8LinearMethod):\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2:\\n        sys.exit(__doc__)\\n    site = Path(sys.argv[1])\\n    texts = {rel: (site / rel).read_text() for rel in {e[0] for e in EDITS}}\\n    if all(MARK in t for t in texts.values()):\\n        print(\\"densefp8_patch: already patched\\")\\n        return\\n    if any(MARK in t for t in texts.values()):\\n        sys.exit(\\"densefp8_patch: PARTIALLY patched; restore the files and re-run\\")\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"densefp8_patch: NOT patched\\")\\n    for rel, old, _ in EDITS:\\n        if texts[rel].count(old) != 1:\\n            sys.exit(f\\"densefp8_patch: anchor x{texts[rel].count(old)} (need 1) in {rel}: {old[:60]!r}; refusing to patch\\")\\n    for rel, old, new in EDITS:\\n        texts[rel] = texts[rel].replace(old, new, 1)\\n    texts[F_FP8] += APPEND\\n    for rel, t in texts.items():\\n        (site / rel).write_text(t)\\n    print(\\"densefp8_patch: patched\\", \\", \\".join(sorted(texts)))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "draft_ep_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s eplb/expert_location.py so a draft MoE layer with MORE experts than the (pruned)\\ntarget loads under expert parallelism: logical_to_all_physical() returns the identity mapping for an expert id beyond\\nthe target-sized map (Pennyroyal already does this for draft layer ids beyond the map). The map is only used at weight\\nload time here (no EPLB: ExpertLocationDispatchInfo.init_new returns None, so routed ids are not remapped at runtime).\\nusage: draft_ep_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [draft-ep patch]\\"\\nANCHOR = \\"        cpu_map = self.logical_to_all_physical_map_cpu\\\\n\\"\\nADD = (f\\"        if logical_expert_id >= cpu_map.shape[1]:  {MARK} draft MoE wider than the pruned target map\\\\n\\"\\n       \\"            return [logical_expert_id]\\\\n\\")\\n\\n\\ndef main():\\n    f = Path(sys.argv[1]) / \\"sglang/srt/eplb/expert_location.py\\"\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"draft_ep_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"draft_ep_patch: NOT patched\\")\\n    if s.count(ANCHOR) != 1:\\n        sys.exit(\\"draft_ep_patch: anchor not found exactly once; refusing to patch\\")\\n    f.write_text(s.replace(ANCHOR, ANCHOR + ADD))\\n    print(\\"draft_ep_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "embedhost_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s Qwen4-Exp model (srt/models/qwen4_exp.py) so the input token embedding table\\n(embed_tokens, vocab 248320 x 2560 bf16, vocab-parallel: 318 MB per TP4 rank) lives in pinned host memory instead of\\non the GPU, gated by SGLANG_EMBED_HOST=1 (unset / 0 = the stock VocabParallelEmbedding, unchanged).\\n\\nWhy: on 4x L4 the KV pool is what bounds concurrency at agent context lengths (v86: 28-34 running at 88% KV use);\\n318 MB/GPU is ~98k more pool tokens (+7%) at 3.24 KB/token/GPU (fp8 KV, pair-owned, + indexer keys).\\nHow: the model already ships Qwen4ExpPinnedHostEmbedding for the PLE n-gram table (qwen4_exp.py:801): a meta-device\\nVocabParallelEmbedding template, a page-locked host table loaded by the stock vocab-parallel weight_loader, and a\\nTriton gather that reads the rows straight from host memory (UVA) and zero-fills ids outside the rank\'s shard, then the\\nsame tensor-parallel all-reduce as VocabParallelEmbedding.forward. Exact: the gathered bf16 rows are copies and the\\nall-reduce adds one non-zero row to zeros. Cost: B rows x 5 KB over PCIe per decode step (~200 KB at B=40), ~5 MB per\\n1024-token prefill chunk. The lm_head (untied, tie_word_embeddings=false) stays on the GPU.\\nusage: embedhost_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [embedhost patch]\\"\\nREL = \\"sglang/srt/models/qwen4_exp.py\\"\\n\\nREQUIRED = [   # read-only anchors: the host-embedding class and a module logger must exist\\n    \\"class Qwen4ExpPinnedHostEmbedding(VocabParallelEmbedding):\\\\n\\",\\n    \\"def _allocate_pinned_table(shape: Tuple[int, ...], dtype: torch.dtype) -> torch.Tensor:\\\\n\\",\\n]\\n\\nOLD = \'\'\'    def _build_embed_tokens(self, config: Qwen4ExpTextConfig) -> nn.Module:\\n        return VocabParallelEmbedding(\\n            config.vocab_size,\\n            config.hidden_size,\\n            org_num_embeddings=config.vocab_size,\\n            use_attn_tp_group=is_dp_attention_enabled(),\\n        )\\n\'\'\'\\n\\nNEW = f\'\'\'    def _build_embed_tokens(self, config: Qwen4ExpTextConfig) -> nn.Module:\\n        import os as _os  {MARK} SGLANG_EMBED_HOST=1: embed_tokens in pinned host memory (colab/perf_patches)\\n        if _os.environ.get(\\"SGLANG_EMBED_HOST\\", \\"0\\") not in (\\"\\", \\"0\\") and not getattr(config, \\"tie_word_embeddings\\", False):\\n            with torch.device(\\"meta\\"):\\n                template = VocabParallelEmbedding(\\n                    config.vocab_size,\\n                    config.hidden_size,\\n                    params_dtype=torch.bfloat16,\\n                    org_num_embeddings=config.vocab_size,\\n                    use_attn_tp_group=is_dp_attention_enabled(),\\n                )\\n            template.register_buffer(\\"weight_scale\\", torch.ones(1, dtype=torch.bfloat16), persistent=False)\\n            emb = Qwen4ExpPinnedHostEmbedding(template)\\n            logger.info(\\n                \\"{MARK} embed_tokens %s x %s bf16 shard in pinned host memory (%.0f MiB per rank)\\",\\n                emb.num_embeddings_per_partition, emb.embedding_dim,\\n                emb.weight.numel() * emb.weight.element_size() / 2**20,\\n            )\\n            return emb\\n        return VocabParallelEmbedding(\\n            config.vocab_size,\\n            config.hidden_size,\\n            org_num_embeddings=config.vocab_size,\\n            use_attn_tp_group=is_dp_attention_enabled(),\\n        )\\n\'\'\'\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"embedhost_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"embedhost_patch: NOT patched\\")\\n    for anchor in REQUIRED + [OLD, \\"from sglang.srt.utils import logger\\\\n\\"]:\\n        if s.count(anchor) < 1 or (anchor == OLD and s.count(anchor) != 1):\\n            sys.exit(f\\"embedhost_patch: anchor x{s.count(anchor)}: {anchor.strip()[:70]!r}; refusing to patch\\")\\n    s = s.replace(OLD, NEW)\\n    f.write_text(s)\\n    print(\\"embedhost_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "hcmix32_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s fused HyperConnection mix (srt/layers/hc_mix_triton.py) so the persistent Triton\\nkernel also serves 17-32 rows, gated by SGLANG_HC_MIX_MAX_ROWS=N (17..32; unset / \\"\\" / <=16 = stock, unchanged).\\n\\nWhy: fused_hc_mix_supported() caps the persistent kernel at _FUSED_MIX_MAX_ROWS = 16 (hc_mix_triton.py:33, :208) and\\nfused_hc_mix() hard-codes rows_pad = 16 (:255). Production decode runs 17-19 live rows on the 20-row CUDA graph, so all\\n97 mixes per step (48 layers x attn/mlp + final mixer) fall back to the 5-kernel torch.compile chain\\n(hyperconnection.py:207-242, called at :312: down GEMM + splitK reduce, silu, up GEMM, sigmoid-mul-mean). The kernel is\\nalready generic in its ROWS constexpr (:59, :68, :125; t_raw is sized by rows_pad, x loads / out stores are masked by\\nnum_rows), so 17-32 rows run the same kernel with ROWS=32. The grid stays one CTA per SM (58 on L4), so the software\\ngrid barrier keeps its co-residency guarantee; only per-CTA smem/registers grow (ROWS=32 doubles the phase-A x tile).\\n\\nWhat changes when enabled (nothing changes when the env is unset):\\n  * fused_hc_mix_supported() also accepts 17..N rows when both weights have the activation dtype (bf16/fp16);\\n    rowwise-FP8 weight pairs (densefp8_patch.py) are accepted too; the stock check still validates their scales.\\n  * fused_hc_mix() pads 17-32 rows to ROWS=32 and launches through _hcmix32_launch(), which passes num_stages\\n    explicitly: SGLANG_HC_MIX_NUM_STAGES=S if set, else the first of 3 (Triton\'s default), 2, 1 that compiles.\\n    3 stages x (32x256 x tile + 32x256 w tile) bf16 = 96 KiB smem is at the sm_89 99 KiB limit, hence the ladder.\\n    If no variant compiles, it logs, computes that one call eagerly (same math as _mix_compute) and disables itself,\\n    so every later >16-row mix takes the stock torch.compile chain.\\n  * <=16 rows always take the untouched stock launch (same kernel source, same constexprs, same Triton cache entry).\\nNumerics: 17-32 rows get exactly the fp32-accumulating atomic path that <=16 rows already use in production (atomic\\nsummation order varies per replay); vs the bf16 torch chain this is a change of rounding only (bench_hcmix32.py).\\nEnv (read once at import, per process): SGLANG_HC_MIX_MAX_ROWS=N, optional SGLANG_HC_MIX_NUM_STAGES=S (1..4).\\nusage: hcmix32_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [hcmix32 patch]\\"\\nREL = \\"sglang/srt/layers/hc_mix_triton.py\\"\\n\\n# Read-only anchors: the kernel/launch contract the ROWS=32 launch in HELPER mirrors argument for argument.\\nREQUIRED = [\\n    \\"def _hc_mix_persistent_kernel(\\\\n\\"\\n    \\"    x_ptr,\\\\n    w_down_ptr,\\\\n    w_up_ptr,\\\\n    t_raw_ptr,\\\\n    out_ptr,\\\\n    counters_ptr,\\\\n\\"\\n    \\"    down_scale_ptr,\\\\n    up_scale_ptr,\\\\n    K,\\\\n    LOWRANK,\\\\n    HS,\\\\n    num_rows,\\\\n    num_ctas,\\\\n\\"\\n    \\"    inv_hc,\\\\n    ROWS: tl.constexpr,\\\\n    HC: tl.constexpr,\\\\n    BLOCK_N: tl.constexpr,\\\\n\\"\\n    \\"    BLOCK_K: tl.constexpr,\\\\n    BLOCK_J: tl.constexpr,\\\\n    BLOCK_R: tl.constexpr,\\\\n\\"\\n    \\"    FP8_WEIGHT: tl.constexpr,\\\\n):\\\\n\\",\\n    \\"    offs_m = tl.arange(0, ROWS)\\\\n    mask_m = offs_m < num_rows\\\\n\\",\\n    \\"    zero_span = ROWS * LOWRANK\\\\n\\",\\n    \\"        acc = tl.zeros((ROWS, HC * BLOCK_J), dtype=tl.float32)\\\\n\\",\\n    \\"    t_raw = torch.empty((rows_pad, lowrank), dtype=torch.float32, device=device)\\\\n\\",\\n    \\"    if rows == 0:\\\\n        return out\\\\n\\",\\n    \\"    _hc_mix_persistent_kernel[(num_ctas,)](\\\\n\\"\\n    \\"        hyper_input_normed,\\\\n        w_down,\\\\n        w_up,\\\\n        t_raw,\\\\n        out,\\\\n\\"\\n    \\"        _get_counters(device),\\\\n        down_scale,\\\\n        up_scale,\\\\n        k,\\\\n        lowrank,\\\\n\\"\\n    \\"        hs,\\\\n        rows,\\\\n        num_ctas,\\\\n        1.0 / hc,\\\\n        ROWS=rows_pad,\\\\n        HC=hc,\\\\n\\"\\n    \\"        BLOCK_N=32,\\\\n        BLOCK_K=256,\\\\n        BLOCK_J=32,\\\\n        BLOCK_R=64,\\\\n\\"\\n    \\"        FP8_WEIGHT=scales is not None,\\\\n        num_warps=8,\\\\n    )\\\\n    return out\\\\n\\",\\n]\\n\\nHELPER = f\'\'\'\\n\\n{MARK} colab/perf_patches/hcmix32_patch.py: optional 17-32 row variant of the persistent HC mix kernel.\\n{MARK} Stock caps it at 16 rows; above that each mix falls back to the 5-kernel torch.compile chain\\n{MARK} (hyperconnection.py _mix_compute). SGLANG_HC_MIX_MAX_ROWS=N (17..32) runs 17..N rows through the same\\n{MARK} kernel with ROWS=32 (bf16/fp16 or rowwise-FP8 weight pairs). Unset / \\"\\" / <=16 = stock.\\n{MARK} SGLANG_HC_MIX_NUM_STAGES=S (1..4) pins num_stages of the ROWS=32 launch; default: the first\\n{MARK} of 3, 2, 1 that compiles. Both read once at import, per process.\\ndef _hcmix32_env():\\n    import logging as _logging\\n    import os as _os\\n\\n    log = _logging.getLogger(__name__)\\n    raw = _os.environ.get(\\"SGLANG_HC_MIX_MAX_ROWS\\", \\"\\").strip()\\n    raw_st = _os.environ.get(\\"SGLANG_HC_MIX_NUM_STAGES\\", \\"\\").strip()\\n    if not raw:\\n        return 0, None\\n    try:\\n        rows = int(raw)\\n    except ValueError:\\n        log.warning(\\"[hcmix32 patch] ignoring SGLANG_HC_MIX_MAX_ROWS=%r; stock 16-row limit\\", raw)\\n        return 0, None\\n    if rows <= _FUSED_MIX_MAX_ROWS:\\n        return 0, None\\n    if rows > 64:\\n        log.warning(\\"[hcmix32 patch] SGLANG_HC_MIX_MAX_ROWS=%d clamped to 64\\", rows)\\n        rows = 64\\n    stages = None\\n    if raw_st:\\n        try:\\n            stages = int(raw_st)\\n        except ValueError:\\n            stages = 0\\n        if not 1 <= stages <= 4:\\n            log.warning(\\"[hcmix32 patch] ignoring SGLANG_HC_MIX_NUM_STAGES=%r\\", raw_st)\\n            stages = None\\n    log.warning(\\n        \\"[hcmix32 patch] fused HC mix up to %d rows (17+ rows: ROWS=32 kernel, num_stages %s)\\",\\n        rows,\\n        stages if stages else \\"auto 3/2/1\\",\\n    )\\n    return rows, stages\\n\\n\\n_HCMIX32_MAX_ROWS, _HCMIX32_STAGES = _hcmix32_env()\\n_HCMIX32_READY = False  # set at the first successful ROWS=32 launch (logged once, also when num_stages is pinned)\\n\\n\\ndef _hcmix32_rows_ok(hyper_input_normed, w_down, w_up) -> bool:\\n    return (\\n        _HCMIX32_MAX_ROWS > 0\\n        and hyper_input_normed.shape[0] <= _HCMIX32_MAX_ROWS\\n        and (\\n            (w_down.dtype == hyper_input_normed.dtype and w_up.dtype == hyper_input_normed.dtype)\\n            or (w_down.dtype == torch.float8_e4m3fn and w_up.dtype == torch.float8_e4m3fn)  # rowwise-FP8 pair (densefp8)\\n        )\\n    )\\n\\n\\ndef _hcmix32_launch(\\n    x, w_down, w_up, t_raw, out, down_scale, up_scale, k, lowrank, hs, rows, num_ctas, hc, fp8\\n):\\n    \\"\\"\\"ROWS=32 launch of the stock persistent kernel (same args as fused_hc_mix\'s stock launch + num_stages).\\"\\"\\"\\n    global _HCMIX32_MAX_ROWS, _HCMIX32_STAGES, _HCMIX32_READY\\n    import logging as _logging\\n\\n    log = _logging.getLogger(__name__)\\n    kernel = _hc_mix_persistent_kernel\\n    for stages in dict.fromkeys(s for s in (_HCMIX32_STAGES, 3, 2, 1) if s):\\n        try:\\n            kernel[(num_ctas,)](\\n                x,\\n                w_down,\\n                w_up,\\n                t_raw,\\n                out,\\n                _get_counters(x.device),\\n                down_scale,\\n                up_scale,\\n                k,\\n                lowrank,\\n                hs,\\n                rows,\\n                num_ctas,\\n                1.0 / hc,\\n                ROWS=32,\\n                HC=hc,\\n                BLOCK_N=32,\\n                BLOCK_K=256,\\n                BLOCK_J=32,\\n                BLOCK_R=64,\\n                FP8_WEIGHT=fp8,\\n                num_warps=8,\\n                num_stages=stages,\\n            )\\n        except Exception as exc:  # compile / out-of-resources: raised before the launch, nothing ran\\n            log.warning(\\n                \\"[hcmix32 patch] ROWS=32 HC mix kernel with num_stages=%d unusable: %s: %s\\",\\n                stages,\\n                type(exc).__name__,\\n                str(exc)[:300],\\n            )\\n            continue\\n        if not _HCMIX32_READY or _HCMIX32_STAGES != stages:\\n            _HCMIX32_READY = True\\n            _HCMIX32_STAGES = stages\\n            log.warning(\\"[hcmix32 patch] ROWS=32 HC mix kernel ready, num_stages=%d\\", stages)\\n        return out\\n    _HCMIX32_MAX_ROWS = 0\\n    log.warning(\\n        \\"[hcmix32 patch] no ROWS=32 HC mix variant compiled; >16-row mixes use the stock torch.compile chain\\"\\n    )\\n    import torch.nn.functional as F  # this one call: eager copy of hyperconnection.py _mix_compute\\n\\n    t = F.silu(F.linear(x, w_down) / hc)\\n    t = torch.sigmoid(F.linear(t, w_up)).unflatten(-1, (hc, hs))\\n    return (t * x.unflatten(-1, (hc, hs))).mean(dim=-2)\\n\'\'\'\\n\\n# (anchor, replacement); every anchor must occur exactly once in the STOCK file. Replacements are applied by position\\n# on the stock text, so inserted text can never be matched by a later anchor.\\nEDITS = [\\n    (\\"_FUSED_MIX_MAX_ROWS = 16\\\\n\\",\\n     \\"_FUSED_MIX_MAX_ROWS = 16\\\\n\\" + HELPER),\\n    (\\"        and hyper_input_normed.shape[0] <= _FUSED_MIX_MAX_ROWS\\\\n\\",\\n     \\"        and (\\\\n\\"\\n     \\"            hyper_input_normed.shape[0] <= _FUSED_MIX_MAX_ROWS\\\\n\\"\\n     f\\"            or _hcmix32_rows_ok(hyper_input_normed, w_down, w_up)  {MARK}\\\\n\\"\\n     \\"        )\\\\n\\"),\\n    (\\"    rows, k = hyper_input_normed.shape\\\\n    lowrank = w_down.shape[0]\\\\n\\",\\n     \\"    rows, k = hyper_input_normed.shape\\\\n\\"\\n     f\\"    if _HCMIX32_MAX_ROWS and rows > 32:  {MARK} >32 rows: 32-row launches (weights re-read from L2)\\\\n\\"\\n     \\"        return torch.cat(\\\\n\\"\\n     \\"            [fused_hc_mix(hyper_input_normed[i : i + 32], w_down, w_up, hc, hs) for i in range(0, rows, 32)], dim=0\\\\n\\"\\n     \\"        )\\\\n\\"\\n     \\"    lowrank = w_down.shape[0]\\\\n\\"),\\n    (\\"    rows_pad = 16\\\\n\\",\\n     \\"    rows_pad = 16\\\\n\\"\\n     f\\"    if _HCMIX32_MAX_ROWS and 16 < rows <= 32:  {MARK} 17-32 rows: ROWS=32 variant\\\\n\\"\\n     \\"        rows_pad = 32\\\\n\\"),\\n    (\\"    _hc_mix_persistent_kernel[(num_ctas,)](\\\\n\\",\\n     f\\"    if rows_pad == 32:  {MARK} explicit num_stages ladder + safe fallback\\\\n\\"\\n     \\"        return _hcmix32_launch(\\\\n\\"\\n     \\"            hyper_input_normed,\\\\n\\"\\n     \\"            w_down,\\\\n\\"\\n     \\"            w_up,\\\\n\\"\\n     \\"            t_raw,\\\\n\\"\\n     \\"            out,\\\\n\\"\\n     \\"            down_scale,\\\\n\\"\\n     \\"            up_scale,\\\\n\\"\\n     \\"            k,\\\\n\\"\\n     \\"            lowrank,\\\\n\\"\\n     \\"            hs,\\\\n\\"\\n     \\"            rows,\\\\n\\"\\n     \\"            num_ctas,\\\\n\\"\\n     \\"            hc,\\\\n\\"\\n     \\"            scales is not None,\\\\n\\"\\n     \\"        )\\\\n\\"\\n     \\"    _hc_mix_persistent_kernel[(num_ctas,)](\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2:\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        missing = [new[:60] for _, new in EDITS if new not in s]\\n        if missing:\\n            sys.exit(f\\"hcmix32_patch: PARTIALLY patched (missing {missing!r}); restore {f} and re-run\\")\\n        print(\\"hcmix32_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"hcmix32_patch: NOT patched\\")\\n    for anchor in REQUIRED:\\n        if s.count(anchor) != 1:\\n            sys.exit(f\\"hcmix32_patch: required anchor x{s.count(anchor)} (need 1): {anchor[:60]!r}; refusing to patch\\")\\n    spots = []\\n    for old, new in EDITS:\\n        if s.count(old) != 1:\\n            sys.exit(f\\"hcmix32_patch: anchor x{s.count(old)} (need 1): {old[:60]!r}; refusing to patch\\")\\n        spots.append((s.index(old), old, new))\\n    for pos, old, new in sorted(spots, reverse=True):\\n        s = s[:pos] + new + s[pos + len(old):]\\n    f.write_text(s)\\n    print(\\"hcmix32_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "hic_check.py": "#!/usr/bin/env python3\\n\\"\\"\\"HiCache host->device reload check against a running SGLang server (stdlib only; ~5 min on 4x L4).\\n\\nWhy: v99 (10-15) crashed on the first host->device load (pair-owned QSA left the layer-done counter at 42 of 48 layers,\\nfixed by the hicpairown patch); a wrong layer count can also skip restoring layers silently. This forces reloads early\\nand checks them exactly instead of waiting for an agent run to stumble on them.\\n  1. targets: T distinct random-token prompts of L tokens, greedy N tokens each, run twice: A = cold prefill, B = GPU hit\\n  2. flood: enough other prompts (> 1.3x the GPU KV pool) that every target is evicted to the host tier\\n  3. C = each target again: its prefix must come back from host (cached_tokens_details) and its greedy text must equal\\n     B\'s when both reused the same prefix length (same compute path; only where the KV/mamba/QSA bytes came from differs)\\n  4. stress: W concurrent workers re-hit random targets/flood prompts (loads during decode) for the remaining minutes\\nok = no failed request + no strict B/C mismatch + at least one host reload seen.\\nusage: hic_check.py ENDPOINT DEVICE_KV_TOKENS OUT_JSON [MINUTES]\\n\\"\\"\\"\\nimport json\\nimport random\\nimport sys\\nimport threading\\nimport time\\nimport urllib.request\\n\\nEP, KV, OUT = sys.argv[1], int(sys.argv[2]), sys.argv[3]\\nMINUTES = float(sys.argv[4]) if len(sys.argv) > 4 else 6.0\\nL, N, T, W = 16384, 24, 6, 8\\nF = int(1.3 * KV / L) + 2\\nT0 = time.time()\\nR = {\\"L\\": L, \\"N\\": N, \\"targets\\": T, \\"flood\\": F, \\"kv\\": KV, \\"fails\\": [], \\"strict_mismatch\\": [], \\"loose_mismatch\\": [],\\n     \\"host_hits\\": 0, \\"rows\\": []}\\n\\n\\ndef prompt(i):\\n    rng = random.Random(1000 + i)\\n    return [rng.randrange(1000, 30000) for _ in range(L)]\\n\\n\\ndef gen(ids, n=N, timeout=900):\\n    body = json.dumps({\\"input_ids\\": ids, \\"sampling_params\\": {\\"temperature\\": 0, \\"max_new_tokens\\": n}}).encode()\\n    req = urllib.request.Request(f\\"{EP}/generate\\", data=body, headers={\\"Content-Type\\": \\"application/json\\"})\\n    with urllib.request.urlopen(req, timeout=timeout) as r:\\n        j = json.loads(r.read())\\n    m = j.get(\\"meta_info\\", {})\\n    return j.get(\\"text\\", \\"\\"), m.get(\\"cached_tokens\\", 0), m.get(\\"cached_tokens_details\\")\\n\\n\\ndef host_of(det):\\n    if isinstance(det, dict):\\n        return int(det.get(\\"host\\", 0) or 0)\\n    return 0\\n\\n\\ndef metrics():\\n    try:\\n        with urllib.request.urlopen(f\\"{EP}/metrics\\", timeout=30) as r:\\n            txt = r.read().decode(errors=\\"replace\\")\\n    except Exception as ex:\\n        return {\\"error\\": repr(ex)}\\n    out = {}\\n    for ln in txt.splitlines():\\n        if ln.startswith(\\"#\\") or not any(k in ln for k in (\\"hicache\\", \\"host\\", \\"cache_hit\\", \\"cached_tokens\\")):\\n            continue\\n        k, _, v = ln.rpartition(\\" \\")\\n        try:\\n            out[k] = float(v)\\n        except ValueError:\\n            pass\\n    return out\\n\\n\\ndef safe(tag, fn):\\n    try:\\n        return fn()\\n    except Exception as ex:\\n        R[\\"fails\\"].append(f\\"{tag}: {type(ex).__name__}: {ex}\\"[:300])\\n        print(\\"FAIL\\", R[\\"fails\\"][-1], flush=True)\\n        return None\\n\\n\\ndef log(msg):\\n    print(f\\"[hic_check +{time.time() - T0:5.0f}s] {msg}\\", flush=True)\\n\\n\\nm0 = metrics()\\nP = [prompt(i) for i in range(T + F)]\\nA, B = {}, {}\\nfor i in range(T):\\n    A[i] = safe(f\\"A{i}\\", lambda: gen(P[i]))\\n    B[i] = safe(f\\"B{i}\\", lambda: gen(P[i]))\\n    log(f\\"target {i}: cold cached {A[i] and A[i][1]} | GPU-hit cached {B[i] and B[i][1]} det {B[i] and B[i][2]}\\")\\nlog(f\\"flood: {F} x {L} tokens (GPU pool {KV})\\")\\nlock = threading.Lock()\\ntodo = list(range(T, T + F))\\n\\n\\ndef flood():\\n    while True:\\n        with lock:\\n            if not todo:\\n                return\\n            k = todo.pop()\\n        safe(f\\"F{k}\\", lambda: gen(P[k], n=8))\\n\\n\\nths = [threading.Thread(target=flood) for _ in range(4)]\\n[t.start() for t in ths]\\n[t.join() for t in ths]\\nlog(\\"flood done\\")\\nfor i in range(T):\\n    c = safe(f\\"C{i}\\", lambda: gen(P[i]))\\n    if not (c and B[i]):\\n        continue\\n    h = host_of(c[2])\\n    R[\\"host_hits\\"] += h > 0\\n    same_prefix = c[1] == B[i][1]\\n    row = {\\"i\\": i, \\"A_cached\\": A[i] and A[i][1], \\"B_cached\\": B[i][1], \\"C_cached\\": c[1], \\"C_det\\": c[2],\\n           \\"B_eq_C\\": c[0] == B[i][0], \\"A_eq_B\\": bool(A[i]) and A[i][0] == B[i][0], \\"same_prefix\\": same_prefix}\\n    R[\\"rows\\"].append(row)\\n    if c[0] != B[i][0]:\\n        (R[\\"strict_mismatch\\"] if same_prefix else R[\\"loose_mismatch\\"]).append(i)\\n    log(f\\"target {i}: reload cached {c[1]} det {c[2]} | B==C {row[\'B_eq_C\']} (same prefix {same_prefix}) | A==B {row[\'A_eq_B\']}\\")\\nstop_at = T0 + MINUTES * 60\\ncnt = {\\"req\\": 0, \\"host\\": 0, \\"eq\\": 0, \\"neq\\": 0}\\n\\n\\ndef stress(w):\\n    rng = random.Random(w)\\n    while time.time() < stop_at and len(R[\\"fails\\"]) < 5:\\n        k = rng.randrange(T + F)\\n        r = safe(f\\"S{w}\\", lambda: gen(P[k], n=16 if k >= T else N))\\n        if r is None:\\n            continue\\n        with lock:\\n            cnt[\\"req\\"] += 1\\n            cnt[\\"host\\"] += host_of(r[2]) > 0\\n            if k < T and B.get(k):\\n                cnt[\\"eq\\" if r[0] == B[k][0] else \\"neq\\"] += 1\\n\\n\\nif time.time() < stop_at - 30 and not R[\\"fails\\"]:\\n    log(f\\"stress: {W} workers until +{MINUTES:.0f} min\\")\\n    ths = [threading.Thread(target=stress, args=(w,)) for w in range(W)]\\n    [t.start() for t in ths]\\n    [t.join() for t in ths]\\nR[\\"stress\\"] = cnt\\nm1 = metrics()\\nR[\\"metrics_delta\\"] = {k: m1[k] - m0.get(k, 0.0) for k in m1 if isinstance(m1.get(k), float) and m1[k] != m0.get(k, 0.0)}\\nR[\\"seconds\\"] = round(time.time() - T0)\\nR[\\"ok\\"] = not R[\\"fails\\"] and not R[\\"strict_mismatch\\"] and (R[\\"host_hits\\"] > 0 or cnt[\\"host\\"] > 0)\\nR[\\"verdict\\"] = (\\"PASS\\" if R[\\"ok\\"] else \\"FAIL\\") + (f\\" fails={len(R[\'fails\'])} strict_mismatch={R[\'strict_mismatch\']} \\"\\n                                                  f\\"loose_mismatch={R[\'loose_mismatch\']} host_reloads={R[\'host_hits\']} \\"\\n                                                  f\\"stress={cnt}\\")\\njson.dump(R, open(OUT, \\"w\\"), indent=1)\\nlog(\\"HIC_CHECK \\" + R[\\"verdict\\"])\\nsys.exit(0 if R[\\"ok\\"] else 1)\\n", "hicmambafb_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch UnifiedRadixCache.init_load_back (srt/mem_cache/unified_radix_cache.py): when a HiCache load-back\\nFAILS on a mamba-hybrid cache, re-prefill the request from the root instead of resuming at the device anchor\\n(HiCache x L4 fallback fix, 2026-10-16).\\n\\nWhy (review workflow 10-16, mamba lens): match_prefix stages the request\'s mamba state from the DEEPEST match N2\\n(best_match_node: COW from its device state, or prepare_load_back loads its host-only state into the request slot),\\nwhile a failed load_back (FULL KV short after evict, or the node-restore mamba alloc failing under slot pressure,\\n41 running x 2 slots on 96) returns the shallower device anchor N1 with N1\'s KV as the prefix. The recurrent state then\\nno longer matches the prefix: zero state (host-only N2: slot freed and re-allocated cleared) or N2\'s state on N1\'s\\nprefix (device N2 COW) -> the 36 GDN layers silently diverge for the rest of the session. Stock never hits this on the\\nH100 TP1 config (ample slots); the L4 rung can.\\nFix: on that failure path, if the cache has a MAMBA component and N2 != N1, drop the prefix to the root, clear any\\nstaged COW (the slot is zeroed instead) and zero the host-hit lengths -> plain full re-prefill = the pre-HiCache miss\\nbehaviour (correct, slower, rare). Any exception inside the fallback is logged and the stock result returned.\\nLogs \\"[hicmambafb]\\" (first 20, then every 100th) so a run shows how often it fires.\\nusage: hicmambafb_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [hicmambafb patch]\\"\\nREL = \\"sglang/srt/mem_cache/unified_radix_cache.py\\"\\nOLD = \'\'\'                return new_indices, best_match_node_id\\n\\n        return (\\n            self.tree_core.empty_match_result.device_indices,\\n            last_best_match_device_node_id,\\n        )\\n\'\'\'\\nNEW = f\'\'\'                return new_indices, best_match_node_id\\n\\n        if best_match_node_id != last_best_match_device_node_id and any(  {MARK}\\n            c.component_type == ComponentType.MAMBA for c in self._components_tuple\\n        ):\\n            try:\\n                req.prefix_indices = req.prefix_indices[:0]\\n                req.mamba_cow_src_index = None\\n                req.mamba_needs_clear = True\\n                req.host_hit_length = 0\\n                req.swa_host_hit_length = 0\\n                req.mamba_host_hit_length = 0\\n                _n = self._hicmambafb_n = getattr(self, \\"_hicmambafb_n\\", 0) + 1\\n                if _n <= 20 or _n % 100 == 0:\\n                    logger.warning(\\n                        \\"[hicmambafb] load-back failed (#%d, node %s, device anchor %s): re-prefill from root\\",\\n                        _n, best_match_node_id, last_best_match_device_node_id,\\n                    )\\n                return (\\n                    self.tree_core.empty_match_result.device_indices,\\n                    self.root_node_handle(getattr(req, \\"extra_key\\", None)),\\n                )\\n            except Exception as _ex:\\n                logger.warning(\\"[hicmambafb] fallback error %r: stock anchor kept\\", _ex)\\n\\n        return (\\n            self.tree_core.empty_match_result.device_indices,\\n            last_best_match_device_node_id,\\n        )\\n\'\'\'\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"hicmambafb_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"hicmambafb_patch: NOT patched\\")\\n    need = [\\"def init_load_back(\\", \\"def root_node_handle(\\", \\"    ComponentType,\\\\n\\", \\"self._components_tuple\\", \\"\\\\nlogger = \\"]\\n    if s.count(OLD) != 1 or not all(n in s for n in need):\\n        sys.exit(f\\"hicmambafb_patch: anchor count {s.count(OLD)} (want 1) or helpers missing; refusing\\")\\n    f.write_text(s.replace(OLD, NEW))\\n    print(\\"hicmambafb_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "hicpairown_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s HiCache stack builder (srt/mem_cache/hybrid_cache/hybrid_pool_assembler.py) so the\\nKV + MAMBA (+ QSA compressed) host tier works with pair-owned QSA (pairown patch, SGLANG_QSA_PAIR_OWN=1).\\n\\nWhy: the builder sets the per-layer transfer count to len(full_layer_mapping | mamba_layer_mapping), i.e. the number of\\nlayer ids this rank stores. Pair-own keeps only 6 of the 12 QSA layers per rank, so the count is 36 + 6 = 42 while layer\\nids run 0..47. The H->D loop (l2_transfer.submit_host_to_device: for layer_id in range(42)) then never reloads layers\\n42..47 and the forward\'s per-layer wait on layer >= 42 indexes past LayerLoadingEvent.load_events (v99 smoke, 15:23 UTC\\n10-15: IndexError in mamba2_layer_index -> wait_until -> load_events[layer_index], scheduler killed, 36 requests lost).\\nFix: transfer layers = global layer ids 0..max over the stored ids AND every pair-owned QSA id (so both ranks of a pair\\nagree on 48). Without pair-own the union already covers 0..47, so the count is unchanged (H100 A/B: 48 either way).\\nExact: no kernel or numeric change; layers this rank does not store map to None and are skipped as before.\\nusage: hicpairown_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [hicpairown patch]\\"\\nREL = \\"sglang/srt/mem_cache/hybrid_cache/hybrid_pool_assembler.py\\"\\n\\nANCHOR = \\"def build_hybrid_mamba_stack(\\\\n\\"\\nHELPER = f\'\'\'def _hic_layer_span(*mappings):  {MARK}\\n    \\"\\"\\"Per-layer transfer count = global layer ids 0..max, not how many ids this rank stores (pair-own stores half the\\n    QSA layers; the H->D loop and the layer-done events must still cover every layer id up to the last one).\\"\\"\\"\\n    ids = set().union(*mappings)\\n    try:\\n        from sglang.srt.layers.attention.qsa import pairown as _pairown\\n\\n        cfg = _pairown._S.get(\\"cfg\\")\\n        if cfg:\\n            ids |= set(cfg[\\"owner\\"])\\n    except Exception:\\n        pass\\n    return max(len(ids), max(ids) + 1) if ids else 0\\n\\n\\n\'\'\'\\nEDITS = [\\n    (\\"    transfer_layer_num = len(full_layer_mapping | mamba_layer_mapping)\\\\n\\",\\n     f\\"    transfer_layer_num = _hic_layer_span(full_layer_mapping, mamba_layer_mapping)  {MARK}\\\\n\\"),\\n    (\\"            transfer_layer_num=len(full_layer_mapping | mamba_layer_mapping),\\\\n\\",\\n     f\\"            transfer_layer_num=_hic_layer_span(full_layer_mapping, mamba_layer_mapping),  {MARK}\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"hicpairown_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"hicpairown_patch: NOT patched\\")\\n    counts = [s.count(ANCHOR)] + [s.count(old) for old, _ in EDITS]\\n    if counts != [1, 1, 1]:\\n        sys.exit(f\\"hicpairown_patch: anchors {counts} (want [1, 1, 1]); refusing to patch\\")\\n    for old, new in EDITS:\\n        s = s.replace(old, new)\\n    s = s.replace(ANCHOR, HELPER + ANCHOR)\\n    f.write_text(s)\\n    print(\\"hicpairown_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "hicqsawait_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch QSATokenToKVPool.get_qsa_compressed_k_buffer (srt/mem_cache/qsa_kv_pool.py) to wait for the\\nHiCache per-layer host->device load like the K/V getters do (HiCache + QSA ordering fix, 2026-10-16).\\n\\nWhy: with --enable-hierarchical-cache a host-backed prefix is copied back layer by layer (KV, mamba, QSA compressed-K)\\nand the forward waits for layer L only inside get_key/value/kv_buffer (HybridLinearKVPool._wait_for_layer). The QSA\\nindexer runs BEFORE attention (qwen4_exp._self_attention_impl / _self_attention_pairown owner branch: topk indices,\\nthen self.attn / pair_attend) and reads the restored compressed keys through get_qsa_compressed_k_buffer (metadata\\nget_prefill/decode_mqa_inputs, qsa_indexer compress store), which had no wait: the last wait was layer L-1\'s, so top-k\\nfor the restored pages could read pages the H2D stream has not written yet (silent wrong top-k, no error).\\nFix: the getter calls self._wait_for_layer(layer_id) first. No-op unless a load-back is in flight (consumer_index < 0\\nfor decode / CUDA graphs); host pool transfers use qsa_compressed_k_buffer_pool directly, not this getter.\\nusage: hicqsawait_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [hicqsawait patch]\\"\\nREL = \\"sglang/srt/mem_cache/qsa_kv_pool.py\\"\\nOLD = (\\"    def get_qsa_compressed_k_buffer(self, layer_id: int) -> torch.Tensor:\\\\n\\"\\n       \\"        return self.qsa_compressed_k_buffer_pool[\\\\n\\")\\nNEW = (\\"    def get_qsa_compressed_k_buffer(self, layer_id: int) -> torch.Tensor:\\\\n\\"\\n       f\\"        self._wait_for_layer(layer_id)  {MARK}\\\\n\\"\\n       \\"        return self.qsa_compressed_k_buffer_pool[\\\\n\\")\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"hicqsawait_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"hicqsawait_patch: NOT patched\\")\\n    if s.count(OLD) != 1 or \\"def _wait_for_layer\\" in s:\\n        sys.exit(f\\"hicqsawait_patch: anchor count {s.count(OLD)} (want 1) or pool overrides _wait_for_layer; refusing\\")\\n    f.write_text(s.replace(OLD, NEW))\\n    print(\\"hicqsawait_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "hisparse_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Placeholder for the HiSparse->QSA offload patches (research/HISPARSE_QSA_FEASIBILITY.md). Nothing is patched yet: the Phase-0\\nbench (bench_hisparse.py) exercises the stock sglang.kernels.ops.kvcache.hisparse kernel directly. Exits 0 in every mode so the\\nbringup/phase-0 plumbing treats the key as applied.\\nusage: hisparse_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\n\\nif len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n    sys.exit(__doc__)\\nprint(\\"hisparse_patch: no source changes yet (bench only)\\")\\n", "idxshard_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s QSA decode indexer so the TP ranks split the compressed-key scan by sequence, gated by\\nSGLANG_QSA_INDEXER_SHARD=1 (unset = stock path, unchanged).\\n\\nWhy: the QSA indexer is a ReplicatedLinear (qsa_indexer.py), so at TP4 every GPU runs the same decode MQA scan over every\\ncompressed key of every running request, 12 times per step (~0.77 KB of keys per context token, 0.416 ms per 100k live\\nKV tokens at ~185 GB/s) and then the same fast_topk(512). The step-time fit step_ms = 26.97 + 1.092 n + 0.416 KV/100k\\nmakes that the term that grows with concurrency x context (10-16 ms at n 32-50 with 80k live ctx).\\n\\nWhat: rank r scans only the compressed pages whose ordinal o (position in the request\'s page list) has o % W == r\\n(page_table[:, r::W], a strided view; pages hold 16 compressed blocks), takes its local top-512 with the shipped\\nfast_topk, maps local positions back to global block ids ((j // 16 * W + r) * 16 + j % 16), all-gathers\\n[rows, 2 * 512] (values + int32 ids bit-cast to fp32) and merges the W * 512 candidates with torch.topk(512). The global\\ntop-512 is a subset of the union of the local top-512s, so the selected block set is exactly the stock set (up to ties\\nat the cut-off value, whose order the stock fast_topk leaves unspecified anyway). Valid ids come first, then -1,\\nas expand_qsa_block_indices requires. Every rank merges the identical gathered tensor, so all ranks agree.\\nPer-rank scan, logits fill and top-k all shrink by W; the cost is one small NCCL all-gather (rows x 4 KB per rank) and a\\nfew tiny kernels per layer. Pool memory is unchanged (every rank still stores and writes every compressed key).\\n\\nEnv (read once at import, per process):\\n  SGLANG_QSA_INDEXER_SHARD=1   enable (needs TP > 1 and attention TP == TP; otherwise logs why and stays stock)\\nDecode / target-verify / draft-extend only; prefill selection is untouched.\\nusage: idxshard_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [idxshard patch]\\"\\nREL_MQA = \\"sglang/srt/layers/attention/qsa/mqa.py\\"\\nREL_IDX = \\"sglang/srt/layers/attention/qsa/qsa_indexer.py\\"\\n\\nAPPEND = f\'\'\'\\n\\n{MARK} colab/perf_patches/idxshard_patch.py: sequence-sharded decode top-k (exact), SGLANG_QSA_INDEXER_SHARD=1.\\nimport os as _idxs_os\\n\\n_IDXSHARD_ENV = _idxs_os.environ.get(\\"SGLANG_QSA_INDEXER_SHARD\\", \\"\\").strip().lower() in (\\"1\\", \\"true\\", \\"yes\\")\\n_IDXSHARD_STATE = {{}}\\n\\n\\ndef idxshard_group():\\n    \\"\\"\\"The TP group when sequence-sharded selection applies, else None (decided once per process).\\"\\"\\"\\n    if \\"group\\" in _IDXSHARD_STATE:\\n        return _IDXSHARD_STATE[\\"group\\"]\\n    import logging as _logging\\n\\n    group, why = None, \\"SGLANG_QSA_INDEXER_SHARD unset\\"\\n    if _IDXSHARD_ENV:\\n        try:\\n            from sglang.srt.distributed import get_tp_group\\n\\n            g = get_tp_group()\\n            why = None\\n            if g.world_size < 2:\\n                why = \\"TP size 1\\"\\n            else:\\n                try:\\n                    from sglang.srt.layers.dp_attention import get_attention_tp_size\\n\\n                    if get_attention_tp_size() != g.world_size:\\n                        why = \\"attention TP group != TP group (DP attention)\\"\\n                except Exception:\\n                    pass\\n            if why is None:\\n                group = g\\n        except Exception as exc:\\n            why = f\\"{{type(exc).__name__}}: {{exc}}\\"\\n    if _IDXSHARD_ENV:\\n        _logging.getLogger(__name__).warning(\\n            \\"[idxshard patch] sequence-sharded QSA decode selection %s\\",\\n            f\\"ON (world {{group.world_size}}, rank {{group.rank_in_group}})\\" if group is not None else f\\"OFF ({{why}})\\",\\n        )\\n    _IDXSHARD_STATE[\\"group\\"] = group\\n    return group\\n\\n\\ndef qsa_mqa_topk_sharded(q, k_cache, page_table, context_lens, rank, world, gather, topk=512):\\n    \\"\\"\\"Exact sequence-sharded decode selection. gather(pay [rows, 2*topk] fp32) -> [world, rows, 2*topk].\\n    Returns global block ids [rows, topk] int32 (valid first, then -1), like fast_topk over the full logits.\\"\\"\\"\\n    from sglang.kernels.ops.elementwise.fast_topk import fast_topk\\n\\n    rows = q.shape[0]\\n    ps = int(k_cache.shape[1])\\n    lpt = page_table[:, rank::world].contiguous()\\n    cl = context_lens.to(device=q.device, dtype=torch.int32)\\n    full = torch.div(cl, ps, rounding_mode=\\"floor\\")\\n    tail = cl - full * ps\\n    nfull_owned = torch.clamp(torch.div(full - rank + (world - 1), world, rounding_mode=\\"floor\\"), min=0)\\n    tail_owned = (torch.remainder(full, world) == rank) & (tail > 0)\\n    local_len = (nfull_owned * ps + torch.where(tail_owned, tail, torch.zeros_like(tail))).to(torch.int32)\\n    logits = qsa_mqa_decode(q, k_cache, lpt, local_len, int(lpt.shape[1]) * ps)\\n    lidx = fast_topk(logits, local_len, topk=topk, row_starts=None)\\n    valid = lidx >= 0\\n    lval = torch.where(valid, logits.gather(1, lidx.clamp(min=0).long()), logits.new_full((), -float(\\"inf\\")))\\n    page_local = torch.div(lidx, ps, rounding_mode=\\"floor\\")\\n    gidx = torch.where(valid, ((page_local * world + rank) * ps + (lidx - page_local * ps)).to(torch.int32),\\n                       torch.full_like(lidx, -1))\\n    pay = torch.cat([lval, gidx.view(torch.float32)], dim=1).contiguous()\\n    allp = gather(pay)\\n    cv = allp[:, :, :topk].permute(1, 0, 2).reshape(rows, world * topk)\\n    ci = allp[:, :, topk:].permute(1, 0, 2).reshape(rows, world * topk).contiguous().view(torch.int32)\\n    tv, ti = torch.topk(cv, topk, dim=1)\\n    out = ci.gather(1, ti)\\n    return torch.where(torch.isfinite(tv), out, torch.full_like(out, -1)).to(torch.int32)\\n\\n\\ndef qsa_mqa_topk_sharded_tp(q, k_cache, page_table, context_lens, topk=512):\\n    group = idxshard_group()\\n    world = group.world_size\\n\\n    def gather(pay):\\n        out = torch.empty((world * pay.shape[0], pay.shape[1]), dtype=pay.dtype, device=pay.device)\\n        group.all_gather_into_tensor(out, pay)\\n        return out.view(world, pay.shape[0], pay.shape[1])\\n\\n    return qsa_mqa_topk_sharded(q, k_cache, page_table, context_lens, group.rank_in_group, world, gather, topk)\\n\'\'\'\\n\\nEDITS_IDX = [\\n    (\\"from sglang.srt.layers.attention.qsa.mqa import qsa_mqa_decode, qsa_mqa_prefill\\\\n\\",\\n     \\"from sglang.srt.layers.attention.qsa.mqa import qsa_mqa_decode, qsa_mqa_prefill\\\\n\\"\\n     f\\"from sglang.srt.layers.attention.qsa import mqa as _idxshard_mqa  {MARK}\\\\n\\"),\\n    (\\"        sequence_lengths: torch.Tensor,\\\\n    ) -> torch.Tensor:\\\\n        logits = qsa_mqa_decode(\\\\n\\",\\n     \\"        sequence_lengths: torch.Tensor,\\\\n    ) -> torch.Tensor:\\\\n\\"\\n     f\\"        if (  {MARK} exact sequence-sharded selection across the TP ranks\\\\n\\"\\n     \\"            self.block_topk == 512\\\\n\\"\\n     \\"            and q.is_cuda\\\\n\\"\\n     \\"            and getattr(_idxshard_mqa, \\\\\\"_IDXSHARD_ENV\\\\\\", False)\\\\n\\"\\n     \\"            and _idxshard_mqa.idxshard_group() is not None\\\\n\\"\\n     \\"        ):\\\\n\\"\\n     \\"            return expand_qsa_block_indices(\\\\n\\"\\n     \\"                _idxshard_mqa.qsa_mqa_topk_sharded_tp(\\\\n\\"\\n     \\"                    q, compressed_cache, compressed_page_table, compressed_lengths, self.block_topk\\\\n\\"\\n     \\"                ),\\\\n\\"\\n     \\"                query_positions,\\\\n\\"\\n     \\"                sequence_lengths,\\\\n\\"\\n     \\"                compress_ratio=self.compress_ratio,\\\\n\\"\\n     \\"                token_topk=self.token_topk,\\\\n\\"\\n     \\"            )\\\\n\\"\\n     \\"        logits = qsa_mqa_decode(\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    root = Path(sys.argv[1])\\n    fm, fi = root / REL_MQA, root / REL_IDX\\n    sm, si = fm.read_text(), fi.read_text()\\n    done = (MARK in sm, MARK in si)\\n    if all(done):\\n        print(\\"idxshard_patch: already patched\\")\\n        return\\n    if any(done):\\n        sys.exit(f\\"idxshard_patch: PARTIALLY patched (mqa {done[0]}, indexer {done[1]}); refusing\\")\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"idxshard_patch: NOT patched\\")\\n    if sm.count(\\"\\\\ndef qsa_mqa_decode(\\\\n\\") != 1 or sm.count(\\"import torch\\\\n\\") < 1:\\n        sys.exit(\\"idxshard_patch: mqa.py anchors changed; refusing to patch\\")\\n    for old, _new in EDITS_IDX:\\n        if si.count(old) != 1:\\n            sys.exit(f\\"idxshard_patch: anchor x{si.count(old)} (need 1): {old.strip()[:70]!r}; refusing to patch\\")\\n    for old, new in EDITS_IDX:\\n        si = si.replace(old, new)\\n    fm.write_text(sm.rstrip(\\"\\\\n\\") + \\"\\\\n\\" + APPEND)\\n    fi.write_text(si)\\n    print(\\"idxshard_patch: patched\\", fm, fi)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "mambafinal_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch SGLang\'s UnifiedRadixCache (srt/mem_cache/unified_radix_cache.py) so the stock \\"sparse-prefill v2\\"\\npolicy (mamba_prefill_final_only: no per-chunk mamba checkpoint, only the cold-start branch + the prefill-end state) can\\nalso run under extra_buffer_lazy when env SGLANG_MAMBA_PREFILL_FINAL_ONLY_LAZY=1 (unset / 0 = stock: off under lazy).\\n\\nWhy: with extra_buffer_lazy every CHUNK (1024) of a chunked prefill calls cache_unfinished_req(chunked=True), which\\nallocates a fresh mamba slot and donates the tracked one to the tree. An 85k-token pin re-prefill thus allocates ~83 slots\\nout of a pool of 96 (36 running x 2 = 72 held), and each allocation LRU-evicts the oldest evictable mamba state -- an\\nidle session\'s leaf, whose KV is dropped with it -- so that session re-prefills from scratch on its next turn and the\\ncascade feeds itself (v92 smoke: 19 mid-session misses, 286k re-prefill tokens). Final-only skips the per-chunk inserts:\\nthe new KV stays request-owned until prefill end, exactly as in the stock non-lazy path.\\nLazy-safe: during prefill lazy mode tracks into one slot (last_track_idx = next_track_idx, no swap) that is only written,\\nnever read, so overwriting it chunk after chunk is what the stock lazy path already does between donations; the\\nprefill-end / finish inserts (chunked=False) are unchanged. Exact: no kernel or numeric change, cache policy only.\\nusage: mambafinal_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [mambafinal patch]\\"\\nREL = \\"sglang/srt/mem_cache/unified_radix_cache.py\\"\\n\\nANCHOR = \\"        self.mamba_prefill_final_only = (\\\\n\\"\\nOLD = \\"            and not params.enable_mamba_extra_buffer_lazy\\\\n\\"\\nNEW = (\\"            and (not params.enable_mamba_extra_buffer_lazy  \\" + MARK + \\" opt-in under lazy\\\\n\\"\\n       \\"                 or os.environ.get(\\\\\\"SGLANG_MAMBA_PREFILL_FINAL_ONLY_LAZY\\\\\\", \\\\\\"0\\\\\\").strip() == \\\\\\"1\\\\\\")\\\\n\\")\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"mambafinal_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"mambafinal_patch: NOT patched\\")\\n    i = s.find(ANCHOR)\\n    j = s.find(OLD, i) if i >= 0 else -1\\n    if s.count(ANCHOR) != 1 or s.count(OLD) != 1 or j < 0 or j - i > 400 or \\"\\\\nimport os\\\\n\\" not in s:\\n        sys.exit(f\\"mambafinal_patch: anchors init x{s.count(ANCHOR)} lazy x{s.count(OLD)}; refusing to patch\\")\\n    s = s[:j] + NEW + s[j + len(OLD):]\\n    f.write_text(s)\\n    print(\\"mambafinal_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "mambaratio_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch SGLang\'s KV-cache configurator (srt/mem_cache/kv_cache_configurator.py) so the mamba-state slots\\nbudgeted per running request can be set by env SGLANG_MAMBA_SLOTS_PER_REQ=<int> (unset / 0 = the stock ratio, unchanged).\\n\\nWhy: _calculate_mamba_ratio() caps max_running_requests at max_mamba_cache_size // ratio, and with extra_buffer_lazy +\\nSGLANG_OPT_MAMBA_SKIP_DECODE_LOCK the stock ratio is 3 (base 3 - 1 skip-lock drop + 1 lazy ping-pong), so MAXREQ 40 needs\\nMAMBA 120 = 1.66 GB/GPU of GDN state (13.8 MB per slot on 4x L4). Under the real v86 agent load (serve.log, 1852 decode\\nlines) the pool held 2.08 slots per running request at 32-35 running (max 74 of 144, usage <= 0.51): lazy mode keeps one\\nworking slot + one tracked checkpoint per request and allocates the transient ping-pong slot only at 64-token track\\nboundaries. With ratio 2, MAMBA 96 keeps MAXREQ 40 (96 // 2 = 48) and frees 24 slots = 0.33 GB/GPU = ~100k KV tokens.\\nSafe by construction: the ratio is only this static cap (and the spec-decode intermediate reservation, unused here);\\nadmission is still gated on the real free + evictable mamba slots (schedule_policy rem_mamba_slots), and a failed\\ntransient boundary alloc degrades tracking in place (schedule_batch.mamba_lazy_prealloc_at_boundary). Exact: no kernel\\nor numeric change.\\nusage: mambaratio_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [mambaratio patch]\\"\\nREL = \\"sglang/srt/mem_cache/kv_cache_configurator.py\\"\\n\\nANCHOR_FN = \\"    def _calculate_mamba_ratio(self) -> int:\\\\n\\"\\nOLD = \\"        return base + additional_ratio\\\\n\\"\\nNEW = (f\\"        import os as _os  {MARK} SGLANG_MAMBA_SLOTS_PER_REQ overrides the slots budgeted per running request\\\\n\\"\\n       \\"        _override = int(_os.environ.get(\\\\\\"SGLANG_MAMBA_SLOTS_PER_REQ\\\\\\", \\\\\\"0\\\\\\") or 0)\\\\n\\"\\n       \\"        if _override > 0:\\\\n\\"\\n       \\"            logger.info(\\\\\\"%s mamba slots per request %d (stock %d)\\\\\\", \\\\\\"\\" + MARK + \\"\\\\\\", _override, base + additional_ratio)\\\\n\\"\\n       \\"            return _override\\\\n\\"\\n       \\"        return base + additional_ratio\\\\n\\")\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"mambaratio_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"mambaratio_patch: NOT patched\\")\\n    i = s.find(ANCHOR_FN)\\n    j = s.find(OLD, i) if i >= 0 else -1\\n    nxt = s.find(\\"\\\\n    def \\", i + len(ANCHOR_FN)) if i >= 0 else -1\\n    if s.count(ANCHOR_FN) != 1 or s.count(OLD) != 1 or j < 0 or (nxt >= 0 and j > nxt) or \\"logger = \\" not in s:\\n        sys.exit(f\\"mambaratio_patch: anchors fn x{s.count(ANCHOR_FN)} return x{s.count(OLD)}; refusing to patch\\")\\n    s = s[:j] + NEW + s[j + len(OLD):]\\n    f.write_text(s)\\n    print(\\"mambaratio_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "marlin_ep_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s GPTQ-Marlin MoE kernel (hardware_backend/gpu/quantization/gptq_kernels.py) for expert\\nparallelism. Under --ep-size > 1 the standard dispatcher rewrites topk_ids to LOCAL ids with -1 for experts owned by other\\nranks, but GPTQMarlinMoEKernel.apply never hands the Marlin runner an expert_map, so fused_marlin_moe runs with\\nis_ep=False (and, at M == 1, the single-token align path) and the -1 ids index weights at expert -1: garbage -> NaN\\n(v59 tensor dump on 4xL4 tp4/ep4: layer 0 experts finite, layer 1 experts all-NaN, every output token = \'!\'). The\\ncompressed-tensors WNA16 MoE scheme already passes the dispatcher\'s local_expert_mapping; this mirrors it.\\nusage: marlin_ep_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [marlin-ep patch]\\"\\nANCHOR = (\\"            weight_bits=self.quant_config.weight_bits,\\\\n\\"\\n          \\"            is_k_full=self.is_k_full,\\\\n\\"\\n          \\"        )\\\\n\\")\\nADD = (f\\"        _disp = getattr(layer, \\\\\\"dispatcher\\\\\\", None)  {MARK} EP: mask non-local (-1) experts in Marlin\\\\n\\"\\n       \\"        _emap = getattr(_disp, \\\\\\"local_expert_mapping\\\\\\", None)\\\\n\\"\\n       \\"        if _emap is not None:\\\\n\\"\\n       \\"            quant_info.expert_map = _emap\\\\n\\"\\n       \\"            quant_info.global_num_experts = self.moe_runner_config.num_experts\\\\n\\")\\n\\n\\ndef main():\\n    f = Path(sys.argv[1]) / \\"sglang/srt/hardware_backend/gpu/quantization/gptq_kernels.py\\"\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"marlin_ep_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"marlin_ep_patch: NOT patched\\")\\n    if s.count(ANCHOR) != 1:\\n        sys.exit(\\"marlin_ep_patch: anchor not found exactly once; refusing to patch\\")\\n    f.write_text(s.replace(ANCHOR, ANCHOR + ADD))\\n    print(\\"marlin_ep_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "marlinclean_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s GPTQ-Marlin MoE glue (srt/layers/moe/fused_moe_triton/fused_marlin_moe.py) with an\\nEP-only cleanup, gated by SGLANG_MARLIN_EP_CLEAN=1 (read once at import, per process). Unset / 0 = the stock code path.\\n\\nThe clean path is taken only when expert_map is set (our colab/marlin_ep_patch.py passes the dispatcher\'s\\nlocal_expert_mapping under --ep-size > 1) and the weights are not MXFP4/NVFP4. topk_ids are then already LOCAL ids in\\n[-1, E) (standard dispatcher, standard.py:227; -1 = expert owned by another rank) and Marlin\'s is_ep kernel skips every\\nblock whose expert id is -1, so the rows of those (token, slot) pairs are never written in either GEMM. Changes:\\n  1. moe_align_block_size gets E + 1 buckets (E = local experts = w1.shape[0], 96 at 384/EP4) instead of\\n     global_num_experts + 1 (385): the ids never exceed E - 1, the per-block expert ids / padding are identical.\\n     (Done here instead of in marlin_ep_patch.py, which keeps passing the global count.)\\n  2. intermediate_cache13 is torch.empty instead of torch.zeros (fused_marlin_moe.py:290): rows of skipped slots are\\n     never read by GEMM2 (it gathers A rows through the same sorted ids, skipping -1 blocks) nor by the masked sum.\\n  3. the intermediate_cache3.zero_() before GEMM2 (fused_marlin_moe.py:365-366) is dropped.\\n  4. GEMM2 no longer multiplies by the top-k weight (bf16 epilogue); the final reduce is the wheel\'s masked\\n     moe_fused_mul_sum (kernels/ops/moe/moe_fused_mul_sum.py, is_ep=True: slots with topk_ids < 0 are masked loads, so\\n     garbage in skipped rows is never read) instead of moe_sum_reduce over zeroed rows (fused_marlin_moe.py:423-427).\\n     The weight multiply moves from bf16 (GEMM2 epilogue) to fp32 (sum kernel): output differs by bf16 rounding only.\\n     expert_map is deliberately NOT passed to moe_fused_mul_sum: it is the GLOBAL->local map, topk_ids are local.\\nusage: marlinclean_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [marlinclean patch]\\"\\nREL = \\"sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py\\"\\nSUM_REL = \\"sglang/kernels/ops/moe/moe_fused_mul_sum.py\\"\\n\\n# Read-only anchors (text, expected count): the semantics the clean path relies on.\\nREQUIRED = [\\n    (\\"        is_ep=expert_map is not None,\\\\n\\", 2),  # both GEMMs run the -1-skipping kernel iff expert_map is set\\n    (\\"    intermediate_cache1 = intermediate_cache13[: M * topk_ids.shape[1] * gemm1_n]\\\\n\\", 1),\\n    (\\"    intermediate_cache3 = intermediate_cache13[: M * topk_ids.shape[1] * K]\\\\n\\", 1),\\n    (\\"            topk_ids, block_size_m, global_num_experts\\\\n\\", 1),  # align buckets = global_num_experts + 1\\n    (\\"    ).view(-1, topk, K)\\\\n\\", 1),  # cache3 is [M, topk, K] contiguous at the final reduce\\n    (\\"    output = zero_copy_context.get_moe_output(hidden_states)\\\\n\\", 1),\\n]\\n# moe_fused_mul_sum contract: is_ep masks slots with id < 0; signature order used below.\\nSUM_REQUIRED = [\\n    (\\"def moe_fused_mul_sum(\\\\n\\", 1),\\n    (\\"    topk_ids: torch.Tensor | None = None,\\\\n\\", 1),\\n    (\\"    expert_map: torch.Tensor | None = None,\\\\n\\", 1),\\n    (\\"    routed_scaling_factor: float | None = None,\\\\n\\", 1),\\n    (\\"    is_ep: bool = False,\\\\n\\", 1),\\n    (\\"        elif is_ep:\\\\n\\", 1),\\n    (\\"            expert_mask = id_val >= 0\\\\n\\", 1),\\n]\\n\\nHEADER = f\'\'\'\\n{MARK} colab/perf_patches/marlinclean_patch.py. SGLANG_MARLIN_EP_CLEAN=1 (read once at import) enables the EP-only\\n{MARK} cleanup in fused_marlin_moe (E+1 align buckets, empty cache, no zero_(), masked fp32 top-k sum); unset = stock.\\nimport logging as _mclean_logging  {MARK}\\nimport os as _mclean_os  {MARK}\\n\\n_MARLIN_EP_CLEAN = _mclean_os.environ.get(\\"SGLANG_MARLIN_EP_CLEAN\\", \\"\\").strip().lower() in (  {MARK}\\n    \\"1\\",\\n    \\"true\\",\\n    \\"yes\\",\\n    \\"on\\",\\n)\\n_MCLEAN_NOTED = []  {MARK} one log line per process on the first clean call\\n\\n\\ndef _mclean_note(global_num_experts, num_local):  {MARK}\\n    if not _MCLEAN_NOTED:\\n        _MCLEAN_NOTED.append(1)\\n        _mclean_logging.getLogger(__name__).warning(\\n            \\"[marlinclean patch] EP clean path active: align buckets %d -> %d, cache13 empty, no zero_(), \\"\\n            \\"masked fp32 top-k sum\\",\\n            global_num_experts + 1,\\n            num_local + 1,\\n        )\\n\\n\'\'\'\\n\\nEDITS = [\\n    # 1. module-level env flag (after the module\'s own _is_cuda probe)\\n    (\\"_is_cuda = is_cuda()\\\\n\\",\\n     \\"_is_cuda = is_cuda()\\\\n\\" + HEADER),\\n    # 2. decide the clean path once per call; local align bucket count\\n    (\\"    if global_num_experts == -1:\\\\n\\"\\n     \\"        global_num_experts = E\\\\n\\",\\n     \\"    if global_num_experts == -1:\\\\n\\"\\n     \\"        global_num_experts = E\\\\n\\"\\n     f\\"    {MARK} EP: topk_ids are LOCAL ids in [-1, E) (dispatcher local_expert_mapping); expert_map is only the\\\\n\\"\\n     f\\"    {MARK} is_ep flag here, so moe_align needs E + 1 buckets, not global + 1.\\\\n\\"\\n     \\"    _ep_clean = (\\\\n\\"\\n     \\"        _MARLIN_EP_CLEAN\\\\n\\"\\n     \\"        and expert_map is not None\\\\n\\"\\n     \\"        and not is_mxfp4_marlin\\\\n\\"\\n     \\"        and w1_global_scale is None\\\\n\\"\\n     \\"    )\\\\n\\"\\n     \\"    if _ep_clean and global_num_experts > E:\\\\n\\"\\n     \\"        _mclean_note(global_num_experts, E)\\\\n\\"\\n     \\"        global_num_experts = E\\\\n\\"),\\n    # 3. zeros -> empty on the clean path (rows of skipped slots are never read)\\n    (\\"    intermediate_cache13 = torch.zeros(\\\\n\\",\\n     f\\"    intermediate_cache13 = (torch.empty if _ep_clean else torch.zeros)(  {MARK} clean: never-written rows unread\\\\n\\"),\\n    # 4. GEMM2 leaves the top-k weight to the fp32 masked sum on the clean path\\n    (\\"        top_k=1,\\\\n\\"\\n     \\"        mul_topk_weights=True,\\\\n\\",\\n     \\"        top_k=1,\\\\n\\"\\n     f\\"        mul_topk_weights=not _ep_clean,  {MARK} clean: weight applied in fp32 by moe_fused_mul_sum\\\\n\\"),\\n    # 5. drop the cache3 memset on the clean path\\n    (\\"    if expert_map is not None:\\\\n\\"\\n     \\"        intermediate_cache3.zero_()\\\\n\\",\\n     f\\"    if expert_map is not None and not _ep_clean:  {MARK} clean: masked sum never reads skipped rows\\\\n\\"\\n     \\"        intermediate_cache3.zero_()\\\\n\\"),\\n    # 6. masked weighted top-k sum instead of moe_sum_reduce over zeroed rows\\n    (\\"    else:\\\\n\\"\\n     \\"        if routed_scaling_factor is None:\\\\n\\"\\n     \\"            routed_scaling_factor = 1.0\\\\n\\"\\n     \\"\\\\n\\"\\n     \\"        moe_sum_reduce(\\\\n\\",\\n     \\"    else:\\\\n\\"\\n     f\\"        if _ep_clean:  {MARK} masked fp32 sum: slots with local id < 0 are masked loads (never read)\\\\n\\"\\n     \\"            from sglang.kernels.ops.moe.moe_fused_mul_sum import moe_fused_mul_sum\\\\n\\"\\n     \\"\\\\n\\"\\n     \\"            if M > 0:\\\\n\\"\\n     \\"                _out = output if output.is_contiguous() else None\\\\n\\"\\n     \\"                _res = moe_fused_mul_sum(\\\\n\\"\\n     \\"                    inputs=intermediate_cache3,\\\\n\\"\\n     \\"                    topk_weights=topk_weights.contiguous(),\\\\n\\"\\n     \\"                    outputs=_out,\\\\n\\"\\n     \\"                    topk_ids=topk_ids.contiguous(),\\\\n\\"\\n     f\\"                    expert_map=None,  {MARK} NOT the global->local map: ids are already local\\\\n\\"\\n     \\"                    routed_scaling_factor=routed_scaling_factor,\\\\n\\"\\n     \\"                    is_ep=True,\\\\n\\"\\n     \\"                )\\\\n\\"\\n     \\"                if _out is None:\\\\n\\"\\n     \\"                    output.copy_(_res)\\\\n\\"\\n     \\"            return output\\\\n\\"\\n     \\"        if routed_scaling_factor is None:\\\\n\\"\\n     \\"            routed_scaling_factor = 1.0\\\\n\\"\\n     \\"\\\\n\\"\\n     \\"        moe_sum_reduce(\\\\n\\"),\\n]\\n\\n\\ndef _check_anchors(s, anchors, what):\\n    for anchor, n in anchors:\\n        if s.count(anchor) != n:\\n            raise ValueError(f\\"{what}: anchor x{s.count(anchor)} (need {n}): {anchor.strip()[:72]!r}\\")\\n\\n\\ndef apply_text(s):\\n    \\"\\"\\"Return the patched fused_marlin_moe.py text. Raises ValueError if any anchor is not found exactly as expected.\\n    The caller handles idempotency (MARK already present). Also used by bench_marlinclean.py.\\"\\"\\"\\n    _check_anchors(s, REQUIRED, \\"fused_marlin_moe.py\\")\\n    _check_anchors(s, [(old, 1) for old, _ in EDITS], \\"fused_marlin_moe.py\\")\\n    for old, new in EDITS:\\n        s = s.replace(old, new)\\n    return s\\n\\n\\ndef revert_text(s):\\n    \\"\\"\\"Inverse of apply_text: the stock text of a file patched by THIS version of the patch. Raises ValueError unless\\n    the result round-trips exactly (apply_text(revert_text(s)) == s). bench_marlinclean.py uses it to rebuild the true\\n    unpatched reference on a venv that is already patched.\\"\\"\\"\\n    orig = s\\n    for old, new in reversed(EDITS):\\n        if s.count(new) != 1:\\n            raise ValueError(f\\"revert: patched block x{s.count(new)} (need 1): {new.strip()[:72]!r}\\")\\n        s = s.replace(new, old)\\n    if MARK in s or apply_text(s) != orig:\\n        raise ValueError(\\"revert: result does not round-trip (file patched by another version?)\\")\\n    return s\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    site = Path(sys.argv[1])\\n    f = site / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"marlinclean_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"marlinclean_patch: NOT patched\\")\\n    try:\\n        _check_anchors((site / SUM_REL).read_text(), SUM_REQUIRED, \\"moe_fused_mul_sum.py\\")\\n        new = apply_text(s)\\n    except (OSError, ValueError) as exc:\\n        sys.exit(f\\"marlinclean_patch: {exc}; refusing to patch\\")\\n    f.write_text(new)\\n    print(\\"marlinclean_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "mqagrid_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s QSA decode indexer (srt/layers/attention/qsa/mqa.py) so the TileLang paged-decode MQA\\nkernel can run with groups_per_cta > 1, gated by SGLANG_QSA_MQA_GROUPS_PER_CTA=G (unset or 1 = stock call, unchanged).\\n\\nWhy: _tilelang_qsa_mqa_decode_kernel already takes groups_per_cta (mqa.py:210, grid mqa.py:234-238, loop mqa.py:246),\\nbut tilelang_qsa_mqa_decode never passes it (mqa.py:372-374), so the grid is (rows, ceil(max_pages / sub_pages)). Under\\nCUDA graphs max_pages is the static graph page-table width: CTX 131072 / ratio 4 / compressed page 16 = 2048 pages,\\nsub_pages 4 -> 512 CTAs per row. At 6k-24k context only 24-96 of those 512 groups hold keys; the other 80-95% of CTAs\\nstill launch, copy Q to shared memory and exit, 12 times per decode step. With G > 1 one CTA scores G consecutive\\n64-key groups (the kernel\'s own T.Pipelined loop), the grid shrinks by G and most empty CTAs disappear. Each logit is\\nstill produced by the same per-group GEMM + head reduction, so logits should be bit-identical (bench_mqagrid.py checks).\\n\\nEnv (read once at import, per process):\\n  SGLANG_QSA_MQA_GROUPS_PER_CTA=G   G > 1 enables; unset / \\"\\" / 1 = the stock kernel call with the stock kwargs\\n  SGLANG_QSA_MQA_NUM_STAGES=S       optional, only honoured with G > 1: overrides the kernel\'s num_stages=3\\nA G > 1 kernel that fails to COMPILE logs a warning and falls back to the stock grid (runtime faults are not caught).\\nThe torch.full(-inf) logits init (mqa.py:351-356) is deliberately kept: the kernel writes only positions < context_len.\\nusage: mqagrid_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [mqagrid patch]\\"\\nREL = \\"sglang/srt/layers/attention/qsa/mqa.py\\"\\n\\n# Read-only anchors: the patch is only meaningful if the kernel still has this groups_per_cta contract.\\nREQUIRED = [\\n    \\"        groups_per_cta: int = 1,\\\\n\\",\\n    \\"                T.ceildiv(T.ceildiv(max_pages, sub_pages), groups_per_cta),\\\\n\\",\\n    \\"                for gi in T.Pipelined(groups_per_cta, num_stages=num_stages):\\\\n\\",\\n    \\"                    group = group_block * groups_per_cta + gi\\\\n\\",\\n]\\n\\nHELPER = f\'\'\'{MARK} TileLang decode-MQA grid, colab/perf_patches/mqagrid_patch.py. One CTA scores G consecutive 64-key\\n{MARK} groups, so the grid is (rows, ceil(groups / G)) instead of (rows, groups). Env, read once at import:\\n{MARK}   SGLANG_QSA_MQA_GROUPS_PER_CTA=G  (unset/1 = stock call with the stock kwargs = the stock TileLang kernel)\\n{MARK}   SGLANG_QSA_MQA_NUM_STAGES=S      (only with G > 1; overrides the kernel\'s num_stages=3)\\ndef _mqagrid_env_kwargs():\\n    import logging as _logging\\n    import os as _os\\n\\n    log = _logging.getLogger(__name__)\\n    raw_g = _os.environ.get(\\"SGLANG_QSA_MQA_GROUPS_PER_CTA\\", \\"\\").strip()\\n    raw_s = _os.environ.get(\\"SGLANG_QSA_MQA_NUM_STAGES\\", \\"\\").strip()\\n    if not raw_g:\\n        return {{}}\\n    try:\\n        groups = int(raw_g)\\n        stages = int(raw_s) if raw_s else None\\n    except ValueError:\\n        groups, stages = 0, None\\n    if groups == 1:\\n        return {{}}\\n    if not 1 < groups <= 4096 or (stages is not None and not 0 <= stages <= 8):\\n        log.warning(\\n            \\"[mqagrid patch] ignoring SGLANG_QSA_MQA_GROUPS_PER_CTA=%r SGLANG_QSA_MQA_NUM_STAGES=%r; stock grid\\",\\n            raw_g,\\n            raw_s,\\n        )\\n        return {{}}\\n    kwargs = {{\\"groups_per_cta\\": groups}}\\n    if stages is not None:\\n        kwargs[\\"num_stages\\"] = stages\\n    log.warning(\\"[mqagrid patch] TileLang QSA decode MQA kernel kwargs %s\\", kwargs)\\n    return kwargs\\n\\n\\n_MQAGRID_KW = _mqagrid_env_kwargs()\\n\\n\\ndef _mqagrid_decode_kernel(heads, head_dim, page_size):\\n    global _MQAGRID_KW\\n    if _MQAGRID_KW:\\n        try:\\n            return _tilelang_qsa_mqa_decode_kernel(\\n                heads=heads, head_dim=head_dim, page_size=page_size, **_MQAGRID_KW\\n            )\\n        except Exception as exc:  # compile failure: keep serving on the stock grid\\n            import logging as _logging\\n\\n            _logging.getLogger(__name__).warning(\\n                \\"[mqagrid patch] decode MQA kernel %s failed to compile (%s: %s); using the stock grid\\",\\n                _MQAGRID_KW,\\n                type(exc).__name__,\\n                str(exc)[:300],\\n            )\\n            _MQAGRID_KW = {{}}\\n    return _tilelang_qsa_mqa_decode_kernel(\\n        heads=heads, head_dim=head_dim, page_size=page_size\\n    )\\n\\n\\n\'\'\'\\n\\nEDITS = [\\n    # module-level env parsing + kernel getter, inserted right before the decode wrapper\\n    (\\"\\\\n\\\\ndef tilelang_qsa_mqa_decode(\\\\n\\",\\n     \\"\\\\n\\\\n\\" + HELPER + \\"def tilelang_qsa_mqa_decode(\\\\n\\"),\\n    # the decode wrapper\'s kernel call (mqa.py:372-374): same kwargs, routed through the getter\\n    (\\"    _tilelang_qsa_mqa_decode_kernel(\\\\n\\"\\n     \\"        heads=kernel_heads, head_dim=head_dim, page_size=page_size\\\\n\\"\\n     \\"    )(\\\\n\\",\\n     f\\"    _mqagrid_decode_kernel(  {MARK} stock kernel unless SGLANG_QSA_MQA_GROUPS_PER_CTA > 1\\\\n\\"\\n     \\"        heads=kernel_heads, head_dim=head_dim, page_size=page_size\\\\n\\"\\n     \\"    )(\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    f = Path(sys.argv[1]) / REL\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"mqagrid_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"mqagrid_patch: NOT patched\\")\\n    for anchor in REQUIRED:\\n        if s.count(anchor) != 1:\\n            sys.exit(f\\"mqagrid_patch: kernel anchor x{s.count(anchor)} (need 1): {anchor.strip()[:70]!r}; refusing to patch\\")\\n    for old, new in EDITS:\\n        if s.count(old) != 1:\\n            sys.exit(f\\"mqagrid_patch: anchor x{s.count(old)} (need 1): {old.strip()[:70]!r}; refusing to patch\\")\\n    for old, new in EDITS:\\n        s = s.replace(old, new)\\n    f.write_text(s)\\n    print(\\"mqagrid_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "pairown_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal so the two TP ranks that share a KV head (\\"KV-head pair\\", ranks (0,1) and (2,3) at TP4 with\\n2 KV heads) store each QSA layer\'s K/V, compressed index keys and key-state ring ONCE per pair instead of on both ranks.\\nGated by SGLANG_QSA_PAIR_OWN=1 (unset = stock path, nothing changes).\\n\\nWhy: at TP4 the 2 KV heads are replicated, so both ranks of a pair hold byte-identical K/V (6.1 KB/token/GPU over the 12 QSA\\nlayers) and byte-identical compressed index keys (0.77 KB/token/GPU: the indexer is a ReplicatedLinear). KV capacity is\\nthe concurrency limit (n ~ 20 at ~50k live ctx), so storing every layer once per pair roughly doubles it.\\n\\nWhat: QSA layer ordinal j (position in the sorted full-attention layer ids) is owned by pair rank j % 2.\\n  * pool: only the owned layer ids are given to QSATokenToKVPool, so K/V, compressed keys and key-state rings shrink to 6\\n    layers and the pool configurator sizes the cell for 6 layers (capacity ~2x, equal on all ranks).\\n  * layer (Qwen4ExpAttentionDecoderLayer._self_attention_impl): every rank still runs the qkv projection (q, k, v, gate).\\n    Exchange 1 (pair all-gather) sends each rank\'s 6 q heads to the owner; the owner runs the indexer + sparse attention\\n    for BOTH ranks\' q heads (12 heads, one KV head: the Triton kernels derive the group size from the shapes) and writes\\n    the new K/V; exchange 2 returns the partner\'s raw attention output; each rank applies its OWN sigmoid gate and o_proj\\n    (+ the usual TP all-reduce). The non-owner skips the indexer, the KV write and the attention.\\n  * exact in math: identical q/k/v/top-k as stock; only where the work runs and what is stored changes.\\nCost: 2 small pair all-gathers per QSA layer (24 per step) ~ 0.05 ms each at decode shapes; the critical path per layer is the\\nowner\'s attention (same KV bytes read as stock), so step time is ~stock + the exchanges, while KV capacity doubles.\\n\\nEnv (read once per process):\\n  SGLANG_QSA_PAIR_OWN=1            enable (needs TP = 2 x KV heads, attention TP == TP; else logs why and stays stock)\\n  SGLANG_QSA_PAIR_OWN_GROUP=pair   exchange group: \\"pair\\" = a dedicated 2-rank NCCL group (default), \\"tp\\" = the TP group\\nusage: pairown_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [pairown patch]\\"\\nREL_MOD = \\"sglang/srt/layers/attention/qsa/pairown.py\\"\\nREL_KVC = \\"sglang/srt/mem_cache/kv_cache_configurator.py\\"\\nREL_POOLCFG = \\"sglang/srt/model_executor/pool_configurator.py\\"\\nREL_Q4 = \\"sglang/srt/models/qwen4_exp.py\\"\\n\\nMODULE_SRC = r\'\'\'# [pairown patch] installed by colab/perf_patches/pairown_patch.py (SGLANG_QSA_PAIR_OWN=1)\\n\\"\\"\\"QSA pair-owned KV (layer ownership across the KV-head replica pair); see pairown_patch.py\'s docstring for the design.\\"\\"\\"\\nimport copy\\nimport logging\\nimport os\\n\\nimport torch\\n\\n_ENV = os.environ.get(\\"SGLANG_QSA_PAIR_OWN\\", \\"\\").strip().lower() in (\\"1\\", \\"true\\", \\"yes\\")\\n_GROUP_MODE = os.environ.get(\\"SGLANG_QSA_PAIR_OWN_GROUP\\", \\"pair\\").strip().lower()\\n_S = {}\\n_log = logging.getLogger(__name__)\\n\\n\\ndef configure(tp_size, tp_rank, total_kv_heads, full_ids):\\n    \\"\\"\\"Pure-python ownership map (no collectives). Returns the config dict, or None when pair ownership does not apply.\\"\\"\\"\\n    if \\"cfg\\" in _S:\\n        return _S[\\"cfg\\"]\\n    cfg, why = None, \\"SGLANG_QSA_PAIR_OWN unset\\"\\n    if _ENV:\\n        reps = tp_size // total_kv_heads if total_kv_heads > 0 and tp_size % total_kv_heads == 0 else 0\\n        ids = sorted(int(i) for i in full_ids)\\n        if reps != 2:\\n            why = f\\"needs exactly 2 TP ranks per KV head (tp {tp_size}, kv heads {total_kv_heads})\\"\\n        elif len(ids) < 2:\\n            why = f\\"needs >= 2 full-attention layers (got {len(ids)})\\"\\n        else:\\n            owner = {lid: j % reps for j, lid in enumerate(ids)}\\n            pair_rank, kvh = tp_rank % reps, tp_rank // reps\\n            cfg = {\\n                \\"R\\": reps,\\n                \\"tp_size\\": tp_size,\\n                \\"tp_rank\\": tp_rank,\\n                \\"kvh\\": kvh,\\n                \\"pair_rank\\": pair_rank,\\n                \\"owner\\": owner,\\n                \\"ranks\\": [kvh * reps + i for i in range(reps)],\\n                \\"all_ranks\\": [[h * reps + i for i in range(reps)] for h in range(total_kv_heads)],\\n                \\"owned\\": [lid for lid in ids if owner[lid] == pair_rank],\\n            }\\n            why = None\\n        _log.warning(\\n            \\"[pairown patch] QSA pair-owned KV %s\\",\\n            f\\"ON (tp rank {tp_rank}, pair {cfg[\'ranks\']}, owns layers {cfg[\'owned\']} of {ids})\\" if cfg else f\\"OFF ({why})\\",\\n        )\\n    _S[\\"cfg\\"] = cfg\\n    return cfg\\n\\n\\ndef get_cfg():\\n    return _S.get(\\"cfg\\")\\n\\n\\ndef owned_layer_ids(ids):\\n    \\"\\"\\"Subset of `ids` this rank stores; ids unknown to the ownership map (draft layer) are kept.\\"\\"\\"\\n    cfg = _S.get(\\"cfg\\")\\n    if cfg is None:\\n        return list(ids)\\n    return [i for i in ids if cfg[\\"owner\\"].get(i, cfg[\\"pair_rank\\"]) == cfg[\\"pair_rank\\"]]\\n\\n\\ndef ensure_group():\\n    \\"\\"\\"Collective: every TP rank must call it at the same point (new_group + NCCL communicator creation).\\"\\"\\"\\n    cfg = _S.get(\\"cfg\\")\\n    if cfg is None or \\"group\\" in _S:\\n        return\\n    from sglang.srt.distributed import get_tp_group\\n\\n    tp = get_tp_group()\\n    if _GROUP_MODE == \\"tp\\":\\n        _S[\\"group\\"], _S[\\"gidx0\\"] = tp, cfg[\\"ranks\\"][0]\\n    else:\\n        from sglang.srt.distributed.parallel_state import get_world_group, init_model_parallel_group\\n\\n        world = get_world_group()\\n        backend = torch.distributed.get_backend(world.device_group)\\n        group = init_model_parallel_group(\\n            cfg[\\"all_ranks\\"],\\n            world.local_rank,\\n            backend,\\n            use_custom_allreduce=False,\\n            use_mscclpp_allreduce=False,\\n            use_torch_symm_mem_allreduce=False,\\n            group_name=\\"qsa_pair\\",\\n        )\\n        _S[\\"group\\"], _S[\\"gidx0\\"] = group, 0\\n    _log.warning(\\"[pairown patch] exchange group %s (world %d)\\", _GROUP_MODE, _S[\\"group\\"].world_size)\\n\\n\\ndef prepare(kvc):\\n    \\"\\"\\"Called once from KVCacheConfigurator._resolve_memory_pool_config, before memory profiling, on every rank.\\"\\"\\"\\n    if not _ENV:\\n        return None\\n    if getattr(kvc, \\"is_draft_worker\\", False) or getattr(kvc, \\"mambaish_config\\", None) is None:\\n        return None\\n    from sglang.srt.distributed import get_tp_group\\n\\n    tp = get_tp_group()\\n    try:\\n        from sglang.srt.layers.dp_attention import get_attention_tp_size\\n\\n        if get_attention_tp_size() != tp.world_size:\\n            _log.warning(\\"[pairown patch] OFF (attention TP != TP)\\")\\n            _S.setdefault(\\"cfg\\", None)\\n            return None\\n    except Exception:\\n        pass\\n    cfg = configure(\\n        tp.world_size,\\n        tp.rank_in_group,\\n        int(kvc.model_config.get_total_num_kv_heads()),\\n        list(kvc.mambaish_config.full_attention_layer_ids),\\n    )\\n    if cfg is not None:\\n        ensure_group()\\n    return cfg\\n\\n\\nclass Plan:\\n    __slots__ = (\\"layer_id\\", \\"owner\\", \\"me\\", \\"is_owner\\", \\"R\\", \\"shim\\")\\n\\n    def __init__(self, layer_id, owner, me, R, shim):\\n        self.layer_id, self.owner, self.me, self.R = layer_id, owner, me, R\\n        self.is_owner = owner == me\\n        self.shim = shim\\n\\n\\ndef layer_plan(layer_mod):\\n    \\"\\"\\"Plan for a Qwen4ExpAttentionDecoderLayer, or None to run the stock path.\\"\\"\\"\\n    cfg = _S.get(\\"cfg\\")\\n    if cfg is None or \\"group\\" not in _S:\\n        return None\\n    lid = layer_mod.layer_id\\n    plans = _S.setdefault(\\"plans\\", {})\\n    plan = plans.get(lid)\\n    if plan is None:\\n        owner = cfg[\\"owner\\"].get(lid)\\n        if owner is None or not getattr(layer_mod, \\"is_qsa\\", False):\\n            plans[lid] = False\\n            return None\\n        shim = None\\n        if owner == cfg[\\"pair_rank\\"]:\\n            shim = copy.copy(layer_mod.attn)  # same layer_id / scaling / head_dim, wider q\\n            shim.tp_q_head_num = layer_mod.attn.tp_q_head_num * cfg[\\"R\\"]\\n        plan = plans[lid] = Plan(lid, owner, cfg[\\"pair_rank\\"], cfg[\\"R\\"], shim)\\n    return plan or None\\n\\n\\ndef gather(x):\\n    \\"\\"\\"All-gather x [rows, C] over the pair -> [R, rows, C] (slot i = pair rank i). CUDA-graph capturable.\\"\\"\\"\\n    group, R = _S[\\"group\\"], _S[\\"cfg\\"][\\"R\\"]\\n    out = torch.empty((group.world_size * x.shape[0], x.shape[1]), dtype=x.dtype, device=x.device)\\n    comm = getattr(group, \\"pynccl_comm\\", None)\\n    if comm is not None and torch.cuda.is_current_stream_capturing():\\n        with comm.change_state(enable=True):\\n            comm.all_gather(out, x)\\n    else:\\n        group.all_gather_into_tensor(out, x)\\n    i0 = _S[\\"gidx0\\"]\\n    return out.view(group.world_size, x.shape[0], x.shape[1])[i0 : i0 + R]\\n\\n\\ndef pair_attend(plan, layer_mod, q, k, v, forward_batch, attend, gather_fn=None):\\n    \\"\\"\\"Raw (pre-gate) attention output [rows, local_q_heads * head_dim] for THIS rank\'s q heads.\\n    `attend(q_wide, k3, v3, shim_layer)` runs the sparse attention on the owner (q_wide = both ranks\' heads, rank order).\\"\\"\\"\\n    gather_fn = gather_fn or gather\\n    rows = q.shape[0]\\n    qc = q.reshape(rows, -1)\\n    if not qc.is_contiguous():\\n        qc = qc.contiguous()\\n    if rows == 0:\\n        return qc\\n    qall = gather_fn(qc)  # exchange 1: [R, rows, C]\\n    if plan.is_owner:\\n        q_wide = qall.permute(1, 0, 2).reshape(rows, -1)\\n        a = layer_mod.attn\\n        k3 = k.reshape(-1, a.tp_k_head_num, a.qk_head_dim)\\n        v3 = v.reshape(-1, a.tp_v_head_num, a.v_head_dim)\\n        out = attend(q_wide, k3, v3, plan.shim).reshape(rows, plan.R, -1)\\n        mine = out[:, plan.me].to(qc.dtype).contiguous()\\n        send = out[:, 1 - plan.me].to(qc.dtype).contiguous()\\n        gather_fn(send)  # exchange 2: the partner reads our slot\\n        return mine\\n    res = gather_fn(torch.empty_like(qc))  # exchange 2: the owner\'s slot carries our output\\n    return res[plan.owner]\\n\'\'\'\\n\\nLAYER_METHOD = \'\'\'\\n    def _self_attention_pairown(self, positions, hidden_states, forward_batch, plan):  {MARK}\\n        \\"\\"\\"Pair-owned QSA layer: only the owner runs the indexer + sparse attention (for both ranks\' q heads).\\"\\"\\"\\n        from sglang.srt.layers.attention.qsa import pairown as _po\\n\\n        is_owner = plan.is_owner\\n        overlap_indexer = (\\n            is_owner\\n            and self.alt_stream is not None\\n            and get_is_capture_mode()\\n            and hidden_states.shape[0] < _QSA_INDEXER_OVERLAP_TOKEN_THRESHOLD\\n        )\\n        topk_indices = None\\n        if overlap_indexer:\\n            current_stream = torch.cuda.current_stream()\\n            self.alt_stream.wait_stream(current_stream)\\n            with torch.cuda.stream(self.alt_stream):\\n                topk_indices = self._compute_qsa_topk_indices(\\n                    hidden_states, positions, forward_batch\\n                )\\n\\n        q, k, v, gate = self._prepare_qkv_gate(\\n            positions=positions,\\n            hidden_states=hidden_states,\\n            forward_batch=forward_batch,\\n        )\\n\\n        if overlap_indexer:\\n            current_stream.wait_stream(self.alt_stream)\\n            topk_indices.record_stream(current_stream)\\n        elif is_owner:\\n            topk_indices = self._compute_qsa_topk_indices(\\n                hidden_states, positions, forward_batch\\n            )\\n\\n        attn_output = _po.pair_attend(\\n            plan,\\n            self,\\n            q,\\n            k,\\n            v,\\n            forward_batch,\\n            lambda q_wide, k3, v3, shim: get_attn_backend().forward(\\n                q_wide, k3, v3, shim, forward_batch, True, topk_indices=topk_indices\\n            ),\\n        )\\n        if gate is not None:\\n            if attn_output.is_cuda:\\n                attn_output = fused_sigmoid_mul(attn_output, gate, inplace=True)\\n            else:\\n                gate = gate.reshape(gate.shape[0], -1) if gate.ndim == 3 else gate\\n                attn_output = attn_output * torch.sigmoid(gate)\\n        output, _ = self.o_proj(attn_output)\\n        return output\\n\'\'\'.replace(\\"{MARK}\\", MARK)\\n\\n# (file, old, new) -- every `old` must occur exactly once.\\nEDITS = [\\n    (REL_KVC,\\n     \\"        available_bytes = self._profile_available_bytes(pre_model_load_memory)\\\\n\\"\\n     \\"        config = self.config_from_budget(available_bytes)\\\\n\\",\\n     f\\"        from sglang.srt.layers.attention.qsa import pairown as _pairown  {MARK}\\\\n\\\\n\\"\\n     \\"        _pairown.prepare(self)\\\\n\\"\\n     \\"        available_bytes = self._profile_available_bytes(pre_model_load_memory)\\\\n\\"\\n     \\"        config = self.config_from_budget(available_bytes)\\\\n\\"),\\n    (REL_KVC,\\n     \\"                if self.layer_info.start_layer <= i < self.layer_info.end_layer\\\\n\\"\\n     \\"            ]\\\\n\\"\\n     \\"        )\\\\n\\"\\n     \\"        quant_method = self._build_fp4_quant_method(\\\\n\\"\\n     \\"            num_layers=len(full_attention_layer_ids)\\\\n\\"\\n     \\"        )\\\\n\\",\\n     \\"                if self.layer_info.start_layer <= i < self.layer_info.end_layer\\\\n\\"\\n     \\"            ]\\\\n\\"\\n     \\"        )\\\\n\\"\\n     f\\"        if not self.is_draft_worker:  {MARK} keep only the layers this rank owns\\\\n\\"\\n     \\"            from sglang.srt.layers.attention.qsa import pairown as _pairown\\\\n\\\\n\\"\\n     \\"            _pairown.prepare(self)\\\\n\\"\\n     \\"            full_attention_layer_ids = _pairown.owned_layer_ids(full_attention_layer_ids)\\\\n\\"\\n     \\"        quant_method = self._build_fp4_quant_method(\\\\n\\"\\n     \\"            num_layers=len(full_attention_layer_ids)\\\\n\\"\\n     \\"        )\\\\n\\"),\\n    (REL_POOLCFG,\\n     \\"            num_layers = len(effective_layer_ids)\\\\n\\"\\n     \\"        else:\\\\n\\"\\n     \\"            num_layers = kvc.layer_info.num_effective_layers\\\\n\\",\\n     f\\"            if not kvc.is_draft_worker:  {MARK} cell size covers only the owned layers\\\\n\\"\\n     \\"                from sglang.srt.layers.attention.qsa import pairown as _pairown\\\\n\\\\n\\"\\n     \\"                _pairown.prepare(kvc)\\\\n\\"\\n     \\"                effective_layer_ids = _pairown.owned_layer_ids(effective_layer_ids)\\\\n\\"\\n     \\"            num_layers = len(effective_layer_ids)\\\\n\\"\\n     \\"        else:\\\\n\\"\\n     \\"            num_layers = kvc.layer_info.num_effective_layers\\\\n\\"),\\n    (REL_Q4,\\n     \\"    ) -> torch.Tensor:\\\\n\\"\\n     \\"        overlap_indexer = (\\\\n\\"\\n     \\"            self.is_qsa\\\\n\\",\\n     \\"    ) -> torch.Tensor:\\\\n\\"\\n     \\"        if self.is_qsa:\\\\n\\"\\n     f\\"            from sglang.srt.layers.attention.qsa import pairown as _po  {MARK}\\\\n\\\\n\\"\\n     \\"            _plan = _po.layer_plan(self)\\\\n\\"\\n     \\"            if _plan is not None:\\\\n\\"\\n     \\"                return self._self_attention_pairown(\\\\n\\"\\n     \\"                    positions, hidden_states, forward_batch, _plan\\\\n\\"\\n     \\"                )\\\\n\\"\\n     \\"        overlap_indexer = (\\\\n\\"\\n     \\"            self.is_qsa\\\\n\\"),\\n    (REL_Q4,\\n     \\"        output, _ = self.o_proj(attn_output)\\\\n\\"\\n     \\"        return output\\\\n\\"\\n     \\"\\\\n\\"\\n     \\"    def forward(\\\\n\\"\\n     \\"        self,\\\\n\\"\\n     \\"        positions: torch.Tensor,\\\\n\\"\\n     \\"        hidden_states: torch.Tensor,\\\\n\\"\\n     \\"        residual: Optional[torch.Tensor],\\\\n\\"\\n     \\"        forward_batch: ForwardBatch,\\\\n\\"\\n     \\"        **kwargs: Any,\\\\n\\"\\n     \\"    ):\\\\n\\"\\n     \\"        hidden_states, residual = self._prepare_qwen4_exp_attn(\\\\n\\",\\n     \\"        output, _ = self.o_proj(attn_output)\\\\n\\"\\n     \\"        return output\\\\n\\"\\n     + LAYER_METHOD +\\n     \\"\\\\n\\"\\n     \\"    def forward(\\\\n\\"\\n     \\"        self,\\\\n\\"\\n     \\"        positions: torch.Tensor,\\\\n\\"\\n     \\"        hidden_states: torch.Tensor,\\\\n\\"\\n     \\"        residual: Optional[torch.Tensor],\\\\n\\"\\n     \\"        forward_batch: ForwardBatch,\\\\n\\"\\n     \\"        **kwargs: Any,\\\\n\\"\\n     \\"    ):\\\\n\\"\\n     \\"        hidden_states, residual = self._prepare_qwen4_exp_attn(\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    root = Path(sys.argv[1])\\n    files = sorted({e[0] for e in EDITS} | {REL_MOD})\\n    texts = {rel: ((root / rel).read_text() if (root / rel).exists() else None) for rel in files}\\n    done = {rel: (texts[rel] is not None and MARK in texts[rel]) for rel in files if rel != REL_MOD}\\n    mod_done = texts[REL_MOD] is not None and MARK in texts[REL_MOD]\\n    if all(done.values()) and mod_done:\\n        print(\\"pairown_patch: already patched\\")\\n        return\\n    if any(done.values()) or mod_done:\\n        sys.exit(f\\"pairown_patch: PARTIALLY patched ({done}, module {mod_done}); refusing\\")\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"pairown_patch: NOT patched\\")\\n    for rel, old, _new in EDITS:\\n        if texts[rel] is None:\\n            sys.exit(f\\"pairown_patch: missing {rel}; refusing to patch\\")\\n        if texts[rel].count(old) != 1:\\n            sys.exit(f\\"pairown_patch: anchor x{texts[rel].count(old)} (need 1) in {rel}: {old.strip()[:70]!r}; refusing to patch\\")\\n    for need in (\\"get_is_capture_mode\\", \\"_QSA_INDEXER_OVERLAP_TOKEN_THRESHOLD\\", \\"fused_sigmoid_mul\\", \\"get_attn_backend\\"):\\n        if need not in texts[REL_Q4]:\\n            sys.exit(f\\"pairown_patch: qwen4_exp.py has no {need}; refusing to patch\\")\\n    for rel, old, new in EDITS:\\n        texts[rel] = texts[rel].replace(old, new)\\n    for rel in files:\\n        if rel != REL_MOD:\\n            (root / rel).write_text(texts[rel])\\n    (root / REL_MOD).write_text(MODULE_SRC)\\n    print(\\"pairown_patch: patched\\", \\", \\".join(files))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "ple_fp8_scale.py": "\\"\\"\\"Scaled BF16 -> FP8 PLE quantisation at load time (optional duck2 patch).\\n\\nInstalled by apply_patches.py --ple-scale as ``sglang/srt/models/duck2_ple_fp8.py``.\\n\\nStock behaviour (Pennyroyal qwen4_exp.py:2094-2109 and 2160-2172): when\\ntext_config.ple_embedding_dtype == \\"float8_e4m3fn\\" and the checkpoint PLE\\nshards are BF16 (Intel / woochangsim W4A16), every row is cast with\\n``.to(float8_e4m3fn)`` and weight_scale stays 1. That cast does not saturate,\\nso |w| > 448 becomes NaN, and |w| < 2^-6 falls into e4m3 subnormals with\\ncoarse steps. The official FP8 PLE (RadixArk NVFP4 release, Kaggle dataset\\nsonphamorg/arc3-flashnext-ple-fp8-v1) instead uses one global per-table scale.\\n\\nWith SGLANG_PLE_FP8_SCALE=<s> set, this module quantises w -> clamp(w/s, \\u00b1448)\\nand sets the module\'s weight_scale buffer to s. The scale is applied after\\nthe PLE all-reduce (qwen4_exp.py:569-570, 1256-1257), so s must be identical on\\nall TP ranks. It is therefore taken from the environment and never computed per\\nrank. The notebook derives it before launch from one PLE shard:\\ns = 2 * amax(shard_0) / 448. The 2x margin leaves room for outliers in other\\nshards, and the clamp saturates any value that still overflows instead of\\nproducing NaN.\\nWithout the env var nothing changes (stock raw cast).\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport os\\nfrom typing import Optional\\n\\nimport torch\\n\\nlogger = logging.getLogger(__name__)\\n_FP8 = torch.float8_e4m3fn\\n_FP8_MAX = 448.0\\n_CHUNK_ROWS = 8192\\n\\n\\ndef ple_fp8_scale() -> Optional[float]:\\n    raw = os.environ.get(\\"SGLANG_PLE_FP8_SCALE\\", \\"\\").strip()\\n    if not raw:\\n        return None\\n    s = float(raw)\\n    if not (s > 0.0 and s == s and s != float(\\"inf\\")):\\n        raise ValueError(f\\"SGLANG_PLE_FP8_SCALE must be a positive float, got {raw!r}\\")\\n    # store/apply exactly what the bf16 weight_scale buffer can represent\\n    return float(torch.tensor(s, dtype=torch.bfloat16).float())\\n\\n\\ndef quantize_ple_rows(emb, src: torch.Tensor) -> torch.Tensor:\\n    \\"\\"\\"BF16/FP32 rows -> FP8 rows with the global scale (or src unchanged if unset).\\"\\"\\"\\n    s = ple_fp8_scale()\\n    if s is None:\\n        return src\\n    if not getattr(emb, \\"_duck2_ple_scale_set\\", False):\\n        with torch.no_grad():\\n            emb.weight_scale.fill_(s)\\n        emb._duck2_ple_scale_set = True\\n        logger.info(\\"duck2 PLE: quantising BF16 PLE rows to fp8 with global scale %.6g\\", s)\\n    out = torch.empty(src.shape, dtype=_FP8, device=src.device)\\n    inv = 1.0 / s\\n    for i in range(0, src.shape[0], _CHUNK_ROWS):\\n        blk = src[i : i + _CHUNK_ROWS].float().mul_(inv).clamp_(-_FP8_MAX, _FP8_MAX)\\n        out[i : i + _CHUNK_ROWS].copy_(blk.to(_FP8))\\n    return out\\n\\n\\ndef keep_quantised_scale(buffer_name: str, loaded_weight: torch.Tensor) -> bool:\\n    \\"\\"\\"True -> skip the checkpoint\'s PLE weight_scale (we own it).\\n\\n    Only a placeholder scale of 1.0 may be skipped. A real scale means the\\n    checkpoint is already FP8, and setting SGLANG_PLE_FP8_SCALE is then an error.\\n    \\"\\"\\"\\n    if buffer_name != \\"weight_scale\\" or ple_fp8_scale() is None:\\n        return False\\n    v = float(loaded_weight.float().reshape(-1)[0])\\n    if abs(v - 1.0) > 1e-6:\\n        raise ValueError(\\n            \\"SGLANG_PLE_FP8_SCALE is set but the checkpoint already carries a PLE \\"\\n            f\\"weight_scale={v}; it is an FP8 PLE checkpoint - unset SGLANG_PLE_FP8_SCALE\\"\\n        )\\n    return True\\n", "prefill_bench.py": "#!/usr/bin/env python3\\n\\"\\"\\"Prefill cost on a running, otherwise idle SGLang server (v86: 1024-token chunks run at ~2.3k tok/s = ~0.44 s each,\\nand every decode request waits for them, ~8% of a harness run\'s wall time).\\n\\n1. Time to first token of single /generate requests with N new tokens (max_new_tokens 1, greedy, raw token ids cut from\\n   tokenized ARC task text, a different slice per request, /flush_cache before each): the slope gives the per-token\\n   cost, the intercept the fixed cost of a forward; chunks of --chunked-prefill-size show as steps.\\n2. One torch-profiler capture (POST /start_profile, num_steps forwards) of a multi-chunk prefill, reduced per TP rank\\n   to GPU busy / idle time, inclusive time per prof_summarize.py category and the top kernels (idle = the GPU waiting\\n   on the host: eager launch overhead, syncs, PLE host work).\\nOutput: --out JSON (also printed as one PREFILL_BENCH_DONE line). Stdlib only (+ prof_drive / prof_summarize, same dir).\\nusage: prefill_bench.py --url http://127.0.0.1:8000 --challenges eval.json --out prefill.json --prof-dir /tmp/prof/prefill\\n\\"\\"\\"\\nimport argparse\\nimport collections\\nimport glob\\nimport json\\nimport os\\nimport statistics as st\\nimport sys\\nimport threading\\nimport time\\n\\nHERE = os.path.dirname(os.path.abspath(__file__))\\nsys.path.insert(0, HERE)\\nimport prof_drive as PD  # noqa: E402\\nimport prof_summarize as PS  # noqa: E402\\n\\n\\ndef token_pool(url, tasks, need, log):\\n    ids, names = [], sorted(tasks)\\n    for i in range(0, len(names), 10):\\n        text = \\"\\".join(PD.task_text(k, tasks[t]) for k, t in enumerate(names[i:i + 10], start=i))\\n        r = PD.http(url, \\"/tokenize\\", {\\"prompt\\": text, \\"add_special_tokens\\": False}, timeout=300)\\n        ids += r[\\"tokens\\"] if isinstance(r, dict) and \\"tokens\\" in r else r\\n        if len(ids) >= need:\\n            break\\n    log(f\\"token pool: {len(ids)} ids from {min(len(names), i + 10)} tasks\\")\\n    return ids\\n\\n\\ndef ttft(url, ids):\\n    PD.http(url, \\"/flush_cache\\", {}, timeout=120, raw=True)\\n    t0 = time.time()\\n    r = PD.http(url, \\"/generate\\", {\\"input_ids\\": ids, \\"sampling_params\\": {\\"max_new_tokens\\": 1, \\"temperature\\": 0.0}},\\n                timeout=1800)\\n    dt = time.time() - t0\\n    mi = (r.get(\\"meta_info\\") or {}) if isinstance(r, dict) else {}\\n    return {\\"n\\": len(ids), \\"s\\": round(dt, 4), \\"prompt_tokens\\": mi.get(\\"prompt_tokens\\"),\\n            \\"cached_tokens\\": mi.get(\\"cached_tokens\\"), \\"e2e\\": mi.get(\\"e2e_latency\\")}\\n\\n\\ndef host_side(path, top_n=25):\\n    \\"\\"\\"Host-side view of the same trace: CUDA runtime/driver calls (syncs, memcpys, launches) and aten ops, total\\n    inclusive ms + count per name (nested aten ops double-count: read as \'where the host time goes\', not a partition).\\"\\"\\"\\n    import gzip\\n    with (gzip.open(path, \\"rt\\") if path.endswith(\\".gz\\") else open(path)) as f:\\n        ev = json.load(f).get(\\"traceEvents\\", [])\\n    acc = collections.defaultdict(lambda: collections.defaultdict(lambda: [0, 0.0]))\\n    for e in ev:\\n        if e.get(\\"ph\\") != \\"X\\":\\n            continue\\n        c = str(e.get(\\"cat\\", \\"\\")).lower()\\n        if c in (\\"cuda_runtime\\", \\"cuda_driver\\", \\"cpu_op\\", \\"user_annotation\\", \\"python_function\\"):\\n            a = acc[c][str(e.get(\\"name\\", \\"\\"))[:90]]\\n            a[0] += 1\\n            a[1] += float(e.get(\\"dur\\") or 0.0)\\n    return {c: [[n, k, round(v / 1e3, 2)] for n, (k, v) in sorted(d.items(), key=lambda x: -x[1][1])[:top_n]]\\n            for c, d in acc.items()}\\n\\n\\ndef reduce_rank(path, top_n=30):\\n    kern, _, info = PS.load_trace(path)\\n    if not kern:\\n        return {\\"file\\": os.path.basename(path), \\"error\\": \\"no kernels\\", **info}\\n    t0, t1 = kern[0].ts, max(k.te for k in kern)\\n    busy = PS.union_len([(k.ts, k.te) for k in kern])\\n    cat, det, top = collections.Counter(), collections.Counter(), collections.defaultdict(lambda: [0, 0.0])\\n    for k in kern:\\n        d = k.te - k.ts\\n        cat[k.top] += d\\n        det[f\\"{k.top}/{k.det}\\"] += d\\n        top[k.name[:100]][0] += 1\\n        top[k.name[:100]][1] += d\\n    # forward boundaries: GPU gaps > 2 ms (prefill runs eagerly; consecutive forwards are separated by scheduling)\\n    gaps = [(b.ts - a.te) for a, b in zip(kern, kern[1:]) if b.ts - a.te > 2000]\\n    return {\\"file\\": os.path.basename(path), \\"span_ms\\": round((t1 - t0) / 1e3, 2), \\"busy_ms\\": round(busy / 1e3, 2),\\n            \\"idle_ms\\": round((t1 - t0 - busy) / 1e3, 2), \\"kernels\\": len(kern),\\n            \\"gaps_gt2ms\\": len(gaps), \\"gap_ms_total\\": round(sum(gaps) / 1e3, 2),\\n            \\"cat_ms\\": {c: round(v / 1e3, 2) for c, v in cat.most_common()},\\n            \\"detail_ms\\": {c: round(v / 1e3, 2) for c, v in det.most_common(25)},\\n            \\"top\\": [[n, c, round(v / 1e3, 2)] for n, (c, v) in sorted(top.items(), key=lambda x: -x[1][1])[:top_n]]}\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__.split(\\"\\\\n\\")[0])\\n    ap.add_argument(\\"--url\\", default=\\"http://127.0.0.1:8000\\")\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--prof-dir\\", default=\\"/tmp/prof/prefill\\")\\n    ap.add_argument(\\"--sizes\\", default=\\"64,256,512,1024,2048,4096,8192,16384\\")\\n    ap.add_argument(\\"--reps\\", type=int, default=2)\\n    ap.add_argument(\\"--profile-tokens\\", type=int, default=5120)\\n    ap.add_argument(\\"--profile-steps\\", type=int, default=5, help=\\"0 = no profiler capture\\")\\n    ap.add_argument(\\"--ranks\\", type=int, default=4)\\n    ap.add_argument(\\"--trace-timeout-s\\", type=float, default=600)\\n    a = ap.parse_args()\\n    t_start = time.time()\\n    log = lambda m: print(f\\"[prefill_bench {time.time() - t_start:6.1f}s] {m}\\", flush=True)\\n    rec = {\\"args\\": vars(a), \\"ttft\\": [], \\"notes\\": []}\\n\\n    def save():\\n        with open(a.out + \\".tmp\\", \\"w\\") as f:\\n            json.dump(rec, f, indent=1)\\n        os.replace(a.out + \\".tmp\\", a.out)\\n\\n    tasks = json.load(open(a.challenges))\\n    sizes = [int(x) for x in a.sizes.split(\\",\\") if x]\\n    need = sum(sizes) * a.reps + a.profile_tokens + 4096\\n    pool = token_pool(a.url, tasks, need, log)\\n    off = 0\\n\\n    def take(n):\\n        nonlocal off\\n        if off + n > len(pool):\\n            off = 0\\n        s = pool[off:off + n]\\n        off += n\\n        return s\\n\\n    ttft(a.url, take(256))   # warm-up (JIT, allocator)\\n    for n in sizes:\\n        for r in range(a.reps):\\n            try:\\n                x = ttft(a.url, take(n))\\n            except Exception as ex:\\n                x = {\\"n\\": n, \\"error\\": f\\"{type(ex).__name__}: {str(ex)[:200]}\\"}\\n            rec[\\"ttft\\"].append(x)\\n            log(f\\"ttft n={n} rep {r}: {x.get(\'s\')} s ({x.get(\'prompt_tokens\')} prompt, {x.get(\'cached_tokens\')} cached)\\")\\n        save()\\n    by = collections.defaultdict(list)\\n    for x in rec[\\"ttft\\"]:\\n        if \\"s\\" in x:\\n            by[x[\\"n\\"]].append(x[\\"s\\"])\\n    rec[\\"ttft_median_s\\"] = {n: round(st.median(v), 4) for n, v in sorted(by.items())}\\n    rec[\\"tok_s\\"] = {n: round(n / v, 1) for n, v in rec[\\"ttft_median_s\\"].items()}\\n    log(f\\"median ttft {rec[\'ttft_median_s\']} | tok/s {rec[\'tok_s\']}\\")\\n    save()\\n\\n    if a.profile_steps <= 0:      # TTFT sweep only\\n        rec[\\"wall_s\\"] = round(time.time() - t_start, 1)\\n        save()\\n        print(\\"PREFILL_BENCH_DONE \\" + json.dumps({k: rec.get(k) for k in (\\"ttft_median_s\\", \\"tok_s\\", \\"wall_s\\", \\"notes\\")}),\\n              flush=True)\\n        return\\n    os.makedirs(a.prof_dir, exist_ok=True)\\n    PD.http(a.url, \\"/flush_cache\\", {}, timeout=120, raw=True)\\n    body = {\\"output_dir\\": a.prof_dir, \\"num_steps\\": a.profile_steps, \\"activities\\": [\\"CPU\\", \\"GPU\\"],\\n            \\"with_stack\\": False, \\"record_shapes\\": False, \\"profile_prefix\\": \\"prefill\\"}\\n    try:\\n        # with num_steps the scheduler answers /start_profile only once the steps are done: call it from a thread\\n        resp = {}\\n        th = threading.Thread(target=lambda: resp.update(r=PD.http(a.url, \\"/start_profile\\", body, timeout=900, raw=True)),\\n                              daemon=True)\\n        th.start()\\n        time.sleep(3)\\n        t0 = time.time()\\n        PD.http(a.url, \\"/generate\\", {\\"input_ids\\": take(a.profile_tokens),\\n                                     \\"sampling_params\\": {\\"max_new_tokens\\": 1, \\"temperature\\": 0.0}}, timeout=1800)\\n        rec[\\"profile_request_s\\"] = round(time.time() - t0, 3)\\n        th.join(timeout=600)\\n        files, wait = PD.wait_traces(a.prof_dir, \\"prefill\\", a.ranks, a.trace_timeout_s)\\n        rec[\\"profile\\"] = {\\"resp\\": str(resp.get(\\"r\\")).strip()[:200], \\"files\\": [os.path.basename(f) for f in files],\\n                          \\"wait_s\\": wait}\\n        rec[\\"profile\\"][\\"ranks\\"] = [reduce_rank(f) for f in files if \\".trace.json\\" in f]\\n        _tr = [f for f in files if \\".trace.json\\" in f]\\n        if _tr:\\n            try:\\n                rec[\\"profile\\"][\\"host_rank0\\"] = host_side(_tr[0])\\n                for c in (\\"cuda_runtime\\", \\"cpu_op\\"):\\n                    for n, k, v in rec[\\"profile\\"][\\"host_rank0\\"].get(c, [])[:10]:\\n                        log(f\\"  host {c}: {v:9.2f} ms {k:6d}x {n}\\")\\n            except Exception as ex:\\n                rec[\\"notes\\"].append(f\\"host_side failed: {type(ex).__name__}: {str(ex)[:200]}\\")\\n        for rr in rec[\\"profile\\"][\\"ranks\\"]:\\n            log(f\\"{rr.get(\'file\')}: span {rr.get(\'span_ms\')} ms busy {rr.get(\'busy_ms\')} idle {rr.get(\'idle_ms\')} | \\"\\n                f\\"{rr.get(\'cat_ms\')}\\")\\n        if rec[\\"profile\\"][\\"ranks\\"]:\\n            for n, c, v in rec[\\"profile\\"][\\"ranks\\"][0].get(\\"top\\", [])[:15]:\\n                log(f\\"  {v:9.2f} ms {c:6d}x {n}\\")\\n    except Exception as ex:\\n        rec[\\"notes\\"].append(f\\"profile failed: {type(ex).__name__}: {str(ex)[:300]}\\")\\n        log(rec[\\"notes\\"][-1])\\n    rec[\\"wall_s\\"] = round(time.time() - t_start, 1)\\n    save()\\n    small = {k: rec.get(k) for k in (\\"ttft_median_s\\", \\"tok_s\\", \\"profile_request_s\\", \\"wall_s\\", \\"notes\\")}\\n    if rec.get(\\"profile\\", {}).get(\\"ranks\\"):\\n        small[\\"rank0\\"] = {k: rec[\\"profile\\"][\\"ranks\\"][0].get(k) for k in (\\"span_ms\\", \\"busy_ms\\", \\"idle_ms\\", \\"cat_ms\\",\\n                                                                            \\"gaps_gt2ms\\", \\"gap_ms_total\\")}\\n    print(\\"PREFILL_BENCH_DONE \\" + json.dumps(small), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "prof_drive.py": "#!/usr/bin/env python3\\n\\"\\"\\"Closed-loop decode load + optional torch-profiler capture against a running SGLang server (memo sec. 4, boot A/B/C).\\n\\nKeeps exactly B long ARC-style requests decoding (ignore_eos, T 1.0 / top_p 0.95 / top_k 20, thinking on) for each B\\nin --Bs, in the given order. All requests start together; stepping B down aborts the newest ones (/abort_request),\\nstepping up starts new ones; a request that ends early is replaced. Per B:\\n  settle --settle-s, then an UNPROFILED window of --window-s: the server\'s own TP0 \\"Decode batch\\" lines in serve.log\\n  (metrics_reporter.py:816-882) give ms/step = 1000 * #running-req / gen-throughput for every log interval that\\n  contains no prefill and runs exactly B requests; also #full token (KV in use, for context regression), fwd\\n  occupancy (if SGLANG_ENABLE_METRICS_DEVICE_TIMER=1), retractions, and a client-side token-rate cross-check.\\n  If B is in --profile-at: POST /start_profile {num_steps, activities, with_stack:false, record_shapes:false,\\n  output_dir:<prof-dir>/b<B>, profile_prefix:\\"b<B>\\"} (auto-stops after num_steps, profiler_manager.py:144-155) and\\n  wait until one stable trace per TP rank exists; prof_summarize.py reduces them at the end (in-process).\\nOutputs: <out-dir>/drive.json (rewritten after every B), <out-dir>/server_lines.txt (raw TP0 decode/prefill lines per\\nwindow), <prof-dir>/b<B>/*.trace.json.gz, <prof-dir>/b<B>/summary.json. Stdlib only.\\n\\nusage (boot A in the memo):\\n  python prof_drive.py --challenges /kaggle/input/.../arc-agi_evaluation_challenges.json \\\\\\n      --Bs 20,19,18,17,16,15,14,13,12,8,1 --ctx 8192 --window-s 45 --profile-at 19,13,1 \\\\\\n      --server-log /kaggle/working/serve.log --prof-dir /tmp/prof\\nNeeds max_running_requests >= max(Bs) and enough KV for max(Bs) x (ctx + generated). The server must see --prof-dir\\n(same machine). Context grows ~19 tok/s per request during the run: use the per-window #full token (step_ms_ctx_adj)\\nwhen comparing windows or boots.\\n\\"\\"\\"\\nimport argparse\\nimport glob\\nimport json\\nimport os\\nimport random\\nimport re\\nimport statistics as st\\nimport sys\\nimport threading\\nimport time\\nimport urllib.error\\nimport urllib.request\\n\\nHERE = os.path.dirname(os.path.abspath(__file__))\\nARC_PROMPT = (\\"You are participating in a puzzle solving competition. You are an expert at solving puzzles.\\\\n\\\\n\\"\\n              \\"Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or \\"\\n              \\"transformation in the training examples that maps the input to the output, then apply that pattern to \\"\\n              \\"the test input to give a final output.\\\\n\\\\nRespond in the format of the training output examples\\\\n\\\\n\\")\\n\\n\\ndef http(url, path, payload=None, timeout=60, raw=False):\\n    req = urllib.request.Request(url + path, data=None if payload is None else json.dumps(payload).encode(),\\n                                 headers={\\"Content-Type\\": \\"application/json\\"},\\n                                 method=\\"GET\\" if payload is None else \\"POST\\")\\n    with urllib.request.urlopen(req, timeout=timeout) as r:\\n        b = r.read()\\n    if raw:\\n        return b.decode(errors=\\"replace\\")\\n    try:\\n        return json.loads(b)\\n    except ValueError:\\n        return b.decode(errors=\\"replace\\")\\n\\n\\ndef task_text(i, t):\\n    ex = \\"\\".join(f\\"--Example {j}-- \\\\n\\\\n INPUT: \\\\n\\\\n{json.dumps(p[\'input\'])}\\\\n\\\\nOUTPUT: \\\\n\\\\n{json.dumps(p[\'output\'])}\\\\n\\\\n\\"\\n                 for j, p in enumerate(t[\\"train\\"]))\\n    return (f\\"=== Puzzle {i} ===\\\\n--Training Examples--\\\\n{ex}--End of Training Examples--\\\\n\\\\n--Test Input--\\\\n\\"\\n            f\\"{json.dumps(t[\'test\'][0][\'input\'])}\\\\n--End of Test Input--\\\\n\\\\n\\")\\n\\n\\ndef synthetic_tasks(n=160, seed=0):\\n    rng = random.Random(seed)\\n    out = {}\\n    for i in range(n):\\n        h, w = rng.randint(6, 24), rng.randint(6, 24)\\n        g = lambda: [[rng.choice([0, 0, 0, 1, 2, 3, 5, 8]) for _ in range(w)] for _ in range(h)]\\n        out[f\\"synth{i:04d}\\"] = {\\"train\\": [{\\"input\\": g(), \\"output\\": g()} for _ in range(3)], \\"test\\": [{\\"input\\": g()}]}\\n    return out\\n\\n\\ndef build_prompts(url, model, tasks, n, ctx, kw, log):\\n    \\"\\"\\"n distinct prompts of ~ctx tokens: ARC tasks packed greedily (rotated start per prompt), tokenized by the\\n    server\'s own chat template (/tokenize with messages -> the exact ids /v1/chat/completions would use).\\"\\"\\"\\n    ids = sorted(tasks)\\n    texts = [task_text(0, tasks[t]) for t in ids]\\n    mk = {\\"model\\": model} if model else {}\\n    cnt = http(url, \\"/tokenize\\", dict(mk, prompt=texts, add_special_tokens=False), timeout=300)[\\"count\\"]\\n    order = sorted(range(len(ids)), key=lambda i: (i * 7919) % len(ids))\\n    prompts, meta = [], []\\n    for p in range(n):\\n        start, chosen, tot = (p * max(1, len(ids) // max(n, 1))) % len(ids), [], 300\\n        for step in range(len(ids)):\\n            i = order[(start + step) % len(ids)]\\n            if tot >= 0.95 * ctx:\\n                break\\n            if tot + cnt[i] <= 1.05 * ctx or not chosen:\\n                chosen.append(i)\\n                tot += cnt[i]\\n        body = ARC_PROMPT + \\"\\".join(task_text(k + 1, tasks[ids[i]]) for k, i in enumerate(chosen)) + \\\\\\n            f\\"Solve Puzzle {len(chosen)}.\\\\n\\\\nYour response:\\"\\n        r = http(url, \\"/tokenize\\", dict(mk, messages=[{\\"role\\": \\"user\\", \\"content\\": body}], chat_template_kwargs=kw),\\n                 timeout=300)\\n        prompts.append(r[\\"tokens\\"])\\n        meta.append({\\"tasks\\": [ids[i] for i in chosen], \\"tokens\\": len(r[\\"tokens\\"])})\\n    log(f\\"prompts: {n} x {[m[\'tokens\'] for m in meta]} tokens\\")\\n    return prompts, meta\\n\\n\\nCT_RE = re.compile(rb\'\\"completion_tokens\\":\\\\s*(\\\\d+)\')\\nFR_RE = re.compile(rb\'\\"finish_reason\\":\\\\s*(\\\\{[^{}]*\\\\}|\\"[^\\"]*\\")\')\\n\\n\\nclass Stream(threading.Thread):\\n    def __init__(self, url, rid, ids, sp, timeout):\\n        super().__init__(daemon=True)\\n        self.url, self.rid, self.ids, self.sp, self.timeout = url, rid, ids, sp, timeout\\n        self.n, self.first_t, self.end_t, self.done, self.err, self.finish, self.aborted = 0, None, None, False, None, None, False\\n        self.t0 = time.time()\\n\\n    def run(self):\\n        body = {\\"input_ids\\": self.ids, \\"sampling_params\\": self.sp, \\"rid\\": self.rid, \\"stream\\": True}\\n        try:\\n            req = urllib.request.Request(self.url + \\"/generate\\", data=json.dumps(body).encode(),\\n                                         headers={\\"Content-Type\\": \\"application/json\\"})\\n            with urllib.request.urlopen(req, timeout=self.timeout) as r:\\n                for line in r:\\n                    if not line.startswith(b\\"data:\\"):\\n                        continue\\n                    if b\\"[DONE]\\" in line:\\n                        break\\n                    m = CT_RE.search(line)\\n                    if m:\\n                        self.n = int(m.group(1))\\n                        if self.first_t is None:\\n                            self.first_t = time.time()\\n                    m = FR_RE.search(line)\\n                    if m:\\n                        self.finish = m.group(1).decode(errors=\\"replace\\")[:160]\\n                    if b\'\\"error\\"\' in line[:40]:\\n                        self.err = line[:300].decode(errors=\\"replace\\")\\n        except Exception as ex:\\n            self.err = f\\"{type(ex).__name__}: {str(ex)[:200]}\\"\\n        finally:\\n            self.end_t, self.done = time.time(), True\\n\\n\\nclass Pool:\\n    def __init__(self, a, url, prompts, ctx_len, log):\\n        self.a, self.url, self.prompts, self.ctx_len, self.log = a, url, prompts, ctx_len, log\\n        self.active, self.ended, self.next_prompt, self.started = [], [], 0, 0\\n        self.sp = {\\"temperature\\": a.temperature, \\"top_p\\": a.top_p, \\"top_k\\": a.top_k, \\"ignore_eos\\": True}\\n\\n    def _start(self):\\n        ids = self.prompts[self.next_prompt % len(self.prompts)]\\n        self.next_prompt += 1\\n        sp = dict(self.sp, max_new_tokens=max(16, min(self.a.max_new, self.ctx_len - len(ids) - 64)))\\n        s = Stream(self.url, f\\"profdrive-{os.getpid()}-{self.started}\\", ids, sp, self.a.http_timeout)\\n        self.started += 1\\n        s.start()\\n        self.active.append(s)\\n        return s\\n\\n    def abort(self, s):\\n        s.aborted = True\\n        try:\\n            http(self.url, \\"/abort_request\\", {\\"rid\\": s.rid}, timeout=30, raw=True)\\n        except Exception as ex:\\n            self.log(f\\"abort {s.rid} failed: {ex}\\")\\n\\n    def reap(self):\\n        gone = [s for s in self.active if s.done]\\n        self.active = [s for s in self.active if not s.done]\\n        self.ended += gone\\n        return [s for s in gone if not s.aborted]\\n\\n    def set_target(self, B, wait_s):\\n        self.reap()\\n        while len(self.active) > B:   # newest first: keeps the survivors\' contexts homogeneous\\n            self.abort(self.active.pop())\\n        new = [self._start() for _ in range(B - len(self.active))]\\n        t0 = time.time()\\n        while new and time.time() - t0 < wait_s and not all(s.first_t or s.done for s in new):\\n            time.sleep(1)\\n        return {\\"started\\": len(new), \\"prefill_wait_s\\": round(time.time() - t0, 1),\\n                \\"not_started\\": sum(1 for s in new if not (s.first_t or s.done))}\\n\\n    def maintain(self, B):\\n        lost = self.reap()\\n        for s in lost:\\n            self.log(f\\"request {s.rid} ended early (n={s.n}, finish={s.finish}, err={s.err}); replacing\\")\\n        while len(self.active) < B:\\n            self._start()\\n        return lost\\n\\n    def tokens(self):   # all tokens ever streamed (ended/aborted streams keep their count)\\n        return sum(s.n for s in self.active) + sum(s.n for s in self.ended)\\n\\n    def close(self):\\n        for s in list(self.active):\\n            self.abort(s)\\n        self.active = []\\n\\n\\nLINE_RE = re.compile(r\\"\\\\[(\\\\d{4}-\\\\d\\\\d-\\\\d\\\\d \\\\d\\\\d:\\\\d\\\\d:\\\\d\\\\d)([^\\\\]]*)\\\\] (Decode|Prefill) batch\\")\\n\\n\\ndef num(pat, line, cast=float):\\n    m = re.search(pat, line)\\n    return cast(m.group(1)) if m else None\\n\\n\\ndef parse_server_lines(text):\\n    \\"\\"\\"TP0 decode/prefill lines -> rows; a decode row is \'clean\' if the previous logged event was a decode line.\\"\\"\\"\\n    rows, prev_prefill = [], True   # the first decode line\'s interval started before the window: not clean\\n    for line in text.splitlines():\\n        m = LINE_RE.search(line)\\n        if not m:\\n            if \\"Retract\\" in line or \\"retract\\" in line:\\n                rows.append({\\"kind\\": \\"retract\\", \\"line\\": line.strip()[:200]})\\n            continue\\n        tags = m.group(2)\\n        if \\"TP\\" in tags and not re.search(r\\"\\\\bTP0\\\\b\\", tags):\\n            continue\\n        if m.group(3) == \\"Prefill\\":\\n            prev_prefill = True\\n            rows.append({\\"kind\\": \\"prefill\\", \\"ts\\": m.group(1)})\\n            continue\\n        r = {\\"kind\\": \\"decode\\", \\"ts\\": m.group(1), \\"running\\": num(r\\"#running-req: (\\\\d+)\\", line, int),\\n             \\"full_token\\": num(r\\"#(?:full )?token: (\\\\d+)\\", line, int),\\n             \\"tps\\": num(r\\"gen throughput \\\\(token/s\\\\): ([\\\\d.]+)\\", line),\\n             \\"occ\\": num(r\\"fwd occupancy: ([\\\\d.]+)%\\", line), \\"queue\\": num(r\\"#queue-req: (\\\\d+)\\", line, int),\\n             \\"graph\\": num(r\\"cuda graph: (True|False)\\", line, str), \\"iter\\": num(r\\"Decode batch \\\\[(\\\\d+)\\\\]\\", line, int),\\n             \\"clean\\": not prev_prefill}\\n        prev_prefill = False\\n        rows.append(r)\\n    return rows\\n\\n\\ndef window_stats(rows, B, ctx_target, ms_per_100k):\\n    dec = [r for r in rows if r[\\"kind\\"] == \\"decode\\"]\\n    use = [r for r in dec if r[\\"clean\\"] and r[\\"running\\"] == B and r[\\"tps\\"] and r[\\"tps\\"] > 0]\\n    out = {\\"decode_lines\\": len(dec), \\"used_lines\\": len(use), \\"prefill_lines\\": sum(r[\\"kind\\"] == \\"prefill\\" for r in rows),\\n           \\"retract_lines\\": sum(r[\\"kind\\"] == \\"retract\\" for r in rows),\\n           \\"lines_wrong_bs\\": sum(1 for r in dec if r[\\"running\\"] != B),\\n           \\"graph_false_lines\\": sum(1 for r in dec if r[\\"graph\\"] == \\"False\\")}\\n    if not use:\\n        return out\\n    ms = sorted(1000.0 * r[\\"running\\"] / r[\\"tps\\"] for r in use)\\n    ft = [r[\\"full_token\\"] for r in use if r[\\"full_token\\"] is not None]\\n    occ = [r[\\"occ\\"] for r in use if r[\\"occ\\"] is not None]\\n    p = lambda q: round(ms[min(len(ms) - 1, int(round(q * (len(ms) - 1))))], 2)\\n    out.update(step_ms_p50=round(st.median(ms), 2), step_ms_mean=round(st.mean(ms), 2), step_ms_p10=p(0.1),\\n               step_ms_p90=p(0.9), tok_s_p50=round(1000.0 * B / st.median(ms), 1),\\n               full_token_p50=int(st.median(ft)) if ft else None, fwd_occupancy_p50=round(st.median(occ), 2) if occ else None)\\n    if ft:\\n        out[\\"step_ms_ctx_adj\\"] = round(out[\\"step_ms_p50\\"] - ms_per_100k * (st.median(ft) - B * ctx_target) / 1e5, 2)\\n    return out\\n\\n\\nclass Log:\\n    def __init__(self, path):\\n        self.path = path\\n\\n    def size(self):\\n        try:\\n            return os.path.getsize(self.path)\\n        except OSError:\\n            return 0\\n\\n    def read(self, a, b):\\n        try:\\n            with open(self.path, \\"rb\\") as f:\\n                f.seek(a)\\n                return f.read(max(0, b - a)).decode(errors=\\"replace\\")\\n        except OSError:\\n            return \\"\\"\\n\\n\\ndef wait_traces(d, prefix, ranks, timeout):\\n    t0, last, stable = time.time(), None, 0\\n    while time.time() - t0 < timeout:\\n        fs = sorted(glob.glob(os.path.join(d, f\\"{prefix}-*TP-*.trace.json*\\")))\\n        sz = [os.path.getsize(f) for f in fs]\\n        if len(fs) >= ranks and all(sz) and sz == last:\\n            stable += 1\\n            if stable >= 3:\\n                return fs, round(time.time() - t0, 1)\\n        else:\\n            stable = 0\\n        last = sz\\n        time.sleep(2)\\n    return sorted(glob.glob(os.path.join(d, f\\"{prefix}-*\\"))), None\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__.split(\\"\\\\n\\")[0])\\n    ap.add_argument(\\"--url\\", default=\\"http://127.0.0.1:8000\\")\\n    ap.add_argument(\\"--challenges\\", help=\\"ARC challenges json (default: synthetic ARC-like grids, with a warning)\\")\\n    ap.add_argument(\\"--Bs\\", default=\\"20,19,18,17,16,15,14,13,12,8,1\\", help=\\"batch sizes, run in this order\\")\\n    ap.add_argument(\\"--ctx\\", type=int, default=8192, help=\\"target prompt tokens per request\\")\\n    ap.add_argument(\\"--n-prompts\\", type=int, default=0, help=\\"distinct prompts (default max(Bs) + 4 spares)\\")\\n    ap.add_argument(\\"--window-s\\", type=float, default=45)\\n    ap.add_argument(\\"--settle-s\\", type=float, default=10)\\n    ap.add_argument(\\"--profile-at\\", default=\\"\\", help=\\"comma list of B values to profile (e.g. 19,13,1)\\")\\n    ap.add_argument(\\"--profile-steps\\", type=int, default=32)\\n    ap.add_argument(\\"--profile-activities\\", default=\\"CPU,GPU\\")\\n    ap.add_argument(\\"--prof-dir\\", default=os.environ.get(\\"SGLANG_TORCH_PROFILER_DIR\\", \\"/tmp/prof\\"))\\n    ap.add_argument(\\"--out-dir\\", default=\\"\\", help=\\"drive.json / server_lines.txt (default: --prof-dir)\\")\\n    ap.add_argument(\\"--server-log\\", default=\\"/kaggle/working/serve.log\\")\\n    ap.add_argument(\\"--max-new\\", type=int, default=30000)\\n    ap.add_argument(\\"--temperature\\", type=float, default=1.0)\\n    ap.add_argument(\\"--top-p\\", type=float, default=0.95)\\n    ap.add_argument(\\"--top-k\\", type=int, default=20)\\n    ap.add_argument(\\"--no-thinking\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--ctx-ms-per-100k\\", type=float, default=0.51, help=\\"memo sec. 0: step cost per 100k KV tokens\\")\\n    ap.add_argument(\\"--prefill-timeout-s\\", type=float, default=1200)\\n    ap.add_argument(\\"--trace-timeout-s\\", type=float, default=900)\\n    ap.add_argument(\\"--http-timeout\\", type=float, default=900)\\n    ap.add_argument(\\"--no-summarize\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--keep-running\\", action=\\"store_true\\", help=\\"do not abort the requests at the end\\")\\n    a = ap.parse_args()\\n    Bs = [int(x) for x in a.Bs.split(\\",\\") if x.strip()]\\n    prof_at = {int(x) for x in a.profile_at.split(\\",\\") if x.strip()}\\n    out_dir = a.out_dir or a.prof_dir\\n    os.makedirs(out_dir, exist_ok=True)\\n    out_path, lines_path = os.path.join(out_dir, \\"drive.json\\"), os.path.join(out_dir, \\"server_lines.txt\\")\\n    t_run = time.time()\\n\\n    def log(msg):\\n        print(f\\"[prof_drive {time.strftime(\'%H:%M:%S\')}] {msg}\\", flush=True)\\n\\n    info = {}\\n    for path in (\\"/server_info\\", \\"/get_server_info\\"):\\n        try:\\n            info = http(a.url, path, timeout=30)\\n            if isinstance(info, dict):\\n                break\\n        except Exception:\\n            continue\\n    info = info if isinstance(info, dict) else {}\\n    model, ctx_len = info.get(\\"served_model_name\\"), info.get(\\"context_length\\")\\n    try:\\n        m = http(a.url, \\"/v1/models\\", timeout=30)[\\"data\\"][0]\\n        model, ctx_len = model or m.get(\\"id\\"), ctx_len or m.get(\\"max_model_len\\")\\n    except Exception:\\n        pass\\n    ctx_len = int(ctx_len or 131072)\\n    ranks = int(info.get(\\"tp_size\\") or 4) * int(info.get(\\"pp_size\\") or 1)\\n    keys = (\\"cuda_graph_bs_decode\\", \\"cuda_graph_max_bs_decode\\", \\"max_running_requests\\", \\"decode_log_interval\\",\\n            \\"tp_size\\", \\"ep_size\\", \\"context_length\\", \\"kv_cache_dtype\\", \\"attention_backend\\", \\"moe_runner_backend\\",\\n            \\"disable_overlap_schedule\\", \\"chunked_prefill_size\\", \\"max_total_num_tokens\\", \\"page_size\\")\\n    rec = {\\"t0\\": t_run, \\"args\\": vars(a), \\"server\\": {k: info.get(k) for k in keys if k in info}, \\"model\\": model,\\n           \\"runs\\": [], \\"notes\\": []}\\n    maxreq = info.get(\\"max_running_requests\\")\\n    if maxreq and max(Bs) > int(maxreq):\\n        raise SystemExit(f\\"max_running_requests={maxreq} < max(Bs)={max(Bs)}: boot the MAXREQ>=20 rung\\")\\n    slog = Log(a.server_log)\\n    if not os.path.exists(a.server_log):\\n        rec[\\"notes\\"].append(f\\"server log {a.server_log} not found: no server-side step times (client rate only)\\")\\n        log(rec[\\"notes\\"][-1])\\n    if a.challenges and os.path.exists(a.challenges):\\n        tasks = json.load(open(a.challenges))\\n    else:\\n        tasks = synthetic_tasks()\\n        rec[\\"notes\\"].append(\\"no --challenges file: synthetic ARC-like grids (routing may differ from real ARC)\\")\\n        log(rec[\\"notes\\"][-1])\\n    kw = {} if a.no_thinking else {\\"enable_thinking\\": True, \\"reasoning_effort\\": \\"xhigh\\"}\\n    n_prompts = a.n_prompts or max(Bs) + 4\\n    prompts, pmeta = build_prompts(a.url, model, tasks, n_prompts, a.ctx, kw, log)\\n    rec[\\"prompts\\"] = pmeta\\n    est = len(Bs) * (a.settle_s + a.window_s) + len(prof_at) * 60\\n    log(f\\"server {rec[\'server\']} | Bs {Bs} | profile at {sorted(prof_at)} | est. {est / 60:.1f} min + prefill\\")\\n\\n    def save():\\n        tmp = out_path + \\".tmp\\"\\n        with open(tmp, \\"w\\") as f:\\n            json.dump(rec, f, indent=1)\\n        os.replace(tmp, out_path)\\n\\n    pool = Pool(a, a.url, prompts, ctx_len, log)\\n    lf = open(lines_path, \\"a\\")\\n    lf.write(f\\"##### prof_drive run {time.strftime(\'%Y-%m-%d %H:%M:%S\')} Bs={Bs}\\\\n\\")\\n    try:\\n        for B in Bs:\\n            run = {\\"B\\": B, \\"t\\": round(time.time() - t_run, 1)}\\n            run[\\"ramp\\"] = pool.set_target(B, a.prefill_timeout_s)\\n            if run[\\"ramp\\"][\\"started\\"]:\\n                log(f\\"B={B}: started {run[\'ramp\'][\'started\']} requests, prefill wait {run[\'ramp\'][\'prefill_wait_s\']} s\\"\\n                    f\\" ({run[\'ramp\'][\'not_started\']} still without a token)\\")\\n            time.sleep(a.settle_s)\\n            pool.maintain(B)\\n            off0, tok0, t0 = slog.size(), pool.tokens(), time.time()\\n            lost = 0\\n            while time.time() - t0 < a.window_s:\\n                time.sleep(1)\\n                lost += len(pool.maintain(B))\\n                if lost > 2 * B + 4:\\n                    raise RuntimeError(f\\"{lost} requests ended early in one window: server unhealthy?\\")\\n            off1, tok1, t1 = slog.size(), pool.tokens(), time.time()\\n            text = slog.read(off0, off1)\\n            rows = parse_server_lines(text)\\n            lf.write(f\\"### B={B} window {time.strftime(\'%H:%M:%S\', time.localtime(t0))}-\\"\\n                     f\\"{time.strftime(\'%H:%M:%S\', time.localtime(t1))}\\\\n\\")\\n            lf.write(\\"\\".join(l + \\"\\\\n\\" for l in text.splitlines() if \\"batch\\" in l or \\"etract\\" in l))\\n            lf.flush()\\n            run[\\"window\\"] = window_stats(rows, B, a.ctx, a.ctx_ms_per_100k)\\n            ctps = (tok1 - tok0) / max(t1 - t0, 1e-6)\\n            run[\\"client\\"] = {\\"tok_s\\": round(ctps, 1), \\"step_ms\\": round(1000.0 * B / ctps, 2) if ctps > 0 else None,\\n                             \\"active\\": len(pool.active), \\"replaced\\": lost,\\n                             \\"context_tokens_mean\\": round(st.mean(len(s.ids) + s.n for s in pool.active), 0)\\n                             if pool.active else None}\\n            if B in prof_at:\\n                pdir, k = os.path.join(a.prof_dir, f\\"b{B}\\"), 0\\n                while glob.glob(os.path.join(pdir, \\"*.trace.json*\\")):   # never mix with traces of an earlier run\\n                    k += 1\\n                    pdir = os.path.join(a.prof_dir, f\\"b{B}_{k}\\")\\n                os.makedirs(pdir, exist_ok=True)\\n                body = {\\"output_dir\\": pdir, \\"num_steps\\": a.profile_steps,\\n                        \\"activities\\": [x for x in a.profile_activities.split(\\",\\") if x],\\n                        \\"with_stack\\": False, \\"record_shapes\\": False, \\"profile_prefix\\": f\\"b{B}\\"}\\n                tp = time.time()\\n                try:\\n                    resp = http(a.url, \\"/start_profile\\", body, timeout=300, raw=True)\\n                except Exception as ex:\\n                    resp = f\\"ERROR {type(ex).__name__}: {ex}\\"\\n                files, wait = wait_traces(pdir, f\\"b{B}\\", ranks, a.trace_timeout_s) if not resp.startswith(\\"ERROR\\") \\\\\\n                    else ([], None)\\n                run[\\"profile\\"] = {\\"prefix\\": f\\"b{B}\\", \\"dir\\": pdir, \\"resp\\": resp.strip()[:200],\\n                                  \\"files\\": [os.path.basename(f) for f in files], \\"wait_s\\": wait,\\n                                  \\"total_s\\": round(time.time() - tp, 1)}\\n                pool.maintain(B)\\n                time.sleep(a.settle_s / 2)\\n            rec[\\"runs\\"].append(run)\\n            save()\\n            w = run[\\"window\\"]\\n            log(f\\"B={B:>2}: step {w.get(\'step_ms_p50\')} ms (n={w.get(\'used_lines\')}/{w.get(\'decode_lines\')}, \\"\\n                f\\"ctx-adj {w.get(\'step_ms_ctx_adj\')}, #full token {w.get(\'full_token_p50\')}, occ \\"\\n                f\\"{w.get(\'fwd_occupancy_p50\')}) | client {run[\'client\'][\'step_ms\']} ms\\"\\n                + (f\\" | profile {run[\'profile\'][\'files\'][:1]}.. wait {run[\'profile\'][\'wait_s\']} s\\" if \\"profile\\" in run else \\"\\"))\\n    finally:\\n        if not a.keep_running:\\n            pool.close()\\n        lf.close()\\n        rec[\\"wall_s\\"] = round(time.time() - t_run, 1)\\n        save()\\n    if prof_at and not a.no_summarize:\\n        try:\\n            sys.path.insert(0, HERE)\\n            import prof_summarize as PS\\n            rec[\\"summaries\\"] = {}\\n            for run in rec[\\"runs\\"]:\\n                p = run.get(\\"profile\\")\\n                if not p or not p[\\"files\\"]:\\n                    continue\\n                res = PS.summarize_paths([p[\\"dir\\"]], baseline_step_ms=run[\\"window\\"].get(\\"step_ms_p50\\"))\\n                with open(os.path.join(p[\\"dir\\"], \\"summary.json\\"), \\"w\\") as f:\\n                    json.dump(res, f, indent=1)\\n                print(PS.format_table(res), flush=True)\\n                for g in res:\\n                    s = g.get(\\"summary\\", {})\\n                    rec[\\"summaries\\"][g[\\"group\\"]] = {\\"B\\": run[\\"B\\"], **s, \\"cross_rank\\": g.get(\\"cross_rank\\")}\\n            save()\\n        except Exception as ex:\\n            rec[\\"notes\\"].append(f\\"summarize failed: {type(ex).__name__}: {ex}\\")\\n            save()\\n    table = [{\\"B\\": r[\\"B\\"], \\"step_ms\\": r[\\"window\\"].get(\\"step_ms_p50\\"), \\"ctx_adj\\": r[\\"window\\"].get(\\"step_ms_ctx_adj\\"),\\n              \\"full_token\\": r[\\"window\\"].get(\\"full_token_p50\\"), \\"n\\": r[\\"window\\"].get(\\"used_lines\\"),\\n              \\"client_ms\\": r[\\"client\\"][\\"step_ms\\"], \\"prof\\": \\"profile\\" in r} for r in rec[\\"runs\\"]]\\n    print(\\"PROF_DRIVE_DONE \\" + json.dumps({\\"out\\": out_path, \\"wall_s\\": rec[\\"wall_s\\"], \\"table\\": table}), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "prof_summarize.py": "#!/usr/bin/env python3\\n\\"\\"\\"Reduce SGLang /start_profile torch-profiler traces (one .trace.json.gz per TP rank) to per-decode-step GPU time by\\ncategory: moe, dense_gemm, nccl, attn_qsa, gdn, hc_mix, sampling, other, idle (+ finer \\"detail\\" buckets, top kernels,\\nNCCL calls/step, and the cross-rank split of NCCL time into pure transfer vs waiting for the slowest rank).\\n\\nTrace files (profiler_manager.py:326-358): <prefix>-<profile_id>-TP-<r>[-EP-<r>].trace.json.gz. Files are grouped by\\neverything before \\"-TP-\\", so one directory may hold several profiles (prof_drive.py writes one dir per batch size).\\n\\nStep split (\\"--split auto\\"):\\n  graph      one decode step = one CUDA-graph replay. CUPTI gives every kernel of a replay the correlation id of its\\n             cudaGraphLaunch call; groups of >= --min-graph-kernels kernels sharing such an id are replays (partial\\n             replays at the profile edges are dropped). Step window = [replay i first kernel, replay i+1 first kernel):\\n             kernels in the window that are not in replay i (input prep, sampling, D2H copies) count as \\"sampling\\".\\n  allgather  fallback (eager decode or no graph correlation): windows between consecutive NCCL AllGather ends (the\\n             logits gather is the last collective of each step); everything is classified by name.\\nPer step, wall time is attributed exclusively: every instant goes to the kernels running then (split evenly when\\nstreams overlap, e.g. the PLE prefetch / QSA indexer side streams), and instants with no kernel are \\"idle\\".\\nSo moe + dense_gemm + nccl + attn_qsa + gdn + hc_mix + sampling + other + idle == step period. \\"inclusive\\" sums\\nthe kernel durations instead. NCCL kernel time includes waiting for the slowest rank (EP imbalance): see\\ncross_rank.nccl_pure_ms (min over ranks per collective) and nccl_wait_ms_by_rank.\\nHC heuristic: the only torch.compile code in decode is the HC mix/combine fallback (hyperconnection.py:207-243,312),\\nso inductor kernels (triton_poi/per/red/tem_*) are hc_mix, and a run of dense GEMM kernels (mm [+ cuBLAS splitK\\nreduce]) immediately followed on its stream by an inductor kernel is one of the fallback\'s two skinny GEMMs\\n(hc_fallback_gemm: mm_down -> poi(div,silu) -> mm_up -> red(sigmoid,mul,mean)). --no-hc-heuristic disables that.\\n\\nusage: prof_summarize.py PATH [PATH ...] [--out summary.json] [--baseline-step-ms 53.5] [--drive drive.json]\\n       PATH = a trace dir (searched one level deep) or trace files.  Stdlib only.\\n       prof_summarize.py --cg-selftest     (Phase 0, GPU, venv python: checks the step-split assumption on this torch)\\n\\"\\"\\"\\nimport argparse\\nimport bisect\\nimport collections\\nimport glob\\nimport gzip\\nimport json\\nimport os\\nimport re\\nimport statistics as st\\nimport sys\\n\\nTOP = [\\"moe\\", \\"dense_gemm\\", \\"nccl\\", \\"attn_qsa\\", \\"gdn\\", \\"hc_mix\\", \\"sampling\\", \\"other\\"]\\nRULES = [  # (top, detail, regex), first match wins; unmatched -> other/misc. Raw top kernels are always emitted.\\n    (\\"nccl\\", \\"allreduce\\", r\\"all_?reduce|cross_device_reduce|one_shot|two_shot\\"),\\n    (\\"nccl\\", \\"allgather\\", r\\"all_?gather\\"),\\n    (\\"nccl\\", \\"other_coll\\", r\\"nccl|reduce_?scatter|broadcast|sendrecv|multimem\\"),\\n    (\\"moe\\", \\"moe_gemm\\", r\\"marlin|moe_wna16|fused_moe_kernel|moegemm|grouped_gemm\\"),\\n    (\\"moe\\", \\"moe_route\\", r\\"moe_fused_gate|topk_softmax|topk_sigmoid|gate_topk|moe_topk|moe_align|align_block|\\"\\n                         r\\"count_and_sort|sort_token|fill_padded|moe_route|pack_topk|expert_(?:count|map|distrib)\\"),\\n    (\\"moe\\", \\"moe_reduce\\", r\\"moe_sum|moe_fused_mul_sum|moe_reduce|moe_finalize|moe_combine\\"),\\n    (\\"moe\\", \\"moe_other\\", r\\"moe|expert\\"),\\n    (\\"hc_mix\\", \\"hc_mix_fused\\", r\\"hc_mix|hcmix\\"),\\n    (\\"hc_mix\\", \\"hc_combine\\", r\\"hc_?combine\\"),\\n    (\\"hc_mix\\", \\"hc_norm\\", r\\"grouped_?gemma|grouped_\\\\w*rms\\"),\\n    (\\"hc_mix\\", \\"hc_fallback\\", r\\"^triton_(?:poi|per|red|tem)_\\"),\\n    (\\"gdn\\", \\"gdn_recurrent\\", r\\"gated_delta|delta_rule|recurrent|kda\\"),\\n    (\\"gdn\\", \\"gdn_conv\\", r\\"causal_conv1d|conv1d\\"),\\n    (\\"gdn\\", \\"gdn_norm\\", r\\"layer_norm_fwd_1pass|layernorm_gated|norm_gated|rms_norm_gated\\"),\\n    (\\"gdn\\", \\"gdn_other\\", r\\"gdn|l2norm|mamba|ssm_state|selective_state\\"),\\n    (\\"attn_qsa\\", \\"qsa_decode\\", r\\"qsa_sparse_decode|sparse_decode|sparse_gqa|decode_att|flash_?decod\\"),\\n    (\\"attn_qsa\\", \\"qsa_mqa\\", r\\"mqa|^kernel_kernel$|^main_kernel$|tilelang|indexer\\"),\\n    (\\"attn_qsa\\", \\"qsa_topk\\", r\\"fast_topk|topk_transform|radix_topk\\"),\\n    (\\"sampling\\", \\"sampling\\", r\\"sampl|top_?k_?top_?p|topktopp|top_p|min_p|multinomial|argmax|penalt|logit|renorm|\\"\\n                             r\\"gumbel|exponential_|softmax|logsumexp\\"),\\n    (\\"attn_qsa\\", \\"qsa_other\\", r\\"qsa|attn|attention|rotary|rope|flash|merge_state|compress\\"),\\n    (\\"dense_gemm\\", \\"dense_gemm\\", r\\"gemm|gemv|xmma|cutlass|cublas|s16816|s1688|splitk|sm[789]\\\\d_|ampere_|matmul|\\"\\n                                 r\\"_mm_|dot_kernel|ldg8|tensorop|nvjet\\"),\\n    (\\"other\\", \\"norm\\", r\\"rmsnorm|rms_norm|layernorm|layer_norm|norm_kernel\\"),\\n    (\\"other\\", \\"ple\\", r\\"_ple_|^ple_|ngram|short_conv|gate_value|_hash\\"),\\n    (\\"other\\", \\"act\\", r\\"silu|gelu|act_and_mul|swiglu|sigmoid\\"),\\n    (\\"other\\", \\"elementwise\\", r\\"elementwise|vectorized|unrolled|reduce_kernel|copy|catarray|index|gather|scatter|\\"\\n                             r\\"fill|arange|where|clamp|cumsum\\"),\\n]\\nRULES = [(t, d, re.compile(p, re.I)) for t, d, p in RULES]\\n_CAT_CACHE = {}\\n\\n\\ndef classify(name, ecat):\\n    if ecat == \\"gpu_memcpy\\":\\n        return \\"other\\", \\"memcpy\\"\\n    if ecat == \\"gpu_memset\\":\\n        return \\"other\\", \\"memset\\"\\n    r = _CAT_CACHE.get(name)\\n    if r is None:\\n        r = next(((t, d) for t, d, rx in RULES if rx.search(name)), (\\"other\\", \\"misc\\"))\\n        _CAT_CACHE[name] = r\\n    return r\\n\\n\\nclass K:\\n    __slots__ = (\\"ts\\", \\"te\\", \\"name\\", \\"ecat\\", \\"stream\\", \\"corr\\", \\"top\\", \\"det\\")\\n\\n    def __init__(self, ts, te, name, ecat, stream, corr):\\n        self.ts, self.te, self.name, self.ecat, self.stream, self.corr = ts, te, name, ecat, stream, corr\\n        self.top, self.det = classify(name, ecat)\\n\\n\\ndef load_trace(path):\\n    with (gzip.open(path, \\"rt\\") if path.endswith(\\".gz\\") else open(path)) as f:\\n        d = json.load(f)\\n    ev = d.get(\\"traceEvents\\", [])\\n    base_ns = float(d.get(\\"baseTimeNanoseconds\\") or 0)\\n    tss = [float(e[\\"ts\\"]) for e in ev[:2000] if e.get(\\"ph\\") == \\"X\\" and \\"ts\\" in e]\\n    # kineto writes ts relative to baseTimeNanoseconds; older traces are absolute epoch us (~1.7e15)\\n    off = base_ns / 1e3 if base_ns and tss and st.median(tss) < 1e12 else 0.0\\n    kern, launches = [], {}\\n    for e in ev:\\n        if e.get(\\"ph\\") != \\"X\\":\\n            continue\\n        c = str(e.get(\\"cat\\", \\"\\")).lower()\\n        if c in (\\"kernel\\", \\"gpu_memcpy\\", \\"gpu_memset\\"):\\n            a = e.get(\\"args\\") or {}\\n            ts = float(e[\\"ts\\"]) + off\\n            kern.append(K(ts, ts + float(e.get(\\"dur\\") or 0.0), str(e.get(\\"name\\", \\"\\")), c,\\n                          a.get(\\"stream\\", e.get(\\"tid\\")), a.get(\\"correlation\\")))\\n        elif c in (\\"cuda_runtime\\", \\"cuda_driver\\") and \\"GraphLaunch\\" in str(e.get(\\"name\\", \\"\\")):\\n            corr = (e.get(\\"args\\") or {}).get(\\"correlation\\")\\n            if corr is not None:\\n                launches[corr] = float(e[\\"ts\\"]) + off\\n    kern.sort(key=lambda k: k.ts)\\n    return kern, launches, {\\"events\\": len(ev), \\"ts_offset_applied\\": bool(off)}\\n\\n\\ndef union_len(iv):\\n    tot, cs, ce = 0.0, None, None\\n    for s, e in sorted(iv):\\n        if ce is None or s > ce:\\n            if ce is not None:\\n                tot += ce - cs\\n            cs, ce = s, e\\n        elif e > ce:\\n            ce = e\\n    return tot + (ce - cs if ce is not None else 0.0)\\n\\n\\ndef split_steps(kern, launches, mode, min_k):\\n    \\"\\"\\"-> (method, [(t0, t1, kernels_in_window, in_graph_id_set_or_None)], info)\\"\\"\\"\\n    info = {}\\n    if mode in (\\"auto\\", \\"graph\\") and launches:\\n        groups = collections.defaultdict(list)\\n        for k in kern:\\n            if k.corr in launches:\\n                groups[k.corr].append(k)\\n        reps = [g for g in groups.values() if len(g) >= min_k]\\n        if reps:\\n            n_mode = collections.Counter(len(g) for g in reps).most_common(1)[0][0]\\n            full = sorted((g for g in reps if len(g) >= 0.9 * n_mode), key=lambda g: g[0].ts)\\n            info.update(graph_launches=len(launches), replays=len(reps), replays_dropped_partial=len(reps) - len(full),\\n                        kernels_per_replay_mode=n_mode)\\n            if len(full) >= 3:\\n                starts = [g[0].ts for g in full]\\n                return \\"graph\\", [(starts[i], starts[i + 1], full[i]) for i in range(len(full) - 1)], info\\n        if mode == \\"graph\\":\\n            return \\"graph\\", [], dict(info, error=\\"fewer than 3 full graph replays\\")\\n    ags = [k for k in kern if k.top == \\"nccl\\" and k.det == \\"allgather\\"]\\n    info[\\"allgathers\\"] = len(ags)\\n    if len(ags) < 3:\\n        return \\"none\\", [], dict(info, error=\\"no graph replays and fewer than 3 AllGather kernels to split on\\")\\n    ends = [k.te for k in ags]\\n    return \\"allgather\\", [(ends[i], ends[i + 1], None) for i in range(len(ends) - 1)], info\\n\\n\\ndef analyse_rank(path, mode=\\"auto\\", min_k=50, hc_heuristic=True, top_n=40):\\n    kern, launches, meta = load_trace(path)\\n    method, steps_raw, info = split_steps(kern, launches, mode, min_k)\\n    out = {\\"file\\": os.path.basename(path), \\"split\\": method, **info, **meta}\\n    if not steps_raw:\\n        out[\\"error\\"] = info.get(\\"error\\", \\"no steps\\")\\n        return out, None\\n    kts = [k.ts for k in kern]\\n    steps, comm_seq = [], []\\n    for t0, t1, graph in steps_raw:\\n        win = kern[bisect.bisect_left(kts, t0):bisect.bisect_left(kts, t1)]\\n        gset = set(map(id, graph)) if graph is not None else None\\n        if hc_heuristic:   # a run of dense GEMMs (mm [+ splitK reduce]) followed by an inductor kernel = HC fallback\\n            by_stream = collections.defaultdict(list)\\n            for k in (graph if graph is not None else win):\\n                by_stream[k.stream].append(k)\\n            for seq in by_stream.values():\\n                i = 0\\n                while i < len(seq):\\n                    if seq[i].top != \\"dense_gemm\\":\\n                        i += 1\\n                        continue\\n                    j = i\\n                    while j < len(seq) and seq[j].top == \\"dense_gemm\\":\\n                        j += 1\\n                    if j < len(seq) and seq[j].det == \\"hc_fallback\\":\\n                        for k in seq[i:j]:\\n                            k.top, k.det = \\"hc_mix\\", \\"hc_fallback_gemm\\"\\n                    i = j\\n\\n        def cat(k):\\n            if gset is not None and id(k) not in gset:\\n                return \\"sampling\\", \\"outside_graph:\\" + (k.det if k.top in (\\"sampling\\", \\"other\\") else k.top)\\n            return k.top, k.det\\n\\n        cats = [cat(k) for k in win]\\n        evs = []\\n        for i, k in enumerate(win):\\n            s, e = max(k.ts, t0), min(k.te, t1)\\n            if e > s:\\n                evs.append((s, 1, i))\\n                evs.append((e, 0, i))\\n        evs.sort()\\n        excl, incl, det_us, det_n, nm_us, nm_n = (collections.Counter() for _ in range(6))\\n        active, last, idle, overlap = set(), t0, 0.0, 0.0\\n        for t, typ, i in evs:\\n            dt = t - last\\n            if dt > 0:\\n                if active:\\n                    sh = dt / len(active)\\n                    for j in active:\\n                        excl[cats[j][0]] += sh\\n                    if len(active) > 1:\\n                        overlap += dt\\n                else:\\n                    idle += dt\\n                last = t\\n            if typ == 1:\\n                active.add(i)\\n            else:\\n                active.discard(i)\\n        idle += max(0.0, t1 - last)\\n        for k, (tp, dt_) in zip(win, cats):\\n            d = min(k.te, t1) - max(k.ts, t0)\\n            if d <= 0:\\n                continue\\n            incl[tp] += d\\n            det_us[f\\"{tp}/{dt_}\\"] += d\\n            det_n[f\\"{tp}/{dt_}\\"] += 1\\n            nm_us[(k.name[:140], tp, dt_)] += d\\n            nm_n[(k.name[:140], tp, dt_)] += 1\\n        comm = [k for k in (graph if graph is not None else win) if k.top == \\"nccl\\"]\\n        comm_seq.append((t0, t1 - t0, [k.te - k.ts for k in comm], [k.det for k in comm]))\\n        g_span = (max(k.te for k in graph) - graph[0].ts) if graph is not None else None\\n        steps.append({\\"period\\": t1 - t0, \\"idle\\": idle, \\"overlap\\": overlap, \\"excl\\": excl, \\"incl\\": incl,\\n                      \\"det_us\\": det_us, \\"det_n\\": det_n, \\"nm_us\\": nm_us, \\"nm_n\\": nm_n,\\n                      \\"n_graph\\": len(graph) if graph is not None else None,\\n                      \\"n_win\\": len(win), \\"graph_span\\": g_span,\\n                      \\"busy\\": union_len([(max(k.ts, t0), min(k.te, t1)) for k in win if k.te > t0])})\\n    med = st.median(s[\\"period\\"] for s in steps)\\n    ok_idx = [i for i, s in enumerate(steps) if s[\\"period\\"] <= 1.5 * med]   # drop prefill-contaminated / stalled\\n    keep = [steps[i] for i in ok_idx]\\n    comm_seq = [comm_seq[i] for i in ok_idx]\\n    n = len(keep)\\n    ms = lambda v: round(v / 1e3, 3)\\n    msum = lambda key: collections.Counter({c: sum(s[key][c] for s in keep) / n for c in\\n                                             set().union(*[set(s[key]) for s in keep])})\\n    excl, incl, dus, dn, nus, nn = (msum(x) for x in (\\"excl\\", \\"incl\\", \\"det_us\\", \\"det_n\\", \\"nm_us\\", \\"nm_n\\"))\\n    mean_idle = sum(s[\\"idle\\"] for s in keep) / n\\n    excl_ms = {c: ms(excl.get(c, 0.0)) for c in TOP}\\n    excl_ms[\\"idle\\"] = ms(mean_idle)\\n    nc = collections.Counter()\\n    for _, _, _, kinds in comm_seq:\\n        nc.update(kinds)\\n    out.update({\\n        \\"n_steps\\": n, \\"n_steps_dropped_outlier\\": len(steps) - n,\\n        \\"step_ms\\": {\\"mean\\": ms(sum(s[\\"period\\"] for s in keep) / n), \\"p50\\": ms(st.median(s[\\"period\\"] for s in keep)),\\n                    \\"min\\": ms(min(s[\\"period\\"] for s in keep)), \\"max\\": ms(max(s[\\"period\\"] for s in keep))},\\n        \\"cat_ms\\": excl_ms,\\n        \\"cat_ms_inclusive\\": {c: ms(incl.get(c, 0.0)) for c in TOP},\\n        \\"busy_ms\\": ms(sum(s[\\"busy\\"] for s in keep) / n), \\"overlap_ms\\": ms(sum(s[\\"overlap\\"] for s in keep) / n),\\n        \\"graph_span_ms\\": ms(sum(s[\\"graph_span\\"] for s in keep) / n) if keep[0][\\"graph_span\\"] is not None else None,\\n        \\"graph_kernels_per_step\\": round(sum(s[\\"n_graph\\"] for s in keep) / n, 1) if keep[0][\\"n_graph\\"] else None,\\n        \\"kernels_per_step\\": round(sum(s[\\"n_win\\"] for s in keep) / n, 1),\\n        \\"detail\\": {k: {\\"ms\\": ms(v), \\"calls\\": round(dn[k], 2), \\"us_per_call\\": round(v / max(dn[k], 1e-9), 2)}\\n                   for k, v in sorted(dus.items(), key=lambda kv: -kv[1])},\\n        \\"nccl_calls_per_step\\": {k: round(v / len(comm_seq), 2) for k, v in nc.items()},\\n        \\"top_kernels\\": [{\\"name\\": nm, \\"cat\\": f\\"{tp}/{dt_}\\", \\"us_per_step\\": round(v, 1),\\n                         \\"calls_per_step\\": round(nn[(nm, tp, dt_)], 2),\\n                         \\"us_per_call\\": round(v / max(nn[(nm, tp, dt_)], 1e-9), 2)}\\n                        for (nm, tp, dt_), v in sorted(nus.items(), key=lambda kv: -kv[1])[:top_n]],\\n    })\\n    out[\\"nccl_kernel_names\\"] = sorted({x[\\"name\\"] for x in out[\\"top_kernels\\"] if x[\\"cat\\"].startswith(\\"nccl/\\")})\\n    return out, comm_seq\\n\\n\\ndef cross_rank(seqs):\\n    \\"\\"\\"seqs[r] = [(t0, period, [nccl durations in order], [kinds])]. Steps matched by start time (same host clock),\\n    falling back to index order. pure = min over ranks of each collective, wait_r = excess on rank r.\\"\\"\\"\\n    if len(seqs) < 2 or any(not s for s in seqs):\\n        return {\\"error\\": \\"need >= 2 ranks with steps\\"}\\n    per = st.median(p for _, p, _, _ in seqs[0])\\n    starts = [[x[0] for x in s] for s in seqs]\\n    tuples = []\\n    for i, (t0, _, _, _) in enumerate(seqs[0]):\\n        row = [seqs[0][i]]\\n        for r in range(1, len(seqs)):\\n            j = bisect.bisect_left(starts[r], t0)\\n            best = min((jj for jj in (j - 1, j) if 0 <= jj < len(starts[r])),\\n                       key=lambda jj: abs(starts[r][jj] - t0), default=None)\\n            if best is None or abs(starts[r][best] - t0) > 0.5 * per:\\n                break\\n            row.append(seqs[r][best])\\n        if len(row) == len(seqs):\\n            tuples.append(row)\\n    method = \\"time\\"\\n    if len(tuples) < 0.5 * len(seqs[0]):\\n        method = \\"index\\"\\n        tuples = [list(x) for x in zip(*seqs)]\\n    pure, wait, pure_by = [], [[] for _ in seqs], collections.defaultdict(list)\\n    skipped = 0\\n    for row in tuples:\\n        if len({len(x[2]) for x in row}) != 1 or not row[0][2]:\\n            skipped += 1\\n            continue\\n        mins = [min(x[2][j] for x in row) for j in range(len(row[0][2]))]\\n        pure.append(sum(mins))\\n        for kind in set(row[0][3]):\\n            pure_by[kind].append(sum(m for m, kk in zip(mins, row[0][3]) if kk == kind))\\n        for r, x in enumerate(row):\\n            wait[r].append(sum(d - m for d, m in zip(x[2], mins)))\\n    if not pure:\\n        return {\\"error\\": \\"collective counts differ across ranks in every step\\", \\"align\\": method}\\n    return {\\"align\\": method, \\"steps_paired\\": len(pure), \\"steps_skipped\\": skipped,\\n            \\"nccl_pure_ms\\": round(st.mean(pure) / 1e3, 3),\\n            \\"nccl_pure_ms_by_kind\\": {k: round(st.mean(v) / 1e3, 3) for k, v in pure_by.items()},\\n            \\"nccl_wait_ms_by_rank\\": [round(st.mean(w) / 1e3, 3) for w in wait]}\\n\\n\\ndef group_files(paths):\\n    files = []\\n    for p in paths:\\n        if os.path.isdir(p):\\n            fs = glob.glob(os.path.join(p, \\"*.trace.json*\\")) or glob.glob(os.path.join(p, \\"*\\", \\"*.trace.json*\\"))\\n            files += fs\\n        else:\\n            files.append(p)\\n    groups = collections.defaultdict(list)\\n    for f in sorted(set(files)):\\n        b = os.path.basename(f)\\n        m = re.search(r\\"^(.*?)-TP-(\\\\d+)\\", b)\\n        if m:\\n            groups[(os.path.dirname(f), m.group(1))].append((int(m.group(2)), f))\\n        else:\\n            groups[(os.path.dirname(f), b)].append((0, f))\\n    return {k: [f for _, f in sorted(v)] for k, v in sorted(groups.items())}\\n\\n\\ndef find_baseline(drive_path, key):\\n    try:\\n        d = json.load(open(drive_path))\\n    except Exception:\\n        return None\\n    for run in d.get(\\"runs\\", []):\\n        pr = (run.get(\\"profile\\") or {}).get(\\"prefix\\")\\n        if pr and key.startswith(pr + \\"-\\"):\\n            return (run.get(\\"window\\") or {}).get(\\"step_ms_p50\\")\\n    return None\\n\\n\\ndef summarize_paths(paths, mode=\\"auto\\", min_k=50, hc_heuristic=True, top_n=40, baseline_step_ms=None, drive=None):\\n    out = []\\n    for (d, key), files in group_files(paths).items():\\n        ranks, seqs = [], []\\n        for f in files:\\n            try:\\n                r, seq = analyse_rank(f, mode, min_k, hc_heuristic, top_n)\\n            except Exception as ex:   # truncated gzip, bad JSON ...\\n                r, seq = {\\"file\\": os.path.basename(f), \\"error\\": f\\"{type(ex).__name__}: {str(ex)[:200]}\\"}, None\\n            ranks.append(r)\\n            if seq:\\n                seqs.append(seq)\\n        res = {\\"group\\": key, \\"dir\\": d, \\"files\\": [os.path.basename(f) for f in files], \\"ranks\\": ranks}\\n        good = [r for r in ranks if \\"cat_ms\\" in r]\\n        if len(seqs) >= 2:\\n            res[\\"cross_rank\\"] = cross_rank(seqs)\\n        if good:\\n            res[\\"summary\\"] = {\\n                \\"rank0_step_ms\\": good[0][\\"step_ms\\"][\\"mean\\"], \\"rank0_cat_ms\\": good[0][\\"cat_ms\\"],\\n                \\"mean_cat_ms\\": {c: round(st.mean(r[\\"cat_ms\\"][c] for r in good), 3) for c in TOP + [\\"idle\\"]},\\n                \\"moe_ms_by_rank\\": [r[\\"cat_ms\\"][\\"moe\\"] for r in good],\\n                \\"nccl_calls_per_step\\": good[0][\\"nccl_calls_per_step\\"],\\n                \\"hc_detail_calls\\": {k: v[\\"calls\\"] for k, v in good[0][\\"detail\\"].items() if k.startswith(\\"hc_mix/\\")},\\n            }\\n            b = baseline_step_ms\\n            if b is None and drive:\\n                b = find_baseline(drive, key)\\n            if b is None:\\n                for cand in (os.path.join(d, \\"drive.json\\"), os.path.join(os.path.dirname(d), \\"drive.json\\")):\\n                    if os.path.exists(cand):\\n                        b = find_baseline(cand, key)\\n                        if b:\\n                            break\\n            if b:\\n                res[\\"summary\\"][\\"unprofiled_step_ms\\"] = b\\n                res[\\"summary\\"][\\"profiler_overhead_pct\\"] = round(100 * (good[0][\\"step_ms\\"][\\"mean\\"] / b - 1), 1)\\n        out.append(res)\\n    return out\\n\\n\\ndef format_table(results):\\n    lines = []\\n    hdr = \\"rank  step_ms \\" + \\" \\".join(f\\"{c[:6]:>7}\\" for c in TOP + [\\"idle\\"]) + \\"   nccl calls/step\\"\\n    for res in results:\\n        s = res.get(\\"summary\\", {})\\n        cr = res.get(\\"cross_rank\\", {})\\n        good = [r for r in res[\\"ranks\\"] if \\"cat_ms\\" in r]\\n        split = good[0][\\"split\\"] if good else \\"-\\"\\n        lines.append(f\\"== {res[\'group\']} ({len(res[\'files\'])} files, split={split}, steps={good[0][\'n_steps\'] if good else 0})\\"\\n                     + (f\\" unprofiled {s[\'unprofiled_step_ms\']} ms, profiler overhead {s[\'profiler_overhead_pct\']}%\\"\\n                        if \\"unprofiled_step_ms\\" in s else \\"\\"))\\n        lines.append(hdr)\\n        for i, r in enumerate(res[\\"ranks\\"]):\\n            if \\"cat_ms\\" not in r:\\n                lines.append(f\\"{i:<5} ERROR {r.get(\'error\')}\\")\\n                continue\\n            c = r[\\"cat_ms\\"]\\n            lines.append(f\\"{i:<5} {r[\'step_ms\'][\'mean\']:7.2f} \\" + \\" \\".join(f\\"{c[k]:7.2f}\\" for k in TOP + [\\"idle\\"])\\n                         + \\"   \\" + \\",\\".join(f\\"{k}:{v:g}\\" for k, v in sorted(r[\\"nccl_calls_per_step\\"].items())))\\n        if \\"nccl_pure_ms\\" in cr:\\n            lines.append(f\\"nccl pure {cr[\'nccl_pure_ms\']} ms/step {cr[\'nccl_pure_ms_by_kind\']} | wait by rank \\"\\n                         f\\"{cr[\'nccl_wait_ms_by_rank\']} ({cr[\'align\']}-aligned, {cr[\'steps_paired\']} steps)\\")\\n        if s.get(\\"hc_detail_calls\\"):\\n            lines.append(f\\"hc calls/step {s[\'hc_detail_calls\']}\\")\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef cg_selftest():\\n    \\"\\"\\"GPU check of the graph-split assumption with this torch/CUPTI: 6 replays of a 60-kernel graph separated by an\\n    out-of-graph kernel must come back as 6 replays sharing one correlation id each, and 5 analysed steps.\\"\\"\\"\\n    import shutil\\n    import tempfile\\n    import torch\\n    from torch.profiler import ProfilerActivity, profile\\n    x = torch.randn(64, 1024, device=\\"cuda\\", dtype=torch.bfloat16)\\n    w = torch.randn(1024, 1024, device=\\"cuda\\", dtype=torch.bfloat16) * 0.03\\n\\n    def body():\\n        y = x\\n        for _ in range(30):\\n            y = torch.relu(y @ w)\\n        return y\\n\\n    s = torch.cuda.Stream()\\n    s.wait_stream(torch.cuda.current_stream())\\n    with torch.cuda.stream(s):\\n        for _ in range(3):\\n            body()\\n    torch.cuda.current_stream().wait_stream(s)\\n    g = torch.cuda.CUDAGraph()\\n    with torch.cuda.graph(g):\\n        out = body()\\n    torch.cuda.synchronize()\\n    d = tempfile.mkdtemp(prefix=\\"cgself\\")\\n    path = os.path.join(d, \\"cgself-0-TP-0.trace.json\\")\\n    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as p:\\n        for _ in range(6):\\n            g.replay()\\n            torch.softmax(out.float(), dim=-1)   # out-of-graph \\"sampling\\" kernel\\n        torch.cuda.synchronize()\\n    p.export_chrome_trace(path)\\n    kern, launches, meta = load_trace(path)\\n    corr = collections.Counter(k.corr for k in kern if k.corr in launches)\\n    res = {\\"kernels\\": len(kern), \\"graph_launch_events\\": len(launches), \\"replays_seen\\": len(corr),\\n           \\"kernels_per_replay\\": sorted(corr.values()), **meta}\\n    summ = summarize_paths([d], min_k=40)\\n    r0 = summ[0][\\"ranks\\"][0] if summ else {}\\n    res.update(split=r0.get(\\"split\\"), n_steps=r0.get(\\"n_steps\\"), step_ms=r0.get(\\"step_ms\\"), cat_ms=r0.get(\\"cat_ms\\"),\\n               error=r0.get(\\"error\\"))\\n    res[\\"nsys\\"] = shutil.which(\\"nsys\\")\\n    res[\\"ncu\\"] = shutil.which(\\"ncu\\")\\n    res[\\"n_steps_dropped_outlier\\"] = r0.get(\\"n_steps_dropped_outlier\\")\\n    ok = (res[\\"replays_seen\\"] == 6 and res[\\"split\\"] == \\"graph\\"\\n          and (res[\\"n_steps\\"] or 0) + (res[\\"n_steps_dropped_outlier\\"] or 0) == 5)\\n    print(json.dumps({\\"cg_selftest\\": res, \\"ok\\": ok}), flush=True)\\n    return ok\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__.split(\\"\\\\n\\")[0])\\n    ap.add_argument(\\"paths\\", nargs=\\"*\\")\\n    ap.add_argument(\\"--out\\")\\n    ap.add_argument(\\"--split\\", choices=[\\"auto\\", \\"graph\\", \\"allgather\\"], default=\\"auto\\")\\n    ap.add_argument(\\"--min-graph-kernels\\", type=int, default=50)\\n    ap.add_argument(\\"--no-hc-heuristic\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--top\\", type=int, default=40)\\n    ap.add_argument(\\"--baseline-step-ms\\", type=float)\\n    ap.add_argument(\\"--drive\\", help=\\"prof_drive.py drive.json (unprofiled step per profile prefix)\\")\\n    ap.add_argument(\\"--cg-selftest\\", action=\\"store_true\\")\\n    a = ap.parse_args()\\n    if a.cg_selftest:\\n        sys.exit(0 if cg_selftest() else 1)\\n    if not a.paths:\\n        ap.error(\\"give a trace dir or files\\")\\n    res = summarize_paths(a.paths, a.split, a.min_graph_kernels, not a.no_hc_heuristic, a.top, a.baseline_step_ms,\\n                          a.drive)\\n    js = json.dumps(res, indent=1)\\n    if a.out:\\n        with open(a.out, \\"w\\") as f:\\n            f.write(js)\\n    print(format_table(res), flush=True)\\n    if not a.out:\\n        print(js[:20000])\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "prune_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s qwen4_exp.py (Qwen4ExpForConditionalGeneration.load_weights = the TARGET model; the MTP\\ndrafter has its own loader) for load-time expert pruning, gated by SGLANG_PRUNE_KEEP=/path/keep_N.json\\n({\\"num_experts\\": N, \\"layers\\": [[sorted kept original expert ids] x 48]}); a no-op when the variable is unset.\\nThe weight iterator is wrapped so that, per layer L, kept expert E becomes expert keep[L].index(E) and pruned experts are\\nskipped, and the router model.language_model.layers.L.mlp.gate.weight [512, H] is cut to the kept rows in the same order.\\nThe server must also build the target with num_experts=N: --json-model-override-args \'{\\"text_config\\":{\\"num_experts\\":N}}\'\\n(+ --speculative-draft-model-override-args \'{}\' when MTP is on, so the drafter keeps its 512 experts).\\nusage: prune_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [prune patch]\\"\\nANCHOR = \'        num_experts = getattr(self.config, \\"num_experts\\", None)\\\\n\'\\nCALL = f\\"        weights = _prune_expert_weights(weights)  {MARK}\\\\n\\"\\nHELPER = f\'\'\'\\n\\n{MARK} load-time expert pruning, env SGLANG_PRUNE_KEEP=/path/keep_N.json; no-op when unset\\ndef _prune_expert_weights(weights):\\n    import json as _json\\n    import logging as _logging\\n    import os as _os\\n    import re as _re\\n\\n    path = _os.environ.get(\\"SGLANG_PRUNE_KEEP\\")\\n    if not path:\\n        yield from weights\\n        return\\n    keep = _json.load(open(path))\\n    layers = keep[\\"layers\\"]\\n    remap = [{{e: i for i, e in enumerate(k)}} for k in layers]\\n    rows = [torch.tensor(k, dtype=torch.long) for k in layers]\\n    exp_re = _re.compile(r\\"^(model\\\\\\\\.(?:language_model\\\\\\\\.)?layers\\\\\\\\.(\\\\\\\\d+)\\\\\\\\.mlp\\\\\\\\.experts\\\\\\\\.)(\\\\\\\\d+)(\\\\\\\\..+)$\\")\\n    gate_re = _re.compile(r\\"^model\\\\\\\\.(?:language_model\\\\\\\\.)?layers\\\\\\\\.(\\\\\\\\d+)\\\\\\\\.mlp\\\\\\\\.gate\\\\\\\\.weight$\\")\\n    n_ren = n_skip = n_gate = 0\\n    for name, w in weights:\\n        m = exp_re.match(name)\\n        if m is not None:\\n            new = remap[int(m.group(2))].get(int(m.group(3)))\\n            if new is None:\\n                n_skip += 1\\n                continue\\n            n_ren += 1\\n            yield f\\"{{m.group(1)}}{{new}}{{m.group(4)}}\\", w\\n            continue\\n        g = gate_re.match(name)\\n        if g is not None:\\n            n_gate += 1\\n            yield name, w.index_select(0, rows[int(g.group(1))].to(w.device))\\n            continue\\n        yield name, w\\n    _logging.getLogger(__name__).info(\\n        \\"[prune patch] %s: N=%s, renamed %d expert tensors, skipped %d, sliced %d routers\\",\\n        path, keep.get(\\"num_experts\\"), n_ren, n_skip, n_gate)\\n\'\'\'\\n\\n\\ndef main():\\n    f = Path(sys.argv[1]) / \\"sglang/srt/models/qwen4_exp.py\\"\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"prune_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"prune_patch: NOT patched\\")\\n    if s.count(ANCHOR) != 1:\\n        sys.exit(\\"prune_patch: anchor not found exactly once; refusing to patch\\")\\n    f.write_text(s.replace(ANCHOR, CALL + ANCHOR) + HELPER)\\n    print(\\"prune_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "qsa_sm89_decode.py": "\\"\\"\\"QSA sparse decode / verify attention for GPUs without trtllm-gen or a usable FA kernel.\\n\\nInstalled by apply_patches.py as ``sglang/srt/layers/attention/qsa/sm89_decode.py``\\ninside the Pennyroyal SGLang (0.5.19+gd00d88efc8d6) venv.\\n\\nWhy it exists (Pennyroyal source, paths under sglang/srt/):\\n  * QwenSparseAttnBackend._forward_paged_attention (layers/attention/\\n    qwen_sparse_attn_backend.py:1869-1961) serves decode, target-verify and\\n    draft-extend for every QSA layer. It uses FlashInfer trtllm-gen only on\\n    SM100/SM120 (:53-73); everywhere else it packs the top-k K/V of each query\\n    row into a scratch buffer (:1926-1948) and calls a dense varlen kernel\\n    resolved at :76-110: ``flash_attn`` (FA2, not in the wheelhouse) or the\\n    flash-attn-4 CuTe interface.\\n  * On sm_89 that fails: with fp8_e4m3 KV the packed K/V are fp8 while q is\\n    bf16 (the pool returns a float8 view, mem_cache/memory_pool.py:2440-2455),\\n    and FA4-CuTe asserts equal q/k/v dtypes and \\"FP8 is only supported on\\n    SM100\\"; with bf16 KV its SM80 class uses a 128x64 hd256 tile = 128 KB of\\n    shared memory, checked only against the sm_80 capacity (\\"TODO: sm86 and\\n    sm89\\"), which exceeds sm_89\'s 99 KB per block.\\n  * The QSA layers are always routed to this backend, whatever\\n    --attention-backend says (layers/attention/attention_registry.py:459-467),\\n    so no flag avoids the path.\\n\\nThis module replaces only the dense fallback. One Triton program per (query row,\\nKV head) reads the selected tokens straight from the paged KV cache through\\nreq_to_token. There is no scratch copy, no host sync and the grid is static,\\nso it is CUDA-graph safe. A vectorised PyTorch implementation is kept as a\\nfallback (SGLANG_QSA_SM89_DECODE=torch) and a loop reference is kept for tests.\\n\\nSemantics are those of qsa_sparse_attention_reference (layers/attention/qsa/\\nkernel.py:491-515). A selected logical position p of row r is used iff\\n0 <= p < sequence_length[r]. Every query row attends to all of its valid\\nselected tokens (the FA call used causal=True with max_seqlen_q=1, which\\nmasks nothing). softmax scale = layer.scaling. FP8 KV is upcast without a\\nscale, because the QSA backend writes it with set_kv_buffer(layer, loc, k, v)\\nand passes no k/v scale (qwen_sparse_attn_backend.py:1572-1574, 1863-1865).\\nThe stock packing path additionally assumes the valid entries are a prefix of\\nthe row (sparse_attn.py:396). The two agree whenever that holds. Where it does\\nnot, this module matches the reference and the stock path would drop entries.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport os\\nfrom typing import Optional\\n\\nimport torch\\n\\ntry:  # the Triton kernel is optional so the torch fallback/reference import anywhere\\n    import triton\\n    import triton.language as tl\\n\\n    HAS_TRITON = True\\nexcept Exception:  # pragma: no cover - CPU-only dev boxes\\n    triton = None\\n    tl = None\\n    HAS_TRITON = False\\n\\n\\nif HAS_TRITON:\\n\\n    @triton.jit\\n    def _qsa_sparse_decode_kernel(\\n        Q,\\n        K,\\n        V,\\n        Out,\\n        ReqToToken,\\n        ReqIdx,\\n        Indices,\\n        SeqLens,\\n        scale,\\n        topk,\\n        stride_qm,\\n        stride_qh,\\n        stride_kn,\\n        stride_kh,\\n        stride_vn,\\n        stride_vh,\\n        stride_om,\\n        stride_oh,\\n        stride_req,\\n        stride_idx,\\n        GROUP: tl.constexpr,\\n        BLOCK_H: tl.constexpr,\\n        BLOCK_N: tl.constexpr,\\n        HEAD_DIM: tl.constexpr,\\n        DOT_FP16: tl.constexpr,\\n    ):\\n        # bf16 MMA on GPU. DOT_FP16 exists only for the Triton CPU interpreter,\\n        # which mis-computes bf16 tl.dot.\\n        dot_dt = tl.float16 if DOT_FP16 else tl.bfloat16\\n        row = tl.program_id(0)\\n        kvh = tl.program_id(1)\\n        length = tl.load(SeqLens + row).to(tl.int64)\\n        req = tl.load(ReqIdx + row).to(tl.int64)\\n        offs_h = tl.arange(0, BLOCK_H)\\n        offs_d = tl.arange(0, HEAD_DIM)\\n        hmask = offs_h < GROUP\\n        qh = kvh * GROUP + offs_h\\n        q = tl.load(\\n            Q + row * stride_qm + qh[:, None] * stride_qh + offs_d[None, :],\\n            mask=hmask[:, None],\\n            other=0.0,\\n        )\\n        # fold softmax scale and log2(e) into q; exp2 below\\n        q = (q.to(tl.float32) * (scale * 1.4426950408889634)).to(dot_dt)\\n        m_i = tl.full([BLOCK_H], float(\\"-inf\\"), tl.float32)\\n        l_i = tl.zeros([BLOCK_H], tl.float32)\\n        acc = tl.zeros([BLOCK_H, HEAD_DIM], tl.float32)\\n        for start in range(0, topk, BLOCK_N):\\n            cols = start + tl.arange(0, BLOCK_N)\\n            pos = tl.load(\\n                Indices + row * stride_idx + cols, mask=cols < topk, other=-1\\n            ).to(tl.int64)\\n            valid = (pos >= 0) & (pos < length)\\n            slot = tl.load(\\n                ReqToToken + req * stride_req + tl.where(valid, pos, 0),\\n                mask=valid,\\n                other=0,\\n            ).to(tl.int64)\\n            # [BLOCK_N, D] rows are contiguous 256-element lines of the pool\\n            k = tl.load(\\n                K + slot[:, None] * stride_kn + kvh * stride_kh + offs_d[None, :],\\n                mask=valid[:, None],\\n                other=0.0,\\n            ).to(dot_dt)\\n            s = tl.dot(q, tl.trans(k))\\n            s = tl.where(valid[None, :], s, float(\\"-inf\\"))\\n            m_new = tl.maximum(m_i, tl.max(s, 1))\\n            m_safe = tl.where(m_new == float(\\"-inf\\"), 0.0, m_new)\\n            alpha = tl.math.exp2(m_i - m_safe)\\n            p = tl.math.exp2(s - m_safe[:, None])\\n            v = tl.load(\\n                V + slot[:, None] * stride_vn + kvh * stride_vh + offs_d[None, :],\\n                mask=valid[:, None],\\n                other=0.0,\\n            ).to(dot_dt)\\n            acc = acc * alpha[:, None] + tl.dot(p.to(dot_dt), v)\\n            l_i = l_i * alpha + tl.sum(p, 1)\\n            m_i = m_new\\n        out = acc / tl.where(l_i > 0.0, l_i, 1.0)[:, None]\\n        tl.store(\\n            Out + row * stride_om + qh[:, None] * stride_oh + offs_d[None, :],\\n            out.to(Out.dtype.element_ty),\\n            mask=hmask[:, None],\\n        )\\n\\n\\ndef _as_kv_view(buf: torch.Tensor) -> torch.Tensor:\\n    # pools store fp8 as uint8 and hand out a float8 view; accept either\\n    if buf.dtype == torch.uint8:\\n        return buf.view(torch.float8_e4m3fn)\\n    return buf\\n\\n\\ndef qsa_sparse_decode_triton(\\n    q: torch.Tensor,  # [rows, Hq, D] bf16\\n    k_buffer: torch.Tensor,  # [slots, Hkv, D] bf16 or float8_e4m3fn\\n    v_buffer: torch.Tensor,\\n    req_to_token: torch.Tensor,  # [req_pool, max_ctx] int32\\n    req_pool_indices: torch.Tensor,  # [rows]\\n    topk_indices: torch.Tensor,  # [rows, topk] logical positions, -1 = none\\n    sequence_lens: torch.Tensor,  # [rows]\\n    scale: float,\\n    block_n: int = 32,\\n    _dot_fp16: bool = False,\\n) -> torch.Tensor:\\n    if not HAS_TRITON:\\n        raise RuntimeError(\\"triton is not available\\")\\n    rows, num_q_heads, head_dim = q.shape\\n    k_buffer, v_buffer = _as_kv_view(k_buffer), _as_kv_view(v_buffer)\\n    num_kv_heads = k_buffer.shape[1]\\n    assert num_q_heads % num_kv_heads == 0, (num_q_heads, num_kv_heads)\\n    assert k_buffer.stride(-1) == 1 and v_buffer.stride(-1) == 1\\n    assert head_dim == triton.next_power_of_2(head_dim), head_dim\\n    group = num_q_heads // num_kv_heads\\n    q = q.contiguous()\\n    topk_indices = topk_indices.contiguous()\\n    out = torch.empty_like(q)\\n    if rows == 0:\\n        return out\\n    _qsa_sparse_decode_kernel[(rows, num_kv_heads)](\\n        q,\\n        k_buffer,\\n        v_buffer,\\n        out,\\n        req_to_token,\\n        req_pool_indices,\\n        topk_indices,\\n        sequence_lens,\\n        float(scale),\\n        topk_indices.shape[1],\\n        q.stride(0),\\n        q.stride(1),\\n        k_buffer.stride(0),\\n        k_buffer.stride(1),\\n        v_buffer.stride(0),\\n        v_buffer.stride(1),\\n        out.stride(0),\\n        out.stride(1),\\n        req_to_token.stride(0),\\n        topk_indices.stride(0),\\n        GROUP=group,\\n        BLOCK_H=max(16, triton.next_power_of_2(group)),\\n        BLOCK_N=block_n,\\n        HEAD_DIM=head_dim,\\n        DOT_FP16=_dot_fp16,\\n        num_warps=4,\\n        num_stages=2,\\n    )\\n    return out\\n\\n\\ndef _gather_rows(buf: torch.Tensor, slots: torch.Tensor) -> torch.Tensor:\\n    # index fp8 through a uint8 view (fp8 advanced indexing is not implemented\\n    # for every device/torch build)\\n    if buf.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):\\n        return buf.view(torch.uint8)[slots].view(buf.dtype)\\n    return buf[slots]\\n\\n\\ndef qsa_sparse_decode_torch(\\n    q, k_buffer, v_buffer, req_to_token, req_pool_indices, topk_indices,\\n    sequence_lens, scale,\\n):\\n    \\"\\"\\"Vectorised, shape-static (CUDA-graph safe) PyTorch fallback.\\n\\n    Transient memory ~ 2 * rows * topk * Hkv * D * 4 bytes (fp32 K and V);\\n    ~200 MB at 48 rows, topk 2051, D 256, Hkv 1.\\n    \\"\\"\\"\\n    rows, hq, d = q.shape\\n    k_buffer, v_buffer = _as_kv_view(k_buffer), _as_kv_view(v_buffer)\\n    hkv = k_buffer.shape[1]\\n    g = hq // hkv\\n    pos = topk_indices.long()\\n    valid = (pos >= 0) & (pos < sequence_lens.long().unsqueeze(1))\\n    safe = torch.where(valid, pos, torch.zeros_like(pos)).clamp_(\\n        max=req_to_token.shape[1] - 1\\n    )\\n    slots = req_to_token[req_pool_indices.long().unsqueeze(1), safe].long()\\n    slots = torch.where(valid, slots, torch.zeros_like(slots))\\n    k = _gather_rows(k_buffer, slots).float()  # [rows, topk, hkv, d]\\n    v = _gather_rows(v_buffer, slots).float()\\n    qg = q.float().view(rows, hkv, g, d)\\n    s = torch.einsum(\\"rhgd,rthd->rhgt\\", qg, k) * float(scale)\\n    s = s.masked_fill(~valid[:, None, None, :], float(\\"-inf\\"))\\n    p = torch.softmax(s, dim=-1).nan_to_num_(0.0)  # rows with no valid entry -> 0\\n    out = torch.einsum(\\"rhgt,rthd->rhgd\\", p, v)\\n    return out.reshape(rows, hq, d).to(q.dtype)\\n\\n\\ndef qsa_sparse_decode_reference(\\n    q, k_buffer, v_buffer, req_to_token, req_pool_indices, topk_indices,\\n    sequence_lens, scale,\\n):\\n    \\"\\"\\"Loop reference: logical->physical mapping + qsa_sparse_attention_reference.\\"\\"\\"\\n    rows, hq, _ = q.shape\\n    k_buffer, v_buffer = _as_kv_view(k_buffer), _as_kv_view(v_buffer)\\n    hkv = k_buffer.shape[1]\\n    rep = hq // hkv\\n    outs = []\\n    for r in range(rows):\\n        pos = topk_indices[r].long()\\n        ok = (pos >= 0) & (pos < int(sequence_lens[r]))\\n        pos = pos[ok]\\n        if pos.numel() == 0:\\n            outs.append(torch.zeros_like(q[r]))\\n            continue\\n        slots = req_to_token[int(req_pool_indices[r]), pos].long()\\n        keys = _gather_rows(k_buffer, slots).float().repeat_interleave(rep, dim=1)\\n        vals = _gather_rows(v_buffer, slots).float().repeat_interleave(rep, dim=1)\\n        sc = torch.einsum(\\"hd,khd->hk\\", q[r].float(), keys) * float(scale)\\n        pr = torch.softmax(sc, dim=-1)\\n        outs.append(torch.einsum(\\"hk,khd->hd\\", pr, vals).to(q.dtype))\\n    return torch.stack(outs)\\n\\n\\n_IMPL_CACHE: dict = {}\\n\\n\\ndef select_sm89_decode_impl(device=None) -> Optional[str]:\\n    \\"\\"\\"Which implementation replaces the FA fallback, or None for stock behaviour.\\n\\n    SGLANG_QSA_SM89_DECODE = auto (default) | triton | torch | off.\\n    auto -> \\"triton\\" on compute capability < 9.0 (sm_8x, e.g. L4 sm_89), None\\n    elsewhere (SM90 keeps FA2/FA4, SM100/120 never reach this point).\\n    \\"\\"\\"\\n    # SGLANG_QSA_TRITON_DECODE is the older name used by vast/penny_patches\\n    flag = os.environ.get(\\n        \\"SGLANG_QSA_SM89_DECODE\\", os.environ.get(\\"SGLANG_QSA_TRITON_DECODE\\", \\"auto\\")\\n    ).strip().lower()\\n    key = (flag, str(device))\\n    if key in _IMPL_CACHE:\\n        return _IMPL_CACHE[key]\\n    if flag in (\\"off\\", \\"0\\", \\"false\\", \\"no\\", \\"stock\\"):\\n        impl = None\\n    elif flag == \\"torch\\":\\n        impl = \\"torch\\"\\n    elif flag in (\\"triton\\", \\"1\\", \\"true\\", \\"on\\"):\\n        impl = \\"triton\\" if HAS_TRITON else \\"torch\\"\\n    else:\\n        try:\\n            major, _ = torch.cuda.get_device_capability(device)\\n            impl = (\\"triton\\" if HAS_TRITON else \\"torch\\") if major < 9 else None\\n        except Exception:\\n            impl = None\\n    _IMPL_CACHE[key] = impl\\n    return impl\\n\\n\\ndef qsa_sparse_decode(impl: str, *args, **kwargs) -> torch.Tensor:\\n    if impl == \\"triton\\":\\n        return qsa_sparse_decode_triton(*args, **kwargs)\\n    return qsa_sparse_decode_torch(*args, **kwargs)\\n\\n\\n__all__ = [\\n    \\"HAS_TRITON\\",\\n    \\"qsa_sparse_decode\\",\\n    \\"qsa_sparse_decode_reference\\",\\n    \\"qsa_sparse_decode_torch\\",\\n    \\"qsa_sparse_decode_triton\\",\\n    \\"select_sm89_decode_impl\\",\\n]\\n", "qsadump_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal SGLang so QSA DECODE steps can dump the per-layer block top-k (the 512 block ids that\\nfast_topk returns, BEFORE expand_qsa_block_indices turns them into 2051 token positions), gated by\\nSGLANG_QSA_DUMP_DIR=<dir> (unset / \\"\\" = stock, no behaviour change). Offline analysis: qsadump_analyze.py.\\n\\nWhy: the HiSparse-for-QSA feasibility study (research/HISPARSE_QSA_FEASIBILITY.md section 6.5) needs the real\\nconsecutive-step overlap of the top-k block sets to price a block-granular hot buffer (LRU, hit rate -> PCIe misses).\\n\\nWhat the patch changes (3 places; nothing runs unless the env is set at import):\\n  * NEW sglang/srt/layers/attention/qsa/qsa_dump.py   env parse, buffering/flush class (numpy only), the hook.\\n  * qsa/qsa_indexer.py  QSAIndexer.select_decode_tokens: one `if _QSADUMP_ON:` line that parks the fast_topk output\\n        (block_indices [rows, 512] int32, compressed_lengths [rows] int32) on the indexer module (self._qsadump_stash).\\n        This is the only place where the raw block ids exist (they are local variables of that method).\\n  * models/qwen4_exp.py Qwen4ExpAttentionDecoderLayer._compute_qsa_topk_indices: after the indexer ran (and after the\\n        MTP-reuse early return, which is not hooked) one `if _QSADUMP_ON:` line calls the hook with\\n        (self.indexer, topk_indices, forward_batch, indexer_metadata).\\nThe hook records only forward_mode == DECODE with spec_info None (so not prefill/extend/mixed, not target-verify, not\\ndraft-extend, not draft decode), only outside CUDA-graph capture/replay metadata, only on the TP ranks listed in\\nSGLANG_QSA_DUMP_RANKS (default \\"0\\"). Any exception inside the hook is caught and logged; after 3 it disables itself.\\n\\nEnv (read once at import, per process):\\n  SGLANG_QSA_DUMP_DIR=<dir>           enables; parts are written to <dir>/qsa_dump_<rank>_<part>.npz\\n  SGLANG_QSA_DUMP_ROWS=K              first K rows of each decode batch (default 4, 1..256)\\n  SGLANG_QSA_DUMP_FLUSH_STEPS=N       write a part every N decode steps (default 64)\\n  SGLANG_QSA_DUMP_MAX_MB=M            stop recording after M MB of array data (default 400), logged once\\n  SGLANG_QSA_DUMP_RANKS=0[,1,..]      TP ranks that record (default \\"0\\"; the indexer is a ReplicatedLinear, so all\\n                                      ranks select the same blocks; extra ranks are only for verifying that)\\nRun with CUDA graphs disabled for decode (--disable-decode-cuda-graph): the hook copies one small tensor to the host\\nper layer per step (a sync), and under graph replay the Python hook would not run at all.\\n\\nusage: qsadump_patch.py <site-packages dir> [--check]\\n       qsadump_patch.py --selftest      (runs the embedded module\'s buffer/hook tests locally; needs numpy, torch optional)\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [qsadump patch]\\"\\nQSA_DIR = \\"sglang/srt/layers/attention/qsa/\\"\\nMODULE_REL = QSA_DIR + \\"qsa_dump.py\\"\\nINDEXER_REL = QSA_DIR + \\"qsa_indexer.py\\"\\nMODEL_REL = \\"sglang/srt/models/qwen4_exp.py\\"\\n\\nMODULE_SRC = r\'\'\'\\"\\"\\"QSA decode block top-k dump (dormant: active only when SGLANG_QSA_DUMP_DIR is set).\\n\\nInstalled by colab/perf_patches/qsadump_patch.py as sglang/srt/layers/attention/qsa/qsa_dump.py.\\n\\nFile format (np.savez, one part per SGLANG_QSA_DUMP_FLUSH_STEPS decode steps, whole steps per part). All arrays are\\nint32 except rid_table. Flat row records: one record = (decode step, QSA layer, batch row) and is chronological.\\n  blocks        [N, topk]  fast_topk output for that row: LOGICAL block numbers inside the request (block b = tokens\\n                           4b..4b+3, 0 <= b < nblk), unordered, padded with -1 exactly as produced\\n  step          [N]        decode-forward counter of this process (0,1,2,..); one value for all layers of a forward\\n  layer         [N]        global layer id of the QSA layer (self.layer_id)\\n  row           [N]        row index in the decode batch (0..K-1)\\n  req_pool_idx  [N]        forward_batch/indexer-metadata req_pool_indices[row] (physical request slot)\\n  uid           [N]        small integer per request id (forward_batch.rids), index into rid_table; -1 if unknown\\n  seq_len       [N]        sequence_lengths[row]: tokens INCLUDING the one being decoded (query position = seq_len-1)\\n  nblk          [N]        compressed_lengths[row] actually given to fast_topk (= seq_len // compress_ratio)\\n  bs            [N]        number of rows of the decode batch at that step\\n  meta          [7]        version, rank, topk, compress_ratio, rows_k, run_id, part\\n  rid_table     [U] str    request id per uid (grows over the run)\\n\\"\\"\\"\\nimport atexit\\nimport logging\\nimport os\\nimport re\\nimport time\\n\\nimport numpy as np\\n\\nlog = logging.getLogger(__name__)\\nTAG = \\"[qsadump patch]\\"\\nVERSION = 1\\nENV_DIR = \\"SGLANG_QSA_DUMP_DIR\\"\\nENV_ROWS = \\"SGLANG_QSA_DUMP_ROWS\\"\\nENV_FLUSH = \\"SGLANG_QSA_DUMP_FLUSH_STEPS\\"\\nENV_MAX_MB = \\"SGLANG_QSA_DUMP_MAX_MB\\"\\nENV_RANKS = \\"SGLANG_QSA_DUMP_RANKS\\"\\n\\n\\ndef _env_int(name, default, lo, hi):\\n    raw = os.environ.get(name, \\"\\").strip()\\n    if not raw:\\n        return default\\n    try:\\n        val = int(raw)\\n    except ValueError:\\n        val = None\\n    if val is None or not lo <= val <= hi:\\n        log.warning(\\"%s ignoring %s=%r (need an integer in %d..%d); using %d\\", TAG, name, raw, lo, hi, default)\\n        return default\\n    return val\\n\\n\\ndef _env_ranks():\\n    raw = os.environ.get(ENV_RANKS, \\"\\").strip()\\n    if not raw:\\n        return frozenset({0})\\n    try:\\n        ranks = frozenset(int(x) for x in raw.split(\\",\\") if x.strip())\\n        if ranks and min(ranks) >= 0:\\n            return ranks\\n    except ValueError:\\n        pass\\n    log.warning(\\"%s ignoring %s=%r (need e.g. 0 or 0,1); using 0\\", TAG, ENV_RANKS, raw)\\n    return frozenset({0})\\n\\n\\nDUMP_DIR = os.environ.get(ENV_DIR, \\"\\").strip()\\nQSADUMP_ON = bool(DUMP_DIR)\\nif QSADUMP_ON:\\n    ROWS_K = _env_int(ENV_ROWS, 4, 1, 256)\\n    FLUSH_STEPS = _env_int(ENV_FLUSH, 64, 1, 1000000)\\n    MAX_MB = _env_int(ENV_MAX_MB, 400, 1, 10000000)\\n    RANKS = _env_ranks()\\n    log.warning(\\n        \\"%s ARMED: %s=%s rows=%d flush=%d steps cap=%d MB ranks=%s (decode steps only; use --disable-decode-cuda-graph)\\",\\n        TAG, ENV_DIR, DUMP_DIR, ROWS_K, FLUSH_STEPS, MAX_MB, sorted(RANKS),\\n    )\\nelse:\\n    ROWS_K, FLUSH_STEPS, MAX_MB, RANKS = 4, 64, 400, frozenset({0})\\n\\n_COLS = (\\"step\\", \\"layer\\", \\"row\\", \\"req_pool_idx\\", \\"uid\\", \\"seq_len\\", \\"nblk\\", \\"bs\\")\\n\\n\\nclass QsaDumpBuffer:\\n    \\"\\"\\"Numpy-only buffering of (step, layer) records, whole-step parts, byte cap. No torch, no sglang.\\"\\"\\"\\n\\n    def __init__(self, out_dir, rank, topk, ratio=4, rows_k=4, flush_steps=64, max_bytes=400 << 20,\\n                 run_id=0, rid_table=None):\\n        self.out_dir = str(out_dir)\\n        self.rank = int(rank)\\n        self.topk = int(topk)\\n        self.ratio = int(ratio)\\n        self.rows_k = int(rows_k)\\n        self.flush_steps = int(flush_steps)\\n        self.max_bytes = int(max_bytes)\\n        self.run_id = int(run_id)\\n        self.rid_table = rid_table if rid_table is not None else []\\n        self.stopped = False\\n        self.bytes_buffered = 0\\n        self.bytes_total = 0\\n        self.records = 0\\n        self.parts_written = 0\\n        self._chunks = []\\n        self._buf_steps = 0\\n        self._last_step = None\\n        os.makedirs(self.out_dir, exist_ok=True)\\n        self.part = self._next_part()\\n\\n    def _next_part(self):\\n        pat = re.compile(r\\"^qsa_dump_%d_(\\\\d+)\\\\.npz$\\" % self.rank)\\n        parts = [int(m.group(1)) for m in (pat.match(f) for f in os.listdir(self.out_dir)) if m]\\n        return max(parts) + 1 if parts else 0\\n\\n    def add(self, step, layer, bs, blocks, seq_len, req_pool_idx, nblk, uid):\\n        \\"\\"\\"One (step, layer) record: blocks [k, topk] int32, the others [k] ints. False once stopped/capped.\\"\\"\\"\\n        if self.stopped:\\n            return False\\n        blocks = np.asarray(blocks, dtype=np.int32)\\n        k = blocks.shape[0]\\n        nbytes = blocks.nbytes + len(_COLS) * k * 4\\n        if self.bytes_total + self.bytes_buffered + nbytes > self.max_bytes:\\n            self.stopped = True\\n            self.flush()\\n            log.warning(\\n                \\"%s cap of %.0f MB reached after %d records / %d parts (last step %s); recording stopped\\",\\n                TAG, self.max_bytes / 2**20, self.records, self.parts_written, self._last_step,\\n            )\\n            return False\\n        if self._last_step is not None and step != self._last_step and self._buf_steps >= self.flush_steps:\\n            self.flush()\\n        if step != self._last_step:\\n            self._buf_steps += 1\\n            self._last_step = step\\n        cols = {\\n            \\"step\\": np.full(k, step, np.int32),\\n            \\"layer\\": np.full(k, layer, np.int32),\\n            \\"row\\": np.arange(k, dtype=np.int32),\\n            \\"req_pool_idx\\": np.asarray(req_pool_idx, dtype=np.int32),\\n            \\"uid\\": np.asarray(uid, dtype=np.int32),\\n            \\"seq_len\\": np.asarray(seq_len, dtype=np.int32),\\n            \\"nblk\\": np.asarray(nblk, dtype=np.int32),\\n            \\"bs\\": np.full(k, bs, np.int32),\\n        }\\n        self._chunks.append((blocks, cols))\\n        self.bytes_buffered += nbytes\\n        self.records += k\\n        return True\\n\\n    def flush(self):\\n        \\"\\"\\"Write the buffered whole steps as one part; returns the path or None. Never raises.\\"\\"\\"\\n        if not self._chunks:\\n            return None\\n        try:\\n            arrays = {\\"blocks\\": np.concatenate([c[0] for c in self._chunks], axis=0)}\\n            for name in _COLS:\\n                arrays[name] = np.concatenate([c[1][name] for c in self._chunks])\\n            arrays[\\"meta\\"] = np.array(\\n                [VERSION, self.rank, self.topk, self.ratio, self.rows_k, self.run_id, self.part], dtype=np.int32\\n            )\\n            arrays[\\"rid_table\\"] = np.array(list(self.rid_table), dtype=np.str_)\\n            final = os.path.join(self.out_dir, \\"qsa_dump_%d_%05d.npz\\" % (self.rank, self.part))\\n            tmp = os.path.join(self.out_dir, \\".qsa_dump_%d_%05d.tmp.npz\\" % (self.rank, self.part))\\n            np.savez(tmp, **arrays)\\n            os.replace(tmp, final)\\n        except Exception as exc:  # disk full, permissions, ...: stop recording, never disturb serving\\n            self.stopped = True\\n            self._chunks = []\\n            log.warning(\\"%s flush failed (%s: %s); recording stopped\\", TAG, type(exc).__name__, str(exc)[:200])\\n            return None\\n        self.bytes_total += self.bytes_buffered\\n        self.bytes_buffered = 0\\n        self._chunks = []\\n        self._buf_steps = 0\\n        self.parts_written += 1\\n        self.part += 1\\n        if self.parts_written == 1 or self.parts_written % 25 == 0:\\n            log.warning(\\n                \\"%s rank %d: %d parts, %d records, %.1f MB of %.0f MB cap (latest %s)\\",\\n                TAG, self.rank, self.parts_written, self.records, self.bytes_total / 2**20,\\n                self.max_bytes / 2**20, final,\\n            )\\n        return final\\n\\n\\nclass _State:\\n    def __init__(self):\\n        self.ready = False\\n        self.active = False\\n        self.dead = False\\n        self.buf = None\\n        self.rank = -1\\n        self.step = 0\\n        self.last_layer = -1\\n        self.uids = {}\\n        self.rid_table = []\\n        self.errors = 0\\n        self.checks_left = 16\\n        self.invalid = 0\\n        self.warned = set()\\n\\n\\n_ST = _State()\\n\\n\\ndef _warn_once(key, msg, *args):\\n    if key not in _ST.warned:\\n        _ST.warned.add(key)\\n        log.warning(TAG + \\" \\" + msg, *args)\\n\\n\\ndef _capturing(t):\\n    if not getattr(t, \\"is_cuda\\", False):\\n        return False\\n    try:\\n        import torch\\n\\n        return bool(torch.cuda.is_current_stream_capturing())\\n    except Exception:\\n        return True  # cannot tell -> do nothing\\n\\n\\n_CAP_FN = []\\n\\n\\ndef _sgl_capture_mode():\\n    if not _CAP_FN:\\n        try:\\n            from sglang.srt.model_executor.runner import get_is_capture_mode\\n\\n            _CAP_FN.append(get_is_capture_mode)\\n        except Exception:\\n            _CAP_FN.append(lambda: False)\\n    try:\\n        return bool(_CAP_FN[0]())\\n    except Exception:\\n        return False\\n\\n\\ndef _tp_rank():\\n    from sglang.srt.distributed import get_tp_group\\n\\n    return int(get_tp_group().rank_in_group)\\n\\n\\ndef _init(indexer):\\n    _ST.ready = True\\n    try:\\n        rank = _tp_rank()\\n    except Exception as exc:\\n        _ST.dead = True\\n        log.warning(\\"%s cannot determine the TP rank (%s: %s); dump disabled\\", TAG, type(exc).__name__, exc)\\n        return\\n    _ST.rank = rank\\n    if rank not in RANKS:\\n        log.warning(\\"%s rank %d: not in %s=%s, no-op\\", TAG, rank, ENV_RANKS, sorted(RANKS))\\n        return\\n    try:\\n        _ST.buf = QsaDumpBuffer(\\n            DUMP_DIR, rank, int(indexer.block_topk), int(indexer.compress_ratio), ROWS_K, FLUSH_STEPS,\\n            MAX_MB << 20, run_id=int(time.time()) - 1700000000, rid_table=_ST.rid_table,\\n        )\\n    except Exception as exc:\\n        _ST.dead = True\\n        log.warning(\\"%s rank %d: cannot create %s (%s: %s); dump disabled\\", TAG, rank, DUMP_DIR,\\n                    type(exc).__name__, exc)\\n        return\\n    atexit.register(_ST.buf.flush)\\n    _ST.active = True\\n    log.warning(\\"%s rank %d: recording decode block top-k (topk %d, ratio %d) into %s\\", TAG, rank,\\n                _ST.buf.topk, _ST.buf.ratio, DUMP_DIR)\\n\\n\\ndef _record(indexer, topk_indices, forward_batch, md):\\n    stash = getattr(indexer, \\"_qsadump_stash\\", None)\\n    if stash is not None:\\n        indexer._qsadump_stash = None  # consume: a stale stash must never be paired with a later call\\n    if not forward_batch.forward_mode.is_decode() or getattr(forward_batch, \\"spec_info\\", None) is not None:\\n        return\\n    if getattr(md, \\"is_cuda_graph\\", False) or _sgl_capture_mode():\\n        return\\n    if stash is None:\\n        _warn_once(\\"nostash\\", \\"no parked fast_topk output (is the qsa_indexer.py part of the patch missing?)\\")\\n        return\\n    blocks, nblk_t = stash\\n    if _capturing(blocks):\\n        return\\n    if not _ST.ready:\\n        _init(indexer)\\n    if _ST.dead or not _ST.active or _ST.buf.stopped:\\n        return\\n    import torch\\n\\n    buf = _ST.buf\\n    topk = buf.topk\\n    if blocks.dim() != 2 or blocks.shape[1] != topk or topk_indices.shape[0] != blocks.shape[0]:\\n        _warn_once(\\"shape\\", \\"unexpected shapes blocks=%s topk_indices=%s; skipping\\", tuple(blocks.shape),\\n                   tuple(topk_indices.shape))\\n        return\\n    bs = int(blocks.shape[0])\\n    k = min(buf.rows_k, bs)\\n    seq = md.sequence_lengths\\n    rpi = md.req_pool_indices if md.req_pool_indices is not None else forward_batch.req_pool_indices\\n    if k <= 0 or seq is None or rpi is None or seq.shape[0] < k or rpi.shape[0] < k or nblk_t.shape[0] < k:\\n        _warn_once(\\"meta\\", \\"missing per-row metadata (seq=%s rpi=%s); skipping\\", seq is None, rpi is None)\\n        return\\n    layer = int(indexer.layer_id)\\n    if layer <= _ST.last_layer:  # layers run in ascending order inside a forward: a wrap = a new decode step\\n        _ST.step += 1\\n    _ST.last_layer = layer\\n    packed = torch.cat(\\n        [\\n            blocks[:k].to(torch.int32),\\n            seq[:k].to(torch.int32).unsqueeze(1),\\n            rpi[:k].to(torch.int32).unsqueeze(1),\\n            nblk_t[:k].to(torch.int32).unsqueeze(1),\\n        ],\\n        dim=1,\\n    ).cpu().numpy()  # ONE host copy (a sync) of [k, topk + 3] int32\\n    blk = np.ascontiguousarray(packed[:, :topk])\\n    seq_np, rpi_np, nblk_np = packed[:, topk].copy(), packed[:, topk + 1].copy(), packed[:, topk + 2].copy()\\n    uid = np.full(k, -1, np.int32)\\n    rids = getattr(forward_batch, \\"rids\\", None)\\n    if rids is not None:\\n        for i in range(min(k, len(rids))):\\n            rid = str(rids[i])\\n            u = _ST.uids.get(rid)\\n            if u is None:\\n                u = len(_ST.rid_table)\\n                _ST.uids[rid] = u\\n                _ST.rid_table.append(rid)\\n            uid[i] = u\\n    # integrity of what the dump claims to be: LOGICAL block numbers 0 <= b < nblk, min(topk, nblk) valid, nblk = seq // r\\n    valid = blk >= 0\\n    bad = (\\n        (valid.sum(1) != np.minimum(topk, nblk_np))\\n        | (blk >= nblk_np[:, None]).any(1)\\n        | (nblk_np != seq_np // buf.ratio)\\n    )\\n    if bad.any():\\n        _ST.invalid += int(bad.sum())\\n        if _ST.invalid <= 5:\\n            i = int(np.argmax(bad))\\n            log.warning(\\n                \\"%s layer %d step %d row %d: UNEXPECTED top-k (valid=%d nblk=%d seq_len=%d max=%d); %d bad so far\\",\\n                TAG, layer, _ST.step, i, int(valid[i].sum()), int(nblk_np[i]), int(seq_np[i]), int(blk[i].max()),\\n                _ST.invalid,\\n            )\\n    if _ST.checks_left > 0:  # first records only: the parked block ids must be the ones expand_* turned into tokens\\n        _ST.checks_left -= 1\\n        exp = topk_indices[:k].cpu().numpy()\\n        same = np.array_equal(\\n            np.where(valid, exp[:, : topk * buf.ratio : buf.ratio], -1), np.where(valid, blk * buf.ratio, -1)\\n        )\\n        dup = sum(len(set(r[r >= 0].tolist())) != int((r >= 0).sum()) for r in blk)\\n        log.warning(\\n            \\"%s selfcheck layer %d step %d: parked blocks == expanded token ids/ratio: %s; rows with duplicate ids: %d; \\"\\n            \\"seq_len(row0)=%d nblk(row0)=%d bs=%d\\",\\n            TAG, layer, _ST.step, same, dup, int(seq_np[0]), int(nblk_np[0]), bs,\\n        )\\n    buf.add(_ST.step, layer, bs, blk, seq_np, rpi_np, nblk_np, uid)\\n\\n\\ndef record_decode_topk(indexer, topk_indices, forward_batch, indexer_metadata):\\n    \\"\\"\\"Hook called from Qwen4ExpAttentionDecoderLayer._compute_qsa_topk_indices (only if QSADUMP_ON). Never raises.\\"\\"\\"\\n    if _ST.dead:\\n        return\\n    try:\\n        _record(indexer, topk_indices, forward_batch, indexer_metadata)\\n    except Exception as exc:\\n        _ST.errors += 1\\n        log.warning(\\"%s hook error %d/3 (%s: %s)\\", TAG, _ST.errors, type(exc).__name__, str(exc)[:300])\\n        if _ST.errors >= 3:\\n            _ST.dead = True\\n            log.warning(\\"%s disabled after repeated errors\\", TAG)\\n\'\'\'\\n\\n# (anchor, replacement); every anchor must occur exactly once in the STOCK file.\\nINDEXER_EDITS = [\\n    (\\"from sglang.srt.layers.attention.qsa.mqa import qsa_mqa_decode, qsa_mqa_prefill\\\\n\\",\\n     \\"from sglang.srt.layers.attention.qsa.mqa import qsa_mqa_decode, qsa_mqa_prefill\\\\n\\"\\n     f\\"from sglang.srt.layers.attention.qsa.qsa_dump import QSADUMP_ON as _QSADUMP_ON  {MARK}\\\\n\\"),\\n    (\\"        return expand_qsa_block_indices(\\\\n\\"\\n     \\"            block_indices,\\\\n\\"\\n     \\"            query_positions,\\\\n\\"\\n     \\"            sequence_lengths,\\\\n\\",\\n     f\\"        if _QSADUMP_ON:  {MARK} SGLANG_QSA_DUMP_DIR: park the raw fast_topk output for the layer hook\\\\n\\"\\n     \\"            self._qsadump_stash = (block_indices, compressed_lengths)\\\\n\\"\\n     \\"        return expand_qsa_block_indices(\\\\n\\"\\n     \\"            block_indices,\\\\n\\"\\n     \\"            query_positions,\\\\n\\"\\n     \\"            sequence_lengths,\\\\n\\"),\\n]\\nMODEL_EDITS = [\\n    (\\"from sglang.srt.utils import logger\\\\n\\",\\n     \\"from sglang.srt.utils import logger\\\\n\\"\\n     f\\"from sglang.srt.layers.attention.qsa.qsa_dump import QSADUMP_ON as _QSADUMP_ON  {MARK}\\\\n\\"\\n     f\\"from sglang.srt.layers.attention.qsa.qsa_dump import record_decode_topk as _qsadump_record  {MARK}\\\\n\\"),\\n    (\\"        return topk_indices\\\\n\\\\n    def self_attention(\\\\n\\",\\n     f\\"        if _QSADUMP_ON:  {MARK} SGLANG_QSA_DUMP_DIR: dump decode block top-k (no-op otherwise)\\\\n\\"\\n     \\"            _qsadump_record(self.indexer, topk_indices, forward_batch, indexer_metadata)\\\\n\\"\\n     \\"        return topk_indices\\\\n\\\\n    def self_attention(\\\\n\\"),\\n]\\nFILES = {INDEXER_REL: INDEXER_EDITS, MODEL_REL: MODEL_EDITS}\\n\\n\\n# ----------------------------------------------------------------------------------------------- selftest\\ndef _load_module(env):\\n    \\"\\"\\"Exec MODULE_SRC as a fresh module under the given env (local tests; no sglang needed).\\"\\"\\"\\n    import os\\n    import types\\n\\n    old = {k: os.environ.get(k) for k in env}\\n    for k, v in env.items():\\n        if v is None:\\n            os.environ.pop(k, None)\\n        else:\\n            os.environ[k] = v\\n    try:\\n        mod = types.ModuleType(\\"qsa_dump_under_test\\")\\n        exec(compile(MODULE_SRC, \\"qsa_dump.py\\", \\"exec\\"), mod.__dict__)\\n    finally:\\n        for k, v in old.items():\\n            if v is None:\\n                os.environ.pop(k, None)\\n            else:\\n                os.environ[k] = v\\n    return mod\\n\\n\\ndef selftest():\\n    import os\\n    import tempfile\\n\\n    import numpy as np\\n\\n    tmp = tempfile.mkdtemp(prefix=\\"qsadump_selftest_\\")\\n    m = _load_module({\\"SGLANG_QSA_DUMP_DIR\\": None})\\n    assert m.QSADUMP_ON is False, \\"dormant when the env is unset\\"\\n    m = _load_module({\\"SGLANG_QSA_DUMP_DIR\\": tmp, \\"SGLANG_QSA_DUMP_ROWS\\": \\"x\\", \\"SGLANG_QSA_DUMP_MAX_MB\\": \\"-3\\"})\\n    assert m.QSADUMP_ON and m.ROWS_K == 4 and m.MAX_MB == 400 and m.RANKS == frozenset({0}), \\"bad env -> defaults\\"\\n\\n    # 1) buffer: whole-step parts, content round trip, -1 padding kept, part numbering resumes, cap stops once\\n    rng = np.random.default_rng(0)\\n    d1 = tempfile.mkdtemp(prefix=\\"qsadump_buf_\\")\\n    rid = [\\"r0\\", \\"r1\\"]\\n    b = m.QsaDumpBuffer(d1, rank=0, topk=8, ratio=4, rows_k=3, flush_steps=3, max_bytes=1 << 30, run_id=7, rid_table=rid)\\n    sent = []\\n    for step in range(7):\\n        for layer in (3, 7):\\n            blk = rng.integers(0, 50, size=(3, 8)).astype(np.int32)\\n            blk[1, 5:] = -1\\n            sent.append((step, layer, blk.copy()))\\n            assert b.add(step, layer, 3, blk, [100 + step] * 3, [5, 6, 7], [25] * 3, [0, 1, -1])\\n    b.flush()\\n    files = sorted(f for f in os.listdir(d1) if f.endswith(\\".npz\\"))\\n    assert files == [\\"qsa_dump_0_00000.npz\\", \\"qsa_dump_0_00001.npz\\", \\"qsa_dump_0_00002.npz\\"], files\\n    z0 = np.load(os.path.join(d1, files[0]))\\n    assert z0[\\"step\\"].tolist() == sorted(z0[\\"step\\"].tolist()) and set(z0[\\"step\\"].tolist()) == {0, 1, 2}, \\"whole steps\\"\\n    assert z0[\\"blocks\\"].shape == (3 * 2 * 3, 8) and z0[\\"blocks\\"].dtype == np.int32\\n    assert all(z0[n].dtype == np.int32 for n in (\\"step\\", \\"layer\\", \\"row\\", \\"req_pool_idx\\", \\"uid\\", \\"seq_len\\", \\"nblk\\", \\"bs\\", \\"meta\\"))\\n    assert z0[\\"meta\\"].tolist() == [1, 0, 8, 4, 3, 7, 0] and z0[\\"rid_table\\"].tolist() == [\\"r0\\", \\"r1\\"]\\n    allblk = np.concatenate([np.load(os.path.join(d1, f))[\\"blocks\\"] for f in files])\\n    assert np.array_equal(allblk, np.concatenate([s[2] for s in sent])), \\"round trip incl. -1 padding\\"\\n    assert not any(f.startswith(\\".\\") for f in os.listdir(d1)), \\"no temp file left\\"\\n    b2 = m.QsaDumpBuffer(d1, rank=0, topk=8)\\n    assert b2.part == 3, \\"part numbering resumes after existing files\\"\\n    assert m.QsaDumpBuffer(d1, rank=1, topk=8).part == 0\\n    d2 = tempfile.mkdtemp(prefix=\\"qsadump_cap_\\")\\n    small = m.QsaDumpBuffer(d2, 0, 8, 4, 3, flush_steps=2, max_bytes=3 * (3 * 8 * 4 + 8 * 3 * 4))\\n    oks = [small.add(s, 3, 3, np.zeros((3, 8), np.int32), [1] * 3, [1] * 3, [1] * 3, [0] * 3) for s in range(6)]\\n    assert oks == [True, True, True, False, False, False] and small.stopped, oks\\n    assert len(os.listdir(d2)) >= 1 and sum(np.load(os.path.join(d2, f))[\\"blocks\\"].shape[0] for f in os.listdir(d2)) == 9\\n\\n    # 2) hook with real CPU torch tensors and stand-in sglang objects\\n    try:\\n        import torch\\n    except ImportError:\\n        print(\\"qsadump_patch selftest: buffer tests passed; torch not installed, hook tests skipped\\")\\n        return\\n\\n    class Mode:\\n        def __init__(self, decode):\\n            self.decode = decode\\n\\n        def is_decode(self):\\n            return self.decode\\n\\n    class NS:\\n        def __init__(self, **kw):\\n            self.__dict__.update(kw)\\n\\n    d3 = tempfile.mkdtemp(prefix=\\"qsadump_hook_\\")\\n    m = _load_module({\\"SGLANG_QSA_DUMP_DIR\\": d3, \\"SGLANG_QSA_DUMP_ROWS\\": \\"2\\", \\"SGLANG_QSA_DUMP_FLUSH_STEPS\\": \\"2\\"})\\n    m._tp_rank = lambda: 0\\n    layers = (3, 7, 11)\\n    idx = {l: NS(layer_id=l, block_topk=512, compress_ratio=4, _qsadump_stash=None) for l in layers}\\n    bs, seq = 3, torch.tensor([9000, 8001, 20000], dtype=torch.int32)\\n    nblk = seq // 4\\n    truth = []\\n\\n    def one_forward(mode_decode=True, spec=None, graph=False, parked=True):\\n        for l in layers:\\n            blocks = torch.full((bs, 512), -1, dtype=torch.int32)\\n            for r in range(bs):\\n                n = min(512, int(nblk[r]))\\n                blocks[r, :n] = torch.randperm(int(nblk[r]))[:n].to(torch.int32)\\n            tokens = torch.full((bs, 2051), -1, dtype=torch.int32)\\n            v = blocks >= 0\\n            tokens[:, 0:2048:4] = torch.where(v, blocks * 4, torch.tensor(-1, dtype=torch.int32))\\n            fb = NS(forward_mode=Mode(mode_decode), spec_info=spec, rids=[\\"a\\", \\"b\\", \\"c\\"],\\n                    req_pool_indices=torch.tensor([11, 12, 13]))\\n            md = NS(sequence_lengths=seq, req_pool_indices=torch.tensor([11, 12, 13], dtype=torch.int64),\\n                    is_cuda_graph=graph)\\n            if parked:\\n                idx[l]._qsadump_stash = (blocks, nblk)\\n            m.record_decode_topk(idx[l], tokens, fb, md)\\n            if mode_decode and spec is None and not graph and parked:\\n                truth.append((l, blocks[:2].numpy().copy()))\\n\\n    one_forward(mode_decode=False)  # extend/prefill: nothing\\n    one_forward(spec=object())  # draft / verify style batch: nothing\\n    one_forward(graph=True)  # cuda-graph metadata: nothing\\n    assert m._ST.buf is None and m._ST.step == 0, \\"no-op paths must not even initialise\\"\\n    for _ in range(5):\\n        seq = seq + 1\\n        nblk = seq // 4\\n        one_forward()\\n    m._ST.buf.flush()\\n    parts = sorted(f for f in os.listdir(d3) if f.endswith(\\".npz\\"))\\n    assert parts == [\\"qsa_dump_0_00000.npz\\", \\"qsa_dump_0_00001.npz\\", \\"qsa_dump_0_00002.npz\\"], parts\\n    z = {n: np.concatenate([np.load(os.path.join(d3, f))[n] for f in parts]) for n in (\\"blocks\\", \\"step\\", \\"layer\\", \\"row\\", \\"req_pool_idx\\", \\"uid\\", \\"seq_len\\", \\"nblk\\", \\"bs\\")}\\n    assert z[\\"blocks\\"].shape == (5 * 3 * 2, 512) and z[\\"bs\\"].tolist() == [3] * 30\\n    assert z[\\"step\\"].tolist() == [s for s in range(5) for _ in range(6)], \\"one step id per forward\\"\\n    assert z[\\"layer\\"].tolist() == [l for _ in range(5) for l in layers for _ in range(2)]\\n    assert z[\\"row\\"].tolist() == [0, 1] * 15 and z[\\"req_pool_idx\\"].tolist() == [11, 12] * 15\\n    assert z[\\"uid\\"].tolist() == [0, 1] * 15 and z[\\"seq_len\\"][:2].tolist() == [9001, 8002]\\n    assert np.array_equal(z[\\"blocks\\"], np.concatenate([t[1] for t in truth]))\\n    assert m._ST.invalid == 0 and not m._ST.dead\\n\\n    # a wrong parked tensor (inconsistent with the lengths) is counted, never raised\\n    idx[3]._qsadump_stash = (torch.full((bs, 512), 10**6, dtype=torch.int32), nblk)\\n    m.record_decode_topk(idx[3], torch.zeros((bs, 2051), dtype=torch.int32),\\n                         NS(forward_mode=Mode(True), spec_info=None, rids=None, req_pool_indices=None),\\n                         NS(sequence_lengths=seq, req_pool_indices=torch.tensor([11, 12, 13]), is_cuda_graph=False))\\n    assert m._ST.invalid == 2 and not m._ST.dead, m._ST.invalid\\n\\n    # rank filter: other ranks never create a buffer or a file\\n    d4 = tempfile.mkdtemp(prefix=\\"qsadump_rank_\\")\\n    m = _load_module({\\"SGLANG_QSA_DUMP_DIR\\": d4})\\n    m._tp_rank = lambda: 2\\n    one_forward_args = (idx[3], torch.zeros((bs, 2051), dtype=torch.int32),\\n                        NS(forward_mode=Mode(True), spec_info=None, rids=None, req_pool_indices=torch.tensor([1, 2, 3])),\\n                        NS(sequence_lengths=seq, req_pool_indices=torch.tensor([1, 2, 3]), is_cuda_graph=False))\\n    idx[3]._qsadump_stash = (torch.zeros((bs, 512), dtype=torch.int32), nblk)\\n    m.record_decode_topk(*one_forward_args)\\n    assert m._ST.buf is None and os.listdir(d4) == []\\n    # errors inside the hook are swallowed and the hook turns itself off after 3\\n    m = _load_module({\\"SGLANG_QSA_DUMP_DIR\\": d4})\\n    m._tp_rank = lambda: 0\\n    for _ in range(4):\\n        idx[3]._qsadump_stash = (None, None)\\n        m.record_decode_topk(*one_forward_args)\\n    assert m._ST.dead and m._ST.errors == 3\\n    print(\\"qsadump_patch selftest: PASS (buffer, whole-step parts, cap, resume, hook gating, rank filter, error isolation)\\")\\n\\n\\n# ----------------------------------------------------------------------------------------------- patcher\\ndef _state(site):\\n    \\"\\"\\"(module_ok, {rel: (n_edits_applied, text)}) for the files that exist.\\"\\"\\"\\n    mod = site / MODULE_REL\\n    module_ok = mod.is_file() and mod.read_text() == MODULE_SRC\\n    texts = {}\\n    for rel, edits in FILES.items():\\n        s = (site / rel).read_text()\\n        texts[rel] = (sum(new in s for _old, new in edits), s)\\n    return module_ok, texts\\n\\n\\ndef main():\\n    args = [a for a in sys.argv[1:] if not a.startswith(\\"--\\")]\\n    if \\"--selftest\\" in sys.argv:\\n        selftest()\\n        return\\n    if not args:\\n        print(__doc__)\\n        sys.exit(2)\\n    site = Path(args[0])\\n    check = \\"--check\\" in sys.argv\\n    for rel in FILES:\\n        if not (site / rel).is_file():\\n            sys.exit(f\\"qsadump_patch: {site / rel} not found; refusing to patch\\")\\n    module_ok, texts = _state(site)\\n    applied = {rel: n for rel, (n, _s) in texts.items()}\\n    full = all(applied[rel] == len(FILES[rel]) for rel in FILES)\\n    none = all(applied[rel] == 0 for rel in FILES)\\n    if full and module_ok:\\n        print(\\"qsadump_patch: already patched\\")\\n        return\\n    if check:\\n        sys.exit(f\\"qsadump_patch: NOT patched (module current {module_ok}, edits applied {applied})\\")\\n    if not full and not none:\\n        sys.exit(f\\"qsadump_patch: PARTIALLY patched (edits applied {applied}); restore the stock files first\\")\\n    new_texts = {}\\n    if none:\\n        for rel, edits in FILES.items():\\n            s = texts[rel][1]\\n            for old, _new in edits:\\n                if s.count(old) != 1:\\n                    sys.exit(f\\"qsadump_patch: anchor x{s.count(old)} (need 1) in {rel}: {old.strip()[:70]!r}; refusing to patch\\")\\n            spots = sorted(((s.index(old), old, new) for old, new in edits), reverse=True)\\n            for pos, old, new in spots:  # by position on the stock text, so inserted text is never re-matched\\n                s = s[:pos] + new + s[pos + len(old):]\\n            try:\\n                compile(s, rel, \\"exec\\")\\n            except SyntaxError as exc:\\n                sys.exit(f\\"qsadump_patch: patched {rel} would not compile ({exc}); refusing to patch\\")\\n            new_texts[rel] = s\\n    try:\\n        compile(MODULE_SRC, MODULE_REL, \\"exec\\")\\n    except SyntaxError as exc:\\n        sys.exit(f\\"qsadump_patch: embedded module does not compile ({exc})\\")\\n    # module first, so a patched call site never references a missing module\\n    (site / MODULE_REL).write_text(MODULE_SRC)\\n    for rel, s in new_texts.items():\\n        (site / rel).write_text(s)\\n    print(\\"qsadump_patch: patched\\", \\", \\".join(str(site / r) for r in [MODULE_REL, *new_texts]))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "qsasplitk_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently add a split-K mode to the duck2 sm_89 QSA sparse decode (SGLANG_QSA_SM89_DECODE=triton_splitk).\\n\\nThe duck2 Triton kernel (duck2/patches/qsa_sm89_decode.py, installed by duck2/patches/apply_patches.py as\\nsglang/srt/layers/attention/qsa/sm89_decode.py) runs one program per (query row, KV head): grid (rows, 1) at TP4,\\nso a decode step with 13-20 rows uses 13-20 of the L4\'s 58 SMs and each program walks the 2051 top-k entries in 65\\ndependent BLOCK_N=32 iterations. This patch adds a drop-in module (embedded below as MODULE_SRC) with a grid\\n(rows, kv_heads, NSPLIT) partial-softmax kernel plus a merge kernel, and routes to it from the call site that\\napply_patches.py inserted into QwenSparseAttnBackend._forward_paged_attention. sm89_decode.py itself is NOT\\nmodified, so re-running apply_patches.py (which re-copies that file) does not undo this patch.\\n\\nDormant by default: the inserted call site hands the duck2 choice to splitk_route(), which returns it unchanged\\n(same objects) unless SGLANG_QSA_SM89_DECODE=triton_splitk. Tuning env (only read with triton_splitk):\\nSGLANG_QSA_SPLITK_CTAS, _NSPLIT, _BLOCK_N, _WARPS, _STAGES (see MODULE_SRC docstring / qsasplitk_NOTES.md).\\n\\nOrder: run AFTER duck2/patches/apply_patches.py (refuses otherwise).\\nusage: qsasplitk_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [qsasplitk patch]\\"\\nDUCK2_MARK = \\"# [duck2 sm89 patch]\\"\\nBACKEND_REL = \\"sglang/srt/layers/attention/qwen_sparse_attn_backend.py\\"\\nBASE_REL = \\"sglang/srt/layers/attention/qsa/sm89_decode.py\\"\\nMODULE_REL = \\"sglang/srt/layers/attention/qsa/sm89_decode_splitk.py\\"\\n\\n# Text that duck2/patches/apply_patches.py (QSA_NEW) inserted into the backend; must occur exactly once.\\nANCHOR = (\\n    \\"        sm89_impl = select_sm89_decode_impl(q.device)\\\\n\\"\\n    \\"        if sm89_impl is not None:\\\\n\\"\\n    \\"            return qsa_sparse_decode(\\\\n\\"\\n    \\"                sm89_impl,\\\\n\\"\\n)\\nNEW = (\\n    \\"        sm89_impl = select_sm89_decode_impl(q.device)\\\\n\\"\\n    f\\"        {MARK} SGLANG_QSA_SM89_DECODE=triton_splitk -> split-K kernel in\\\\n\\"\\n    \\"        # qsa/sm89_decode_splitk.py; any other value returns (sm89_impl, qsa_sparse_decode) unchanged.\\\\n\\"\\n    \\"        from sglang.srt.layers.attention.qsa.sm89_decode_splitk import splitk_route\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"        sm89_impl, qsa_sparse_decode = splitk_route(sm89_impl, qsa_sparse_decode)\\\\n\\"\\n    \\"        if sm89_impl is not None:\\\\n\\"\\n    \\"            return qsa_sparse_decode(\\\\n\\"\\n    \\"                sm89_impl,\\\\n\\"\\n)\\n\\nMODULE_SRC = r\'\'\'\\"\\"\\"Split-K variant of the duck2 sm_89 QSA sparse decode kernel (SGLANG_QSA_SM89_DECODE=triton_splitk).\\n\\nInstalled by colab/perf_patches/qsasplitk_patch.py as\\nsglang/srt/layers/attention/qsa/sm89_decode_splitk.py, next to the duck2\\nsm89_decode.py, which stays untouched. The patched call site in\\nQwenSparseAttnBackend._forward_paged_attention passes the duck2 choice through\\nsplitk_route(), which returns it unchanged unless the env value is triton_splitk.\\n\\nWhy: the duck2 kernel runs one program per (query row, KV head). At TP4 there is\\none KV head per rank, so B decode rows launch B CTAs on the L4\'s 58 SMs and each\\nCTA walks all 2051 top-k entries in 65 dependent BLOCK_N=32 iterations (index ->\\nreq_to_token -> K/V rows). With B <= 20 most SMs idle and the call is\\nlatency-bound.\\n\\nHere the top-k axis is cut into NSPLIT contiguous chunks of whole BLOCK_N blocks:\\ngrid (rows, kv_heads, NSPLIT). Each program runs the duck2 loop body over its\\nchunk and writes a partial softmax state per query head (m = max in log2 units,\\nl = sum of exp2, acc = unnormalised P@V; fp32). A merge kernel, grid\\n(rows, q_heads), combines them: out = sum_s 2^(m_s-M) acc_s / sum_s 2^(m_s-M) l_s\\nwith M = max_s m_s. Per-entry math (scale*log2e folded into q, bf16 MMA, exp2,\\nfp32 accumulation, unscaled fp8 upcast) is the duck2 kernel\'s; only the grouping\\nof the fp32 sums changes. Rows without any valid entry give exactly 0. When\\nNSPLIT is 1 (many rows: MTP verify, large batches) the first kernel normalises\\nand stores directly (the duck2 computation) and no merge kernel is launched.\\n\\nNSPLIT depends only on static shapes (rows, kv heads, topk), so the grid is fixed\\nper captured CUDA graph: no host sync, no data-dependent launch. Default:\\nNSPLIT = floor(target_ctas / (rows * kv_heads)), target_ctas = 2 x SM count,\\nclamped to [1, min(64, n_blocks)], then rebalanced so that no split is empty.\\n\\nEnv (read once per process, at the first call):\\n  SGLANG_QSA_SM89_DECODE=triton_splitk  enable (any other value: duck2 choice)\\n  SGLANG_QSA_SPLITK_CTAS=N     target CTAs per call (default 2 x SMs; 1 = never split)\\n  SGLANG_QSA_SPLITK_NSPLIT=N   fixed split count for every shape (default 0 = from CTAS)\\n  SGLANG_QSA_SPLITK_BLOCK_N=N  top-k entries per iteration, power of 2 in [16, 128] (default 32)\\n  SGLANG_QSA_SPLITK_WARPS=N    num_warps of the split kernel, power of 2 in [1, 16] (default 4)\\n  SGLANG_QSA_SPLITK_STAGES=N   num_stages of the split kernel, 1..6 (default 2)\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport os\\n\\nimport torch\\n\\ntry:  # optional so the module imports on CPU-only boxes\\n    import triton\\n    import triton.language as tl\\n\\n    HAS_TRITON = True\\nexcept Exception:  # pragma: no cover\\n    triton = None\\n    tl = None\\n    HAS_TRITON = False\\n\\nlogger = logging.getLogger(__name__)\\n\\nENV_MODE = \\"SGLANG_QSA_SM89_DECODE\\"\\nMODE = \\"triton_splitk\\"\\nMAX_SPLIT = 64\\n_MERGE_CHUNK = 8\\n\\nif HAS_TRITON:\\n\\n    @triton.jit\\n    def _qsa_splitk_partial_kernel(\\n        Q,\\n        K,\\n        V,\\n        Out,\\n        MidO,\\n        MidM,\\n        MidL,\\n        ReqToToken,\\n        ReqIdx,\\n        Indices,\\n        SeqLens,\\n        scale,\\n        topk,\\n        cols_per_split,\\n        stride_qm,\\n        stride_qh,\\n        stride_kn,\\n        stride_kh,\\n        stride_vn,\\n        stride_vh,\\n        stride_om,\\n        stride_oh,\\n        stride_req,\\n        stride_idx,\\n        stride_mo_m,\\n        stride_mo_h,\\n        stride_mo_s,\\n        stride_ml_m,\\n        stride_ml_h,\\n        GROUP: tl.constexpr,\\n        BLOCK_H: tl.constexpr,\\n        BLOCK_N: tl.constexpr,\\n        HEAD_DIM: tl.constexpr,\\n        FINAL: tl.constexpr,\\n        DOT_FP16: tl.constexpr,\\n    ):\\n        dot_dt = tl.float16 if DOT_FP16 else tl.bfloat16\\n        row = tl.program_id(0)\\n        kvh = tl.program_id(1)\\n        split = tl.program_id(2)\\n        length = tl.load(SeqLens + row).to(tl.int64)\\n        req = tl.load(ReqIdx + row).to(tl.int64)\\n        offs_h = tl.arange(0, BLOCK_H)\\n        offs_d = tl.arange(0, HEAD_DIM)\\n        hmask = offs_h < GROUP\\n        qh = kvh * GROUP + offs_h\\n        q = tl.load(\\n            Q + row * stride_qm + qh[:, None] * stride_qh + offs_d[None, :],\\n            mask=hmask[:, None],\\n            other=0.0,\\n        )\\n        q = (q.to(tl.float32) * (scale * 1.4426950408889634)).to(dot_dt)\\n        m_i = tl.full([BLOCK_H], float(\\"-inf\\"), tl.float32)\\n        l_i = tl.zeros([BLOCK_H], tl.float32)\\n        acc = tl.zeros([BLOCK_H, HEAD_DIM], tl.float32)\\n        col_lo = split * cols_per_split\\n        col_hi = tl.minimum(col_lo + cols_per_split, topk)\\n        for start in range(col_lo, col_hi, BLOCK_N):\\n            cols = start + tl.arange(0, BLOCK_N)\\n            pos = tl.load(\\n                Indices + row * stride_idx + cols, mask=cols < topk, other=-1\\n            ).to(tl.int64)\\n            valid = (pos >= 0) & (pos < length)\\n            slot = tl.load(\\n                ReqToToken + req * stride_req + tl.where(valid, pos, 0),\\n                mask=valid,\\n                other=0,\\n            ).to(tl.int64)\\n            k = tl.load(\\n                K + slot[:, None] * stride_kn + kvh * stride_kh + offs_d[None, :],\\n                mask=valid[:, None],\\n                other=0.0,\\n            ).to(dot_dt)\\n            s = tl.dot(q, tl.trans(k))\\n            s = tl.where(valid[None, :], s, float(\\"-inf\\"))\\n            m_new = tl.maximum(m_i, tl.max(s, 1))\\n            m_safe = tl.where(m_new == float(\\"-inf\\"), 0.0, m_new)\\n            alpha = tl.math.exp2(m_i - m_safe)\\n            p = tl.math.exp2(s - m_safe[:, None])\\n            v = tl.load(\\n                V + slot[:, None] * stride_vn + kvh * stride_vh + offs_d[None, :],\\n                mask=valid[:, None],\\n                other=0.0,\\n            ).to(dot_dt)\\n            acc = acc * alpha[:, None] + tl.dot(p.to(dot_dt), v)\\n            l_i = l_i * alpha + tl.sum(p, 1)\\n            m_i = m_new\\n        if FINAL:\\n            out = acc / tl.where(l_i > 0.0, l_i, 1.0)[:, None]\\n            tl.store(\\n                Out + row * stride_om + qh[:, None] * stride_oh + offs_d[None, :],\\n                out.to(Out.dtype.element_ty),\\n                mask=hmask[:, None],\\n            )\\n        else:\\n            tl.store(\\n                MidO\\n                + row * stride_mo_m\\n                + qh[:, None] * stride_mo_h\\n                + split * stride_mo_s\\n                + offs_d[None, :],\\n                acc,\\n                mask=hmask[:, None],\\n            )\\n            ml = row * stride_ml_m + qh * stride_ml_h + split\\n            tl.store(MidM + ml, m_i, mask=hmask)\\n            tl.store(MidL + ml, l_i, mask=hmask)\\n\\n    @triton.jit\\n    def _qsa_splitk_merge_kernel(\\n        MidO,\\n        MidM,\\n        MidL,\\n        Out,\\n        nsplit,\\n        stride_mo_m,\\n        stride_mo_h,\\n        stride_mo_s,\\n        stride_ml_m,\\n        stride_ml_h,\\n        stride_om,\\n        stride_oh,\\n        SPLIT_P2: tl.constexpr,\\n        CHUNK: tl.constexpr,\\n        HEAD_DIM: tl.constexpr,\\n    ):\\n        row = tl.program_id(0)\\n        h = tl.program_id(1)\\n        offs_d = tl.arange(0, HEAD_DIM)\\n        base_ml = row * stride_ml_m + h * stride_ml_h\\n        base_o = row * stride_mo_m + h * stride_mo_h\\n        offs_s = tl.arange(0, SPLIT_P2)\\n        smask = offs_s < nsplit\\n        m_all = tl.load(MidM + base_ml + offs_s, mask=smask, other=float(\\"-inf\\"))\\n        l_all = tl.load(MidL + base_ml + offs_s, mask=smask, other=0.0)\\n        m_max = tl.max(m_all, 0)\\n        m_safe = tl.where(m_max == float(\\"-inf\\"), 0.0, m_max)\\n        denom = tl.sum(tl.math.exp2(m_all - m_safe) * l_all, 0)\\n        acc = tl.zeros([HEAD_DIM], tl.float32)\\n        for s0 in range(0, nsplit, CHUNK):\\n            offs_c = s0 + tl.arange(0, CHUNK)\\n            cmask = offs_c < nsplit\\n            m_c = tl.load(MidM + base_ml + offs_c, mask=cmask, other=float(\\"-inf\\"))\\n            w_c = tl.math.exp2(m_c - m_safe)\\n            o_c = tl.load(\\n                MidO + base_o + offs_c[:, None] * stride_mo_s + offs_d[None, :],\\n                mask=cmask[:, None],\\n                other=0.0,\\n            )\\n            acc += tl.sum(o_c * w_c[:, None], 0)\\n        out = acc / tl.where(denom > 0.0, denom, 1.0)\\n        tl.store(\\n            Out + row * stride_om + h * stride_oh + offs_d,\\n            out.to(Out.dtype.element_ty),\\n        )\\n\\n\\ndef _as_kv_view(buf: torch.Tensor) -> torch.Tensor:\\n    # pools store fp8 as uint8 and hand out a float8 view; accept either (same as duck2)\\n    if buf.dtype == torch.uint8:\\n        return buf.view(torch.float8_e4m3fn)\\n    return buf\\n\\n\\ndef _cdiv(a: int, b: int) -> int:\\n    return -(-a // b)\\n\\n\\ndef _pow2(x: int) -> bool:\\n    return x > 0 and (x & (x - 1)) == 0\\n\\n\\ndef _env_int(name: str, default: int, lo: int, hi: int, pow2: bool = False) -> int:\\n    raw = os.environ.get(name, \\"\\").strip()\\n    if not raw:\\n        return default\\n    try:\\n        val = int(raw)\\n    except ValueError:\\n        val = None\\n    if val is None or not lo <= val <= hi or (pow2 and not _pow2(val)):\\n        logger.warning(\\"[qsasplitk] ignoring %s=%r; using %d\\", name, raw, default)\\n        return default\\n    return val\\n\\n\\n_CFG: dict = {}\\n\\n\\ndef splitk_config() -> dict:\\n    \\"\\"\\"Process-wide defaults (env read once).\\"\\"\\"\\n    if not _CFG:\\n        _CFG.update(\\n            target_ctas=_env_int(\\"SGLANG_QSA_SPLITK_CTAS\\", 0, 0, 1 << 20),\\n            nsplit=_env_int(\\"SGLANG_QSA_SPLITK_NSPLIT\\", 0, 0, MAX_SPLIT),\\n            block_n=_env_int(\\"SGLANG_QSA_SPLITK_BLOCK_N\\", 32, 16, 128, pow2=True),\\n            num_warps=_env_int(\\"SGLANG_QSA_SPLITK_WARPS\\", 4, 1, 16, pow2=True),\\n            num_stages=_env_int(\\"SGLANG_QSA_SPLITK_STAGES\\", 2, 1, 6),\\n        )\\n    return _CFG\\n\\n\\n_SM_COUNT: dict = {}\\n\\n\\ndef _sm_count(device: torch.device) -> int:\\n    idx = device.index if device.index is not None else torch.cuda.current_device()\\n    if idx not in _SM_COUNT:\\n        _SM_COUNT[idx] = torch.cuda.get_device_properties(idx).multi_processor_count\\n    return _SM_COUNT[idx]\\n\\n\\ndef plan_splits(\\n    rows: int,\\n    num_kv_heads: int,\\n    topk: int,\\n    block_n: int,\\n    nsplit: int = 0,\\n    target_ctas: int = 0,\\n    num_sms: int = 58,\\n):\\n    \\"\\"\\"(nsplit, cols_per_split) from static shapes only; every split gets >= 1 block.\\"\\"\\"\\n    nblocks = max(1, _cdiv(topk, block_n))\\n    if nsplit <= 0:\\n        target = target_ctas if target_ctas > 0 else 2 * num_sms\\n        nsplit = target // max(1, rows * num_kv_heads)\\n    nsplit = max(1, min(nsplit, nblocks, MAX_SPLIT))\\n    blocks_per_split = _cdiv(nblocks, nsplit)\\n    nsplit = _cdiv(nblocks, blocks_per_split)\\n    return nsplit, blocks_per_split * block_n\\n\\n\\n_LOGGED: set = set()\\n\\n\\ndef _log_once(key, msg, *args):\\n    if key not in _LOGGED:\\n        _LOGGED.add(key)\\n        logger.info(msg, *args)\\n\\n\\ndef qsa_sparse_decode_triton_splitk(\\n    q: torch.Tensor,  # [rows, Hq, D] bf16\\n    k_buffer: torch.Tensor,  # [slots, Hkv, D] bf16 or float8_e4m3fn (or uint8 storage)\\n    v_buffer: torch.Tensor,\\n    req_to_token: torch.Tensor,  # [req_pool, max_ctx] int32\\n    req_pool_indices: torch.Tensor,  # [rows]\\n    topk_indices: torch.Tensor,  # [rows, topk] logical positions, -1 = none\\n    sequence_lens: torch.Tensor,  # [rows]\\n    scale: float,\\n    *,\\n    nsplit=None,\\n    target_ctas=None,\\n    block_n=None,\\n    num_warps=None,\\n    num_stages=None,\\n    _dot_fp16: bool = False,\\n) -> torch.Tensor:\\n    if not HAS_TRITON:\\n        raise RuntimeError(\\"triton is not available\\")\\n    cfg = splitk_config()\\n    nsplit = cfg[\\"nsplit\\"] if nsplit is None else int(nsplit)\\n    target_ctas = cfg[\\"target_ctas\\"] if target_ctas is None else int(target_ctas)\\n    block_n = cfg[\\"block_n\\"] if block_n is None else int(block_n)\\n    num_warps = cfg[\\"num_warps\\"] if num_warps is None else int(num_warps)\\n    num_stages = cfg[\\"num_stages\\"] if num_stages is None else int(num_stages)\\n    rows, num_q_heads, head_dim = q.shape\\n    k_buffer, v_buffer = _as_kv_view(k_buffer), _as_kv_view(v_buffer)\\n    num_kv_heads = k_buffer.shape[1]\\n    assert num_q_heads % num_kv_heads == 0, (num_q_heads, num_kv_heads)\\n    assert k_buffer.stride(-1) == 1 and v_buffer.stride(-1) == 1\\n    assert head_dim == triton.next_power_of_2(head_dim), head_dim\\n    assert _pow2(block_n) and block_n >= 16, block_n\\n    group = num_q_heads // num_kv_heads\\n    q = q.contiguous()\\n    topk_indices = topk_indices.contiguous()\\n    out = torch.empty_like(q)\\n    if rows == 0:\\n        return out\\n    topk = topk_indices.shape[1]\\n    ns, cols_per_split = plan_splits(\\n        rows, num_kv_heads, topk, block_n, nsplit, target_ctas, _sm_count(q.device)\\n    )\\n    _log_once(\\n        (rows, num_kv_heads, topk, ns, block_n),\\n        \\"[qsasplitk] rows=%d kv_heads=%d topk=%d -> nsplit=%d (%d cols/split, block_n=%d, warps=%d, stages=%d)\\",\\n        rows, num_kv_heads, topk, ns, cols_per_split, block_n, num_warps, num_stages,\\n    )\\n    final = ns == 1\\n    if final:\\n        mid_o = mid_m = mid_l = out  # unused by the FINAL kernel\\n        so = (0, 0, 0)\\n        sm = (0, 0)\\n    else:\\n        mid_o = torch.empty(\\n            (rows, num_q_heads, ns, head_dim), dtype=torch.float32, device=q.device\\n        )\\n        mid_m = torch.empty((rows, num_q_heads, ns), dtype=torch.float32, device=q.device)\\n        mid_l = torch.empty_like(mid_m)\\n        so = (mid_o.stride(0), mid_o.stride(1), mid_o.stride(2))\\n        sm = (mid_m.stride(0), mid_m.stride(1))\\n    _qsa_splitk_partial_kernel[(rows, num_kv_heads, ns)](\\n        q,\\n        k_buffer,\\n        v_buffer,\\n        out,\\n        mid_o,\\n        mid_m,\\n        mid_l,\\n        req_to_token,\\n        req_pool_indices,\\n        topk_indices,\\n        sequence_lens,\\n        float(scale),\\n        topk,\\n        cols_per_split,\\n        q.stride(0),\\n        q.stride(1),\\n        k_buffer.stride(0),\\n        k_buffer.stride(1),\\n        v_buffer.stride(0),\\n        v_buffer.stride(1),\\n        out.stride(0),\\n        out.stride(1),\\n        req_to_token.stride(0),\\n        topk_indices.stride(0),\\n        so[0],\\n        so[1],\\n        so[2],\\n        sm[0],\\n        sm[1],\\n        GROUP=group,\\n        BLOCK_H=max(16, triton.next_power_of_2(group)),\\n        BLOCK_N=block_n,\\n        HEAD_DIM=head_dim,\\n        FINAL=final,\\n        DOT_FP16=_dot_fp16,\\n        num_warps=num_warps,\\n        num_stages=num_stages,\\n    )\\n    if not final:\\n        _qsa_splitk_merge_kernel[(rows, num_q_heads)](\\n            mid_o,\\n            mid_m,\\n            mid_l,\\n            out,\\n            ns,\\n            so[0],\\n            so[1],\\n            so[2],\\n            sm[0],\\n            sm[1],\\n            out.stride(0),\\n            out.stride(1),\\n            SPLIT_P2=max(2, triton.next_power_of_2(ns)),\\n            CHUNK=_MERGE_CHUNK,\\n            HEAD_DIM=head_dim,\\n            num_warps=4,\\n            num_stages=1,\\n        )\\n    return out\\n\\n\\ndef _splitk_dispatch(impl, *args, **kwargs):\\n    # same calling convention as sm89_decode.qsa_sparse_decode(impl, *args)\\n    return qsa_sparse_decode_triton_splitk(*args, **kwargs)\\n\\n\\ndef splitk_route(impl, base_dispatch):\\n    \\"\\"\\"(impl, dispatch) for the patched call site: unchanged unless SGLANG_QSA_SM89_DECODE=triton_splitk.\\n\\n    The duck2 selector maps the unknown value triton_splitk to its auto branch\\n    (\\"triton\\" on sm_8x); like an explicit \\"triton\\", triton_splitk is honoured on\\n    any capability that reaches this fallback. Without Triton the duck2 choice\\n    (torch fallback) is kept.\\n    \\"\\"\\"\\n    if os.environ.get(ENV_MODE, \\"\\").strip().lower() != MODE or not HAS_TRITON:\\n        return impl, base_dispatch\\n    _log_once(\\n        \\"selected\\",\\n        \\"[qsasplitk] %s=%s: split-K QSA sparse decode (duck2 choice was %r; cfg %s)\\",\\n        ENV_MODE, MODE, impl, splitk_config(),\\n    )\\n    return MODE, _splitk_dispatch\\n\\n\\n__all__ = [\\n    \\"HAS_TRITON\\",\\n    \\"MODE\\",\\n    \\"plan_splits\\",\\n    \\"qsa_sparse_decode_triton_splitk\\",\\n    \\"splitk_config\\",\\n    \\"splitk_route\\",\\n]\\n\'\'\'\\n\\n\\ndef main():\\n    args = [a for a in sys.argv[1:] if not a.startswith(\\"--\\")]\\n    if not args:\\n        print(__doc__)\\n        sys.exit(2)\\n    site = Path(args[0])\\n    check = \\"--check\\" in sys.argv\\n    backend, base, module = site / BACKEND_REL, site / BASE_REL, site / MODULE_REL\\n    if not backend.is_file():\\n        sys.exit(f\\"qsasplitk_patch: {backend} not found; refusing to patch\\")\\n    s = backend.read_text()\\n    module_ok = module.is_file() and module.read_text() == MODULE_SRC\\n    if MARK in s and module_ok:\\n        print(\\"qsasplitk_patch: already patched\\")\\n        return\\n    if check:\\n        sys.exit(f\\"qsasplitk_patch: NOT patched (call-site mark {MARK in s}, module current {module_ok})\\")\\n    if DUCK2_MARK not in s or not base.is_file():\\n        sys.exit(\\"qsasplitk_patch: duck2 sm89 patch not applied (run duck2/patches/apply_patches.py first); \\"\\n                 \\"refusing to patch\\")\\n    if MARK not in s and s.count(ANCHOR) != 1:\\n        sys.exit(f\\"qsasplitk_patch: anchor x{s.count(ANCHOR)} (need 1): {ANCHOR[:60]!r}; refusing to patch\\")\\n    # module first, so the call site never references a missing module\\n    module.write_text(MODULE_SRC)\\n    if MARK not in s:\\n        backend.write_text(s.replace(ANCHOR, NEW))\\n    print(\\"qsasplitk_patch: patched\\", backend, \\"+\\", module)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "test_qsa_sm89_decode.py": "#!/usr/bin/env python3\\n\\"\\"\\"Unit tests for the duck2 sm_89 patches. Needs no SGLang install, only torch (+ triton on GPU).\\n\\n  python test_qsa_sm89_decode.py            # CPU: torch fallback + PLE quantiser\\n                                            # GPU: also Triton kernel, CUDA graph, timing\\n  TRITON_INTERPRET=1 python test_...py      # CPU Triton interpreter (slow, small shapes)\\n\\nShapes mirror Flash-Next at TP4: Hq=24/4=6, Hkv=max(1, 2/4)=1, D=256, topk row\\nwidth = indexer_budget 2048 + compress_ratio 4 - 1 = 2051 (qsa_indexer.py:559).\\nPass criteria: max |out - ref| <= 2e-2 * max|ref| + 1e-2 (bf16 MMA), CUDA-graph\\nreplay bit-identical to eager, rows with no valid entry exactly 0.\\n\\"\\"\\"\\nimport os\\nimport sys\\nimport time\\n\\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\\nimport torch  # noqa: E402\\n\\nfrom qsa_sm89_decode import (  # noqa: E402\\n    HAS_TRITON,\\n    qsa_sparse_decode_reference,\\n    qsa_sparse_decode_torch,\\n)\\n\\nINTERP = os.environ.get(\\"TRITON_INTERPRET\\") == \\"1\\"\\nCUDA = torch.cuda.is_available() and not INTERP\\nDEV = \\"cuda\\" if CUDA else \\"cpu\\"\\nFAILS = []\\n\\n\\ndef check(name, out, ref, zero_rows=()):\\n    err = (out.float() - ref.float()).abs().max().item()\\n    tol = 2e-2 * ref.float().abs().max().item() + 1e-2\\n    zeros_ok = all(out[r].abs().max().item() == 0 for r in zero_rows)\\n    ok = err <= tol and zeros_ok and torch.isfinite(out.float()).all().item()\\n    print(f\\"  {\'ok \' if ok else \'BAD\'} {name}: max_err={err:.3e} tol={tol:.3e} zero_rows_ok={zeros_ok}\\")\\n    if not ok:\\n        FAILS.append(name)\\n\\n\\ndef make_case(rows, topk, kv_dtype, hq=6, hkv=1, d=256, ctx=6000, pool_reqs=8, seed=0,\\n              prefix_layout=False):\\n    g = torch.Generator().manual_seed(seed)\\n    slots_n = pool_reqs * ctx + 64\\n    q = torch.randn(rows, hq, d, generator=g).to(torch.bfloat16)\\n    k = (torch.randn(slots_n, hkv, d, generator=g) * 0.5).to(kv_dtype)\\n    v = (torch.randn(slots_n, hkv, d, generator=g) * 2.0).to(kv_dtype)\\n    req_to_token = (torch.randperm(slots_n - 64, generator=g)[: pool_reqs * ctx] + 64)\\n    req_to_token = req_to_token.view(pool_reqs, ctx).to(torch.int32)\\n    req_idx = torch.randint(0, pool_reqs, (rows,), generator=g).to(torch.int64)\\n    seq = torch.randint(1, ctx, (rows,), generator=g).to(torch.int32)\\n    if rows > 2:\\n        seq[1] = 3      # short row: almost everything invalid\\n        seq[2] = 0      # no visible token -> output must be 0\\n    idx = torch.full((rows, topk), -1, dtype=torch.int32)\\n    for r in range(rows):\\n        n = min(int(seq[r]), topk - 3)\\n        if n:\\n            idx[r, :n] = torch.randperm(int(seq[r]), generator=g)[:n].to(torch.int32)\\n        if not prefix_layout and n > 8:\\n            idx[r, n // 2] = -1                       # hole in the middle\\n            idx[r, topk - 1] = int(seq[r]) - 1       # valid entry after the -1 tail\\n            idx[r, topk - 2] = int(seq[r]) + 5       # >= seq: must be ignored\\n    t = lambda x: x.to(DEV)  # noqa: E731\\n    zero_rows = [2] if rows > 2 else []\\n    return [t(q), t(k), t(v), t(req_to_token), t(req_idx), t(idx), t(seq), d ** -0.5], zero_rows\\n\\n\\ndef stock_semantics(args):\\n    \\"\\"\\"What the stock compact+FA path computes: only the first valid_count columns,\\n    every column < valid_count must be valid (prefix layout).\\"\\"\\"\\n    q, k, v, r2t, req, idx, seq, scale = args\\n    pos = idx.long()\\n    valid = (pos >= 0) & (pos < seq.long()[:, None])\\n    count = valid.sum(1, keepdim=True)\\n    cols = torch.arange(idx.shape[1], device=idx.device)[None, :]\\n    keep = valid & (cols < count)\\n    idx2 = torch.where(keep, idx, torch.full_like(idx, -1))\\n    return qsa_sparse_decode_reference(q, k, v, r2t, req, idx2, seq, scale)\\n\\n\\ndef test_torch_fallback():\\n    print(\\"[torch fallback vs reference]\\")\\n    for kv in (torch.bfloat16, torch.float8_e4m3fn):\\n        args, zr = make_case(rows=7, topk=2051 if CUDA else 300, kv_dtype=kv, ctx=3000)\\n        check(f\\"torch {kv}\\", qsa_sparse_decode_torch(*args), qsa_sparse_decode_reference(*args), zr)\\n    args, zr = make_case(rows=5, topk=200, kv_dtype=torch.bfloat16, ctx=1000, prefix_layout=True)\\n    check(\\"reference == stock packed semantics (prefix layout)\\",\\n          qsa_sparse_decode_reference(*args), stock_semantics(args), zr)\\n\\n\\ndef test_triton():\\n    from qsa_sm89_decode import qsa_sparse_decode_triton\\n\\n    print(\\"[triton kernel vs reference]\\")\\n    cases = [(1, 2051), (7, 2051), (48, 2051), (5, 17)] if CUDA else [(5, 70)]\\n    for kv in (torch.bfloat16, torch.float8_e4m3fn):\\n        for rows, topk in cases:\\n            args, zr = make_case(rows=rows, topk=topk, kv_dtype=kv, ctx=6000 if CUDA else 200)\\n            out = qsa_sparse_decode_triton(*args, _dot_fp16=INTERP)\\n            check(f\\"triton {kv} rows={rows} topk={topk}\\", out, qsa_sparse_decode_reference(*args), zr)\\n    # GQA with several KV heads (TP1/TP2 layouts): Hq=24/Hkv=2 and 12/1\\n    for hq, hkv in ((24, 2), (12, 1)):\\n        args, zr = make_case(rows=4, topk=2051 if CUDA else 40, kv_dtype=torch.bfloat16,\\n                             hq=hq, hkv=hkv, ctx=4000 if CUDA else 100)\\n        out = qsa_sparse_decode_triton(*args, _dot_fp16=INTERP)\\n        check(f\\"triton GQA hq={hq} hkv={hkv}\\", out, qsa_sparse_decode_reference(*args), zr)\\n\\n\\ndef test_cuda_graph_and_speed():\\n    from qsa_sm89_decode import qsa_sparse_decode_triton\\n\\n    print(\\"[CUDA graph replay + timing]\\")\\n    args, _ = make_case(rows=48, topk=2051, kv_dtype=torch.float8_e4m3fn, ctx=30000, pool_reqs=12)\\n    eager = qsa_sparse_decode_triton(*args)\\n    s = torch.cuda.Stream()\\n    s.wait_stream(torch.cuda.current_stream())\\n    with torch.cuda.stream(s):\\n        for _ in range(2):\\n            qsa_sparse_decode_triton(*args)\\n    torch.cuda.current_stream().wait_stream(s)\\n    graph = torch.cuda.CUDAGraph()\\n    with torch.cuda.graph(graph):\\n        static_out = qsa_sparse_decode_triton(*args)\\n    # mutate inputs in place (new indices / lengths) and replay\\n    args[5].copy_(torch.roll(args[5], 1, dims=1))\\n    args[6].sub_(1).clamp_(min=0)\\n    graph.replay()\\n    torch.cuda.synchronize()\\n    fresh = qsa_sparse_decode_triton(*args)\\n    same = torch.equal(static_out, fresh)\\n    print(f\\"  {\'ok \' if same else \'BAD\'} graph replay == eager after input update\\")\\n    if not same:\\n        FAILS.append(\\"cuda graph\\")\\n    check(\\"graph vs reference\\", static_out, qsa_sparse_decode_reference(*args), [2])\\n    del eager\\n    for name, fn in ((\\"triton\\", qsa_sparse_decode_triton), (\\"torch\\", qsa_sparse_decode_torch)):\\n        for _ in range(3):\\n            fn(*args)\\n        torch.cuda.synchronize()\\n        t0 = time.perf_counter()\\n        n = 50\\n        for _ in range(n):\\n            fn(*args)\\n        torch.cuda.synchronize()\\n        us = (time.perf_counter() - t0) / n * 1e6\\n        print(f\\"  {name}: {us:.0f} us per QSA layer call at 48 rows x 2051 (x12 layers -> {12*us/1000:.2f} ms/step)\\")\\n\\n\\ndef test_ple_quantiser():\\n    print(\\"[PLE scaled fp8 quantiser]\\")\\n    import importlib\\n\\n    os.environ[\\"SGLANG_PLE_FP8_SCALE\\"] = \\"0.004\\"\\n    import ple_fp8_scale as pfs\\n\\n    importlib.reload(pfs)\\n\\n    class Emb:  # stand-in for the PLE embedding module\\n        weight_scale = torch.ones(1, dtype=torch.bfloat16)\\n\\n    emb = Emb()\\n    g = torch.Generator().manual_seed(1)\\n    w = (torch.randn(20000, 64, generator=g) * 0.05).to(torch.bfloat16)\\n    w[0, 0] = 5.0  # outlier beyond 448 * s = 1.79 -> must saturate, not NaN\\n    qf = pfs.quantize_ple_rows(emb, w)\\n    s = float(emb.weight_scale)\\n    deq = qf.float() * s\\n    finite = torch.isfinite(deq).all().item()\\n    rel = ((deq[1:] - w[1:].float()).norm() / w[1:].float().norm()).item()\\n    raw = w.to(torch.float8_e4m3fn).float()\\n    raw_rel = ((raw[1:] - w[1:].float()).norm() / w[1:].float().norm()).item()\\n    ok = finite and abs(s - 0.004) < 1e-4 and rel < 0.04 and float(deq[0, 0]) == 448 * s\\n    print(f\\"  {\'ok \' if ok else \'BAD\'} scale={s:.6g} rel_err scaled={rel:.4f} vs raw cast={raw_rel:.4f} \\"\\n          f\\"outlier->{float(deq[0, 0]):.3f} finite={finite}\\")\\n    if not ok:\\n        FAILS.append(\\"ple quantiser\\")\\n    keep = pfs.keep_quantised_scale(\\"weight_scale\\", torch.ones(1))\\n    try:\\n        pfs.keep_quantised_scale(\\"weight_scale\\", torch.tensor([0.0123]))\\n        guard = False\\n    except ValueError:\\n        guard = True\\n    print(f\\"  {\'ok \' if keep and guard else \'BAD\'} checkpoint scale handling (skip 1.0, reject real scale)\\")\\n    if not (keep and guard):\\n        FAILS.append(\\"ple scale guard\\")\\n    os.environ.pop(\\"SGLANG_PLE_FP8_SCALE\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    torch.manual_seed(0)\\n    print(f\\"device={DEV} triton={HAS_TRITON} interpreter={INTERP}\\")\\n    test_torch_fallback()\\n    test_ple_quantiser()\\n    if HAS_TRITON and (CUDA or INTERP):\\n        test_triton()\\n    else:\\n        print(\\"[triton kernel] skipped (needs CUDA or TRITON_INTERPRET=1 with triton installed)\\")\\n    if CUDA and HAS_TRITON:\\n        test_cuda_graph_and_speed()\\n    print(\\"FAILED: \\" + \\", \\".join(FAILS) if FAILS else \\"ALL OK\\")\\n    sys.exit(1 if FAILS else 0)\\n", "think_force_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal\'s strict-thinking grammar (constrained/reasoner_grammar_backend.py) so that a request\\nwhose thinking budget (custom_params.thinking_budget, --enable-strict-thinking) runs out is forced to emit\\n</think> + SGLANG_FORCED_THINK_SUFFIX (e.g. \\"\\\\n\\\\n[[\\" to start the answer grid) before free generation resumes.\\nWithout it, a forced </think> mid-thought often leads to more reasoning in the answer (cut by max_tokens) or a bare\\n\\"To solve\\" + EOS: G4 test 2026-10-08, 3 of 10 forced ARC attempts parsed a grid. The suffix is tokenized once by the\\nbackend; natural (unforced) think ends are untouched. Empty/unset env = stock behaviour.\\nusage: think_force_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [think-force patch]\\"\\nEDITS = [\\n    # module-level holder for the tokenized suffix (set by the backend, read by every grammar object)\\n    (\\"class ReasonerGrammarObject(BaseGrammarObject):\\\\n\\",\\n     f\\"_FORCED_SUFFIX = {{\\\\\\"ids\\\\\\": ()}}  {MARK} tokenized SGLANG_FORCED_THINK_SUFFIX\\\\n\\\\n\\\\n\\"\\n     \\"class ReasonerGrammarObject(BaseGrammarObject):\\\\n\\"),\\n    (\\"        self.max_think_tokens = max_think_tokens\\\\n\\",\\n     \\"        self.max_think_tokens = max_think_tokens\\\\n\\"\\n     f\\"        self._forced_end = False  {MARK} set when the budget forced the think end\\\\n\\"),\\n    # the budget-exhausted branch: remember that this think end is forced\\n    (\\"            else:\\\\n\\"\\n     \\"                self._do_token_filter(\\\\n\\"\\n     \\"                    vocab_mask,\\\\n\\"\\n     \\"                    [self.think_end_ids[self._matched_think_end_tokens]],\\\\n\\",\\n     \\"            else:\\\\n\\"\\n     f\\"                self._forced_end = True  {MARK}\\\\n\\"\\n     \\"                self._do_token_filter(\\\\n\\"\\n     \\"                    vocab_mask,\\\\n\\"\\n     \\"                    [self.think_end_ids[self._matched_think_end_tokens]],\\\\n\\"),\\n    # right after a forced think end: force the suffix token by token\\n    (\\"        if self._is_generation() and self.grammar is not None:\\\\n\\"\\n     \\"            self.grammar.fill_vocab_mask(vocab_mask, idx)\\\\n\\",\\n     \\"        _sfx = _FORCED_SUFFIX[\\\\\\"ids\\\\\\"]\\\\n\\"\\n     f\\"        if self._is_generation() and self._forced_end and 0 <= self.tokens_after_end < len(_sfx):  {MARK}\\\\n\\"\\n     \\"            self._do_token_filter(vocab_mask, [_sfx[self.tokens_after_end]], idx, is_allowed=True)\\\\n\\"\\n     \\"            return\\\\n\\"\\n     \\"        if self._is_generation() and self.grammar is not None:\\\\n\\"\\n     \\"            self.grammar.fill_vocab_mask(vocab_mask, idx)\\\\n\\"),\\n    (\\"        new_obj._thinking_match_history = list(self._thinking_match_history)\\\\n\\",\\n     \\"        new_obj._thinking_match_history = list(self._thinking_match_history)\\\\n\\"\\n     f\\"        new_obj._forced_end = self._forced_end  {MARK}\\\\n\\"),\\n    (\\"        self.max_think_tokens = envs.SGLANG_MAX_THINK_TOKENS.get()\\\\n\\",\\n     \\"        self.max_think_tokens = envs.SGLANG_MAX_THINK_TOKENS.get()\\\\n\\"\\n     f\\"        import os as _os  {MARK}\\\\n\\"\\n     \\"        _s = _os.environ.get(\\\\\\"SGLANG_FORCED_THINK_SUFFIX\\\\\\", \\\\\\"\\\\\\").encode().decode(\\\\\\"unicode_escape\\\\\\")\\\\n\\"\\n     \\"        _FORCED_SUFFIX[\\\\\\"ids\\\\\\"] = tuple(tokenizer.encode(_s, add_special_tokens=False)) if _s else ()\\\\n\\"\\n     \\"        logger.info(\\\\\\"[think-force patch] forced think suffix %r -> %s\\\\\\", _s, _FORCED_SUFFIX[\\\\\\"ids\\\\\\"])\\\\n\\"),\\n]\\n\\n\\ndef main():\\n    f = Path(sys.argv[1]) / \\"sglang/srt/constrained/reasoner_grammar_backend.py\\"\\n    s = f.read_text()\\n    if MARK in s:\\n        print(\\"think_force_patch: already patched\\")\\n        return\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"think_force_patch: NOT patched\\")\\n    for old, new in EDITS:\\n        if s.count(old) != 1:\\n            sys.exit(f\\"think_force_patch: anchor x{s.count(old)} (need 1): {old[:60]!r}; refusing to patch\\")\\n        s = s.replace(old, new)\\n    f.write_text(s)\\n    print(\\"think_force_patch: patched\\", f)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "topklogits_patch.py": "#!/usr/bin/env python3\\n\\"\\"\\"Idempotently patch Pennyroyal SGLang so a TP>1 decode step samples from per-rank top-K logits instead of\\nall-gathering the full [B, V] vocab logits, gated by SGLANG_TOPK_LOGITS_EXCHANGE=K (unset / \\"\\" / 0 = stock).\\n\\nWhy: every decode step all-gathers the lm_head output [B, V/TP] -> [B, V] (V = 248320; NCCL all_gather inside the\\ndecode CUDA graph, then a movedim copy and an fp32 copy into the shared logits buffer, logits_processor.py:678-695),\\nthen the sampler divides, softmaxes and samples over all V columns (sampler.py:215-237, 285-292). On 4x L4 (PCIe,\\nmultimem off) that is ~1-1.6 ms per step at B 13-20, plus ~0.5 ms of full-vocab sampling passes. The harness only\\nuses top-k sampling (top_k 20, top_p 0.95, T 1.0; salvage is greedy), which needs just the top-k tokens and the\\nsoftmax normalizer.\\n\\nWith K set, for forward_mode == DECODE only:\\n  * logits_processor.py: the decode lm_head output stays vocab-sharded. Each rank returns its own [B, W] shard\\n    (W = V_padded / TP) as fp32, written into the shared next_token_logits_buffer (same lifetime as the stock\\n    logits). No NCCL op for logits inside the decode graph.\\n  * model_runner.py sample(): hands the shard to sampler.topk_logits_exchange_sample() before any logits use.\\n  * sampler.py: if the batch only needs top-k sampling (top_k <= K for every row, no logprobs, grammar,\\n    penalties, logit_bias, min_p, custom logit processor, seed, sampling masks, observer; flashinfer/pytorch\\n    backend), each rank sends [top-K values | global ids | logsumexp(logits/T) | world*K Exp(1) draws] (fp32,\\n    B x (2K+1+world*K)) through one small all-gather; every rank merges the identical gathered tensor the same\\n    way: the union of the per-rank top-K holds every token the stock sampler can pick and every token more probable\\n    than it, so the stock filter rule is evaluated exactly (flashinfer \\"joint\\": #{p_j > p_i} < top_k and\\n    sum{p_j > p_i} < top_p, with p = full-vocab softmax(logits/T); pytorch backend: sorted position < top_k and\\n    exclusive cumsum <= top_p), then an exponential race with rank 0\'s draws picks the token. All-greedy batches\\n    exchange (max, first argmax) per rank and reproduce torch.argmax over the full row exactly (ties included).\\n    Otherwise it performs the stock all-gather itself (same op, same dtype, same [:, :vocab] slice -> bit-identical\\n    stock next_token_logits) and the stock path runs unchanged.\\n  * sampling_batch_info.py: a CPU copy of the per-request top_k list (kept through filter/merge), so the\\n    eligibility check needs no GPU sync. The decision uses only batch-level CPU state that is identical on every\\n    TP rank, so all ranks take the same collective.\\nDisabled automatically (warning) for: TP 1, DP attention / dp lm_head / lm_head all-to-all, final logit\\nsoftcapping, return_full_logits, RL on-policy mode, speculative decoding.\\n\\nusage: topklogits_patch.py <site-packages dir> [--check]\\n\\"\\"\\"\\nimport sys\\nfrom pathlib import Path\\n\\nMARK = \\"# [topklogits patch]\\"\\nENV = \\"SGLANG_TOPK_LOGITS_EXCHANGE\\"\\n\\n# Shared env parse, used verbatim in logits_processor.py (TKX_K) and sampling_batch_info.py (_TKX_ON).\\nENV_PARSE_SRC = \'\'\'def _tkx_env_k():\\n    import logging as _tkx_logging\\n    import os as _tkx_os\\n\\n    raw = _tkx_os.environ.get(\\"SGLANG_TOPK_LOGITS_EXCHANGE\\", \\"\\").strip()\\n    if not raw:\\n        return 0\\n    try:\\n        k = int(raw)\\n    except ValueError:\\n        k = -1\\n    if k == 0:\\n        return 0\\n    if not 1 <= k <= 1024:\\n        _tkx_logging.getLogger(__name__).warning(\\n            \\"[topklogits patch] ignoring SGLANG_TOPK_LOGITS_EXCHANGE=%r (need 1..1024); stock logits all-gather\\",\\n            raw,\\n        )\\n        return 0\\n    return k\\n\'\'\'\\n\\n# ---------------------------------------------------------------- logits_processor.py (module level)\\nLP_HELPER = (\'\'\'@MARK@ vocab-sharded decode logits, colab/perf_patches/topklogits_patch.py. With\\n@MARK@ SGLANG_TOPK_LOGITS_EXCHANGE=K a TP>1 DECODE forward skips the [B, V] logits all-gather and returns this\\n@MARK@ rank\'s [B, V/TP] lm_head shard (fp32, in the shared logits buffer); sampler.topk_logits_exchange_sample\\n@MARK@ then exchanges per-rank top-K (or does the stock all-gather when the batch needs full logits).\\n\'\'\' + ENV_PARSE_SRC + \'\'\'\\n\\nTKX_K = _tkx_env_k()\\n_TKX_STATE = {}  # W: local logits width, vocab: stock slice width, dtype: lm_head output dtype\\n\\n\\ndef _tkx_logits_enabled(proc) -> bool:\\n    if TKX_K <= 0:\\n        return False\\n    why = None\\n    if not proc.do_tensor_parallel_all_gather:\\n        why = \\"no TP logits all-gather (tp_size 1 or skip_all_gather)\\"\\n    elif proc.use_attn_tp_group:\\n        why = \\"enable_dp_lm_head\\"\\n    elif proc.do_tensor_parallel_all_gather_dp_attn:\\n        why = \\"DP attention\\"\\n    elif proc.use_tp_lm_head_all_to_all:\\n        why = \\"enable_tp_lm_head_all_to_all\\"\\n    elif proc.final_logit_softcapping:\\n        why = \\"final_logit_softcapping\\"\\n    elif proc.return_full_logits:\\n        why = \\"return_full_logits\\"\\n    elif proc.rl_on_policy_target is not None:\\n        why = \\"rl_on_policy_target\\"\\n    else:\\n        try:\\n            from sglang.srt.runtime_context import get_server_args\\n\\n            spec = getattr(get_server_args(), \\"speculative_algorithm\\", None)\\n        except Exception as exc:  # no server args (offline use): stay stock\\n            spec = \\"unknown (%s)\\" % type(exc).__name__\\n        if spec:\\n            why = \\"speculative decoding (%s)\\" % spec\\n    if why is not None:\\n        logger.warning(\\n            \\"[topklogits patch] SGLANG_TOPK_LOGITS_EXCHANGE=%d ignored for this LogitsProcessor: %s\\", TKX_K, why\\n        )\\n        return False\\n    _TKX_STATE[\\"vocab\\"] = proc.vocab_size\\n    logger.warning(\\n        \\"[topklogits patch] decode logits stay vocab-sharded; the sampler exchanges per-rank top-%d\\", TKX_K\\n    )\\n    return True\\n\\n\\ndef _tkx_shard_carrier(logits, buf, vocab_size):\\n    \\"\\"\\"This rank\'s [B, W] lm_head output as fp32 (exact cast), written into the shared logits buffer when there is\\n    one (as a contiguous [B, W] block at its start), so it has the same lifetime as the stock next_token_logits.\\"\\"\\"\\n    rows, width = logits.shape[0], logits.shape[-1]\\n    _TKX_STATE[\\"W\\"] = width\\n    _TKX_STATE[\\"dtype\\"] = logits.dtype\\n    _TKX_STATE[\\"vocab\\"] = vocab_size\\n    if (\\n        buf is not None\\n        and buf.dim() == 2\\n        and buf.dtype == torch.float32\\n        and buf.shape[0] == rows\\n        and buf.shape[1] >= width\\n    ):\\n        out = buf.view(-1)[: rows * width].view(rows, width) if buf.is_contiguous() else buf[:, :width]\\n        out.copy_(logits)\\n        return out\\n    return logits.float()\\n\\n\\n\'\'\').replace(\\"@MARK@\\", MARK)\\n\\n# ---------------------------------------------------------------- sampler.py: pure torch math (bench-tested)\\nPURE_SRC = \'\'\'def tkx_greedy_payload(x, col_offset, valid_w):\\n    \\"\\"\\"[B, 2] fp32 per rank: (max logit, global id of its first occurrence) over the rank\'s valid columns.\\"\\"\\"\\n    rows = x.shape[0]\\n    pay = torch.empty((rows, 2), dtype=torch.float32, device=x.device)\\n    if valid_w <= 0:\\n        pay[:, 0] = float(\\"-inf\\")\\n        pay[:, 1] = 0.0\\n        return pay\\n    xv = x[:, :valid_w]\\n    idx = torch.argmax(xv, dim=-1, keepdim=True)\\n    pay[:, 0:1] = torch.gather(xv, 1, idx)\\n    pay[:, 1:2] = idx + col_offset\\n    return pay\\n\\n\\ndef tkx_greedy_merge(gathered):\\n    \\"\\"\\"[world, B, 2] -> int64 [B]; ties go to the lowest rank, i.e. torch.argmax over the concatenated row.\\"\\"\\"\\n    vals = gathered[:, :, 0].t()\\n    ids = gathered[:, :, 1].t()\\n    best = torch.argmax(vals, dim=-1, keepdim=True)\\n    return torch.gather(ids, 1, best).squeeze(1).to(torch.int64)\\n\\n\\ndef tkx_sample_payload(x, temperatures, k, n_draws, col_offset, valid_w):\\n    \\"\\"\\"[B, 2k+1+n_draws] fp32 per rank: k largest logits | their global ids | logsumexp(logits / T) | Exp(1) draws.\\n    Every rank draws the same number of random values, so the TP ranks\' generators stay in lockstep.\\"\\"\\"\\n    rows = x.shape[0]\\n    pay = torch.empty((rows, 2 * k + 1 + n_draws), dtype=torch.float32, device=x.device)\\n    kk = min(k, max(valid_w, 0))\\n    if kk > 0:\\n        xv = x[:, :valid_w]\\n        vals, idx = torch.topk(xv, kk, dim=-1, largest=True, sorted=False)\\n        pay[:, :kk] = vals\\n        pay[:, k : k + kk] = idx + col_offset\\n        pay[:, 2 * k] = torch.logsumexp(xv / temperatures, dim=-1)\\n    else:\\n        pay[:, 2 * k] = float(\\"-inf\\")\\n    if kk < k:\\n        pay[:, kk:k] = float(\\"-inf\\")\\n        pay[:, k + kk : 2 * k] = 0.0\\n    pay[:, 2 * k + 1 :] = torch.empty(\\n        (rows, n_draws), dtype=torch.float32, device=x.device\\n    ).exponential_()\\n    return pay\\n\\n\\ndef tkx_sample_weights(gathered, temperatures, top_ks, top_ps, k, joint=True):\\n    \\"\\"\\"[world, B, 2k+1+world*k] (rank order) -> (global ids [B, world*k] as fp32, weights [B, world*k], inexact [B]).\\n\\n    Candidates = union of the per-rank top-k. With top_k <= k every token the stock sampler can pick is in it, and\\n    so is every token more probable than it, so the strictly-greater counts / masses below are exact.\\n    weights = full-vocab softmax(logits / T) of the allowed candidates, 0 for the others.\\n    joint=True  = flashinfer top_k_top_p_sampling_from_probs(filter_apply_order=\\"joint\\"): token i is allowed iff\\n                  #{j: p_j > p_i} < top_k and sum{p_j: p_j > p_i} < top_p.\\n    joint=False = sglang\'s pytorch backend: sorted position < top_k and exclusive cumsum <= top_p.\\n    inexact: the only way the union can miss an allowed token is an exact logit tie at some rank\'s k-th value (more\\n    than k - top_k tokens of one rank tied inside the allowed set). Rows whose allowed set reaches down to the\\n    largest per-rank k-th value are flagged (sound: every row that can differ from the stock rule is flagged).\\"\\"\\"\\n    world, rows, _ = gathered.shape\\n    n = world * k\\n    vals = gathered[:, :, :k].permute(1, 0, 2).reshape(rows, n)\\n    ids = gathered[:, :, k : 2 * k].permute(1, 0, 2).reshape(rows, n)\\n    lse = torch.logsumexp(gathered[:, :, 2 * k], dim=0).unsqueeze(1)\\n    cut = gathered[:, :, :k].amin(dim=-1).amax(dim=0).unsqueeze(1)  # largest per-rank k-th value\\n    vals, order = torch.sort(vals, dim=-1, descending=True)\\n    ids = torch.gather(ids, 1, order)\\n    probs = torch.exp(vals / temperatures - lse)\\n    if joint:\\n        gt = probs.unsqueeze(1) > probs.unsqueeze(2)  # [B, i, j]: p_j > p_i\\n        n_gt = gt.sum(dim=-1)\\n        m_gt = (gt.to(probs.dtype) * probs.unsqueeze(1)).sum(dim=-1)\\n        allow = (n_gt < top_ks.view(-1, 1)) & (m_gt < top_ps.view(-1, 1))\\n    else:\\n        excl = torch.cumsum(probs, dim=-1) - probs\\n        pos = torch.arange(n, device=probs.device).view(1, -1)\\n        allow = (pos < top_ks.view(-1, 1)) & ~(excl > top_ps.view(-1, 1))\\n    weights = torch.where(allow, probs, torch.zeros_like(probs))\\n    inexact = ((weights > 0) & (vals <= cut)).any(dim=-1)\\n    return ids, weights, inexact\\n\\n\\ndef tkx_sample_merge(gathered, temperatures, top_ks, top_ps, k, joint=True):\\n    \\"\\"\\"[world, B, 2k+1+world*k] -> (int32 token ids [B], inexact [B]), bitwise identical on every rank: an\\n    exponential race over tkx_sample_weights with rank 0\'s Exp(1) draws (P(token) = weight / sum of weights).\\"\\"\\"\\n    ids, weights, inexact = tkx_sample_weights(gathered, temperatures, top_ks, top_ps, k, joint)\\n    draws = gathered[0, :, 2 * k + 1 : 2 * k + 1 + ids.shape[1]].clamp_min(1e-30)\\n    pick = torch.argmax(weights / draws, dim=-1, keepdim=True)\\n    return torch.gather(ids, 1, pick).squeeze(1).to(torch.int32), inexact\\n\'\'\'\\n\\n# ---------------------------------------------------------------- sampler.py: SGLang glue\\nGLUE_SRC = \'\'\'_TKX_STATS = {\\"compact\\": 0, \\"fallback\\": 0, \\"reasons\\": {}, \\"inexact\\": None}\\n\\n\\ndef _tkx_lp():\\n    from sglang.srt.layers import logits_processor as lp\\n\\n    return lp\\n\\n\\ndef _tkx_ineligible(model_runner, forward_batch, observer, rows, k):\\n    \\"\\"\\"Why this decode batch needs the full-vocab logits, or None. Only batch-level CPU state that is identical\\n    on every TP rank is used, so every rank takes the same branch (both branches are TP collectives).\\"\\"\\"\\n    si = forward_batch.sampling_info\\n    if si is None:\\n        return \\"no sampling_info\\"\\n    if rows <= 0:\\n        return \\"empty batch\\"\\n    if forward_batch.return_logprob:\\n        return \\"logprobs\\"\\n    ks = getattr(si, \\"tkx_top_ks_cpu\\", None)\\n    if ks is None or len(ks) != rows:\\n        return \\"no CPU top_k list\\"\\n    if not si.is_all_greedy and max(ks) > k:\\n        return \\"top_k > K (or unset)\\"\\n    if si.need_min_p_sampling:\\n        return \\"min_p\\"\\n    if si.grammars:\\n        return \\"grammar\\"\\n    if si.logit_bias is not None:\\n        return \\"logit_bias\\"\\n    if si.has_custom_logit_processor:\\n        return \\"custom logit processor\\"\\n    if si.sampling_seed is not None:\\n        return \\"sampling seed (deterministic)\\"\\n    if si.acc_additive_penalties is not None or si.acc_scaling_penalties is not None:\\n        return \\"penalties\\"\\n    orch = si.penalizer_orchestrator\\n    if orch is not None and orch.is_required:\\n        return \\"penalties\\"\\n    if any(si.return_sampling_masks or ()):\\n        return \\"sampling masks\\"\\n    if observer is not None and observer.is_active(si):\\n        return \\"sampling observer\\"\\n    sampler = model_runner.sampler\\n    if type(sampler) is not Sampler:\\n        return \\"custom sampler\\"\\n    if sampler.rl_on_policy_target is not None or sampler.use_ascend_backend:\\n        return \\"rl / ascend sampler\\"\\n    if not si.is_all_greedy and get_exec().kernel.sampling_backend not in (\\"flashinfer\\", \\"pytorch\\"):\\n        return \\"sampling backend\\"\\n    return None\\n\\n\\ndef _tkx_note(kind, reason=None):\\n    st = _TKX_STATS\\n    st[kind] += 1\\n    if reason is not None:\\n        seen = st[\\"reasons\\"].get(reason, 0)\\n        st[\\"reasons\\"][reason] = seen + 1\\n        if seen == 0:\\n            logger.warning(\\"[topklogits patch] full-vocab fallback (first time): %s\\", reason)\\n    total = st[\\"compact\\"] + st[\\"fallback\\"]\\n    if total == 1 or total % 20000 == 0:\\n        inexact = st[\\"inexact\\"]\\n        logger.warning(\\n            \\"[topklogits patch] decode steps: top-K exchange %d (rows hitting a k-th-value tie: %s), \\"\\n            \\"full-vocab fallback %d %s\\",\\n            st[\\"compact\\"],\\n            \\"n/a\\" if inexact is None else int(inexact),  # one host sync per 20000 steps\\n            st[\\"fallback\\"],\\n            st[\\"reasons\\"],\\n        )\\n\\n\\ndef _tkx_all_gather(pay, group):\\n    world = group.world_size\\n    out = torch.empty((world * pay.shape[0], pay.shape[1]), dtype=pay.dtype, device=pay.device)\\n    group.all_gather_into_tensor(out, pay)\\n    return out.view(world, pay.shape[0], pay.shape[1])\\n\\n\\ndef _tkx_full_logits(x, st, group):\\n    \\"\\"\\"Stock decode next_token_logits from the shard: the same TP all-gather of the lm_head output dtype, the same\\n    [:, :vocab] slice, then fp32 -> bit-identical to the unpatched logits.\\"\\"\\"\\n    dtype = st.get(\\"dtype\\", torch.float32)\\n    full = group.all_gather(x.to(dtype).contiguous(), dim=-1)\\n    return full[:, : st[\\"vocab\\"]].float().contiguous()\\n\\n\\ndef topk_logits_exchange_sample(model_runner, logits_output, forward_batch, observer):\\n    \\"\\"\\"Decode sampling from vocab-sharded logits. Returns the next token ids, or None after putting the stock\\n    full-vocab logits back into logits_output (the caller then runs the stock path).\\"\\"\\"\\n    lp = _tkx_lp()\\n    st = lp._TKX_STATE\\n    x = logits_output.next_token_logits\\n    width = st.get(\\"W\\")\\n    if width is None or x.dim() != 2 or x.shape[-1] != width or width >= st.get(\\"vocab\\", 0):\\n        return None  # full-vocab logits (prefill, or the shard path is off): stock path\\n    group = get_tp_group()\\n    k = lp.TKX_K\\n    rows = x.shape[0]\\n    reason = _tkx_ineligible(model_runner, forward_batch, observer, rows, k)\\n    if reason is None and width * group.world_size < st[\\"vocab\\"]:\\n        reason = \\"unexpected shard layout\\"\\n    if reason is not None:\\n        logits_output.next_token_logits = _tkx_full_logits(x, st, group)\\n        _tkx_note(\\"fallback\\", reason)\\n        return None\\n    si = forward_batch.sampling_info\\n    si.grammar_mask = None\\n    sanitize_nan_logits(x, \\"sampler: next_token_logits (top-K exchange shard)\\")\\n    world = group.world_size\\n    off = group.rank_in_group * width\\n    valid = max(0, min(width, st[\\"vocab\\"] - off))\\n    if si.is_all_greedy:\\n        ids = tkx_greedy_merge(_tkx_all_gather(tkx_greedy_payload(x, off, valid), group))\\n    else:\\n        pay = tkx_sample_payload(x, si.temperatures, k, world * k, off, valid)\\n        ids, inexact = tkx_sample_merge(\\n            _tkx_all_gather(pay, group),\\n            si.temperatures,\\n            si.top_ks,\\n            si.top_ps,\\n            k,\\n            joint=get_exec().kernel.sampling_backend == \\"flashinfer\\",\\n        )\\n        if _TKX_STATS[\\"inexact\\"] is None:\\n            _TKX_STATS[\\"inexact\\"] = torch.zeros((), dtype=torch.int64, device=ids.device)\\n        _TKX_STATS[\\"inexact\\"].add_(inexact.sum())\\n    model_runner.sampler._sync_token_ids_across_tp(ids, si)\\n    _tkx_note(\\"compact\\")\\n    return ids\\n\'\'\'\\n\\nSAMPLER_BLOCK = (\\n    f\\"{MARK} per-rank top-K logits exchange for decode sampling, colab/perf_patches/topklogits_patch.py.\\\\n\\"\\n    f\\"{MARK} Only reached when SGLANG_TOPK_LOGITS_EXCHANGE=K is set (model_runner.sample -> topk_logits_exchange_sample).\\\\n\\"\\n    + PURE_SRC + \\"\\\\n\\\\n\\" + GLUE_SRC\\n)\\n\\nSBI_HELPER = (f\\"\\\\n{MARK} CPU copy of per-request top_k for the top-K logits exchange (no GPU sync)\\\\n\\"\\n              + ENV_PARSE_SRC + \\"\\\\n\\\\n_TKX_ON = _tkx_env_k() > 0\\\\n\\")\\n\\nFILES = {\\n    \\"sglang/srt/layers/logits_processor.py\\": [\\n        (\\"\\\\n\\\\nclass LogitsProcessor(nn.Module):\\\\n\\",\\n         \\"\\\\n\\\\n\\" + LP_HELPER + \\"class LogitsProcessor(nn.Module):\\\\n\\"),\\n        (\\"        self.input_logprob_processor = InputLogprobProcessor()\\\\n\\",\\n         \\"        self.input_logprob_processor = InputLogprobProcessor()\\\\n\\"\\n         f\\"        self._tkx_enabled = _tkx_logits_enabled(self)  {MARK}\\\\n\\"),\\n        (\\"        used_tp_lm_head_all_to_all = False\\\\n\\"\\n         \\"        if self.do_tensor_parallel_all_gather:\\\\n\\"\\n         \\"            if self.use_attn_tp_group:\\\\n\\",\\n         f\\"        if self._tkx_enabled and logits_metadata.forward_mode.is_decode():  {MARK}\\\\n\\"\\n         \\"            return _tkx_shard_carrier(\\\\n\\"\\n         \\"                logits,\\\\n\\"\\n         \\"                logits_metadata.next_token_logits_buffer if use_logits_buffer else None,\\\\n\\"\\n         \\"                self.vocab_size,\\\\n\\"\\n         \\"            )\\\\n\\"\\n         \\"\\\\n\\"\\n         \\"        used_tp_lm_head_all_to_all = False\\\\n\\"\\n         \\"        if self.do_tensor_parallel_all_gather:\\\\n\\"\\n         \\"            if self.use_attn_tp_group:\\\\n\\"),\\n    ],\\n    \\"sglang/srt/layers/sampler.py\\": [\\n        (\\"\\\\n\\\\ndef register_sampler_backend(backend: str, factory: Callable[[], \\\\\\"Sampler\\\\\\"]) -> None:\\\\n\\",\\n         \\"\\\\n\\\\n\\" + SAMPLER_BLOCK\\n         + \\"\\\\n\\\\ndef register_sampler_backend(backend: str, factory: Callable[[], \\\\\\"Sampler\\\\\\"]) -> None:\\\\n\\"),\\n    ],\\n    \\"sglang/srt/model_executor/model_runner.py\\": [\\n        (\\"from sglang.srt.layers.logits_processor import LogitsProcessorOutput\\\\n\\",\\n         \\"from sglang.srt.layers.logits_processor import LogitsProcessorOutput\\\\n\\"\\n         f\\"from sglang.srt.layers.logits_processor import TKX_K as _TKX_K  {MARK}\\\\n\\"),\\n        (\\"        logits_output.auxiliary_device_output = None\\\\n\\"\\n         \\"        observer = self.sampling_observer\\\\n\\"\\n         \\"        # Preserve two-argument overrides when observation is inactive.\\\\n\\",\\n         \\"        logits_output.auxiliary_device_output = None\\\\n\\"\\n         \\"        observer = self.sampling_observer\\\\n\\"\\n         f\\"        if _TKX_K > 0 and logits_output.next_token_logits is not None:  {MARK}\\\\n\\"\\n         \\"            from sglang.srt.layers.sampler import topk_logits_exchange_sample\\\\n\\"\\n         \\"\\\\n\\"\\n         \\"            _tkx_ids = topk_logits_exchange_sample(\\\\n\\"\\n         \\"                self, logits_output, forward_batch, observer\\\\n\\"\\n         \\"            )\\\\n\\"\\n         \\"            if _tkx_ids is not None:  # sampled from the per-rank top-K exchange\\\\n\\"\\n         \\"                self.ngram_embedding_manager.update_after_decode(\\\\n\\"\\n         \\"                    next_token_ids=_tkx_ids,\\\\n\\"\\n         \\"                    forward_batch=forward_batch,\\\\n\\"\\n         \\"                )\\\\n\\"\\n         \\"                return _tkx_ids\\\\n\\"\\n         \\"        # Preserve two-argument overrides when observation is inactive.\\\\n\\"),\\n    ],\\n    \\"sglang/srt/sampling/sampling_batch_info.py\\": [\\n        (\\"logger = logging.getLogger(__name__)\\\\n\\",\\n         \\"logger = logging.getLogger(__name__)\\\\n\\" + SBI_HELPER),\\n        (\\"    sampling_mask_max_top_k: int = 0\\\\n\\",\\n         \\"    sampling_mask_max_top_k: int = 0\\\\n\\"\\n         f\\"    tkx_top_ks_cpu: Optional[List[int]] = None  {MARK} only with SGLANG_TOPK_LOGITS_EXCHANGE\\\\n\\"),\\n        (\\"            sampling_mask_max_top_k=sampling_mask_max_top_k,\\\\n\\"\\n         \\"        )\\\\n\\"\\n         \\"        ret.adjusted_from_schedule_batch(batch, vocab_size)\\\\n\\",\\n         \\"            sampling_mask_max_top_k=sampling_mask_max_top_k,\\\\n\\"\\n         f\\"            tkx_top_ks_cpu=(  {MARK}\\\\n\\"\\n         \\"                [r.sampling_params.top_k for r in reqs] if _TKX_ON else None\\\\n\\"\\n         \\"            ),\\\\n\\"\\n         \\"        )\\\\n\\"\\n         \\"        ret.adjusted_from_schedule_batch(batch, vocab_size)\\\\n\\"),\\n        (\\"\\\\n        self.adjusted_filter_batch(keep_indices, keep_indices_device)\\\\n\\",\\n         f\\"\\\\n        if self.tkx_top_ks_cpu is not None:  {MARK}\\\\n\\"\\n         \\"            self.tkx_top_ks_cpu = [self.tkx_top_ks_cpu[i] for i in keep_indices]\\\\n\\"\\n         \\"\\\\n        self.adjusted_filter_batch(keep_indices, keep_indices_device)\\\\n\\"),\\n        (\\"        self.is_all_greedy &= other.is_all_greedy\\\\n\\",\\n         f\\"        self.tkx_top_ks_cpu = (  {MARK}\\\\n\\"\\n         \\"            self.tkx_top_ks_cpu + other.tkx_top_ks_cpu\\\\n\\"\\n         \\"            if self.tkx_top_ks_cpu is not None and other.tkx_top_ks_cpu is not None\\\\n\\"\\n         \\"            else None\\\\n\\"\\n         \\"        )\\\\n\\"\\n         \\"        self.is_all_greedy &= other.is_all_greedy\\\\n\\"),\\n    ],\\n}\\n\\n\\ndef main():\\n    if len(sys.argv) < 2 or sys.argv[1].startswith(\\"-\\"):\\n        sys.exit(__doc__)\\n    root = Path(sys.argv[1])\\n    texts = {rel: (root / rel).read_text() for rel in FILES}\\n    marked = [rel for rel, s in texts.items() if MARK in s]\\n    if len(marked) == len(FILES):\\n        print(\\"topklogits_patch: already patched\\")\\n        return\\n    if marked:\\n        sys.exit(f\\"topklogits_patch: PARTIALLY patched ({\', \'.join(marked)}); restore the stock files first\\")\\n    if \\"--check\\" in sys.argv:\\n        sys.exit(\\"topklogits_patch: NOT patched\\")\\n    for rel, edits in FILES.items():\\n        for old, _new in edits:\\n            n = texts[rel].count(old)\\n            if n != 1:\\n                sys.exit(f\\"topklogits_patch: anchor x{n} (need 1) in {rel}: {old.strip()[:70]!r}; refusing to patch\\")\\n    for rel, edits in FILES.items():\\n        s = texts[rel]\\n        for old, new in edits:\\n            s = s.replace(old, new)\\n        texts[rel] = s\\n    for rel, s in texts.items():\\n        (root / rel).write_text(s)\\n        print(\\"topklogits_patch: patched\\", root / rel)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')

      def run(cmd, *, check=True, env=None, timeout=None):
          cmd = list(map(str, cmd))
          print("$", shlex.join(cmd))
          result = subprocess.run(cmd, text=True, capture_output=True, env=env, timeout=timeout)
          if result.stdout.strip(): print(result.stdout.strip()[-6000:])
          if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
          if check and result.returncode:
              raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
          return result

      def find_unique(root, name, *, directory=False, required=True):
          root = Path(root)
          direct = root if root.name == name else root / name
          valid = lambda p: p.is_dir() if directory else p.is_file()
          if valid(direct): return direct
          hits = sorted(p for p in root.rglob(name) if valid(p))
          if len(hits) > 1:
              raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
          if hits: return hits[0]
          if required: raise FileNotFoundError(f"{name} not found under {root}")
          return None

      def sha256(path):
          h = hashlib.sha256()
          with Path(path).open("rb") as f:
              for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
          return h.hexdigest()

      # v111 (boot-and-misc L2): the v105c PLE local copy starts HERE, alongside the offline install, instead of after the model
      # view is built (v109: install + checks 130 s, then a 140 s copy wait). Same 10 files, same readers, same destination;
      # nothing is relinked here. The model cell adopts the finished copy (size + header check, then relinks its view) and the
      # boot cell still waits for it before rung 0. Any problem here -> PLE_EARLY stays None / ok False -> the model cell
      # copies again exactly as v108 does.
      PLE_EARLY, PLE_EARLY_STATE = None, {"bytes": 0, "total": 0, "ok": None, "names": [], "src": {}, "t_start": time.time(),
                                          "t_done": None}
      if (CFG.get("PLE_LOCAL") or {}).get("EARLY") and CFG["PLE_MODE"] in ("auto", "B"):
          try:
              _ple_rec = find("ple-fp8-receipt.json", required=False)      # = the model cell's PLE_DIR rule
              _ple_dir = _ple_rec.parent if _ple_rec else next((p.parent for p in find("model-plefp8-00000.safetensors", many=True)), None)
              if _ple_dir is not None:
                  _ple_pw = json.loads((_ple_dir / "model.safetensors.index.json").read_text())["weight_map"]
                  PLE_EARLY_STATE["names"] = sorted({v for v in _ple_pw.values() if v.startswith("model-plefp8-")})
                  PLE_EARLY_STATE["src"] = {f: str((_ple_dir / f).resolve()) for f in PLE_EARLY_STATE["names"]}

                  def _ple_early_copy(st=PLE_EARLY_STATE, dst=Path(CFG["PLE_LOCAL"]["DIR"]), threads=CFG["PLE_LOCAL"]["THREADS"],
                                      chunk=32 << 20):
                      t, lock = time.time(), threading.Lock()
                      try:
                          dst.mkdir(parents=True, exist_ok=True)
                          jobs = []
                          for f in st["names"]:
                              n = os.path.getsize(st["src"][f])
                              with open(dst / f, "wb") as fh:
                                  fh.truncate(n)
                              jobs += [(st["src"][f], dst / f, o, min(chunk, n - o)) for o in range(0, n, chunk)]
                              st["total"] += n

                          def _one(job):
                              s, d, o, ln = job
                              fi, fo = os.open(s, os.O_RDONLY), os.open(d, os.O_WRONLY)
                              try:
                                  got = 0
                                  while got < ln:
                                      b = os.pread(fi, min(ln - got, 8 << 20), o + got)
                                      if not b:
                                          raise IOError(f"short read {s} at {o + got}")
                                      os.pwrite(fo, b, o + got)
                                      got += len(b)
                              finally:
                                  os.close(fi); os.close(fo)
                              with lock:
                                  st["bytes"] += ln
                          with ThreadPoolExecutor(max_workers=threads) as pool:
                              list(pool.map(_one, jobs))
                          st["t_done"] = time.time()
                          st["ok"] = True
                          dt = time.time() - t
                          print(f"PLE early copy: {len(st['names'])} files {st['total']/1e9:.1f} GB in {dt:.0f} s "
                                f"({st['total']/1e6/max(dt, 1e-6):.0f} MB/s, {threads} readers) at +{(time.time()-T0)/60:.1f} min",
                                flush=True)
                      except Exception as ex:
                          st["ok"] = False
                          print(f"PLE early copy FAILED after {time.time()-t:.0f} s ({type(ex).__name__}: {ex}); the model cell "
                                f"copies again", flush=True)
                  _ple_thr = threading.Thread(target=_ple_early_copy, daemon=True)
                  _ple_thr.start()
                  PLE_EARLY = _ple_thr                       # set only once the thread runs (the model cell joins it)
                  print(f"PLE early copy started with the install: {len(PLE_EARLY_STATE['names'])} files from {_ple_dir} -> "
                        f"{CFG['PLE_LOCAL']['DIR']} ({CFG['PLE_LOCAL']['THREADS']} readers) at +{(time.time()-T0)/60:.1f} min",
                        flush=True)
              else:
                  print("PLE early copy: no FP8 PLE dataset found (the model cell decides option A/B as before)", flush=True)
          except Exception as _ple_ex:                       # PLE_EARLY stays None unless the copy thread already runs
              print(f"PLE early copy not started ({type(_ple_ex).__name__}: {_ple_ex}); the model cell copies as before", flush=True)

      WHEELHOUSE_DIR = next((p.parent for p in find("requirements.lock", many=True) if (p.parent / "wheels").is_dir()), None)
      assert WHEELHOUSE_DIR, f"Pennyroyal wheelhouse (requirements.lock + wheels/) not found under {INP}"
      print("wheelhouse:", WHEELHOUSE_DIR, flush=True)
      precache(str(WHEELHOUSE_DIR))
      Path(PREFIX).mkdir(parents=True, exist_ok=True)
      WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
      LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
      sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
      assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
      wheel = sglang_wheels[0]
      install_marker = Path(PREFIX) / "installed-bundle.txt"

      install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
      install_env.pop("PYTHONPATH", None)
      t_inst = time.time()
      if not Path(SGLANG).exists() or not install_marker.is_file():
          print("=== installing Pennyroyal (offline) ===")
          uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
          assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
          uv_binary = Path(PREFIX) / "bin/uv"
          uv_binary.parent.mkdir(parents=True, exist_ok=True)
          with zipfile.ZipFile(uv_wheels[0]) as archive:
              members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
              assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
              binary = archive.read(members[0])
              assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
              uv_binary.write_bytes(binary)
          uv_binary.chmod(0o755)
          uv = [str(uv_binary)]
          if not Path(PYTHON).exists():
              run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
          install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
          run(install + ["-r", str(LOCK)], env=install_env)
          run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
          install_marker.write_text('install')
      else:
          print(f"Matching offline bundle already installed at {VENV}")
      print(f"install {time.time()-t_inst:.0f}s", flush=True)

      # ---- bundled CUDA toolkit and linker symlinks (verbatim) ----
      cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
      if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
          raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
      CUDA_HOME = str(cu13[0])
      SITE = cu13[0].parent.parent
      libdir = Path(CUDA_HOME) / "lib"
      lib64 = Path(CUDA_HOME) / "lib64"
      if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
      for so in sorted(libdir.glob("*.so.*")):
          link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
          if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
      assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
      if not (libdir / "libcuda.so").exists():
          candidates = []
          for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
              candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
          if not candidates:
              code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
              maps = run([PYTHON, "-c", code], env=install_env).stdout
              candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
          if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
          link = libdir / "libcuda.so"
          if link.is_symlink(): link.unlink()
          link.symlink_to(candidates[0].resolve())
      CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
      assert CXX, "Host C++ compiler not found"
      CC = CXX.replace("g++", "gcc")
      assert shutil.which(CC), f"Matching C compiler missing: {CC}"

      # ---- runtime environment: sm_89, no SM120 knobs ----
      env = dict(os.environ)
      env.pop("PYTHONPATH", None)
      cache = f"{PREFIX}/cache"
      for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit", "tilelang"):
          Path(cache, name).mkdir(parents=True, exist_ok=True)
      env.update({
          "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
          "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
          "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
          "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
          "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
          "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
          "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
          "TORCH_CUDA_ARCH_LIST": "8.9",
          "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
          "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
          "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
          "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer", "TILELANG_CACHE_DIR": f"{cache}/tilelang",
          "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
          "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
          "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
          "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
          "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
          "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
          "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
          "SGLANG_QSA_SM89_DECODE": "triton",
      })
      for k in [k for k in env if k.startswith("SGLANG_SM120") or k == "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM"]:
          env.pop(k)
      run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
      print("Host compiler:", CXX)
      GPU_CHECK = "; ".join([
          "import sglang, torch",
          "print('sglang', sglang.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)",
          "assert torch.cuda.is_available(), 'CUDA unavailable'",
          "print('GPU:', torch.cuda.get_device_name(0), '| count', torch.cuda.device_count(), '| capability', torch.cuda.get_device_capability(0))",
          "print('CAP=%d.%d COUNT=%d' % (*torch.cuda.get_device_capability(0), torch.cuda.device_count()))"])
      gpu = run([PYTHON, "-c", GPU_CHECK], env=env).stdout
      m = re.search(r"CAP=(\d+)\.(\d+) COUNT=(\d+)", gpu)
      assert m and int(m.group(3)) >= CFG["TP"], f"need {CFG['TP']} GPUs: {gpu[-300:]}"
      if (int(m.group(1)), int(m.group(2))) != (8, 9):
          print(f"WARNING: compute capability {m.group(1)}.{m.group(2)}, this notebook is tuned for L4 (8.9)", flush=True)
      print(sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1"), "driver | wheelhouse is cu130", flush=True)

      PATCH_DIR = Path("/tmp/duck2-patches"); PATCH_DIR.mkdir(parents=True, exist_ok=True)
      # ---- driver / PTX smoke on GPU 0 (~1 min): a mismatch fails here instead of after a 55-min rung ----
      DRIVER_CHECK = """
      import time, torch
      t = time.time()
      a = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16)
      assert torch.isfinite((a @ a).float()).all().item(); torch.cuda.synchronize()
      print("MATMUL_OK", flush=True)
      try:
          import triton, triton.language as tl
          @triton.jit
          def _double(x, y, n, B: tl.constexpr):
              o = tl.program_id(0) * B + tl.arange(0, B)
              m = o < n
              tl.store(y + o, tl.load(x + o, mask=m) * 2, mask=m)
          x = torch.arange(1000, device="cuda", dtype=torch.float32); y = torch.empty_like(x)
          _double[(4,)](x, y, 1000, B=256); torch.cuda.synchronize()
          assert torch.equal(y, x * 2)
          print("TRITON_OK", triton.__version__, flush=True)
      except Exception as e:
          print("TRITON_FAIL", type(e).__name__, str(e)[:800], flush=True)
      try:
          import flashinfer, flashinfer.sampling as fs
          p = torch.softmax(torch.randn(4, 32000, device="cuda"), -1)
          s = fs.sampling_from_probs(p); torch.cuda.synchronize()
          assert s.shape[0] == 4
          print("FLASHINFER_OK", flashinfer.__version__, flush=True)
      except Exception as e:
          print("FLASHINFER_FAIL", type(e).__name__, str(e)[:800], flush=True)
      print("DRIVER_CHECK_DONE %.1fs" % (time.time() - t), flush=True)
      """
      if CFG["DRIVER_TEST"]:
          import textwrap   # the stage guard indents this cell, string contents included
          (PATCH_DIR / "driver_check.py").write_text(textwrap.dedent(DRIVER_CHECK))
          try:
              r = run([PYTHON, PATCH_DIR / "driver_check.py"], env=dict(env, CUDA_VISIBLE_DEVICES="0"), check=False, timeout=900)
              out = r.stdout + r.stderr
          except subprocess.TimeoutExpired:
              out = "DRIVER_CHECK timed out"
          print("driver check:", " ".join(t for t in ("MATMUL_OK", "TRITON_OK", "TRITON_FAIL", "FLASHINFER_OK", "FLASHINFER_FAIL")
                                          if t in out), flush=True)
          if "MATMUL_OK" not in out:
              print(out[-3000:], flush=True)
              raise RuntimeError("torch bf16 matmul failed on GPU 0 (driver / CUDA 13 runtime mismatch?)")
          if "TRITON_OK" not in out or "FLASHINFER_OK" not in out:
              print("!!! WARNING: Triton or FlashInfer failed on this driver; the serve rungs will likely fail too\n"
                    + out[-3000:], flush=True)

      # ---- duck2 patches: sm_89 QSA sparse decode (required), scaled FP8 PLE cast (env-gated, option A only) ----
      for n, src in PATCHES.items():
          (PATCH_DIR / n).write_text(src)
      run([PYTHON, PATCH_DIR / "apply_patches.py", SITE, "--ple-scale"], env=env)
      run([PYTHON, PATCH_DIR / "apply_patches.py", SITE, "--ple-scale", "--check"], env=env)
      run([PYTHON, PATCH_DIR / "prune_patch.py", SITE], env=env)
      run([PYTHON, PATCH_DIR / "prune_patch.py", SITE, "--check"], env=env)
      run([PYTHON, PATCH_DIR / "draft_ep_patch.py", SITE], env=env)
      run([PYTHON, PATCH_DIR / "draft_ep_patch.py", SITE, "--check"], env=env)
      run([PYTHON, PATCH_DIR / "marlin_ep_patch.py", SITE], env=env)
      run([PYTHON, PATCH_DIR / "marlin_ep_patch.py", SITE, "--check"], env=env)
      run([PYTHON, PATCH_DIR / "think_force_patch.py", SITE], env=env)
      run([PYTHON, PATCH_DIR / "think_force_patch.py", SITE, "--check"], env=env)
      PERF_APPLIED = [k for k in ['hcmix32', 'qsasplitk', 'mqagrid', 'topklogits', 'marlinclean', 'densefp8', 'idxshard', 'hisparse', 'pairown', 'embedhost', 'mambaratio', 'mambafinal', 'hicpairown', 'hicqsawait', 'hicmambafb']   # non-fatal: a refusing perf patch is just skipped
                      if run([PYTHON, PATCH_DIR / f'{k}_patch.py', SITE], env=env, check=False).returncode == 0
                      and run([PYTHON, PATCH_DIR / f'{k}_patch.py', SITE, '--check'], env=env, check=False).returncode == 0]
      print('perf patches applied (dormant):', PERF_APPLIED, flush=True)
      if not {'hicpairown', 'hicqsawait', 'hicmambafb'} <= set(PERF_APPLIED):
          _hic = [r['NAME'] for r in CFG['RUNGS'] if '--enable-hierarchical-cache' in r.get('SERVE_ARGS', [])]
          if _hic and len(_hic) < len(CFG['RUNGS']):
              CFG['RUNGS'] = [r for r in CFG['RUNGS'] if r['NAME'] not in _hic]
              print('hicpairown/hicqsawait/hicmambafb patch NOT applied: dropped HiCache rungs', _hic, flush=True)
      QSA_DEFAULT = "triton"
      if CFG["QSA_GPU_TEST"]:
          te = dict(env, CUDA_VISIBLE_DEVICES="0")
          try:
              r = run([PYTHON, PATCH_DIR / "test_qsa_sm89_decode.py"], env=te, check=False, timeout=900)
              ok = r.returncode == 0 and "ALL OK" in r.stdout
              print("QSA sm_89 GPU unit test:", "PASS" if ok else f"FAIL rc={r.returncode}", flush=True)
              if not ok:
                  print((r.stderr or "")[-3000:])
                  QSA_DEFAULT = "torch"
                  print("-> QSA decode falls back to the torch gather path (SGLANG_QSA_SM89_DECODE=torch)", flush=True)
          except subprocess.TimeoutExpired:
              QSA_DEFAULT = "torch"; print("QSA GPU test timed out -> torch path", flush=True)
      print(f"install cell done at +{(time.time()-T0)/60:.1f} min", flush=True)

      if CFG.get("QSADUMP") and RUN_MODE != "full":     # dormant unless SGLANG_QSA_DUMP_DIR is set in the rung env
          run([PYTHON, PATCH_DIR / "qsadump_patch.py", SITE], env=env)
          run([PYTHON, PATCH_DIR / "qsadump_patch.py", SITE, "--check"], env=env)
          print("qsadump patch applied", flush=True)
  except Exception as _ex:
    stage_failed('install', _ex, fatal=True)


[sys] RAM 3.1/188.7 | swap none | proc 0.1 GB, 57 threads, 55 fds | GPU0 0.0/22.5 GPU1 0.0/22.5 GPU2 0.0/22.5 GPU3 0.0/22.5 | disk / 7108.1/8156.9, /tmp 7108.1/8156.9, /kaggle/working 0.0/19.5
skipping stage install: earlier stage failed: ['quick_version']


In [5]:
# [duck2:inproj]  11-01 TPS build: GDN in_proj fix (patch_inproj.py; SGLANG_INPROJ_FP8/NODUP) and one-pass FP8 dequant for
# prefill-size rowwise-FP8 linears (patch_fp8dequant.py; SGLANG_FP8_FUSED_DEQUANT=1, bit-identical); inert without the env
if FAILED:
    print('skipping stage inproj: earlier stage failed:', FAILED, flush=True)
else:
  try:
      for _fn, _src in (("patch_inproj.py", '#!/usr/bin/env python3\n"""patch_inproj.py (10-31): GDN in_proj fix for Pennyroyal SGLang on sm89 (qwen3_5.py finalize_fused_in_proj).\nfinalize_fused_in_proj stacks in_proj_qkvz + in_proj_ba into one bf16 buffer BEFORE the loader\'s sm89 rowwise-FP8 pass;\nthe FP8 pass then converts in_proj_qkvz into a second, unused FP8 copy (decode/prefill <= 1024 rows use the bf16 buffer).\n  SGLANG_INPROJ_NODUP=1 (lossless): after fusing, in_proj_qkvz keeps a plain UnquantizedLinearMethod -> no FP8 copy\n                                    (~10.5 MB/GDN layer/GPU freed for KV); every row count runs the same bf16 weights.\n  SGLANG_INPROJ_FP8=1  (numerics change): do not fuse; in_proj_qkvz runs the rowwise-FP8 path like the other dense\n                                    projections (in_proj_ba, too small to convert, stays bf16); frees the bf16 buffer.\nUnset: byte-identical behaviour. Anchored edits, idempotent.  Usage: patch_inproj.py <site-packages>"""\nimport sys\nfrom pathlib import Path\np = Path(sys.argv[1]) / "sglang/srt/models/qwen3_5.py"\ns = p.read_text()\nMARK = "# [duck2:inproj]"\nif MARK in s:\n    print("already patched"); sys.exit(0)\na1 = "        if not _is_cuda or self._fused_in_proj_weight is not None:\\n            return\\n"\na2 = "        self._fused_in_proj_weight = fused\\n"\nassert s.count(a1) == 1 and s.count(a2) == 1, "anchor mismatch: different SGLang build"\ns = s.replace(a1, a1 + f"        {MARK} SGLANG_INPROJ_FP8=1: keep in_proj_qkvz on the rowwise-FP8 path (no bf16 fused buffer)\\n"\n              "        if os.environ.get(\\"SGLANG_INPROJ_FP8\\", \\"0\\") == \\"1\\":\\n            return\\n")\ns = s.replace(a2, a2 + f"        {MARK} SGLANG_INPROJ_NODUP=1: the fused bf16 buffer serves every GEMM; skip the later FP8 copy of qkvz\\n"\n              "        if os.environ.get(\\"SGLANG_INPROJ_NODUP\\", \\"0\\") == \\"1\\" and type(qkvz.quant_method) is not UnquantizedLinearMethod:\\n"\n              "            qkvz.quant_method = UnquantizedLinearMethod()\\n")\ncompile(s, str(p), "exec")\np.write_text(s)\nprint("patched", p)\n'), ("patch_fp8dequant.py", '#!/usr/bin/env python3\n"""patch_fp8dequant.py (11-01): prefill-path fix for Pennyroyal SGLang\'s sm89 rowwise-FP8 linears\n(sglang/kernels/ops/gemm/sm120_online_fp8.py, sm89_rowwise_fp8_linear). Above _MAX_KERNEL_ROWS rows (prefill chunks) it\ndequantizes the weight block by block as (w.float() * scale).to(bf16) -- three elementwise kernels through an fp32 copy --\nruns one GEMM per block and copies each block into the output: ~17% of a 1024-token prefill chunk on 4xL4.\nSGLANG_FP8_FUSED_DEQUANT=1: one Triton kernel reads the FP8 weight once and writes the bf16 weight with the same fp32\narithmetic (fp8 -> fp32 exact, * fp32 row scale, round-to-nearest-even to bf16: bit-identical values), then one GEMM over\nthe whole weight. Unset: byte-identical behaviour. Anchored, idempotent. Usage: patch_fp8dequant.py <site-packages>"""\nimport sys\nfrom pathlib import Path\np = Path(sys.argv[1]) / "sglang/kernels/ops/gemm/sm120_online_fp8.py"\ns = p.read_text()\nMARK = "# [duck2:fp8dequant]"\nif MARK in s:\n    print("already patched"); sys.exit(0)\nold = \'\'\'    else:\n        out = torch.empty((rows, n), dtype=torch.bfloat16, device=x.device)\n        block_rows = max(1, min(8192, _DEQUANT_TARGET_BYTES // max(1, weight.shape[1] * 4)))\n\'\'\'\nassert s.count(old) == 1, "anchor mismatch: different SGLang build"\ns = s.replace(old, f\'\'\'    elif _duck2_fused_dequant_ok(weight):   {MARK} one-pass FP8 -> bf16 weight, then one GEMM\n        out = x2 @ _duck2_dequant_rowwise(weight, scale).T\n\'\'\' + old, 1)\nanchor = "def sm89_rowwise_fp8_linear(x, weight, bias=None):"\nassert s.count(anchor) == 1\ns = s.replace(anchor, f\'\'\'{MARK} SGLANG_FP8_FUSED_DEQUANT=1: prefill-size rowwise-FP8 linears dequantize in one pass (see patch_fp8dequant.py)\nimport os as _duck2_os\n\n\n@triton.jit\ndef _duck2_dequant_rowwise_kernel(w_ptr, s_ptr, o_ptr, K, BLOCK_K: tl.constexpr):\n    row = tl.program_id(0)\n    cols = tl.program_id(1) * BLOCK_K + tl.arange(0, BLOCK_K)\n    mask = cols < K\n    w = tl.load(w_ptr + row * K + cols, mask=mask, other=0.0).to(tl.float32)\n    sc = tl.load(s_ptr + row)\n    tl.store(o_ptr + row * K + cols, (w * sc).to(tl.bfloat16), mask=mask)\n\n\ndef _duck2_fused_dequant_ok(weight):\n    return (_duck2_os.environ.get("SGLANG_FP8_FUSED_DEQUANT", "0") == "1" and weight.is_contiguous()\n            and weight.dim() == 2 and weight.dtype == torch.float8_e4m3fn)\n\n\ndef _duck2_dequant_rowwise(weight, scale):\n    n, k = weight.shape\n    out = torch.empty((n, k), dtype=torch.bfloat16, device=weight.device)\n    _duck2_dequant_rowwise_kernel[(n, triton.cdiv(k, 1024))](weight, scale.contiguous(), out, k, BLOCK_K=1024,\n                                                             num_warps=4)\n    return out\n\n\n\'\'\' + anchor, 1)\ncompile(s, str(p), "exec")\np.write_text(s)\nprint("patched", p)\n')):
          (PATCH_DIR / _fn).write_text(_src)
          print(run([PYTHON, str(PATCH_DIR / _fn), str(SITE)], env=env).stdout[-300:], flush=True)
  except Exception as _ex:
      stage_failed("inproj", _ex)


skipping stage inproj: earlier stage failed: ['quick_version']


In [6]:
# [duck2:model]  target view (FP8 PLE), draft view, model precache
if FAILED:
    print('skipping stage model: earlier stage failed:', FAILED, flush=True)
else:
  try:
      import struct
      EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
      DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"
      TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
      TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"

      def indexed_shards(root):
          root = Path(root)
          index = json.loads((root / "model.safetensors.index.json").read_text())
          names = sorted(set(index["weight_map"].values()))
          if not names: raise RuntimeError(f"Empty weight index: {root}")
          for name in names:
              rel = Path(name)
              if rel.is_absolute() or ".." in rel.parts:
                  raise RuntimeError(f"Invalid shard path: {name}")
              p = root / rel
              if not p.is_file() or p.stat().st_size < 8:
                  raise RuntimeError(f"Missing/empty shard: {p}")
          return index, names

      def prepare_draft_view(target, source_root, work_root):
          """Franzen, verbatim: adapt metadata only; link original dataset weights without copying them."""
          target, source_root, work_root = map(Path, (target, source_root, work_root))
          configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
          configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
          candidates = []
          for path in configs:
              cfg = json.loads(path.read_text())
              q = cfg.get("quantization_config", {})
              if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
                  candidates.append((path.parent, cfg))
          if len(candidates) != 1:
              raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
          source, cfg = candidates[0]
          index, shards = indexed_shards(source)
          if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
              raise RuntimeError("Draft index does not contain the expected MTP experts")
          cfg = copy.deepcopy(cfg)
          q = cfg["quantization_config"]
          group = q["config_groups"]["mtp_routed_experts"]
          w = group["weights"]
          if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
              raise RuntimeError("Expected symmetric INT4 group32 draft experts")
          if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
              raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
          if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
              raise RuntimeError("Unexpected draft ignore rules")
          group["targets"] = [EXPERT_TARGET]
          q["ignore"] = [DENSE_IGNORE]
          identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
          suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
          view = work_root / f"draft-view-{suffix}"
          view.mkdir(parents=True, exist_ok=True)
          config_path = view / "config.json"
          config_text = json.dumps(cfg, indent=2) + "\n"
          if config_path.exists() or config_path.is_symlink():
              if config_path.is_symlink() or config_path.read_text() != config_text:
                  raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
          else:
              config_path.write_text(config_text)
          links = {name: source / name for name in shards}
          links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
          for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
              if (target / name).is_file(): links[name] = target / name
          for name, src in links.items():
              dest = view / name
              dest.parent.mkdir(parents=True, exist_ok=True)
              if dest.exists() or dest.is_symlink():
                  if not dest.is_symlink() or dest.resolve() != src.resolve():
                      raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
              else:
                  dest.symlink_to(src.resolve())
          print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
          return view

      def st_header(p):
          with open(p, "rb") as f:
              n = struct.unpack("<Q", f.read(8))[0]
              return json.loads(f.read(n))

      is_ple = lambda k: "ple_embedding.ngram_embedding" in k

      def build_fp8ple_view(intel, ple, view):
          """Intel W4A16 non-PLE shards + RadixArk FP8 PLE shards (= albucino W4A16-FP8PLE), symlinks only."""
          iw = json.loads((intel / "model.safetensors.index.json").read_text())["weight_map"]
          pw = json.loads((ple / "model.safetensors.index.json").read_text())["weight_map"]
          cfg = json.loads((intel / "config.json").read_text())
          parts = int(cfg.get("text_config", {}).get("split_ngram_parts", cfg.get("split_ngram_parts", 512)))
          ple_bf16 = {k for k in iw if is_ple(k)}
          # the safetensors iterator reads every tensor of a linked file, so BF16 PLE files are dropped BY FILE NAME
          ple_files = {v for v in iw.values() if v.startswith("model-ple-")}
          assert ple_files, "no model-ple-* files in the Intel index"
          assert all(iw[k] in ple_files for k in ple_bf16), "BF16 PLE tensors outside model-ple-* files"
          assert all(is_ple(k) for k, v in iw.items() if v in ple_files), "non-PLE tensors inside model-ple-* files"
          extra_mtp_only = {f for f in set(iw.values()) if f == "model_extra_tensors.safetensors"
                            and all(k.startswith("mtp.") for k, v in iw.items() if v == f)}   # BF16 MTP: drafter replaces it
          tgt = {k: v for k, v in iw.items() if v not in ple_files and v not in extra_mtp_only}
          fp8 = {k: v for k, v in pw.items() if v.startswith("model-plefp8-")}
          assert len(fp8) == parts + 1 and all(is_ple(k) for k in fp8), f"FP8 PLE index: {len(fp8)} entries, {parts} parts"
          shard_keys = {k for k in ple_bf16 if ".shard_" in k}
          assert len(shard_keys) == parts, f"{len(shard_keys)} BF16 PLE shards != split_ngram_parts {parts}"
          assert shard_keys == {k for k in fp8 if ".shard_" in k}, "PLE shard names differ"
          assert any(k.endswith("ngram_embedding.weight_scale") for k in fp8)
          assert not set(tgt) & set(fp8)
          rec_path = ple / "ple-fp8-receipt.json"
          receipt = json.loads(rec_path.read_text())["files"] if rec_path.is_file() else {}
          hdr = {}
          for f in sorted(set(fp8.values())):            # header + size checks (no 51 GB sha pass)
              if f in receipt:
                  assert (ple / f).stat().st_size == receipt[f]["bytes"], f
              h = st_header(ple / f)
              assert all(k in h for k, v in fp8.items() if v == f), f
              hdr.update({k: h[k] for k, v in fp8.items() if v == f})
          ihdr = {}
          for f in sorted(ple_files):
              h = st_header(intel / f)
              ihdr.update({k: h[k] for k in shard_keys if iw[k] == f})
          for k in shard_keys:      # the loader places shard N at N x ceil(vocab / parts): rows must match exactly
              assert hdr[k]["dtype"] == "F8_E4M3", (k, hdr[k]["dtype"])
              assert hdr[k]["shape"] == ihdr[k]["shape"], (k, hdr[k]["shape"], ihdr[k]["shape"])
          view.mkdir(parents=True, exist_ok=True)
          links = {f: intel / f for f in set(tgt.values())} | {f: ple / f for f in set(fp8.values())}
          assert not any(f.startswith("model-ple-") for f in links), "a BF16 PLE file would be linked"
          for p in intel.iterdir():                      # tokenizer, chat template, processor, quantization_config ...
              if p.is_file() and p.suffix != ".safetensors" and p.name not in ("config.json", "model.safetensors.index.json"):
                  links[p.name] = p
          for name, src in sorted(links.items()):
              d = view / name
              if d.is_symlink() or d.exists():
                  assert d.resolve() == src.resolve(), d
              else:
                  d.symlink_to(src)
          cfg.setdefault("text_config", {})["ple_embedding_dtype"] = "float8_e4m3fn"   # mandatory with --ple-offload-embedding
          (view / "config.json").write_text(json.dumps(cfg, indent=2) + "\n")
          total = sum((view / f).stat().st_size for f in set(tgt.values()) | set(fp8.values()))
          (view / "model.safetensors.index.json").write_text(json.dumps({"metadata": {"total_size": total}, "weight_map": {**tgt, **fp8}}))
          print(f"FP8-PLE view {view}: {len(tgt)}+{len(fp8)} tensors, {len(set(tgt.values()))}+{len(set(fp8.values()))} shards, "
                f"{total/1e9:.1f} GB", flush=True)
          return view

      INTEL_DIR = next((p.parent for p in find("model.safetensors.index.json", many=True)
                        if "w4a16" in str(p).lower() and any(p.parent.glob("model-ple-*.safetensors"))), None)
      assert INTEL_DIR, f"Intel W4A16 AutoRound checkpoint not found under {INP}"
      DRAFT_MODEL_DIR = find("mtp-routed-experts-int4.safetensors").parent
      _rec = find("ple-fp8-receipt.json", required=False)
      PLE_DIR = _rec.parent if _rec else next((p.parent for p in find("model-plefp8-00000.safetensors", many=True)), None)
      PLE_OPTION = "B" if (CFG["PLE_MODE"] in ("auto", "B") and PLE_DIR is not None) else "A"
      print("INTEL", INTEL_DIR, "\nDRAFT", DRAFT_MODEL_DIR, "\nPLE  ", PLE_DIR, "| option", PLE_OPTION, flush=True)
      target_cfg = json.loads((INTEL_DIR / "config.json").read_text())
      q = target_cfg.get("quantization_config", {})
      assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "INTEL_DIR must be the AutoRound INT4 checkpoint"
      EXTRA_SERVE_ARGS, PLE_ENV = [], {}
      if PLE_OPTION == "B":
          try:
              MODEL_DIR = str(build_fp8ple_view(INTEL_DIR, PLE_DIR, Path("/tmp/flashnext-w4a16-fp8ple")))
          except Exception as ex:
              print(f"FP8-PLE view failed ({type(ex).__name__}: {ex}) -> option A (Intel BF16 PLE + scaled FP8 cast)", flush=True)
              PLE_OPTION = "A"
      if PLE_OPTION == "A":
          # Intel BF16 PLE cast to FP8 at load with one global scale (patch ple_fp8_scale); 47.7 GiB pinned instead of 95.4
          MODEL_DIR = str(INTEL_DIR)
          EXTRA_SERVE_ARGS = ["--json-model-override-args", '{"text_config":{"ple_embedding_dtype":"float8_e4m3fn"}}']
          iw = json.loads((INTEL_DIR / "model.safetensors.index.json").read_text())["weight_map"]
          k0 = next(k for k in sorted(iw) if is_ple(k) and re.search(r"\.shard_0\.", k))
          r = run([PYTHON, "-c", f"from safetensors import safe_open; f = safe_open({str(INTEL_DIR / iw[k0])!r}, 'pt'); "
                               f"t = f.get_tensor({k0!r}).float(); print('PLE_STATS', t.abs().max().item(), t.std().item())"], env=env)
          amax, std = map(float, re.search(r"PLE_STATS (\S+) (\S+)", r.stdout).groups())
          PLE_ENV = {"SGLANG_PLE_FP8_SCALE": f"{2 * amax / 448:.6g}"}
          print(f"option A: shard_0 amax {amax:.4g} std {std:.4g} -> SGLANG_PLE_FP8_SCALE={PLE_ENV['SGLANG_PLE_FP8_SCALE']}", flush=True)
      _, target_shards = indexed_shards(MODEL_DIR)
      print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype: float8_e4m3fn", flush=True)
      DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)
      TOKEN_MAP = None
      if CFG["FRSPEC"]:
          tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
          if tok is not None and sha256(tok) == TOKEN_MAP_SHA and sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA:
              TOKEN_MAP = str(tok)
          print("FR-Spec token map:", TOKEN_MAP or "OFF (missing or sha mismatch)", flush=True)
      else:
          print("FR-Spec token map: OFF (CFG FRSPEC=False on L4)", flush=True)
      if CFG["PRECACHE_MODEL"]:   # core shards + drafter only: PLE shards would evict them from page cache
          core = sorted({str(Path(MODEL_DIR, f).resolve()) for f in target_shards if "plefp8" not in f and not f.startswith("model-ple-")})
          precache_model_thread = threading.Thread(target=precache, args=(*core, str(DRAFT_MODEL_DIR)),
                                                   kwargs={"delay": 30, "threads": 3}, daemon=True)
          precache_model_thread.start()

      PLE_COPY, PLE_COPY_STATE = None, {"bytes": 0, "total": 0, "ok": None}
      if CFG.get("PLE_LOCAL") and PLE_OPTION == "B":
          # v105c: on 10-19/20 the PLE dataset mount read at 7-32 MB/s uncached (390-725 MB/s on earlier boots), and page cache
          # cannot hold core + PLE once the server pins its tables, so every rung re-read it and timed out. The 10 PLE files are
          # copied to local disk by parallel readers before rung 0 starts; the view then links the local copies (also used by
          # crash restarts). On any error the view keeps the dataset links.
          # v111: the copy itself was started by the install cell (PLE_EARLY); it is adopted only if it covers exactly this
          # view's PLE files from the same resolved sources, else (or if it failed) the files are copied again here as in v108.
          PLE_LOCAL_DIR = Path(CFG["PLE_LOCAL"]["DIR"])
          _ple_names = sorted(f for f in set(target_shards) if f.startswith("model-plefp8-"))
          _ple_early = globals().get("PLE_EARLY")
          _ple_adopt = (_ple_early is not None and PLE_EARLY_STATE["names"] == _ple_names
                        and all(PLE_EARLY_STATE["src"].get(f) == str(Path(MODEL_DIR, f).resolve()) for f in _ple_names))
          if _ple_adopt:
              PLE_COPY_STATE = PLE_EARLY_STATE          # the boot cell's wait line reads bytes/total from this dict

          def _ple_copy(names=_ple_names, threads=CFG["PLE_LOCAL"]["THREADS"], chunk=32 << 20):
              t, lock = time.time(), threading.Lock()
              try:
                  if _ple_early is not None:
                      _ple_early.join()                     # never two writers on PLE_LOCAL_DIR
                  src = {f: Path(MODEL_DIR, f).resolve() for f in names}
                  adopted = _ple_adopt and PLE_EARLY_STATE["ok"] is True
                  if not adopted:
                      if _ple_early is not None:
                          print(f"PLE early copy not usable (ok={PLE_EARLY_STATE['ok']}, same files={_ple_adopt}): copying again",
                                flush=True)
                      PLE_COPY_STATE.update(bytes=0, total=0)
                      PLE_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
                      jobs = []
                      for f in names:
                          n = src[f].stat().st_size
                          with open(PLE_LOCAL_DIR / f, "wb") as fh:
                              fh.truncate(n)
                          jobs += [(src[f], PLE_LOCAL_DIR / f, o, min(chunk, n - o)) for o in range(0, n, chunk)]
                          PLE_COPY_STATE["total"] += n

                      def _one(job):
                          s, d, o, ln = job
                          fi, fo = os.open(s, os.O_RDONLY), os.open(d, os.O_WRONLY)
                          try:
                              got = 0
                              while got < ln:
                                  b = os.pread(fi, min(ln - got, 8 << 20), o + got)
                                  if not b:
                                      raise IOError(f"short read {s} at {o + got}")
                                  os.pwrite(fo, b, o + got)
                                  got += len(b)
                          finally:
                              os.close(fi); os.close(fo)
                          with lock:
                              PLE_COPY_STATE["bytes"] += ln
                      with ThreadPoolExecutor(max_workers=threads) as pool:
                          list(pool.map(_one, jobs))
                  for f in names:     # sizes and headers must match before the view is retargeted
                      assert (PLE_LOCAL_DIR / f).stat().st_size == src[f].stat().st_size, f
                      assert st_header(PLE_LOCAL_DIR / f) == st_header(src[f]), f
                  for f in names:
                      tmp = Path(MODEL_DIR, f + ".local")
                      if tmp.is_symlink():
                          tmp.unlink()
                      tmp.symlink_to(PLE_LOCAL_DIR / f)
                      os.replace(tmp, Path(MODEL_DIR, f))
                  dt = time.time() - t
                  PLE_COPY_STATE["ok"] = True
                  if adopted:
                      _cp = PLE_EARLY_STATE["t_done"] - PLE_EARLY_STATE["t_start"]
                      print(f"PLE local copy (started with the install): {len(names)} files {PLE_COPY_STATE['total']/1e9:.1f} GB "
                            f"copied in {_cp:.0f} s ({PLE_COPY_STATE['total']/1e6/max(_cp, 1e-6):.0f} MB/s), adopted after a "
                            f"{dt:.0f} s wait in the model stage; view relinked to {PLE_LOCAL_DIR}", flush=True)
                  else:
                      print(f"PLE local copy: {len(names)} files {PLE_COPY_STATE['total']/1e9:.1f} GB in {dt:.0f} s "
                            f"({PLE_COPY_STATE['total']/1e6/max(dt, 1e-6):.0f} MB/s, {threads} readers); view relinked to {PLE_LOCAL_DIR}",
                            flush=True)
              except Exception as ex:
                  PLE_COPY_STATE["ok"] = False
                  print(f"PLE local copy FAILED after {time.time()-t:.0f} s ({type(ex).__name__}: {ex}); the view keeps the "
                        f"dataset links", flush=True)
          PLE_COPY = threading.Thread(target=_ple_copy, daemon=True)
          PLE_COPY.start()
          print(f"PLE local copy {'adopting the early copy' if _ple_adopt else 'started'}: {len(_ple_names)} files -> {PLE_LOCAL_DIR} "
                f"({CFG['PLE_LOCAL']['THREADS']} readers)", flush=True)

      if CFG.get("PRUNE_N"):
          KEEP_PATH = PATCH_DIR / "keep.json"
          KEEP_PATH.write_text('{"num_experts": 384, "source": "../../../colab/results_prof/analysis/counts.npz decode + 1.0*counts_code.npz decode + 0.5*counts_code.npz prefill (per-layer normalized)", "profiled_mass": null, "layers": [[0, 1, 3, 4, 5, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 42, 43, 44, 46, 48, 49, 50, 53, 57, 58, 59, 60, 61, 63, 65, 66, 67, 69, 70, 71, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 87, 88, 90, 92, 93, 95, 96, 98, 99, 101, 102, 103, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 123, 125, 126, 128, 130, 131, 132, 138, 139, 142, 143, 145, 147, 148, 149, 150, 151, 153, 154, 155, 156, 158, 159, 160, 161, 162, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 208, 209, 210, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 223, 224, 225, 227, 228, 230, 233, 234, 235, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 251, 252, 254, 256, 257, 258, 259, 260, 261, 262, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 275, 277, 278, 279, 280, 281, 282, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 301, 302, 303, 305, 306, 307, 308, 309, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 327, 329, 330, 332, 333, 335, 336, 337, 338, 339, 340, 342, 345, 346, 349, 350, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 366, 367, 368, 369, 371, 372, 375, 376, 378, 379, 380, 381, 382, 383, 384, 385, 389, 390, 391, 392, 393, 394, 395, 397, 399, 401, 403, 404, 406, 407, 408, 409, 410, 411, 412, 416, 418, 419, 420, 423, 426, 427, 428, 429, 430, 431, 432, 433, 435, 436, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 456, 457, 458, 461, 462, 463, 464, 467, 469, 470, 471, 472, 473, 474, 475, 476, 478, 480, 481, 482, 485, 486, 487, 489, 490, 491, 493, 494, 495, 496, 498, 499, 500, 501, 502, 504, 505, 506, 507, 508, 510], [0, 3, 4, 5, 6, 10, 12, 13, 14, 15, 16, 17, 18, 22, 23, 24, 27, 29, 30, 31, 32, 33, 34, 35, 36, 39, 40, 41, 43, 45, 46, 47, 48, 49, 52, 53, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 84, 85, 86, 87, 88, 90, 92, 94, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 112, 113, 115, 119, 120, 123, 125, 126, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 140, 141, 142, 143, 144, 145, 146, 147, 148, 150, 151, 152, 153, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 200, 202, 203, 204, 205, 206, 207, 208, 209, 211, 212, 213, 215, 216, 217, 218, 219, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 234, 235, 236, 237, 238, 239, 240, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 254, 257, 259, 260, 261, 262, 263, 264, 265, 267, 268, 269, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 320, 321, 323, 324, 325, 326, 328, 329, 331, 333, 334, 336, 338, 339, 340, 341, 342, 344, 345, 348, 350, 352, 353, 354, 355, 357, 362, 363, 364, 366, 368, 370, 372, 373, 375, 376, 378, 382, 383, 385, 386, 388, 389, 390, 391, 392, 394, 395, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 410, 414, 415, 417, 418, 420, 422, 423, 425, 426, 427, 430, 431, 432, 434, 435, 437, 438, 439, 440, 443, 444, 445, 446, 447, 449, 451, 452, 453, 454, 455, 456, 458, 461, 462, 463, 464, 465, 466, 468, 470, 471, 474, 475, 476, 477, 479, 481, 482, 483, 485, 487, 488, 489, 490, 491, 492, 493, 494, 496, 497, 498, 499, 500, 502, 504, 505, 508, 509], [0, 1, 2, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 32, 34, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 50, 51, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 69, 73, 74, 76, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 98, 100, 101, 103, 104, 105, 106, 107, 109, 110, 112, 113, 114, 115, 116, 118, 119, 122, 124, 125, 127, 128, 129, 130, 131, 132, 134, 135, 136, 137, 138, 139, 140, 142, 143, 146, 147, 148, 149, 150, 151, 153, 154, 155, 157, 159, 161, 162, 164, 165, 166, 168, 169, 171, 172, 173, 175, 176, 177, 180, 181, 183, 184, 185, 188, 189, 190, 191, 193, 195, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 210, 211, 212, 214, 215, 220, 222, 224, 225, 226, 227, 228, 229, 232, 236, 237, 238, 239, 240, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 260, 261, 262, 263, 264, 265, 266, 268, 270, 271, 272, 273, 276, 277, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 316, 319, 320, 321, 322, 323, 324, 326, 327, 328, 332, 333, 334, 335, 336, 337, 339, 340, 341, 342, 343, 344, 346, 347, 349, 350, 351, 353, 354, 355, 357, 359, 360, 361, 363, 365, 367, 368, 370, 371, 372, 373, 377, 378, 379, 380, 383, 384, 385, 386, 387, 389, 390, 391, 392, 393, 394, 395, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 416, 419, 420, 421, 423, 425, 426, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 441, 442, 444, 447, 448, 449, 450, 451, 452, 453, 454, 456, 457, 458, 460, 461, 462, 463, 464, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 480, 481, 482, 483, 485, 486, 488, 489, 490, 493, 494, 495, 496, 497, 499, 500, 502, 503, 504, 506, 507, 508, 510, 511], [0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 26, 28, 29, 30, 31, 33, 35, 36, 37, 39, 41, 42, 43, 44, 45, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 61, 63, 65, 66, 72, 73, 74, 75, 76, 77, 78, 79, 80, 82, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 100, 101, 102, 103, 104, 105, 107, 108, 109, 110, 111, 114, 115, 116, 118, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 130, 132, 133, 134, 135, 137, 138, 139, 141, 144, 146, 147, 148, 149, 150, 153, 156, 157, 158, 159, 160, 162, 163, 166, 168, 170, 171, 172, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186, 187, 188, 189, 192, 193, 196, 197, 198, 199, 201, 202, 203, 204, 205, 206, 207, 209, 210, 211, 212, 213, 214, 216, 217, 219, 220, 221, 223, 225, 226, 227, 229, 232, 234, 236, 237, 238, 240, 241, 242, 243, 244, 246, 247, 248, 249, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 265, 267, 268, 269, 270, 271, 272, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 287, 288, 290, 291, 292, 293, 295, 296, 298, 299, 300, 301, 303, 305, 306, 307, 308, 309, 310, 311, 312, 314, 315, 316, 318, 319, 320, 321, 322, 323, 326, 327, 328, 330, 332, 333, 334, 335, 337, 338, 339, 340, 341, 342, 346, 348, 349, 350, 351, 354, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 377, 378, 381, 382, 383, 384, 385, 386, 388, 389, 390, 391, 392, 394, 395, 396, 397, 398, 400, 402, 403, 404, 405, 407, 408, 409, 410, 416, 417, 420, 421, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 434, 435, 436, 437, 439, 441, 442, 443, 445, 447, 448, 449, 450, 451, 452, 453, 455, 457, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 473, 474, 476, 477, 478, 479, 480, 481, 482, 483, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 503, 504, 505, 507, 508, 509, 510, 511], [0, 2, 3, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 19, 20, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 39, 40, 41, 42, 44, 46, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 66, 67, 68, 70, 72, 73, 74, 76, 78, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 91, 92, 94, 95, 96, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 141, 143, 144, 145, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 168, 169, 170, 171, 173, 174, 175, 176, 177, 181, 182, 183, 184, 188, 189, 190, 191, 192, 193, 194, 195, 196, 198, 199, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 216, 217, 218, 219, 221, 222, 223, 225, 227, 228, 229, 230, 233, 234, 235, 236, 237, 238, 241, 242, 243, 245, 246, 247, 248, 249, 252, 254, 256, 258, 259, 260, 262, 263, 265, 267, 268, 269, 270, 271, 272, 274, 275, 276, 279, 281, 283, 284, 286, 287, 289, 291, 292, 294, 296, 298, 299, 300, 302, 304, 305, 306, 307, 308, 310, 311, 312, 313, 314, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 331, 332, 333, 338, 339, 340, 341, 342, 344, 346, 348, 349, 353, 355, 357, 358, 359, 360, 361, 363, 364, 365, 367, 370, 371, 373, 374, 375, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 398, 399, 403, 404, 405, 406, 410, 411, 412, 414, 416, 417, 419, 420, 422, 423, 424, 425, 426, 428, 429, 430, 431, 432, 433, 434, 436, 437, 438, 440, 441, 443, 444, 445, 447, 448, 449, 452, 453, 454, 455, 456, 458, 459, 460, 461, 462, 463, 464, 465, 466, 468, 469, 471, 472, 473, 474, 475, 476, 477, 479, 480, 481, 482, 485, 486, 488, 489, 490, 491, 492, 493, 494, 495, 496, 498, 499, 500, 501, 502, 503, 504, 506, 507, 509, 511], [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 21, 23, 24, 26, 27, 29, 32, 33, 35, 36, 37, 38, 41, 42, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 56, 58, 59, 60, 62, 63, 64, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 93, 96, 97, 98, 99, 100, 101, 104, 105, 107, 108, 109, 110, 111, 112, 113, 115, 117, 119, 120, 121, 124, 125, 127, 128, 129, 130, 131, 132, 133, 136, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 149, 151, 153, 154, 155, 156, 158, 159, 160, 161, 162, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 194, 195, 196, 197, 198, 199, 201, 202, 204, 206, 207, 208, 209, 213, 215, 217, 218, 219, 220, 222, 224, 225, 226, 227, 228, 230, 231, 233, 234, 235, 236, 237, 238, 239, 242, 244, 246, 247, 249, 250, 251, 252, 255, 257, 258, 259, 260, 261, 263, 264, 265, 266, 267, 270, 271, 272, 273, 275, 276, 277, 278, 279, 280, 281, 283, 284, 285, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 303, 304, 307, 308, 309, 310, 311, 312, 313, 315, 316, 318, 320, 321, 322, 323, 324, 325, 326, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 344, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 362, 364, 367, 368, 369, 370, 372, 373, 374, 375, 376, 377, 378, 379, 381, 383, 384, 385, 387, 388, 389, 391, 392, 394, 397, 399, 400, 401, 403, 405, 406, 407, 409, 410, 411, 412, 413, 414, 415, 416, 420, 423, 424, 425, 428, 430, 433, 434, 436, 437, 438, 439, 441, 442, 443, 445, 446, 448, 449, 450, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 465, 467, 468, 469, 470, 471, 474, 475, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 508, 509, 510, 511], [0, 1, 3, 4, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 20, 21, 22, 24, 25, 26, 27, 30, 31, 33, 34, 35, 36, 40, 42, 45, 47, 48, 50, 51, 52, 53, 54, 56, 57, 58, 59, 62, 63, 66, 67, 68, 69, 70, 73, 74, 76, 77, 78, 79, 80, 81, 84, 87, 88, 90, 91, 93, 94, 95, 97, 98, 99, 100, 101, 102, 103, 104, 108, 109, 111, 112, 113, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 134, 135, 136, 137, 138, 140, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 156, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 172, 173, 175, 176, 177, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 198, 199, 200, 201, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 216, 217, 219, 220, 222, 223, 224, 226, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 239, 240, 241, 242, 243, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 263, 264, 265, 266, 269, 270, 272, 273, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 292, 293, 294, 296, 298, 299, 300, 301, 302, 303, 304, 306, 308, 309, 310, 312, 313, 314, 316, 317, 318, 319, 320, 321, 324, 325, 326, 327, 328, 329, 331, 332, 333, 334, 335, 336, 338, 340, 342, 343, 344, 345, 347, 349, 350, 351, 352, 353, 354, 355, 357, 358, 359, 361, 362, 363, 364, 365, 368, 369, 370, 371, 372, 373, 374, 377, 379, 381, 382, 383, 384, 385, 386, 387, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 401, 403, 404, 405, 406, 408, 409, 410, 412, 414, 416, 417, 418, 419, 422, 423, 424, 425, 427, 428, 429, 430, 431, 432, 434, 435, 436, 437, 438, 439, 441, 443, 444, 445, 446, 447, 448, 451, 452, 454, 455, 458, 459, 461, 463, 465, 466, 467, 468, 469, 470, 471, 472, 474, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 489, 490, 491, 493, 495, 496, 498, 499, 500, 504, 505, 508, 509, 510, 511], [0, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 16, 17, 20, 21, 22, 23, 24, 26, 27, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 85, 86, 88, 89, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 105, 106, 107, 109, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 132, 133, 134, 135, 136, 138, 139, 140, 141, 142, 143, 145, 146, 147, 148, 150, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 184, 186, 187, 188, 189, 190, 191, 194, 195, 198, 199, 200, 202, 204, 205, 206, 208, 210, 212, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 227, 228, 229, 230, 231, 232, 234, 235, 236, 238, 239, 240, 241, 242, 244, 245, 248, 250, 251, 254, 255, 256, 257, 259, 260, 261, 262, 263, 264, 267, 268, 269, 270, 272, 273, 274, 275, 276, 277, 278, 282, 283, 284, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 302, 303, 304, 305, 307, 308, 311, 312, 314, 315, 316, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 331, 332, 333, 334, 335, 338, 339, 341, 342, 343, 346, 349, 350, 351, 352, 354, 356, 357, 358, 360, 363, 367, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 390, 391, 393, 394, 395, 396, 397, 399, 404, 408, 409, 410, 412, 414, 415, 416, 417, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 432, 436, 437, 438, 439, 441, 442, 444, 445, 449, 450, 451, 452, 453, 454, 455, 457, 458, 459, 462, 463, 465, 466, 468, 469, 470, 473, 474, 476, 477, 478, 479, 480, 481, 482, 484, 485, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 499, 500, 501, 503, 505, 506, 507, 508, 509], [2, 3, 4, 6, 7, 8, 10, 11, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 42, 43, 45, 48, 49, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 64, 66, 67, 68, 69, 70, 73, 75, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 96, 97, 100, 101, 102, 103, 106, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119, 120, 122, 123, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 143, 144, 145, 146, 147, 148, 150, 151, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 165, 167, 169, 171, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 186, 187, 188, 189, 190, 191, 192, 195, 196, 197, 198, 200, 202, 203, 204, 205, 206, 207, 209, 210, 211, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 235, 236, 237, 238, 239, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 252, 254, 255, 258, 260, 261, 262, 263, 265, 266, 268, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 285, 286, 291, 292, 296, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 321, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 334, 335, 336, 337, 338, 339, 340, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 362, 363, 365, 366, 370, 374, 375, 376, 377, 378, 379, 381, 383, 386, 388, 389, 390, 392, 394, 397, 399, 401, 402, 403, 404, 405, 406, 407, 408, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 427, 429, 430, 432, 433, 434, 437, 441, 442, 443, 444, 445, 447, 448, 449, 450, 451, 453, 454, 455, 456, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 476, 478, 482, 483, 484, 486, 487, 489, 491, 493, 494, 495, 496, 497, 498, 499, 500, 502, 503, 505, 509, 510, 511], [0, 2, 3, 4, 5, 6, 7, 8, 10, 14, 15, 17, 18, 19, 20, 21, 25, 27, 28, 29, 31, 32, 34, 35, 36, 37, 39, 41, 42, 43, 45, 46, 48, 49, 50, 51, 53, 56, 58, 59, 60, 62, 63, 65, 66, 68, 70, 71, 72, 74, 76, 77, 78, 80, 81, 83, 85, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 106, 108, 109, 111, 112, 113, 115, 118, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 139, 140, 145, 146, 147, 148, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 173, 174, 176, 177, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 189, 192, 193, 194, 195, 197, 199, 202, 203, 204, 206, 208, 210, 211, 212, 213, 214, 215, 217, 218, 220, 221, 223, 224, 225, 226, 228, 229, 230, 231, 232, 233, 234, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 264, 266, 269, 270, 271, 275, 277, 279, 281, 282, 283, 284, 286, 288, 290, 292, 293, 294, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 307, 308, 310, 311, 312, 313, 314, 315, 316, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 335, 337, 338, 339, 340, 341, 342, 343, 347, 348, 349, 350, 353, 355, 356, 357, 358, 360, 361, 364, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 378, 379, 380, 381, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 399, 400, 403, 404, 405, 407, 408, 410, 411, 412, 413, 417, 419, 420, 421, 424, 426, 427, 429, 430, 431, 432, 433, 434, 435, 436, 439, 440, 441, 442, 443, 445, 446, 447, 449, 450, 451, 452, 453, 456, 457, 458, 459, 460, 461, 462, 464, 465, 468, 469, 470, 471, 472, 474, 475, 476, 478, 479, 480, 481, 482, 483, 484, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 499, 500, 502, 503, 504, 506, 508, 509, 511], [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 23, 24, 25, 26, 28, 29, 30, 31, 35, 36, 37, 38, 39, 41, 42, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 65, 66, 67, 68, 71, 72, 73, 74, 75, 76, 77, 78, 79, 81, 82, 84, 85, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 100, 101, 102, 103, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 116, 117, 118, 120, 122, 123, 124, 125, 126, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 141, 142, 145, 146, 148, 151, 152, 153, 155, 156, 157, 158, 160, 161, 162, 163, 165, 166, 167, 170, 171, 172, 173, 174, 175, 176, 177, 179, 180, 181, 182, 183, 184, 185, 187, 188, 189, 191, 194, 196, 197, 198, 199, 200, 201, 204, 205, 207, 208, 209, 210, 211, 212, 213, 215, 216, 217, 218, 220, 221, 223, 224, 225, 226, 229, 231, 232, 233, 234, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 251, 253, 254, 255, 257, 258, 260, 261, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 280, 288, 291, 293, 294, 295, 299, 300, 301, 302, 303, 304, 305, 306, 308, 309, 310, 311, 313, 314, 315, 317, 318, 320, 322, 323, 324, 325, 326, 327, 328, 329, 330, 332, 333, 334, 335, 336, 337, 340, 341, 342, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 356, 358, 359, 361, 362, 364, 366, 368, 369, 370, 371, 372, 373, 376, 378, 379, 380, 381, 382, 384, 385, 386, 387, 388, 389, 391, 392, 394, 395, 396, 397, 398, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 415, 416, 417, 419, 421, 422, 423, 424, 425, 426, 427, 428, 430, 432, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 454, 456, 457, 458, 460, 461, 463, 465, 466, 467, 468, 469, 471, 472, 473, 474, 478, 481, 482, 483, 485, 487, 488, 489, 490, 491, 492, 494, 496, 497, 499, 500, 501, 502, 503, 504, 505, 507, 509, 510], [0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 15, 16, 17, 18, 19, 24, 25, 26, 28, 30, 32, 33, 34, 35, 36, 37, 39, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 57, 58, 59, 60, 62, 63, 64, 65, 66, 67, 69, 71, 73, 76, 77, 78, 80, 81, 82, 83, 84, 85, 87, 88, 90, 94, 96, 97, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 123, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 139, 140, 141, 142, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 162, 163, 165, 166, 167, 168, 169, 171, 172, 174, 175, 179, 180, 181, 182, 183, 184, 189, 190, 191, 192, 193, 196, 197, 198, 201, 202, 203, 204, 205, 206, 207, 209, 210, 213, 216, 218, 221, 222, 226, 227, 228, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 248, 249, 250, 251, 253, 254, 255, 256, 257, 258, 259, 261, 262, 263, 264, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 281, 283, 284, 285, 286, 287, 288, 290, 291, 292, 294, 295, 296, 297, 299, 300, 301, 302, 303, 304, 305, 308, 311, 312, 313, 314, 315, 317, 318, 319, 320, 321, 322, 325, 326, 327, 328, 329, 330, 332, 333, 334, 335, 336, 337, 339, 340, 342, 343, 344, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 358, 359, 360, 361, 363, 364, 365, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 380, 381, 384, 385, 387, 388, 389, 390, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408, 410, 411, 413, 414, 415, 416, 419, 420, 422, 423, 424, 425, 426, 427, 428, 430, 432, 433, 434, 435, 436, 437, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 453, 454, 455, 456, 457, 460, 462, 463, 464, 467, 468, 469, 470, 472, 473, 474, 475, 477, 478, 481, 483, 486, 487, 488, 489, 491, 493, 494, 495, 497, 498, 499, 502, 503, 504, 506, 507, 509, 510, 511], [3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 17, 18, 19, 21, 22, 25, 26, 27, 28, 30, 31, 32, 33, 34, 37, 38, 39, 40, 41, 42, 44, 46, 47, 48, 49, 50, 52, 53, 55, 57, 58, 60, 61, 62, 64, 65, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 116, 117, 118, 119, 120, 121, 122, 123, 124, 127, 129, 131, 132, 134, 135, 136, 137, 138, 139, 142, 143, 144, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 183, 184, 186, 187, 188, 190, 191, 192, 194, 195, 196, 197, 198, 202, 203, 204, 205, 206, 207, 208, 211, 214, 217, 218, 219, 220, 221, 223, 224, 226, 227, 228, 230, 231, 233, 234, 235, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 261, 262, 263, 265, 266, 267, 270, 271, 272, 273, 276, 277, 278, 279, 280, 282, 284, 285, 286, 287, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 303, 305, 306, 307, 308, 310, 312, 313, 314, 315, 317, 322, 323, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 336, 338, 340, 342, 343, 345, 346, 347, 348, 349, 350, 351, 352, 355, 356, 358, 359, 360, 362, 363, 365, 366, 367, 368, 369, 370, 371, 372, 374, 375, 377, 378, 379, 380, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 408, 409, 410, 411, 414, 416, 417, 418, 419, 423, 424, 425, 426, 427, 429, 430, 431, 432, 434, 438, 439, 440, 442, 443, 444, 445, 447, 450, 453, 454, 455, 457, 459, 460, 462, 463, 464, 467, 470, 471, 472, 473, 474, 475, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 505, 506, 507, 508, 510], [0, 1, 3, 4, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 21, 22, 23, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42, 43, 45, 46, 47, 48, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 93, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 138, 139, 141, 142, 144, 145, 147, 149, 150, 151, 152, 154, 155, 157, 158, 159, 160, 161, 163, 165, 166, 168, 169, 170, 171, 172, 173, 174, 176, 177, 178, 179, 180, 182, 183, 184, 186, 187, 188, 189, 192, 194, 195, 196, 197, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 213, 216, 217, 219, 220, 222, 223, 225, 226, 228, 229, 230, 231, 232, 233, 234, 235, 237, 238, 239, 241, 243, 244, 245, 246, 247, 248, 251, 252, 253, 254, 257, 258, 261, 262, 263, 265, 266, 268, 269, 270, 271, 272, 273, 275, 276, 277, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 296, 297, 299, 300, 302, 303, 304, 305, 306, 307, 309, 310, 313, 316, 317, 318, 320, 321, 322, 324, 328, 329, 330, 331, 332, 333, 334, 335, 336, 338, 340, 341, 343, 344, 345, 346, 347, 348, 351, 352, 355, 356, 358, 359, 360, 361, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 389, 390, 391, 395, 397, 398, 399, 400, 401, 403, 404, 406, 407, 408, 410, 411, 412, 414, 415, 416, 420, 421, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 434, 435, 436, 437, 438, 444, 445, 446, 447, 448, 449, 450, 451, 452, 454, 456, 459, 460, 461, 462, 466, 467, 468, 469, 472, 474, 475, 477, 478, 480, 482, 483, 484, 486, 487, 488, 489, 491, 493, 495, 497, 499, 501, 502, 503, 504, 505, 506, 510], [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 25, 26, 28, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 67, 69, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 83, 85, 86, 87, 89, 90, 91, 92, 93, 94, 96, 98, 99, 100, 101, 102, 103, 104, 105, 107, 108, 109, 110, 111, 112, 113, 114, 117, 118, 120, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 136, 137, 138, 139, 140, 141, 142, 143, 145, 146, 147, 148, 149, 150, 153, 154, 156, 158, 163, 164, 165, 166, 167, 168, 171, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186, 188, 189, 190, 191, 192, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 206, 207, 208, 209, 210, 211, 212, 213, 215, 217, 218, 219, 221, 222, 224, 228, 229, 230, 231, 232, 233, 235, 236, 237, 238, 239, 240, 241, 242, 243, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 265, 267, 268, 270, 272, 273, 274, 275, 276, 277, 278, 279, 282, 283, 284, 287, 288, 289, 291, 292, 293, 294, 295, 296, 300, 301, 302, 303, 304, 305, 306, 307, 310, 312, 314, 316, 317, 319, 320, 321, 323, 324, 325, 326, 328, 331, 332, 333, 334, 335, 337, 340, 343, 344, 345, 346, 348, 349, 350, 351, 352, 353, 354, 356, 357, 360, 362, 363, 364, 365, 366, 367, 368, 369, 371, 372, 373, 374, 375, 376, 378, 381, 382, 383, 384, 386, 387, 388, 389, 393, 394, 395, 396, 397, 398, 399, 400, 401, 403, 404, 405, 406, 407, 409, 410, 411, 412, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 430, 431, 432, 434, 435, 436, 437, 438, 440, 441, 444, 445, 446, 447, 448, 450, 451, 453, 454, 455, 456, 458, 459, 461, 463, 464, 466, 467, 468, 469, 470, 473, 475, 476, 477, 481, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 497, 498, 503, 505, 506, 507, 509, 510, 511], [1, 2, 4, 5, 6, 7, 9, 11, 12, 14, 15, 16, 17, 19, 20, 21, 22, 24, 26, 27, 28, 29, 34, 35, 36, 38, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 55, 56, 57, 58, 59, 60, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 83, 86, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 102, 103, 106, 107, 108, 110, 112, 116, 118, 120, 121, 122, 123, 124, 125, 128, 129, 133, 135, 136, 137, 139, 140, 141, 142, 144, 146, 147, 149, 151, 152, 153, 155, 156, 157, 158, 160, 161, 164, 165, 166, 167, 168, 170, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 186, 187, 188, 189, 190, 191, 193, 194, 198, 199, 200, 202, 203, 206, 208, 209, 211, 212, 214, 215, 217, 218, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 235, 237, 238, 239, 240, 242, 243, 245, 246, 247, 248, 249, 251, 252, 253, 254, 255, 257, 258, 259, 260, 262, 264, 265, 266, 267, 268, 271, 272, 274, 275, 276, 278, 279, 280, 282, 284, 285, 286, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 315, 316, 317, 318, 321, 322, 323, 325, 328, 329, 330, 331, 332, 333, 334, 335, 337, 338, 339, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 357, 358, 359, 360, 361, 362, 363, 364, 366, 368, 369, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 387, 389, 390, 391, 392, 393, 394, 395, 397, 398, 400, 401, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 418, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 452, 453, 454, 456, 457, 458, 459, 460, 462, 465, 466, 467, 468, 469, 470, 472, 476, 477, 478, 479, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 494, 495, 496, 498, 499, 500, 501, 502, 504, 505, 506, 507, 508, 509, 510], [0, 1, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 51, 54, 55, 56, 57, 58, 61, 63, 64, 65, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 78, 81, 83, 84, 87, 88, 89, 90, 92, 95, 96, 97, 98, 100, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 117, 119, 120, 121, 122, 125, 127, 129, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 141, 142, 143, 144, 146, 147, 148, 149, 150, 152, 156, 157, 158, 159, 161, 162, 163, 164, 166, 167, 168, 171, 173, 174, 175, 176, 177, 179, 183, 184, 190, 194, 195, 197, 198, 199, 201, 202, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 216, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 234, 235, 237, 238, 239, 240, 241, 243, 244, 245, 247, 249, 250, 251, 252, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 284, 285, 286, 287, 288, 289, 290, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 305, 306, 307, 308, 311, 313, 314, 315, 316, 317, 320, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 342, 345, 346, 347, 348, 349, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 366, 367, 368, 370, 371, 372, 373, 375, 377, 378, 379, 380, 382, 383, 385, 386, 388, 389, 390, 392, 393, 394, 396, 397, 403, 404, 406, 407, 410, 412, 413, 414, 415, 416, 419, 421, 422, 423, 425, 427, 428, 429, 430, 431, 432, 433, 435, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 453, 454, 455, 456, 458, 459, 461, 462, 466, 467, 469, 470, 471, 472, 473, 474, 475, 476, 478, 479, 480, 481, 482, 483, 484, 486, 487, 488, 489, 490, 492, 493, 494, 495, 496, 497, 499, 501, 502, 504, 506, 507, 508, 509, 510], [0, 1, 2, 3, 6, 7, 9, 10, 11, 12, 14, 16, 17, 18, 19, 22, 24, 25, 27, 28, 29, 30, 32, 34, 35, 36, 37, 38, 39, 41, 42, 43, 44, 47, 48, 49, 50, 52, 55, 57, 58, 59, 60, 61, 62, 64, 65, 66, 68, 69, 70, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 86, 87, 88, 89, 90, 92, 93, 94, 95, 96, 97, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 116, 117, 118, 119, 120, 122, 123, 124, 125, 126, 130, 131, 132, 133, 135, 137, 138, 139, 140, 141, 142, 144, 145, 146, 147, 150, 151, 152, 153, 154, 156, 157, 159, 161, 162, 163, 164, 165, 167, 170, 171, 172, 173, 174, 175, 177, 178, 180, 181, 183, 185, 187, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 223, 224, 225, 227, 228, 230, 231, 232, 233, 234, 236, 238, 240, 241, 242, 243, 244, 245, 246, 247, 248, 251, 252, 253, 254, 255, 257, 258, 260, 261, 263, 264, 265, 267, 268, 270, 272, 273, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 291, 292, 293, 295, 297, 298, 299, 300, 301, 302, 305, 306, 307, 308, 311, 312, 313, 314, 315, 316, 317, 318, 319, 321, 322, 323, 324, 327, 329, 330, 332, 333, 334, 335, 336, 340, 341, 342, 343, 345, 346, 347, 348, 349, 351, 354, 356, 357, 359, 361, 362, 363, 364, 365, 366, 370, 372, 375, 378, 379, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 395, 396, 398, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 413, 415, 416, 417, 418, 419, 420, 422, 423, 424, 425, 426, 427, 430, 431, 434, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 460, 461, 462, 463, 465, 467, 468, 469, 470, 472, 474, 475, 476, 477, 478, 479, 480, 482, 483, 484, 488, 489, 490, 491, 492, 493, 495, 496, 497, 498, 499, 500, 502, 503, 504, 505, 507, 508, 509, 510, 511], [0, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 26, 27, 28, 29, 31, 34, 37, 38, 39, 40, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 70, 72, 73, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 98, 99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 111, 112, 113, 114, 115, 117, 118, 119, 120, 122, 123, 125, 126, 127, 128, 129, 131, 132, 133, 135, 138, 139, 140, 141, 143, 144, 145, 146, 147, 150, 151, 152, 153, 156, 159, 160, 162, 163, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 178, 179, 180, 181, 182, 183, 184, 185, 187, 190, 191, 193, 194, 195, 196, 197, 198, 199, 200, 204, 205, 206, 208, 209, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 251, 252, 253, 255, 256, 260, 263, 264, 265, 266, 269, 271, 272, 274, 275, 276, 277, 278, 280, 282, 283, 284, 285, 286, 287, 288, 291, 292, 295, 296, 300, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 324, 326, 327, 329, 330, 331, 332, 335, 336, 337, 338, 339, 340, 341, 342, 345, 346, 347, 348, 350, 353, 354, 357, 358, 359, 360, 361, 362, 363, 364, 367, 368, 369, 370, 371, 372, 373, 374, 375, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 395, 396, 397, 398, 399, 400, 401, 402, 403, 405, 407, 411, 413, 416, 417, 419, 421, 422, 423, 424, 425, 427, 428, 431, 433, 434, 435, 437, 438, 439, 440, 441, 443, 444, 446, 447, 448, 449, 450, 451, 453, 454, 456, 457, 458, 459, 461, 462, 463, 464, 466, 467, 468, 470, 471, 472, 473, 474, 475, 476, 477, 480, 481, 482, 483, 485, 486, 487, 490, 491, 493, 494, 495, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510, 511], [1, 2, 3, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 41, 43, 44, 46, 47, 48, 49, 50, 51, 52, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 72, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 86, 87, 88, 89, 90, 92, 93, 94, 97, 99, 102, 103, 104, 106, 107, 110, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 130, 131, 132, 134, 135, 136, 137, 139, 140, 141, 142, 143, 145, 146, 147, 148, 149, 150, 152, 153, 155, 156, 157, 158, 160, 161, 163, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 181, 183, 184, 185, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 205, 206, 207, 208, 209, 210, 211, 212, 213, 215, 216, 217, 218, 219, 220, 221, 222, 223, 225, 226, 228, 230, 232, 233, 234, 235, 236, 237, 238, 240, 241, 242, 243, 246, 249, 250, 251, 252, 253, 254, 255, 256, 258, 259, 261, 262, 263, 264, 265, 268, 271, 272, 273, 276, 277, 279, 280, 281, 282, 283, 284, 285, 286, 288, 289, 292, 293, 294, 296, 297, 298, 299, 300, 304, 305, 306, 307, 308, 312, 314, 315, 317, 318, 319, 320, 321, 322, 323, 324, 326, 328, 329, 330, 331, 332, 334, 335, 336, 338, 339, 340, 341, 342, 344, 346, 347, 348, 354, 356, 357, 360, 362, 363, 364, 365, 366, 369, 371, 372, 373, 374, 376, 377, 378, 379, 381, 382, 383, 384, 385, 387, 389, 390, 391, 395, 396, 398, 399, 400, 401, 404, 405, 406, 407, 408, 409, 411, 412, 413, 415, 416, 417, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 429, 433, 434, 437, 438, 439, 440, 443, 444, 445, 447, 448, 450, 451, 452, 453, 454, 456, 458, 460, 461, 462, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 476, 477, 478, 479, 481, 482, 483, 484, 485, 486, 487, 488, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 506, 507, 508, 510], [0, 2, 3, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17, 18, 19, 21, 22, 24, 25, 27, 28, 29, 30, 33, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 47, 48, 50, 51, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 68, 69, 70, 71, 73, 74, 75, 76, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 104, 106, 107, 111, 112, 113, 116, 117, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 131, 132, 133, 136, 138, 140, 141, 142, 143, 144, 145, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 162, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 186, 187, 188, 189, 190, 191, 192, 194, 197, 198, 199, 200, 203, 204, 207, 208, 210, 211, 212, 213, 214, 215, 216, 218, 219, 220, 221, 224, 225, 227, 228, 229, 230, 231, 232, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 251, 252, 253, 256, 257, 258, 259, 260, 261, 262, 263, 265, 266, 268, 270, 271, 272, 273, 274, 275, 278, 281, 282, 283, 285, 287, 288, 289, 290, 291, 292, 293, 294, 296, 297, 298, 300, 301, 303, 304, 305, 308, 309, 310, 311, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 327, 328, 330, 332, 334, 335, 337, 338, 339, 340, 342, 343, 344, 345, 347, 348, 350, 351, 355, 356, 357, 359, 360, 361, 362, 363, 366, 367, 368, 369, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 391, 392, 393, 394, 397, 398, 399, 401, 403, 405, 406, 408, 409, 410, 411, 413, 414, 417, 418, 419, 420, 421, 423, 424, 425, 426, 428, 430, 431, 432, 433, 436, 437, 438, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 454, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 473, 475, 476, 477, 480, 482, 484, 485, 486, 487, 490, 492, 493, 494, 495, 496, 497, 498, 501, 503, 504, 506, 507, 509, 511], [1, 2, 3, 5, 8, 9, 11, 16, 18, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 56, 57, 58, 61, 65, 66, 67, 68, 72, 74, 75, 76, 77, 78, 79, 80, 82, 83, 84, 86, 87, 88, 89, 91, 92, 93, 95, 97, 98, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 129, 130, 131, 133, 135, 136, 137, 138, 139, 141, 142, 144, 145, 146, 147, 148, 150, 151, 152, 153, 154, 156, 157, 158, 159, 160, 162, 163, 164, 165, 166, 167, 169, 170, 171, 172, 173, 174, 175, 176, 178, 179, 183, 185, 186, 187, 189, 190, 192, 193, 194, 196, 198, 199, 201, 202, 203, 204, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 218, 219, 220, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 250, 251, 252, 253, 255, 256, 257, 259, 260, 261, 264, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 281, 282, 283, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 297, 298, 299, 301, 303, 304, 305, 306, 309, 311, 312, 313, 314, 315, 316, 317, 318, 320, 321, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 344, 345, 346, 347, 350, 351, 353, 354, 355, 357, 358, 360, 363, 364, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 382, 384, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 400, 401, 402, 403, 404, 405, 406, 408, 409, 411, 412, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 429, 430, 431, 432, 434, 436, 437, 438, 441, 442, 443, 444, 446, 447, 448, 449, 450, 452, 453, 454, 456, 457, 458, 459, 460, 461, 462, 464, 465, 466, 467, 469, 470, 472, 473, 474, 475, 476, 477, 478, 480, 483, 484, 485, 486, 489, 493, 495, 496, 497, 499, 502, 503, 504, 505, 507], [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 64, 65, 66, 67, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 89, 91, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 105, 107, 108, 109, 110, 111, 112, 117, 118, 120, 121, 122, 123, 124, 125, 126, 130, 131, 132, 133, 136, 138, 139, 140, 141, 142, 143, 144, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 157, 158, 159, 160, 162, 163, 164, 165, 171, 172, 174, 175, 177, 179, 181, 184, 186, 187, 188, 189, 190, 191, 192, 193, 196, 197, 199, 201, 205, 206, 210, 212, 214, 215, 216, 217, 218, 221, 223, 224, 225, 227, 229, 230, 233, 234, 237, 238, 240, 241, 242, 243, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 259, 261, 264, 265, 266, 267, 269, 270, 271, 273, 274, 275, 276, 278, 279, 280, 281, 283, 284, 287, 288, 290, 292, 293, 294, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 307, 308, 309, 313, 314, 316, 317, 319, 320, 321, 322, 324, 325, 326, 327, 328, 329, 330, 331, 332, 334, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 347, 348, 349, 351, 353, 355, 356, 357, 358, 359, 361, 363, 364, 365, 366, 369, 371, 372, 373, 374, 375, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 393, 394, 395, 396, 400, 401, 402, 403, 404, 405, 406, 408, 409, 410, 411, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 443, 444, 445, 446, 447, 449, 450, 452, 453, 456, 457, 458, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 478, 479, 481, 482, 483, 487, 489, 490, 491, 492, 494, 495, 496, 497, 499, 500, 504, 505, 506, 507, 508, 509, 511], [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 37, 39, 41, 42, 43, 44, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 69, 70, 72, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 88, 89, 90, 92, 94, 95, 96, 98, 100, 101, 102, 103, 104, 105, 106, 108, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 134, 137, 138, 139, 140, 141, 142, 144, 145, 146, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 162, 163, 165, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 194, 195, 196, 197, 199, 203, 204, 206, 207, 208, 209, 210, 211, 213, 214, 215, 219, 220, 221, 222, 224, 226, 227, 228, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 242, 243, 244, 245, 247, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 268, 271, 272, 274, 275, 276, 278, 279, 281, 284, 286, 288, 289, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 302, 303, 304, 305, 308, 309, 312, 313, 315, 316, 318, 319, 322, 323, 324, 325, 326, 328, 329, 330, 331, 332, 334, 336, 337, 338, 339, 340, 344, 345, 346, 347, 349, 356, 357, 358, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 373, 374, 377, 378, 379, 380, 381, 382, 383, 386, 388, 389, 390, 391, 392, 394, 396, 398, 399, 401, 402, 403, 404, 406, 407, 408, 409, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 424, 425, 426, 428, 429, 430, 431, 433, 434, 435, 436, 437, 438, 440, 441, 442, 443, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 460, 461, 462, 463, 465, 466, 467, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 482, 483, 484, 485, 486, 488, 490, 492, 493, 494, 497, 500, 501, 503, 504, 507, 509, 510, 511], [0, 3, 4, 5, 6, 7, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 52, 53, 54, 56, 58, 59, 60, 61, 64, 66, 67, 68, 69, 70, 71, 72, 73, 75, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 91, 92, 93, 95, 96, 97, 98, 100, 101, 103, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 120, 121, 122, 123, 126, 127, 128, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 142, 143, 144, 145, 146, 147, 148, 151, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 165, 167, 168, 169, 171, 173, 174, 175, 177, 178, 179, 180, 182, 183, 184, 186, 187, 188, 189, 191, 194, 195, 196, 197, 198, 200, 202, 204, 205, 206, 207, 210, 211, 213, 214, 215, 216, 217, 219, 220, 221, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 252, 254, 256, 258, 260, 261, 262, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 285, 289, 291, 292, 293, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 313, 314, 315, 316, 317, 319, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 334, 335, 336, 337, 338, 340, 344, 346, 347, 348, 350, 353, 356, 357, 358, 359, 360, 361, 362, 363, 365, 366, 370, 372, 375, 376, 377, 378, 379, 381, 382, 383, 386, 387, 388, 389, 390, 392, 394, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 427, 428, 429, 430, 432, 434, 435, 436, 437, 438, 441, 442, 443, 444, 445, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 458, 459, 460, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 482, 483, 484, 486, 487, 489, 491, 492, 493, 494, 496, 497, 498, 499, 500, 503, 505, 509, 510], [0, 2, 3, 4, 6, 7, 8, 9, 10, 14, 15, 17, 19, 21, 22, 25, 26, 27, 29, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 66, 68, 69, 70, 71, 72, 74, 75, 76, 77, 78, 80, 81, 82, 83, 85, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 98, 99, 100, 102, 103, 104, 106, 108, 109, 111, 112, 114, 115, 118, 119, 120, 121, 122, 124, 125, 127, 128, 130, 131, 132, 134, 135, 136, 138, 139, 140, 142, 145, 146, 147, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 176, 177, 178, 179, 180, 181, 182, 184, 185, 187, 189, 192, 193, 194, 195, 196, 197, 199, 202, 203, 204, 206, 208, 210, 211, 212, 213, 214, 215, 218, 220, 221, 223, 224, 225, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 260, 261, 262, 264, 265, 266, 269, 270, 271, 275, 277, 279, 280, 283, 284, 285, 286, 290, 291, 292, 293, 295, 296, 297, 298, 299, 300, 301, 302, 304, 305, 306, 307, 308, 310, 311, 314, 316, 318, 319, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 335, 337, 338, 340, 341, 343, 344, 346, 347, 349, 352, 353, 355, 356, 357, 358, 359, 360, 361, 364, 366, 367, 368, 369, 370, 371, 372, 373, 374, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 389, 391, 393, 394, 395, 396, 399, 400, 403, 404, 405, 407, 408, 410, 411, 412, 413, 417, 419, 420, 421, 422, 423, 424, 426, 427, 429, 430, 431, 432, 433, 434, 439, 440, 441, 442, 443, 445, 446, 447, 450, 451, 452, 453, 455, 456, 457, 458, 460, 461, 462, 464, 465, 466, 467, 468, 469, 470, 471, 472, 474, 475, 476, 478, 479, 480, 481, 482, 483, 484, 486, 487, 488, 489, 491, 492, 493, 494, 495, 496, 497, 499, 500, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 22, 23, 24, 26, 28, 29, 30, 31, 34, 35, 36, 38, 39, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 71, 72, 74, 76, 77, 78, 79, 82, 84, 85, 87, 88, 89, 90, 91, 92, 93, 97, 98, 100, 102, 103, 105, 106, 107, 108, 110, 113, 114, 115, 116, 117, 118, 120, 122, 123, 124, 125, 126, 129, 130, 131, 133, 134, 136, 137, 138, 139, 142, 145, 146, 147, 152, 153, 155, 156, 157, 158, 160, 161, 162, 163, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 188, 191, 194, 196, 197, 198, 199, 200, 201, 205, 207, 208, 209, 210, 211, 212, 213, 215, 217, 218, 220, 221, 223, 224, 226, 228, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 244, 245, 247, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 282, 284, 285, 286, 288, 291, 293, 294, 295, 297, 298, 299, 301, 303, 304, 305, 306, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 320, 323, 324, 325, 326, 327, 328, 330, 332, 333, 334, 335, 336, 337, 340, 341, 343, 344, 346, 347, 348, 349, 350, 351, 352, 353, 358, 359, 360, 361, 362, 365, 368, 369, 370, 371, 372, 373, 375, 378, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 394, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 415, 416, 417, 418, 419, 420, 421, 424, 425, 426, 427, 428, 429, 430, 432, 434, 435, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 457, 460, 461, 465, 466, 467, 468, 469, 471, 472, 474, 476, 477, 479, 480, 482, 483, 484, 485, 487, 488, 489, 490, 491, 492, 494, 495, 496, 497, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], [0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 15, 16, 17, 18, 19, 21, 22, 24, 25, 28, 30, 31, 33, 34, 35, 36, 37, 39, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 71, 72, 73, 76, 77, 78, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 135, 136, 139, 141, 142, 143, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 167, 168, 169, 171, 172, 174, 175, 176, 178, 179, 180, 181, 182, 183, 184, 185, 189, 190, 191, 192, 193, 195, 196, 197, 198, 202, 203, 204, 205, 206, 207, 209, 210, 212, 213, 216, 218, 221, 222, 225, 226, 227, 228, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 246, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 261, 262, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 281, 283, 284, 285, 286, 287, 288, 291, 292, 293, 294, 295, 296, 297, 299, 300, 301, 302, 303, 304, 305, 308, 311, 312, 313, 314, 315, 317, 318, 319, 321, 322, 325, 327, 329, 330, 333, 334, 335, 336, 337, 339, 340, 342, 343, 344, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 357, 359, 360, 361, 363, 364, 365, 367, 368, 369, 370, 372, 373, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 387, 388, 389, 390, 391, 392, 393, 395, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 410, 411, 413, 414, 415, 416, 417, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 432, 433, 434, 435, 437, 439, 440, 441, 442, 443, 444, 445, 448, 449, 451, 454, 455, 457, 460, 464, 466, 467, 468, 469, 470, 472, 473, 474, 475, 477, 481, 483, 486, 489, 491, 493, 494, 495, 497, 498, 499, 502, 503, 506, 507, 509, 510, 511], [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 30, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 44, 46, 47, 48, 50, 52, 53, 57, 58, 60, 61, 62, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 116, 117, 118, 120, 121, 122, 123, 125, 127, 129, 131, 132, 134, 135, 136, 137, 138, 139, 143, 144, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 172, 173, 174, 175, 177, 178, 180, 181, 187, 188, 190, 191, 193, 194, 195, 196, 197, 198, 202, 204, 205, 206, 207, 208, 211, 215, 217, 218, 219, 220, 221, 223, 224, 228, 229, 230, 231, 233, 234, 235, 236, 237, 239, 240, 241, 243, 244, 246, 247, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 265, 266, 267, 268, 270, 272, 273, 276, 277, 278, 279, 280, 282, 284, 285, 286, 287, 289, 290, 291, 292, 293, 294, 296, 298, 299, 303, 304, 305, 306, 307, 310, 312, 313, 314, 315, 317, 318, 322, 323, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 350, 351, 352, 353, 355, 356, 357, 358, 359, 360, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 377, 378, 379, 380, 384, 385, 386, 387, 389, 390, 391, 392, 394, 395, 396, 397, 399, 400, 401, 403, 404, 406, 408, 409, 410, 414, 415, 416, 417, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 442, 443, 444, 445, 447, 448, 453, 454, 455, 456, 457, 458, 460, 461, 462, 463, 464, 465, 467, 469, 470, 471, 472, 473, 474, 475, 478, 480, 482, 483, 484, 486, 487, 488, 489, 490, 492, 493, 494, 495, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510], [0, 1, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 21, 22, 23, 25, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 38, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 120, 122, 123, 124, 125, 126, 127, 128, 129, 131, 134, 135, 136, 137, 138, 139, 141, 142, 143, 144, 145, 147, 149, 151, 152, 153, 154, 155, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 176, 177, 178, 179, 180, 182, 184, 186, 188, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 202, 203, 205, 207, 208, 209, 210, 212, 213, 215, 217, 218, 219, 220, 222, 223, 225, 226, 228, 229, 231, 232, 233, 234, 235, 237, 238, 239, 241, 242, 243, 244, 245, 246, 247, 251, 252, 254, 257, 258, 261, 262, 263, 266, 268, 270, 271, 272, 273, 275, 277, 280, 281, 282, 283, 284, 285, 287, 288, 289, 291, 292, 293, 296, 297, 299, 300, 302, 303, 304, 305, 306, 307, 309, 310, 313, 314, 315, 316, 318, 320, 321, 324, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 341, 342, 343, 344, 346, 347, 348, 351, 352, 355, 356, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 384, 386, 387, 388, 389, 390, 391, 393, 395, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408, 410, 411, 412, 413, 414, 415, 416, 417, 420, 421, 423, 424, 425, 426, 427, 428, 429, 432, 435, 438, 439, 440, 441, 444, 445, 446, 447, 448, 449, 450, 451, 452, 454, 458, 459, 460, 461, 462, 463, 467, 468, 469, 472, 475, 476, 477, 479, 480, 481, 482, 483, 484, 486, 487, 489, 490, 491, 492, 493, 494, 495, 497, 499, 501, 503, 504, 505, 506, 508, 510], [0, 1, 2, 3, 5, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 55, 56, 57, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 76, 79, 80, 81, 83, 85, 86, 89, 90, 91, 92, 93, 94, 96, 98, 99, 100, 102, 103, 104, 105, 107, 108, 109, 111, 112, 113, 114, 117, 118, 119, 120, 123, 124, 126, 127, 129, 130, 131, 132, 133, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 156, 157, 158, 159, 161, 163, 164, 165, 166, 168, 171, 173, 174, 175, 176, 177, 178, 179, 180, 182, 184, 185, 186, 188, 190, 191, 192, 193, 194, 195, 196, 197, 198, 200, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 215, 217, 218, 219, 221, 222, 224, 228, 229, 230, 231, 233, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 247, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 260, 262, 266, 267, 270, 272, 273, 274, 275, 276, 277, 278, 280, 282, 283, 284, 287, 289, 291, 292, 293, 294, 295, 296, 297, 298, 301, 302, 303, 304, 305, 306, 310, 312, 314, 315, 316, 319, 320, 321, 323, 324, 325, 326, 327, 328, 331, 332, 333, 334, 335, 337, 339, 340, 342, 343, 344, 345, 346, 348, 349, 350, 351, 352, 354, 355, 356, 357, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 371, 372, 374, 375, 376, 378, 379, 381, 382, 383, 384, 386, 387, 388, 389, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 416, 419, 420, 421, 422, 423, 424, 425, 427, 428, 430, 431, 432, 434, 435, 436, 437, 438, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 463, 464, 466, 467, 468, 469, 470, 473, 475, 476, 480, 481, 482, 484, 487, 488, 489, 490, 493, 495, 497, 498, 501, 503, 505, 506, 507, 508, 509, 510, 511], [0, 1, 2, 3, 5, 6, 7, 11, 12, 14, 15, 16, 17, 19, 20, 22, 24, 25, 26, 27, 28, 29, 34, 35, 36, 38, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58, 60, 63, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75, 77, 78, 79, 82, 83, 85, 86, 88, 90, 91, 93, 94, 95, 96, 97, 98, 100, 102, 103, 106, 107, 108, 109, 110, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 131, 133, 135, 136, 137, 138, 139, 140, 141, 142, 144, 145, 146, 147, 149, 150, 151, 152, 153, 156, 157, 158, 160, 161, 164, 165, 166, 167, 170, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 183, 184, 186, 187, 188, 189, 190, 191, 193, 194, 196, 198, 199, 200, 201, 202, 203, 206, 208, 209, 210, 211, 212, 214, 215, 217, 218, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 235, 237, 238, 239, 242, 243, 245, 246, 247, 249, 251, 252, 253, 254, 255, 257, 258, 259, 260, 262, 264, 265, 266, 268, 269, 270, 271, 272, 274, 275, 276, 277, 280, 282, 284, 285, 286, 288, 290, 291, 293, 294, 295, 297, 298, 299, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 315, 316, 317, 318, 320, 321, 322, 323, 325, 326, 328, 329, 330, 332, 333, 335, 338, 339, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 359, 361, 362, 363, 364, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 387, 388, 389, 390, 391, 392, 393, 394, 395, 397, 398, 400, 401, 405, 407, 408, 409, 410, 411, 412, 413, 414, 416, 417, 418, 421, 422, 423, 424, 426, 427, 428, 429, 430, 431, 432, 434, 435, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 452, 453, 454, 456, 457, 458, 459, 460, 462, 465, 466, 467, 468, 469, 470, 472, 476, 477, 478, 479, 480, 482, 483, 484, 485, 486, 487, 489, 490, 491, 492, 494, 495, 496, 497, 498, 499, 500, 501, 502, 504, 505, 506, 507, 508, 509, 510], [0, 1, 2, 6, 7, 8, 9, 10, 12, 13, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 61, 62, 63, 64, 65, 66, 67, 69, 70, 71, 72, 73, 74, 75, 76, 78, 80, 83, 84, 85, 86, 87, 88, 89, 90, 92, 94, 95, 96, 97, 98, 101, 102, 103, 106, 107, 108, 109, 110, 111, 112, 114, 115, 117, 119, 120, 121, 123, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 152, 156, 157, 158, 159, 161, 162, 163, 164, 166, 167, 168, 170, 171, 172, 173, 174, 175, 176, 177, 179, 182, 183, 184, 189, 190, 191, 192, 194, 195, 197, 198, 199, 201, 202, 205, 206, 208, 209, 210, 211, 212, 213, 216, 218, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 234, 235, 237, 238, 239, 240, 241, 243, 244, 245, 247, 248, 249, 250, 252, 255, 256, 259, 260, 262, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 285, 286, 287, 288, 289, 290, 292, 293, 294, 295, 296, 297, 299, 300, 301, 302, 305, 306, 308, 310, 311, 313, 314, 315, 316, 317, 320, 322, 323, 324, 325, 326, 327, 328, 329, 330, 332, 333, 334, 336, 337, 338, 339, 340, 343, 345, 346, 347, 348, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 365, 366, 367, 370, 372, 373, 375, 377, 378, 379, 380, 381, 382, 383, 387, 388, 389, 390, 392, 394, 395, 396, 397, 399, 402, 403, 405, 407, 410, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 425, 426, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 453, 454, 455, 459, 461, 462, 466, 467, 469, 470, 471, 472, 473, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 492, 493, 494, 496, 499, 501, 502, 504, 506, 507, 508, 509, 510], [0, 1, 2, 3, 6, 7, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 22, 24, 25, 27, 28, 30, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 54, 55, 57, 58, 59, 60, 61, 62, 65, 66, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 86, 87, 88, 89, 90, 93, 94, 96, 97, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 116, 117, 118, 119, 120, 122, 123, 124, 125, 126, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 144, 145, 146, 147, 149, 150, 152, 153, 155, 156, 159, 160, 162, 163, 164, 165, 167, 170, 172, 173, 174, 175, 176, 177, 180, 181, 183, 185, 187, 189, 190, 191, 192, 193, 194, 196, 197, 198, 201, 202, 203, 204, 205, 206, 209, 210, 211, 212, 214, 215, 216, 218, 219, 220, 223, 224, 225, 227, 228, 229, 230, 231, 232, 233, 234, 236, 238, 240, 242, 243, 244, 245, 246, 247, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 276, 277, 278, 279, 281, 282, 283, 284, 285, 286, 287, 288, 291, 293, 295, 297, 298, 299, 300, 301, 302, 305, 306, 307, 308, 311, 312, 314, 315, 317, 318, 319, 321, 322, 323, 324, 327, 329, 330, 332, 333, 334, 336, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 354, 356, 357, 359, 361, 362, 363, 364, 365, 366, 368, 370, 371, 372, 375, 376, 378, 379, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 398, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 415, 416, 417, 418, 419, 420, 422, 423, 424, 426, 427, 428, 430, 431, 432, 434, 435, 436, 437, 439, 440, 441, 442, 443, 446, 447, 449, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 465, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 482, 483, 484, 485, 488, 489, 490, 491, 492, 495, 496, 498, 499, 500, 501, 503, 504, 505, 506, 508, 509, 510, 511], [1, 2, 3, 5, 6, 8, 10, 11, 12, 13, 15, 17, 18, 19, 20, 21, 22, 24, 26, 28, 29, 34, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 55, 56, 57, 58, 59, 60, 62, 63, 66, 67, 68, 70, 71, 72, 73, 75, 76, 77, 79, 80, 81, 82, 83, 84, 85, 86, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 112, 113, 114, 115, 116, 117, 118, 119, 120, 122, 125, 126, 127, 128, 131, 132, 133, 137, 138, 139, 140, 141, 143, 144, 146, 149, 150, 151, 152, 153, 156, 159, 160, 162, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 178, 179, 180, 182, 183, 184, 185, 187, 188, 190, 191, 192, 193, 195, 196, 197, 198, 199, 200, 203, 204, 206, 208, 209, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 223, 224, 225, 226, 229, 230, 232, 233, 234, 235, 236, 237, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 251, 252, 253, 254, 255, 256, 260, 262, 263, 264, 265, 266, 268, 269, 272, 275, 276, 277, 278, 280, 281, 282, 283, 284, 285, 286, 287, 288, 291, 292, 293, 295, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 324, 326, 327, 329, 330, 331, 332, 333, 334, 336, 337, 338, 339, 340, 341, 342, 346, 347, 350, 353, 354, 355, 358, 359, 360, 361, 362, 363, 364, 366, 367, 369, 370, 372, 373, 374, 375, 376, 378, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 407, 413, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 427, 428, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 443, 444, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 462, 463, 464, 465, 466, 467, 468, 470, 471, 472, 473, 474, 475, 477, 478, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 493, 494, 495, 497, 498, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510, 511], [1, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 43, 44, 46, 47, 48, 49, 50, 51, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 73, 75, 76, 77, 79, 80, 81, 82, 83, 85, 87, 88, 89, 90, 92, 93, 94, 96, 97, 99, 102, 103, 104, 106, 107, 108, 110, 111, 112, 113, 114, 117, 118, 119, 120, 121, 122, 123, 124, 126, 127, 130, 132, 133, 134, 135, 136, 137, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 152, 153, 155, 156, 157, 158, 161, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 181, 183, 184, 185, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 225, 226, 228, 231, 232, 234, 235, 236, 237, 238, 240, 241, 242, 245, 246, 247, 250, 252, 253, 254, 255, 256, 257, 259, 261, 263, 264, 265, 267, 268, 272, 273, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 288, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 300, 303, 304, 306, 307, 308, 312, 314, 315, 317, 318, 319, 320, 321, 322, 323, 324, 326, 327, 328, 329, 330, 331, 332, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 350, 351, 354, 355, 356, 360, 362, 363, 364, 365, 366, 369, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 390, 391, 395, 396, 398, 399, 401, 402, 405, 406, 407, 408, 409, 411, 412, 413, 415, 417, 418, 419, 421, 422, 423, 424, 426, 427, 428, 429, 430, 433, 434, 438, 440, 441, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 456, 458, 461, 462, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 476, 478, 479, 481, 482, 483, 484, 485, 486, 487, 488, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 504, 506, 507, 508, 509, 510, 511], [0, 2, 3, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 18, 20, 21, 22, 24, 25, 27, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 40, 42, 44, 45, 46, 47, 48, 50, 51, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 68, 69, 71, 73, 74, 75, 76, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 101, 102, 104, 106, 107, 108, 112, 113, 115, 116, 117, 120, 121, 122, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 136, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 158, 159, 160, 162, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 185, 186, 187, 189, 190, 191, 192, 194, 197, 198, 199, 200, 202, 203, 204, 207, 208, 210, 211, 212, 213, 214, 215, 216, 218, 219, 220, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 256, 257, 258, 259, 260, 262, 263, 265, 266, 268, 270, 271, 272, 273, 274, 275, 278, 279, 282, 283, 284, 285, 288, 289, 290, 291, 292, 293, 296, 297, 298, 300, 301, 304, 305, 308, 310, 311, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 324, 325, 327, 328, 330, 332, 334, 335, 337, 338, 339, 340, 342, 344, 345, 346, 347, 348, 349, 350, 351, 354, 355, 356, 357, 358, 360, 361, 362, 363, 366, 367, 368, 369, 370, 373, 374, 375, 378, 379, 380, 381, 384, 385, 386, 387, 389, 391, 393, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 408, 409, 411, 412, 413, 414, 415, 416, 417, 418, 419, 421, 423, 424, 425, 426, 428, 429, 430, 431, 432, 433, 434, 436, 437, 440, 441, 442, 443, 445, 447, 448, 449, 450, 451, 454, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 473, 476, 477, 478, 480, 482, 484, 485, 487, 490, 492, 493, 494, 495, 496, 497, 498, 499, 500, 502, 503, 504, 506, 507, 509, 510, 511], [0, 1, 2, 3, 5, 6, 8, 10, 11, 12, 13, 15, 16, 18, 19, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32, 33, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 51, 52, 53, 54, 56, 57, 58, 59, 61, 62, 64, 65, 67, 68, 69, 70, 72, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 86, 87, 88, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 102, 103, 104, 105, 107, 109, 110, 111, 112, 113, 115, 116, 117, 118, 119, 120, 121, 122, 125, 126, 127, 129, 130, 131, 133, 135, 136, 138, 139, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 178, 183, 184, 187, 189, 190, 192, 193, 194, 196, 198, 199, 201, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 216, 218, 220, 223, 225, 226, 227, 229, 231, 233, 234, 236, 237, 239, 240, 241, 243, 244, 245, 246, 247, 248, 250, 252, 253, 255, 256, 257, 258, 260, 261, 264, 266, 267, 268, 270, 271, 272, 274, 275, 276, 277, 279, 280, 281, 282, 283, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 316, 318, 319, 320, 321, 323, 324, 325, 326, 327, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 344, 345, 347, 350, 351, 352, 354, 355, 357, 358, 360, 363, 364, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 384, 385, 387, 388, 389, 391, 392, 393, 394, 395, 396, 398, 400, 401, 402, 403, 404, 405, 407, 408, 409, 411, 412, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 443, 444, 446, 447, 448, 450, 452, 453, 454, 456, 457, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 470, 471, 472, 473, 474, 475, 476, 477, 478, 480, 481, 483, 484, 485, 489, 494, 495, 496, 497, 499, 502, 503, 504, 505, 506], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 32, 33, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 56, 57, 59, 60, 62, 64, 65, 66, 67, 69, 70, 73, 74, 75, 77, 78, 79, 80, 81, 82, 85, 86, 89, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105, 107, 109, 110, 111, 112, 113, 115, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 131, 133, 134, 136, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 154, 155, 157, 158, 159, 160, 162, 163, 164, 165, 166, 167, 170, 171, 172, 173, 174, 175, 177, 178, 179, 181, 183, 184, 186, 187, 188, 189, 190, 192, 193, 196, 197, 201, 202, 204, 205, 206, 207, 210, 211, 212, 213, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 229, 230, 233, 234, 235, 237, 238, 239, 241, 242, 243, 246, 247, 248, 249, 250, 252, 253, 254, 255, 256, 258, 259, 261, 263, 264, 265, 266, 267, 268, 269, 270, 271, 273, 274, 275, 276, 279, 281, 282, 283, 284, 288, 290, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 313, 314, 316, 317, 319, 320, 321, 322, 324, 325, 326, 327, 328, 331, 332, 334, 336, 337, 338, 339, 340, 342, 343, 345, 346, 347, 348, 349, 351, 352, 355, 356, 357, 358, 359, 361, 363, 366, 367, 368, 371, 372, 373, 374, 375, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 389, 390, 393, 394, 395, 396, 400, 401, 402, 403, 404, 405, 406, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 426, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 444, 445, 446, 448, 449, 450, 452, 453, 456, 457, 458, 460, 461, 462, 463, 464, 465, 466, 467, 469, 470, 471, 472, 473, 477, 478, 479, 481, 482, 483, 487, 489, 490, 491, 492, 494, 495, 496, 497, 498, 499, 500, 504, 505, 506, 507, 508, 509, 511], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 17, 18, 19, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 39, 41, 43, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 72, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 88, 89, 90, 92, 95, 96, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 133, 134, 135, 137, 138, 139, 140, 141, 142, 144, 145, 146, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 162, 163, 165, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 185, 186, 187, 189, 190, 191, 192, 194, 196, 197, 199, 203, 204, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 219, 220, 221, 222, 224, 226, 227, 228, 229, 232, 233, 234, 235, 236, 237, 239, 240, 242, 243, 244, 245, 247, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 265, 266, 267, 269, 270, 271, 273, 274, 275, 276, 278, 281, 282, 283, 284, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 308, 309, 312, 313, 315, 316, 318, 319, 323, 325, 326, 328, 329, 330, 331, 332, 334, 336, 337, 338, 339, 342, 344, 346, 347, 348, 351, 352, 353, 356, 357, 358, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 372, 373, 374, 377, 379, 380, 381, 382, 383, 386, 388, 389, 390, 391, 392, 394, 396, 398, 399, 401, 402, 403, 404, 406, 407, 408, 409, 410, 411, 412, 414, 415, 416, 417, 419, 422, 424, 425, 429, 430, 431, 432, 433, 435, 437, 438, 441, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 457, 458, 460, 461, 462, 463, 464, 465, 466, 467, 468, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 488, 490, 492, 493, 494, 496, 499, 500, 501, 502, 504, 507, 509, 510, 511], [0, 2, 4, 6, 7, 8, 9, 10, 11, 12, 14, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 41, 44, 46, 47, 48, 49, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64, 65, 66, 68, 69, 70, 72, 73, 74, 75, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 91, 92, 96, 99, 100, 101, 102, 103, 105, 106, 107, 109, 110, 112, 113, 114, 115, 116, 117, 119, 120, 123, 124, 125, 126, 127, 129, 133, 134, 135, 136, 137, 138, 141, 143, 144, 145, 146, 147, 148, 149, 150, 153, 155, 156, 157, 160, 162, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 184, 185, 186, 188, 189, 190, 191, 194, 195, 197, 198, 199, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 215, 216, 217, 219, 222, 223, 224, 227, 228, 229, 230, 232, 234, 235, 237, 238, 239, 240, 241, 243, 244, 246, 247, 248, 250, 251, 252, 253, 254, 255, 257, 258, 259, 262, 263, 264, 265, 266, 268, 270, 274, 275, 278, 279, 280, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 313, 314, 315, 316, 317, 318, 319, 320, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 334, 335, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 365, 366, 367, 368, 369, 371, 373, 374, 378, 382, 383, 384, 385, 388, 389, 390, 391, 393, 394, 395, 396, 397, 398, 400, 403, 404, 405, 406, 407, 409, 410, 411, 412, 413, 415, 418, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 437, 439, 440, 442, 444, 445, 446, 447, 449, 450, 451, 452, 453, 455, 457, 458, 459, 460, 461, 462, 463, 464, 465, 467, 468, 469, 470, 471, 472, 474, 475, 476, 478, 481, 483, 484, 486, 487, 489, 490, 491, 493, 494, 497, 498, 499, 500, 501, 502, 503, 504, 505, 507, 508, 511], [0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 24, 25, 26, 27, 28, 29, 32, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 46, 49, 50, 51, 52, 53, 54, 55, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 73, 77, 78, 80, 83, 84, 85, 86, 87, 88, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 103, 104, 105, 107, 108, 109, 110, 113, 116, 117, 118, 120, 121, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 136, 138, 139, 140, 141, 142, 143, 145, 147, 149, 151, 152, 155, 157, 158, 160, 162, 163, 164, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 179, 180, 181, 182, 183, 184, 185, 188, 190, 191, 192, 194, 195, 196, 197, 198, 201, 203, 204, 205, 206, 208, 209, 210, 213, 215, 216, 217, 218, 220, 221, 222, 223, 225, 226, 227, 230, 231, 232, 233, 235, 236, 237, 240, 244, 245, 246, 248, 249, 251, 252, 255, 257, 258, 259, 260, 261, 264, 265, 266, 267, 268, 269, 271, 272, 273, 275, 276, 277, 280, 281, 282, 283, 286, 287, 288, 289, 290, 292, 293, 294, 295, 297, 299, 300, 301, 302, 304, 305, 306, 307, 308, 311, 312, 313, 315, 316, 317, 319, 324, 325, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 338, 339, 340, 342, 343, 344, 345, 346, 348, 349, 350, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 363, 364, 365, 366, 368, 369, 373, 374, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 395, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408, 409, 412, 414, 415, 416, 417, 418, 419, 420, 421, 422, 424, 425, 426, 427, 429, 430, 431, 432, 433, 434, 435, 436, 438, 439, 440, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 462, 463, 464, 465, 467, 469, 471, 472, 473, 475, 476, 477, 478, 479, 480, 481, 482, 484, 485, 487, 489, 490, 491, 492, 493, 494, 495, 496, 498, 499, 500, 501, 504, 505, 506, 507, 508, 510], [0, 2, 3, 5, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 27, 30, 33, 34, 36, 37, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 99, 100, 102, 103, 104, 105, 106, 107, 109, 110, 113, 114, 115, 119, 121, 122, 123, 124, 128, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 150, 151, 152, 153, 154, 155, 157, 159, 162, 163, 165, 166, 168, 170, 171, 172, 173, 174, 175, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 201, 202, 203, 205, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 222, 225, 226, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 243, 245, 247, 248, 249, 250, 251, 253, 255, 256, 257, 258, 260, 261, 265, 267, 268, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 282, 283, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 296, 299, 300, 301, 302, 303, 304, 305, 306, 307, 310, 312, 313, 315, 316, 317, 318, 319, 321, 322, 323, 325, 327, 329, 330, 331, 332, 334, 339, 340, 341, 342, 345, 347, 349, 351, 352, 353, 354, 355, 357, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 373, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 396, 397, 398, 400, 402, 403, 404, 406, 407, 408, 410, 411, 412, 413, 414, 416, 417, 418, 419, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 438, 439, 440, 441, 442, 443, 444, 447, 448, 450, 452, 455, 456, 457, 458, 461, 462, 463, 464, 465, 466, 469, 470, 471, 472, 473, 476, 478, 479, 480, 482, 484, 485, 487, 489, 490, 491, 492, 493, 495, 496, 499, 500, 501, 502, 504, 507, 508, 509], [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 23, 24, 25, 26, 27, 29, 30, 31, 32, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 63, 64, 67, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 89, 91, 92, 93, 94, 95, 97, 98, 99, 100, 102, 104, 105, 106, 107, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 139, 141, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 175, 177, 178, 180, 181, 184, 185, 187, 190, 191, 192, 193, 195, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 207, 209, 210, 212, 213, 215, 216, 217, 218, 219, 220, 221, 222, 223, 226, 228, 229, 230, 231, 232, 233, 234, 236, 237, 240, 241, 242, 243, 244, 245, 246, 248, 253, 255, 256, 259, 260, 262, 266, 267, 269, 270, 273, 274, 275, 276, 277, 278, 279, 280, 282, 283, 284, 285, 289, 290, 292, 293, 294, 295, 296, 297, 300, 302, 303, 305, 306, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 326, 329, 332, 334, 335, 336, 337, 338, 339, 340, 342, 344, 345, 346, 349, 350, 351, 352, 353, 354, 355, 356, 357, 359, 361, 363, 364, 365, 367, 369, 370, 371, 372, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 391, 392, 393, 394, 395, 396, 397, 398, 400, 401, 406, 407, 408, 410, 411, 413, 414, 415, 416, 418, 419, 420, 421, 422, 423, 424, 425, 427, 428, 430, 431, 433, 435, 436, 438, 441, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 455, 456, 457, 458, 459, 460, 461, 462, 464, 465, 466, 470, 471, 475, 476, 477, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 493, 494, 495, 496, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510], [0, 1, 4, 5, 8, 9, 12, 14, 16, 17, 21, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 50, 51, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 67, 68, 69, 70, 73, 74, 75, 76, 77, 79, 81, 82, 84, 85, 86, 87, 88, 89, 90, 91, 92, 94, 95, 97, 99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 110, 111, 116, 117, 118, 119, 120, 121, 122, 124, 125, 127, 129, 130, 131, 133, 134, 135, 136, 138, 139, 140, 141, 142, 143, 144, 145, 147, 148, 150, 152, 154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 165, 167, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 181, 182, 183, 184, 186, 188, 189, 190, 191, 192, 195, 196, 197, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 211, 212, 213, 214, 215, 216, 217, 219, 220, 221, 223, 224, 225, 227, 228, 231, 232, 233, 234, 235, 237, 238, 240, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 255, 256, 257, 258, 260, 262, 263, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 280, 281, 283, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 307, 308, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 322, 323, 324, 325, 326, 327, 328, 329, 332, 333, 335, 337, 338, 339, 341, 342, 343, 344, 345, 346, 348, 350, 351, 352, 353, 354, 355, 356, 358, 359, 361, 362, 364, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 379, 380, 381, 383, 384, 385, 386, 387, 389, 390, 393, 394, 395, 396, 397, 399, 401, 402, 404, 405, 408, 409, 410, 411, 412, 413, 415, 417, 418, 419, 420, 422, 424, 430, 432, 433, 434, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 460, 461, 462, 464, 465, 466, 468, 469, 471, 472, 473, 474, 477, 479, 481, 482, 483, 484, 485, 486, 489, 490, 491, 492, 494, 495, 497, 499, 500, 501, 502, 503, 505, 506, 509, 510], [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 14, 15, 16, 18, 21, 22, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 55, 56, 57, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 78, 79, 83, 84, 85, 86, 87, 89, 90, 94, 95, 97, 99, 101, 103, 104, 106, 110, 111, 113, 114, 115, 116, 118, 119, 121, 122, 124, 125, 126, 127, 130, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 145, 146, 147, 148, 149, 150, 153, 154, 155, 157, 158, 159, 160, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 176, 178, 179, 180, 181, 182, 183, 184, 185, 186, 188, 189, 191, 193, 194, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 208, 209, 210, 211, 212, 213, 214, 215, 216, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 237, 238, 239, 240, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 274, 275, 277, 278, 280, 281, 282, 283, 286, 288, 289, 290, 293, 294, 295, 296, 297, 299, 300, 303, 304, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 321, 322, 323, 324, 325, 327, 328, 330, 331, 332, 333, 334, 335, 337, 338, 339, 341, 342, 343, 346, 347, 348, 349, 351, 352, 354, 355, 356, 357, 359, 360, 361, 363, 365, 366, 367, 368, 370, 371, 372, 373, 374, 375, 376, 379, 381, 385, 386, 388, 390, 394, 397, 398, 401, 402, 404, 405, 406, 407, 408, 412, 415, 416, 418, 419, 420, 421, 425, 426, 428, 429, 430, 431, 433, 434, 435, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 462, 464, 465, 466, 468, 470, 471, 472, 473, 475, 476, 477, 479, 480, 481, 482, 483, 484, 485, 487, 488, 489, 492, 494, 495, 496, 497, 500, 501, 502, 503, 504, 507, 508, 509, 510, 511], [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 30, 31, 32, 33, 35, 36, 37, 41, 42, 43, 44, 45, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 76, 77, 78, 80, 81, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 96, 97, 100, 101, 104, 107, 109, 110, 111, 112, 113, 114, 115, 119, 120, 121, 122, 123, 124, 128, 129, 130, 132, 135, 138, 139, 140, 141, 142, 143, 144, 145, 147, 149, 150, 151, 152, 154, 156, 158, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 174, 177, 181, 184, 185, 186, 187, 188, 191, 192, 193, 194, 195, 196, 197, 199, 200, 201, 203, 205, 206, 208, 209, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 233, 235, 236, 237, 239, 240, 241, 243, 245, 246, 247, 248, 249, 250, 252, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 271, 272, 273, 274, 276, 277, 278, 279, 280, 281, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 296, 297, 299, 300, 303, 304, 305, 307, 308, 311, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 327, 329, 330, 331, 332, 333, 334, 336, 337, 338, 339, 340, 341, 342, 343, 344, 346, 348, 349, 350, 351, 353, 355, 356, 357, 358, 359, 361, 362, 363, 364, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 389, 390, 391, 392, 393, 394, 396, 398, 401, 403, 404, 405, 406, 407, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 425, 426, 429, 431, 432, 433, 434, 435, 436, 437, 438, 439, 441, 443, 444, 445, 446, 450, 452, 454, 455, 457, 458, 459, 461, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 477, 483, 484, 485, 487, 489, 490, 491, 492, 493, 495, 496, 497, 499, 502, 503, 504, 505, 506, 507, 508, 510, 511], [1, 2, 3, 4, 6, 8, 9, 11, 12, 13, 14, 16, 17, 18, 20, 22, 24, 25, 26, 27, 29, 31, 33, 34, 35, 36, 37, 39, 40, 42, 47, 48, 49, 50, 52, 54, 55, 56, 58, 60, 61, 63, 64, 65, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 117, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 131, 132, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 162, 163, 164, 166, 168, 169, 171, 173, 175, 176, 177, 178, 179, 181, 184, 185, 186, 188, 189, 190, 194, 195, 196, 199, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 215, 217, 218, 220, 221, 222, 223, 224, 225, 227, 229, 231, 232, 233, 234, 235, 236, 237, 239, 240, 242, 244, 245, 247, 248, 249, 250, 252, 253, 254, 255, 256, 258, 259, 260, 261, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 280, 281, 282, 283, 284, 285, 286, 287, 289, 290, 291, 292, 294, 296, 297, 298, 299, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 313, 315, 317, 318, 319, 320, 321, 323, 324, 325, 326, 328, 329, 330, 331, 332, 333, 334, 336, 337, 338, 340, 342, 343, 344, 345, 346, 347, 348, 349, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 364, 365, 366, 367, 368, 369, 370, 372, 376, 377, 378, 379, 380, 381, 383, 384, 385, 387, 388, 389, 390, 391, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 406, 407, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 423, 424, 425, 427, 431, 433, 438, 440, 441, 442, 443, 444, 445, 446, 448, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 462, 463, 467, 468, 469, 470, 472, 473, 475, 476, 478, 479, 480, 481, 482, 484, 485, 486, 487, 489, 491, 492, 494, 495, 496, 497, 498, 500, 501, 506, 507, 508, 509]]}')
          _ov = {}
          if "--json-model-override-args" in EXTRA_SERVE_ARGS:
              _i = EXTRA_SERVE_ARGS.index("--json-model-override-args")
              _ov = json.loads(EXTRA_SERVE_ARGS[_i + 1]); del EXTRA_SERVE_ARGS[_i:_i + 2]
          _ov.setdefault("text_config", {})["num_experts"] = CFG["PRUNE_N"]
          EXTRA_SERVE_ARGS += ["--json-model-override-args", json.dumps(_ov, separators=(",", ":"))]
          PLE_ENV["SGLANG_PRUNE_KEEP"] = str(KEEP_PATH)
          print("expert pruning:", CFG["PRUNE_N"], "experts/layer |", EXTRA_SERVE_ARGS, flush=True)
  except Exception as _ex:
    stage_failed('model', _ex, fatal=True)


skipping stage model: earlier stage failed: ['quick_version']


In [7]:
# [duck2:serve]  TP4/EP4 Flash-Next serve with a fallback ladder (first healthy rung that passes the echo gate wins)
if FAILED:
    print('skipping stage serve: earlier stage failed:', FAILED, flush=True)
else:
  try:
      PORT = CFG["PORT"]; SERVED_NAME = CFG["SERVED_NAME"]; ENDPOINT = f"http://127.0.0.1:{PORT}"
      FATAL = ("hit an exception", "Received sigquit from a child process", "CUDA out of memory", "OutOfMemoryError",
               "Loaded weights leave no GPU memory", "Not enough memory")
      MEMORY_MARKERS = ("out of memory", "OutOfMemoryError", "Not enough memory", "leave no GPU memory")
      DETERMINISTIC = ("ImportError", "ModuleNotFoundError", "SyntaxError", "NameError", "AttributeError", "TypeError",
                       "CalledProcessError", "CompilationError", "nvcc", "ptxas", "compile")
      def last_line(off=0):
          ls = [l.strip() for l in re.split(r"[\r\n]+", log_text(max(off, log_size() - 8000))) if l.strip()]
          return ls[-1][:220] if ls else ""

      def failure_sig(txt):
          """Normalised last exception line of a failed rung ("MEMORY" for any OOM-type failure, None if none)."""
          if any(k in txt for k in MEMORY_MARKERS):
              return "MEMORY"
          ex = re.findall(r"^\s*([A-Za-z_][\w.]*(?:Error|Exception)): ?(.*)$", txt, re.M)
          if not ex:
              return None
          name, msg = ex[-1]
          return re.sub(r"0x[0-9a-fA-F]+|\d+", "#", f"{name}: {msg}")[:200]

      def kv_tokens(off):
          m = [int(x) for x in re.findall(r"max_total_num_tokens=(\d+)", log_text(off))]
          return min(m) if m else 0

      def size_sessions(kv, maxreq):
          """(in-flight sessions, per-session ctx-limit) so that sum(ctx-limit + turn-max) <= KV_USE x pool."""
          turn = CFG["HARNESS"]["turn-max-tokens"]
          top = max(1, min(maxreq, CFG["MAX_INFLIGHT"]))
          if kv <= 0:                                      # pool size not logged: conservative default
              return min(2, top), CFG["HARNESS"]["ctx-limit"]
          use = CFG["KV_USE"] * kv
          for n in range(top, 0, -1):
              ctx = int(use / n - turn)
              if ctx >= CFG["CTX_MIN"]:
                  return n, min(CFG["CTX_MAX"], ctx)
          return 1, max(8000, min(CFG["CTX_MAX"], int(use - turn)))

      def layout_args(r):
          if r.get("LAYOUT") == "dpattn":
              return ["--tp-size", "4", "--dp-size", "2", "--enable-dp-attention", "--ep-size", "4", "--enable-dp-lm-head"]
          return ["--tp-size", str(CFG["TP"]), "--ep-size", str(CFG["EP"])]

      def graph_max(r):   # decode CUDA-graph batch per attention-DP group
          return r["MAXREQ"] // (2 if r.get("LAYOUT") == "dpattn" else 1)

      def graph_bs(m):
          return sorted({b for b in (1, 2, 4, 8, 12, 16, 20, 24, 32, 40, 48) if b <= m} | {m})

      def rung_args(r):
          maxreq, spec = r["MAXREQ"], r["SPEC"]
          args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
              "--model-loader-extra-config", '{"enable_multithread_load":false}', "--weight-loader-prefetch-checkpoints",
              "--trust-remote-code", "--served-model-name", SERVED_NAME, "--host", "127.0.0.1", "--port", str(PORT),
              *layout_args(r), "--moe-a2a-backend", "none", "--moe-runner-backend", "auto",
              "--dtype", "bfloat16", "--quantization", "auto-round", "--ple-offload-embedding",
              "--attention-backend", "triton", "--kv-cache-dtype", r["KV"], "--page-size", "64",
              "--mem-fraction-static", str(r["MEMFRAC"]), "--context-length", str(CFG["CTX"]),
              "--max-running-requests", str(maxreq), "--chunked-prefill-size", str(r.get("CHUNK", CFG["CHUNK"])),
              # NEVER drop --disable-prefill-cuda-graph (or change the decode-graph setting) on L4 without re-deriving the
              # KV budget: with prefill graphs on, post-capture sizing reserves ~4 GiB/GPU of activation slack for a mamba
              # model (runtime_context.py:1607-1636, kv_cache_configurator.py:1882-1889), more than the whole L4 KV pool.
              "--max-prefill-tokens", str(r.get("CHUNK", CFG["CHUNK"])), "--disable-prefill-cuda-graph",
              "--cuda-graph-max-bs-decode", str(graph_max(r)), "--cuda-graph-bs-decode", *map(str, r.get("GRAPH_BS") or graph_bs(graph_max(r))),
              "--mamba-ssm-dtype", "bfloat16", "--max-mamba-cache-size", str(r["MAMBA"]),
              "--mamba-radix-cache-strategy", r.get("MAMBA_STRATEGY", "extra_buffer"), "--mamba-track-interval", "64",
              "--mamba-backend", "triton", "--linear-attn-backend", "triton",
              "--linear-attn-decode-backend", "triton", "--linear-attn-prefill-backend", "triton",
              "--disable-custom-all-reduce", "--disable-flashinfer-autotune",
              "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
              "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
              "--chat-template", str(Path(MODEL_DIR, "chat_template.jinja")),
              "--default-chat-template-kwargs", '{"preserve_thinking":true}',
              "--schedule-policy", "lpm", "--watchdog-timeout", "1800", "--enable-cache-report", "--enable-metrics",
              *EXTRA_SERVE_ARGS]
          if r.get("NO_GRAPH"):
              args += ["--disable-decode-cuda-graph"]
          if spec:
              args += ["--gdn-mtp-cache-mode", "none",
                  "--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
                  "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"] + 1),
                  "--speculative-draft-model-path", str(DRAFT_VIEW),
                  "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
                  "--speculative-draft-kv-cache-dtype", r["KV"],
                  "--speculative-accept-threshold-single", "1.0", "--speculative-accept-threshold-acc", "1.0"]
              if CFG.get("PRUNE_N"):
                  args += ["--speculative-draft-model-override-args", "{}"]   # drafter keeps its 512 experts
              if TOKEN_MAP:
                  args += ["--speculative-token-map", TOKEN_MAP]
          e = dict(env, SGLANG_QSA_SM89_DECODE=r.get("QSA", QSA_DEFAULT), **PLE_ENV)
          if r.get("LAYOUT") == "dpattn":
              e["CUDA_VISIBLE_DEVICES"] = CFG["DP_GPU_ORDER"]   # attention pairs = consecutive TP ranks
          if r.get("SKIP_LOCK"):
              e["SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK"] = "1"
          e.update({k: str(v) for k, v in (r.get("ENV") or {}).items()})   # per-rung extra env
          if r.get("LMONLY"):
              if "--json-model-override-args" in args:
                  _i = args.index("--json-model-override-args")
                  _ov = json.loads(args[_i + 1]); _ov["language_model_only"] = True
                  args[_i + 1] = json.dumps(_ov, separators=(",", ":"))
              else:
                  args += ["--json-model-override-args", json.dumps({"language_model_only": True})]
          args += [str(x) for x in (r.get("SERVE_ARGS") or [])]   # per-rung extra server flags (e.g. --max-total-tokens)
          if r.get("PRUNE") is False:   # full model: no keep list, no num_experts override
              e.pop("SGLANG_PRUNE_KEEP", None)
              _drop = {"--json-model-override-args", "--speculative-draft-model-override-args"}
              args = [x for i, x in enumerate(args) if x not in _drop and (i == 0 or args[i - 1] not in _drop)]
          return args, e

      def gpu_used_gib():
          try:
              out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                   capture_output=True, text=True, timeout=10).stdout
              return [float(x) / 1024 for x in out.split()]
          except Exception:
              return []

      def stop_server(proc):
          for sig, wait in ((signal.SIGTERM, 20), (signal.SIGKILL, 5)):
              try:
                  os.killpg(proc.pid, sig)
              except Exception:
                  pass
              try:
                  proc.wait(timeout=wait)
              except Exception:
                  pass
          t = time.time()
          while time.time() - t < 180:             # pinned PLE + GPU memory must be released before the next rung
              u = gpu_used_gib()
              if u and max(u) < 1.5:
                  break
              time.sleep(5)
          print("server stopped; GPU used GiB", [round(x, 1) for x in gpu_used_gib()], flush=True)

      def chat(messages, max_tokens=256, timeout=600, **kw):
          body = {"model": SERVED_NAME, "messages": messages, "max_tokens": max_tokens, **kw}
          req = urllib.request.Request(f"{ENDPOINT}/v1/chat/completions", data=json.dumps(body).encode(),
                                       headers={"Content-Type": "application/json"})
          t = time.time()
          with urllib.request.urlopen(req, timeout=timeout) as r:
              d = json.loads(r.read())
          d["_latency"] = time.time() - t
          return d

      ECHO_GRID = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 0, 1], [2, 2, 0, 0, 3, 3], [5, 0, 5, 0, 5, 0], [9, 8, 7, 6, 5, 4]]
      def echo_test():
          g = "\n".join("".join(map(str, row)) for row in ECHO_GRID)
          try:
              d = chat([{"role": "user", "content": f"Copy this grid exactly, one row per line, digits only, nothing else:\n{g}"}],
                       max_tokens=200, timeout=300, temperature=0.0, chat_template_kwargs={"enable_thinking": False})
          except Exception as ex:
              return False, f"{type(ex).__name__}: {ex}"
          txt = d["choices"][0]["message"].get("content") or ""
          return g in txt.replace(" ", ""), txt[:200].replace("\n", "|")

      def start_rung(i, r, hard_deadline):
          args, e = rung_args(r)
          for _ in range(30):                      # port must be free (a previous rung may still be exiting)
              with socket.socket() as s:
                  if s.connect_ex(("127.0.0.1", PORT)) != 0:
                      break
              time.sleep(2)
          off = log_size()
          print(f"\n=== rung {i} {r['NAME']} (QSA {e['SGLANG_QSA_SM89_DECODE']}) at +{(time.time()-T0)/60:.1f} min ===\n"
                + shlex.join(args) + "\n", flush=True)
          with open(LOG, "ab", buffering=0) as logf:
              logf.write(f"\n===== duck2 rung {r['NAME']} =====\n".encode())
              proc = subprocess.Popen(args, env=e, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
          t0 = time.time(); limit = min(hard_deadline, t0 + CFG["RUNG_MAX_MIN"][min(i, len(CFG["RUNG_MAX_MIN"]) - 1)] * 60)
          fatal_at, last_size, last_growth, last_report, why = None, -1, t0, -60, "timeout"
          while time.time() < limit:
              if proc.poll() is not None:
                  why = f"exited rc={proc.returncode}"; break
              size = log_size()
              if size != last_size:
                  last_size, last_growth = size, time.time()
              elif time.time() - last_growth > CFG["STALL_MIN"] * 60:
                  why = f"log stalled {CFG['STALL_MIN']} min"; break
              if fatal_at is None and any(k in log_text(off) for k in FATAL):
                  fatal_at = time.time(); print("fatal marker in serve.log; grace", CFG["FATAL_GRACE_S"], "s", flush=True)
              if fatal_at and time.time() - fatal_at > CFG["FATAL_GRACE_S"]:
                  why = "fatal marker"; break
              try:
                  with urllib.request.urlopen(f"{ENDPOINT}/health", timeout=5) as h:
                      if h.status == 200:
                          print(f"HEALTHY {r['NAME']} after {(time.time()-t0)/60:.1f} min", flush=True)
                          ok, txt = echo_test()
                          print(f"greedy grid echo: {'PASS' if ok else 'FAIL'} | {txt}", flush=True)
                          if ok or not CFG["SANITY_GATE"]:
                              return proc, off, None, False
                          why = "echo gate failed"; break
              except Exception:
                  pass
              if time.time() - last_report >= 60:
                  last_report = time.time()
                  print(f"  {r['NAME']} {(time.time()-t0)/60:4.1f} min | {last_line(off)}", flush=True)
              time.sleep(3)
          print(f"FAILED rung {r['NAME']}: {why} after {(time.time()-t0)/60:.1f} min", flush=True)
          print(key_lines(off, 30), flush=True)
          tail = log_text(off)
          print("---- serve.log tail ----\n" + tail[-6000:], flush=True)
          stop_server(proc)
          sig = failure_sig(tail)
          det = sig not in (None, "MEMORY") and (any(d in sig for d in DETERMINISTIC)
                                                 or (sig.startswith("ValueError") and "load_weights" in tail))
          print(f"failure signature: {sig!r} | deterministic: {det}", flush=True)
          return None, off, sig, det

      def relaunch_server(until):
          """Crash recovery: boot the winning rung again (page cache is warm by now)."""
          global log_off
          i = CFG["LADDER"].index(SERVER_RUNG)
          proc, off, sig, det = start_rung(i, SERVER_RUNG, until)
          if proc is not None:
              log_off = off
              print(f"relaunched {SERVER_RUNG['NAME']} | KV tokens {kv_tokens(off)}", flush=True)
          return proc

  except Exception as _ex:
    stage_failed('serve', _ex, fatal=True)


skipping stage serve: earlier stage failed: ['quick_version']


In [8]:
# [duck2:phase0]  GPU microbenchmarks (server not started) gate the dormant decode-speed patches
if FAILED:
    print('skipping stage phase0: earlier stage failed:', FAILED, flush=True)
else:
  try:
      def _last_json(text):
          for line in reversed((text or "").strip().splitlines()):
              line = line.strip()
              if line.startswith("{") and line.endswith("}"):
                  try:
                      return json.loads(line)
                  except Exception:
                      continue
          return None

      def _env_items(rec):
          if isinstance(rec, dict):
              return {str(a): str(b) for a, b in rec.items()}
          if isinstance(rec, str):
              return dict(x.split("=", 1) for x in rec.split() if "=" in x)
          return {}

      PHASE0 = {"enabled": {}, "benches": {}}
      _gpu = {"hcmix32": "0", "qsasplitk": "1", "mqagrid": "2", "marlinclean": "0", "topklogits": "0,1,2,3", "densefp8": "3", "idxshard": "0,1,2,3", "hisparse": "0,1,2,3", "pairown": "0,1,2,3"}
      _nosite = ["marlinclean"]     # this bench takes no site-packages argument
      _multi = ["topklogits", "idxshard", "hisparse", "pairown"]          # needs all four GPUs (real NCCL): run after the single-GPU benches, one at a time
      _args = {"mqagrid": ["--timeout", str(CFG["PHASE0_S"] - 60)], "densefp8": ["--timeout", str(CFG["PHASE0_S"] - 60)],
               "idxshard": ["--timeout", str(CFG["PHASE0_S"] - 120)], "hisparse": ["--timeout", str(CFG["PHASE0_S"] - 120)], "pairown": ["--timeout", str(CFG["PHASE0_S"] - 120)]}
      _ENVS = {"topklogits": {"SGLANG_TOPK_LOGITS_EXCHANGE": "32"}, "marlinclean": {"SGLANG_MARLIN_EP_CLEAN": "1"},
               "densefp8": {"SGLANG_SM120_ONLINE_MXFP8": "1"}, "idxshard": {"SGLANG_QSA_INDEXER_SHARD": "1"}}
      _t0p = time.time()

      def _launch(k):
          cmd = ["timeout", "-k", "10", str(CFG["PHASE0_S"]), PYTHON, str(PATCH_DIR / f"bench_{k}.py")] + ([] if k in _nosite else [SITE]) + _args.get(k, [])
          return subprocess.Popen(cmd, env=dict(env, CUDA_VISIBLE_DEVICES=_gpu[k]), stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, text=True)

      def _collect(k, pr):
          try:
              out, _ = pr.communicate(timeout=CFG["PHASE0_S"] + 60)
          except Exception as ex:
              pr.kill(); out = f"communicate failed: {ex!r}"
          j = _last_json(out) or {}
          try:
              (work / f"phase0_{k}.txt").write_text((out or "")[-300000:])
          except Exception:
              pass
          PHASE0["benches"][k] = {"rc": pr.returncode, "ok": j.get("ok"), "tail": (out or "")[-2000:],
                                  "summary": {x: j.get(x) for x in ("baseline_us", "patched_us", "saving_ms_per_step", "per_step_saving_us",
                                                                    "min_speedup_rows_le20", "gate_30pct_rows_le20", "recommend_env",
                                                                    "suggest_env", "failed", "max_abs_err", "est_step_saving_ms_48_layers",
                                                                    "error", "saved_ms_per_step", "max_kernel_rel_err", "max_quant_rel_err",
                                                                    "best_cfg", "hc", "proj_step_us", "method_flow", "hcmix32_active", "errors", "correctness", "timing", "nccl_graph_us_12x_rows32", "headline", "io512_all4_us_rows32_churn6", "jit_s", "modes", "sglang_init", "owned_rank0") if x in j}}
          if k == "hcmix32" and j.get("ok") and j.get("recommend_env"):
              PHASE0["enabled"].update(_env_items(j["recommend_env"]))
          elif k == "qsasplitk" and j.get("ok") and j.get("gate_30pct_rows_le20"):
              PHASE0["enabled"]["SGLANG_QSA_SM89_DECODE"] = "triton_splitk"
              PHASE0["enabled"].update(_env_items(j.get("suggest_env")))
          elif k == "mqagrid" and j.get("ok") and j.get("recommend_env"):
              PHASE0["enabled"].update(_env_items(j["recommend_env"]))
          elif k in _ENVS and j.get("ok") and j.get("baseline_us") is not None and j.get("patched_us") is not None \
                  and j["patched_us"] < j["baseline_us"]:
              PHASE0["enabled"].update(_ENVS[k])

      _todo = [b for b in CFG["PHASE0_BENCHES"] if b in globals().get("PERF_APPLIED", [])]
      _procs = {k: _launch(k) for k in _todo if k not in _multi}
      for k, pr in _procs.items():
          _collect(k, pr)
      for k in [b for b in _todo if b in _multi]:
          _collect(k, _launch(k))
      # PHASE0_TRUSTED: env of patches whose bench passed on earlier Kaggle runs (v66, v67), enabled without re-benching
      for k, envs in (CFG.get("PHASE0_TRUSTED") or {}).items():
          if k in globals().get("PERF_APPLIED", []) and k not in PHASE0["benches"]:
              PHASE0["enabled"].update({str(a): str(b) for a, b in envs.items()})
              PHASE0.setdefault("trusted", []).append(k)
      PHASE0["seconds"] = round(time.time() - _t0p, 1)
      for r in CFG["RUNGS"]:
          if r.get("PERF"):
              r["ENV"] = dict(r.get("ENV") or {}, **PHASE0["enabled"])
      write_json_atomic(work / "phase0.json", PHASE0)
      print("PHASE0", json.dumps({k: {"rc": v["rc"], "ok": v["ok"]} for k, v in PHASE0["benches"].items()}),
            "| enabled on rung 0:", PHASE0["enabled"], f"| {PHASE0['seconds']} s", flush=True)
      for k, v in PHASE0["benches"].items():
          print(" ", k, json.dumps(v["summary"])[:700], flush=True)
  except Exception as _ex:
    stage_failed('phase0', _ex, fatal=True)


skipping stage phase0: earlier stage failed: ['quick_version']


In [9]:
# [duck2:pcie]  host<->GPU bandwidth and topology (server not started): sizes the KV-offload option
try:
    PCIE = {}
    if CFG.get("PCIE") and RUN_MODE != "full":
        def _sh(c):
            try:
                return subprocess.run(c, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=60).stdout[-2500:]
            except Exception as ex:
                return repr(ex)
        PCIE["topo"] = _sh("nvidia-smi topo -m")
        PCIE["links"] = _sh("nvidia-smi --query-gpu=index,pcie.link.gen.current,pcie.link.width.current,pcie.link.gen.max --format=csv,noheader")
        PCIE["cpu"] = _sh("lscpu | grep -E 'Model name|^CPU\\(s\\)|Socket|NUMA node\\(s\\)|L3'; free -g | head -2")
        def _run(gpus):
            t0 = time.time() + 6
            ps = {g: subprocess.Popen([PYTHON, str(PATCH_DIR / "bench_pcie.py"), repr(t0)], env=dict(env, CUDA_VISIBLE_DEVICES=str(g)),
                                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, text=True) for g in gpus}
            out = {}
            for g, pr in ps.items():
                try:
                    o, _ = pr.communicate(timeout=240)
                except Exception as ex:
                    pr.kill(); o = repr(ex)
                out[g] = _last_json(o) or {"raw": (o or "")[-300:]}
            return out
        PCIE["solo"] = _run([0])
        PCIE["all4"] = _run([0, 1, 2, 3])
        write_json_atomic(work / "pcie.json", PCIE)
        print("PCIE links:", PCIE["links"].strip().replace("\n", " | "), flush=True)
        print("PCIE solo:", json.dumps(PCIE["solo"]), flush=True)
        print("PCIE all4:", json.dumps(PCIE["all4"]), flush=True)
        print("PCIE topo:\n" + PCIE["topo"], flush=True)
    else:
        print("pcie: off", flush=True)
except Exception as _ex:
    stage_failed('pcie', _ex, fatal=False)


pcie: off


In [10]:
# [duck2:boot]  first healthy rung of CFG["RUNGS"] (health + greedy echo gate), else no server
if FAILED:
    print('skipping stage boot: earlier stage failed:', FAILED, flush=True)
else:
  try:
      CRASH_DIR = str(work / "crash")                # small CUDA coredumps (no device memory) + SGLang's last-request dump
      EXTRA_SERVE_ARGS += ["--crash-dump-folder", CRASH_DIR]
      if CFG["SOLVE"].get("force_think"):      # per-request thinking budgets (custom_params.thinking_budget) need strict thinking
          EXTRA_SERVE_ARGS += ["--enable-strict-thinking"]
          PLE_ENV["SGLANG_MAX_THINK_TOKENS"] = "1000000"   # >= 0 turns the token filter on; requests override the limit
          PLE_ENV["SGLANG_FORCED_THINK_SUFFIX"] = CFG["SOLVE"]["forced_suffix"]   # think_force_patch: answer prefix after a forced </think>
      if globals().get("PLE_COPY") is not None and PLE_COPY.is_alive():   # v105c: rung 0's clock starts after the PLE copy
          _w = time.time(); _until = _w + CFG["PLE_LOCAL"]["WAIT_MIN"] * 60
          while PLE_COPY.is_alive() and time.time() < _until:
              PLE_COPY.join(max(1.0, min(60.0, _until - time.time())))
              if PLE_COPY.is_alive():
                  _b, _n = PLE_COPY_STATE["bytes"], max(PLE_COPY_STATE["total"], 1)
                  print(f"waiting for the PLE local copy: {(time.time()-_w)/60:.1f} min | {_b/1e9:.1f}/{_n/1e9:.1f} GB "
                        f"({_b/1e6/max(time.time()-_w, 1e-6):.0f} MB/s since the wait began)", flush=True)
          print(f"PLE copy wait {(time.time()-_w)/60:.1f} min: "
                f"{'still running, rung 0 starts anyway' if PLE_COPY.is_alive() else 'copy ok' if PLE_COPY_STATE['ok'] else 'copy failed'}",
                flush=True)
      for i, r in enumerate([] if CFG.get("NO_BOOT") else CFG["RUNGS"]):     # NO_BOOT: install + phase-0/pcie microbenchmarks only
          if time.time() > T0 + CFG["SERVE_BUDGET_MIN"] * 60:
              print("serve budget exhausted", flush=True); break
          proc, off, sig, det = start_rung(i, r, time.time() + CFG["RUNG_MAX_MIN"][min(i, 3)] * 60)
          if proc is not None:
              server, log_off, SERVER_RUNG, KV_TOKENS = proc, off, r, kv_tokens(off)
              break
          print(f"rung {r['NAME']} failed: {sig}", flush=True)
      SERVER_OK = server is not None
      print(f"SERVER {'READY ' + SERVER_RUNG['NAME'] if SERVER_OK else 'NONE'} at +{(time.time()-T0)/60:.1f} min | KV tokens {KV_TOKENS}",
            flush=True)
      print(key_lines(log_off, 20), flush=True)
  except Exception as _ex:
    stage_failed('boot', _ex, fatal=True)


skipping stage boot: earlier stage failed: ['quick_version']


In [11]:
# [duck2:bench]  interactive runs with CFG["BENCH"] only: long-context decode benchmark (prof_drive.py), no solve
try:
    import shlex, sys
    BENCH_ON = bool(CFG.get("BENCH")) and RUN_MODE != "full"
    if BENCH_ON and SERVER_OK:
        BC = CFG["BENCH"]
        _bs = [b for b in BC["Bs"] if b <= SERVER_RUNG["MAXREQ"]]
        _kv = int(KV_TOKENS or 0)
        # prompts sized so the pool is ~fill full at the end of the largest-B window (after ~grow decoded tokens), while
        # every request's admission reserve (prompt + min(max_new, CLIP 8192) at new-token ratio 1.0) still fits
        _clip = int((SERVER_RUNG.get("ENV") or {}).get("SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION", 8192))
        _ctx = int(min((BC["fill"] * _kv - BC["grow"]) / max(_bs), 0.96 * _kv / max(_bs) - _clip - 500))
        _out = work / "bench"
        _out.mkdir(exist_ok=True)
        _cmd = [sys.executable, str(PATCH_DIR / "prof_drive.py"), "--url", ENDPOINT, "--challenges", str(EVAL_CHAL or RUN_CHAL),
                "--Bs", ",".join(map(str, _bs)), "--ctx", str(_ctx), "--window-s", str(BC["window_s"]),
                "--settle-s", str(BC["settle_s"]), "--server-log", LOG, "--out-dir", str(_out), "--prof-dir", "/tmp/prof",
                "--max-new", "30000", "--prefill-timeout-s", str(BC["prefill_timeout_s"])]
        if BC.get("profile_at"):      # torch-profiler capture at these B (needs SGLANG_TORCH_PROFILER_DIR=/tmp/prof in the rung env)
            _cmd += ["--profile-at", ",".join(map(str, BC["profile_at"])), "--profile-steps", str(BC.get("profile_steps", 24))]
        else:
            _cmd += ["--no-summarize"]
        print(f"BENCH pool {_kv} tokens | Bs {_bs} | prompt {_ctx} tokens/request\n" + shlex.join(_cmd), flush=True)
        BENCH_SUMMARY = {"rung": SERVER_RUNG["NAME"], "pool": _kv, "Bs": _bs, "ctx": _ctx}
        _t0b = time.time()
        try:
            _p = subprocess.run(_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, text=True,
                                timeout=BC["max_s"])
            _txt, BENCH_SUMMARY["rc"] = _p.stdout or "", _p.returncode
        except subprocess.TimeoutExpired as _ex:
            _txt, BENCH_SUMMARY["rc"] = _ex.stdout if isinstance(_ex.stdout, str) else (_ex.stdout or b"").decode(errors="replace"), "timeout"
        BENCH_SUMMARY["seconds"] = round(time.time() - _t0b, 1)
        print(_txt[-8000:], flush=True)
        (_out / "prof_drive.log").write_text(_txt)
        for _sj in sorted(Path("/tmp/prof").glob("b*/summary.json")):      # per-B kernel-time summaries (the raw traces are large)
            (_out / f"prof_{_sj.parent.name}_summary.json").write_text(_sj.read_text())
        _done = [l for l in _txt.splitlines() if l.startswith("PROF_DRIVE_DONE ")]
        if _done:
            BENCH_SUMMARY["table"] = json.loads(_done[-1].split(" ", 1)[1]).get("table")
        else:
            try:
                BENCH_SUMMARY["runs"] = [{"B": r["B"], **{k: r["window"].get(k) for k in ("step_ms_p50", "tok_s_p50", "full_token_p50",
                                         "used_lines", "retract_lines")}, "client_tok_s": r["client"]["tok_s"]}
                                         for r in json.loads((_out / "drive.json").read_text())["runs"]]
            except Exception as _ex:
                BENCH_SUMMARY["error"] = repr(_ex)
        print("BENCH_SUMMARY", json.dumps(BENCH_SUMMARY), flush=True)
        write_json_atomic(work / "bench_summary.json", BENCH_SUMMARY)
    else:
        print("bench: off" if not BENCH_ON else "bench: no server", flush=True)
except Exception as _ex:
    stage_failed('bench', _ex, fatal=False)


bench: off


In [12]:
# [duck2:agent]  ARC-AGI-2 program-synthesis harness v2 (harness/arc_agent.py, embedded verbatim) against the local server
if FAILED:
    print('skipping stage agent: earlier stage failed:', FAILED, flush=True)
else:
  try:
      AG = CFG["AGENT"]
      HARNESS_SRC = __import__("zlib").decompress(__import__("base64").b64decode('...')).decode()   # (push size)
      (work / "arc_agent.py").write_text(HARNESS_SRC)
      HIC_CHECK, HIC_SKIP = None, False
      solve_t0 = time.time()
      if RUN_MODE == "full":
          DEADLINE = T0 + CFG["FULL_BUDGET_S"] - CFG["END_MARGIN_S"]
      else:
          DEADLINE = min(solve_t0 + AG["SOLVE_MIN"] * 60, T0 + CFG["FULL_BUDGET_S"] - CFG["END_MARGIN_S"])
      RUNG_I = next((i for i, r in enumerate(CFG["RUNGS"]) if r is SERVER_RUNG), 0)
      # sessions keep their whole context cached between turns: size concurrency so ~CTX_PER_SESSION live tokens each fit
      # the KV pool (an evicted prefix is re-prefilled at ~1.5k tok/s, stalling every session)
      INFLIGHT = max(1, min((SERVER_RUNG or CFG["RUNGS"][0])["MAXREQ"], AG["MAX_INFLIGHT"],
                            (KV_TOKENS // AG["CTX_PER_SESSION"]) if KV_TOKENS else AG["MAX_INFLIGHT"]))
      HEALTH_FAILS, RESTARTS, HLOGS, CRASHES, AGENT_SUMMARY = 0, 0, [], [], {}
      STALLED, BATCH_SEEN = "", [None, time.time()]      # watchdog: reason / [latest batch line, when it first showed]
      BATCH_LINE = re.compile(r"[^\n]*\] (?:Prefill|Decode) batch[^\n]*")
      # LBPROBE (competition rerun only): no restarts; the first failure before the last RESTART_MIN_S ends the run at once
      # and the cell [duck2:lbprobe] turns its class into the LB result (see there)
      LBPROBE = RUN_MODE == "full" and bool(AG.get("LBPROBE"))
      LBPROBE_KIND, LBPROBE_INFO = "", {}

      def server_load():
          """(running, waiting) requests as the scheduler reports them, or None if it does not answer in 60 s."""
          try:
              with urllib.request.urlopen(f"{ENDPOINT}/v1/loads?include=core", timeout=60) as r:
                  loads = json.loads(r.read()).get("loads", [])
              return (sum(int(x.get("num_running_reqs", 0)) for x in loads), sum(int(x.get("num_waiting_reqs", 0)) for x in loads))
          except Exception:
              return None

      def server_answers():
          """A 1-token /generate completes within 60 s (tokenizer, scheduler and detokenizer all work)."""
          try:
              req = urllib.request.Request(f"{ENDPOINT}/generate", headers={"Content-Type": "application/json"},
                                           data=json.dumps({"text": "1+1=", "sampling_params": {"max_new_tokens": 1, "temperature": 0}}).encode())
              with urllib.request.urlopen(req, timeout=60) as r:
                  return r.status == 200
          except Exception:
              return False

      def lbprobe_fail(why):
          """Name the failure and end the run: the server process died (died / died_leak: its strict idle check found
          leaked slots), it is alive but stuck (stalled: no batch for STALL_MIN with requests on the scheduler;
          stalled_noanswer: the scheduler does not answer /v1/loads; stalled_io: it is idle but a 1-token /generate fails;
          health: /health refused), or the harness failed
          (harness_crash: rc != 0; harness_hang: sessions 'running' but the scheduler has no request)."""
          global LBPROBE_KIND, MERGE_INTO_SUBMISSION
          if why == "harness":
              kind = "harness_crash"
          elif server is None or server.poll() is not None:
              kind = "died_leak" if "memory leak detected" in log_text(log_off) else "died"
          elif STALLED:
              ld = server_load()
              LBPROBE_INFO["load"] = ld
              kind = ("stalled_noanswer" if ld is None else "stalled" if ld != (0, 0) else
                      "harness_hang" if server_answers() else "stalled_io")
          else:
              kind = "health"
          LBPROBE_KIND = kind
          LBPROBE_INFO.update(kind=kind, why=why, t_min=round((time.time() - solve_t0) / 60, 1), server=last_decode_line(),
                              stalled=STALLED, tail=log_text(max(log_off, log_size() - 3000))[-3000:])
          MERGE_INTO_SUBMISSION = False
          write_json_atomic(SUBMISSION, FALLBACK)
          print(f"!!! LBPROBE: {kind} at +{LBPROBE_INFO['t_min']} min of the solve ({why}) -> ending the run now", flush=True)

      def server_dead():
          """The server process exited, /health refused connections HEALTH_FAILS_MAX status ticks in a row, or it stalled."""
          return server is None or server.poll() is not None or HEALTH_FAILS >= AG["HEALTH_FAILS_MAX"] or bool(STALLED)

      def stall_tick():
          """SGLang's watchdog only covers a stuck forward pass. Queued requests that are never admitted leave it, the idle
          leak check and /health all quiet while every session waits to the deadline. So: STALL_MIN minutes without one new
          Prefill/Decode batch line while the harness has sessions running (or its status froze with sessions running, which
          also stops the batches) -> the server counts as dead and takes the restart path."""
          global STALLED
          if server is None or server.poll() is not None or not AG.get("STALL_MIN"):
              return
          n = log_size()
          ls = BATCH_LINE.findall(log_text(max(log_off, n - 400000)))
          now = time.time()
          if (ls[-1] if ls else None) != BATCH_SEEN[0]:
              BATCH_SEEN[:] = [ls[-1] if ls else None, now]
              return
          try:
              st = json.loads((OUT / "status.json").read_text())
          except Exception:
              return
          idle = (now - BATCH_SEEN[1]) / 60
          if (st.get("running_sessions", 0) > 0 and not st.get("finished") and idle >= AG["STALL_MIN"]
                  and RESTARTS < AG["MAX_RESTARTS"] and DEADLINE - now > AG["RESTART_MIN_S"]):   # only when a restart can help
              STALLED = (f"stalled: no scheduler batch for {idle:.0f} min with {st['running_sessions']} sessions running "
                         f"(harness status {(now - st.get('now', now)) / 60:.0f} min old)")

      def leak_check(wait_s):
          """Smoke only: abort every request and let the scheduler go fully idle. This build's strict idle check
          (SGLANG_ENABLE_STRICT_MEM_CHECK_DURING_IDLE, default on) then audits the KV, mamba and request pools and raises
          'memory leak detected' (killing the server) on any slot the run leaked."""
          if server is None or server.poll() is not None:
              return {"result": "server not running"}
          off, t = log_size(), time.time()
          try:
              req = urllib.request.Request(f"{ENDPOINT}/abort_request", data=json.dumps({"abort_all": True}).encode(),
                                           headers={"Content-Type": "application/json"})
              urllib.request.urlopen(req, timeout=30).read()
          except Exception as ex:
              print("abort_all:", repr(ex), flush=True)
          mark, quiet_since = None, time.time()
          while time.time() - t < wait_s and server.poll() is None:
              ls = BATCH_LINE.findall(log_text(off))
              if (ls[-1] if ls else None) != mark:
                  mark, quiet_since = (ls[-1] if ls else None), time.time()
              time.sleep(5)
          tail = log_text(off)
          leaks = re.findall(r"[^\n]*memory leak detected[^\n]*", tail)
          quiet = time.time() - quiet_since
          res = {"result": "LEAK" if leaks else ("server died" if server.poll() is not None else
                                                  "clean" if quiet >= 20 else "not idle"),
                 "quiet_s": round(quiet), "leak_lines": [l[-600:] for l in leaks[:4]],
                 "alive": server.poll() is None}
          print(f"LEAK CHECK: {res['result']} (no batch for the last {res['quiet_s']} s, server alive {res['alive']})"
                + "".join("\n  " + l for l in res["leak_lines"]), flush=True)
          return res

      def health_tick():
          global HEALTH_FAILS
          if server is None or server.poll() is not None:
              return
          try:
              with urllib.request.urlopen(f"{ENDPOINT}/health", timeout=30):
                  pass
              HEALTH_FAILS = 0
          except urllib.error.HTTPError:
              HEALTH_FAILS = 0                       # it answered
          except Exception:
              HEALTH_FAILS += 1

      def last_decode_line():
          if server is None or server.poll() is not None:
              return f"DEAD rc={None if server is None else server.returncode}"
          if HEALTH_FAILS:
              return f"/health failing x{HEALTH_FAILS}"
          if STALLED:
              return STALLED
          ls = re.findall(r"Decode batch[^\n]*", log_text(max(0, log_size() - 20000)))
          if not ls:
              return ""
          m = {k: re.search(k + r": ([\d.]+)", ls[-1]) for k in ("#running-req", "full token usage", r"gen throughput \(token/s\)", "#queue-req")}
          return " ".join(f"{k.split()[0].strip('#')}={v.group(1)}" for k, v in m.items() if v)

      def print_status(full=False):
          try:
              st = json.loads((OUT / "status.json").read_text())
          except Exception:
              print(f"--- {time.strftime('%H:%M:%S', time.gmtime())} UTC | no status.json yet in {OUT.name}", flush=True); return
          el = (time.time() - st["t_start"]) / 60
          states = {}
          for r in st["tasks"]:
              states[r["state"]] = states.get(r["state"], 0) + 1
          sc, n_ans = nb_score() if SOLS else (None, 0)
          print(f"--- {time.strftime('%H:%M:%S', time.gmtime())} UTC | +{(time.time()-T0)/60:.0f} min (run {len(OUT_DIRS)} solve {el:.0f}, "
                f"{(DEADLINE - time.time())/60:.0f} left) | accepted {st['accepted']}/{st['n_tasks']}"
                + (f" | score {sc:.3f} ({n_ans} answered)" if sc is not None else "")
                + f" | gen {st['gen_tokens']} ({st['gen_tokens'] / max(1, el * 60):.0f} tok/s) | live "
                f"{min(st['running_sessions'], st['max_inflight'])}/{st['max_inflight']} | {states} | server: {last_decode_line()}",
                flush=True)
          rows = st["tasks"] if full else [r for r in st["tasks"] if r["state"] == "running"]
          for r in rows[:40]:
              print(f"  {r['id']} {r['state']:9s} tests {r['best_passed']}/{r['n_tests']} s{r['sessions']} t{r['turns']} "
                    f"gen {r['gen_tokens']:>6} {r['elapsed_s']:>5}s d{r.get('directs', 0)} c{r.get('continuations', 0)} | "
                    f"{r['last_event'][:70]}", flush=True)

      def task_records():
          """{task id: accepted?} over every harness out dir (tasks/<id>.json is written once a task has a session)."""
          rec = {}
          for d in OUT_DIRS:
              for p in sorted((Path(d) / "tasks").glob("*.json")):
                  try:
                      j = json.loads(p.read_text())
                      rec[j["id"]] = rec.get(j["id"], False) or bool(j.get("accepted"))
                  except Exception:
                      pass
          return rec

      def remaining_ids():
          """Not-yet-accepted tasks, never-started ones first."""
          rec = task_records()
          rest = [k for k in RUN_IDS if not rec.get(k)]
          return [k for k in rest if k not in rec] + [k for k in rest if k in rec]

      def launch_harness(ids, first=False):
          """First launch writes to OUT; every relaunch gets a new dir (arc_agent rewrites --out/submission.json at start)."""
          global OUT
          if not first:
              OUT = OUT_DIRS[0].parent / f"{OUT_DIRS[0].name}_r{len(OUT_DIRS)}"
              OUT_DIRS.append(OUT)
          cmd = [sys.executable, str(work / "arc_agent.py"), "--challenges", str(RUN_CHAL), "--task-ids", ",".join(ids),
                 "--out", str(OUT), "--endpoint", ENDPOINT, "--model", SERVED_NAME, "--deadline-epoch", f"{DEADLINE:.0f}",
                 "--max-inflight", str(INFLIGHT), "--max-model-len", str(CFG["CTX"])]
          for k, v in AG["ARGS"].items():
              if v is not False and v is not None:
                  cmd += [f"--{k}"] if v is True else [f"--{k}", str(v)]
          if not first and AG.get("LOAD_STATE", True):  # a relaunch continues every task from the earlier runs' records
              cmd += ["--load-state", ",".join(str(d) for d in OUT_DIRS[:-1])]
          if AG.get("KV_GATE_FRAC") and KV_TOKENS:       # --kv-gate sized to this server's pool (fallback rungs are smaller)
              cmd += ["--kv-gate", str(int(KV_TOKENS * AG["KV_GATE_FRAC"]))]
          print(f"AGENT START {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} | run {len(OUT_DIRS)} | {len(ids)} tasks | "
                f"inflight {INFLIGHT} | deadline in {(DEADLINE - time.time())/60:.0f} min\n" + " ".join(cmd[6:]), flush=True)
          hlog = (work / f"harness{'' if len(OUT_DIRS) == 1 else '_r%d' % (len(OUT_DIRS) - 1)}.log").open("w")
          return subprocess.Popen(cmd, stdout=hlog, stderr=subprocess.STDOUT, start_new_session=True), hlog

      def stop_harness(p, why):
          if p is not None and p.poll() is None:
              for sig, wait in ((signal.SIGTERM, 30), (signal.SIGKILL, 10)):
                  try:
                      os.killpg(p.pid, sig)
                  except Exception:
                      pass
                  try:
                      p.wait(timeout=wait); break
                  except subprocess.TimeoutExpired:
                      pass
              print(f"harness stopped by the notebook ({why})", flush=True)

      def merge_tick():
          if MERGE_INTO_SUBMISSION:
              sub, n = merged_submission(); write_json_atomic(SUBMISSION, sub)
              print(f"  submission.json: {n} tasks from the harness", flush=True)

      def restart_server(why):
          """Boot the same rung again (the page cache is warm); True if it is back. A CRASH_NEXT rung (HiCache) falls back
          to the next rung (same serving config without the host KV tier) instead."""
          global server, log_off, SERVER_RUNG, KV_TOKENS, HEALTH_FAILS, STALLED, RUNG_I
          CRASHES.append({"t_min": round((time.time() - solve_t0) / 60, 1), "why": why, "tail": log_text(log_off)[-1500:]})
          try:
              stop_server(server)
          except Exception as ex:
              print("stop_server:", repr(ex), flush=True)
          if SERVER_RUNG.get("CRASH_NEXT") and RUNG_I + 1 < len(CFG["RUNGS"]):
              RUNG_I += 1; SERVER_RUNG = CFG["RUNGS"][RUNG_I]
              print(f"restart ({why}) on a CRASH_NEXT rung -> falling back to {SERVER_RUNG['NAME']}", flush=True)
          proc, off, sig, det = start_rung(RUNG_I, SERVER_RUNG, DEADLINE - AG["MIN_SOLVE_S"])
          HEALTH_FAILS, STALLED, BATCH_SEEN[:] = 0, "", [None, time.time()]
          if proc is None:
              print(f"restart failed: {sig}", flush=True); server = None; return False
          server, log_off, KV_TOKENS = proc, off, kv_tokens(off)
          print(f"server back on {SERVER_RUNG['NAME']} at +{(time.time()-solve_t0)/60:.0f} min | KV {KV_TOKENS}", flush=True)
          return True

      if SERVER_OK and AG.get("HIC_CHECK") and RUN_MODE != "full":
          # HiCache reload test before the agent (tools_sp/hic_check.py): forced GPU->host evictions + exact reload compare;
          # a failure skips the agent (nothing left to learn, and the quota is rationed)
          try:
              _hc = subprocess.run([sys.executable, str(PATCH_DIR / "hic_check.py"), ENDPOINT, str(KV_TOKENS),
                                    str(work / "hic_check.json"), str(AG["HIC_CHECK"])], capture_output=True, text=True,
                                   timeout=AG["HIC_CHECK"] * 60 + 900)
              print(_hc.stdout[-8000:], _hc.stderr[-3000:], flush=True)
              HIC_CHECK = json.loads((work / "hic_check.json").read_text())
          except Exception as _ex:
              HIC_CHECK = {"ok": False, "verdict": f"hic_check error: {type(_ex).__name__}: {_ex}"}
          print("HIC_CHECK", HIC_CHECK.get("verdict"), "| server dead" if server_dead() else "| server alive", flush=True)
          if not HIC_CHECK.get("ok") or server_dead():
              print("HIC_CHECK failed: skipping the agent", flush=True); HIC_SKIP = True
          solve_t0 = time.time()                # the check is not agent time: the agent still gets SOLVE_MIN
          DEADLINE = min(solve_t0 + AG["SOLVE_MIN"] * 60, T0 + CFG["FULL_BUDGET_S"] - CFG["END_MARGIN_S"])
      if not SERVER_OK or HIC_SKIP:
          print("agent skipped: no server" if not SERVER_OK else "agent skipped: HIC_CHECK failed", flush=True)
          LBPROBE_KIND = "noserver" if LBPROBE else ""
      elif DEADLINE - time.time() < AG["MIN_SOLVE_S"]:
          print(f"agent skipped: only {DEADLINE - time.time():.0f}s left", flush=True)
          LBPROBE_KIND = "late" if LBPROBE else ""
      else:
          hproc = None
          t_kill = DEADLINE + AG.get("KILL_GRACE_S", 420)   # arc_agent joins its workers at deadline + 300 s, then writes report.json
          try:
              hproc, hl = launch_harness(RUN_IDS, first=True); HLOGS.append(hl)
              dead_since, tick, last_status = None, 0, time.time()
              while time.time() < t_kill:
                  try:
                      hproc.wait(timeout=AG["POLL_S"])
                  except subprocess.TimeoutExpired:
                      pass
                  try:                          # one bad tick (disk, parse, missing file) must never orphan the harness
                      if time.time() - last_status >= AG["STATUS_MIN"] * 60:
                          last_status, tick = time.time(), tick + 1
                          health_tick()
                          stall_tick()
                          print_status(full=(RUN_MODE == "smoke" and tick % 5 == 0))
                          merge_tick()
                      if server_dead():
                          if dead_since is None:
                              dead_since = time.time()
                              print(f"!!! SGLang server down ({last_decode_line()})\n{log_text(max(0, log_size() - 4000))}", flush=True)
                          left = DEADLINE - time.time()
                          if LBPROBE and left > AG["RESTART_MIN_S"]:
                              lbprobe_fail("server"); break
                          if RESTARTS < AG["MAX_RESTARTS"] and left > AG["RESTART_MIN_S"]:
                              RESTARTS += 1
                              stop_harness(hproc, "server down"); merge_tick()
                              print(f"=== restart {RESTARTS} with {left/60:.0f} min left ===", flush=True)
                              dead_since = None
                              if not restart_server(STALLED or "down"):
                                  break
                              ids = remaining_ids()
                              if not ids or DEADLINE - time.time() < AG["MIN_SOLVE_S"]:
                                  print("nothing left to run after the restart", flush=True); break
                              hproc, hl = launch_harness(ids); HLOGS.append(hl)
                              continue
                          if time.time() - dead_since > AG.get("DEAD_STOP_S", 300):
                              print("server down for 5 min and no restart possible: stopping the harness", flush=True)
                              break
                      if hproc.poll() is not None:
                          rc = hproc.returncode
                          print(f"harness run {len(OUT_DIRS)} exited rc={rc}", flush=True)
                          if LBPROBE and rc != 0 and DEADLINE - time.time() > AG["RESTART_MIN_S"]:
                              lbprobe_fail("harness"); break
                          ids = remaining_ids() if rc != 0 else []
                          if (ids and not server_dead() and RESTARTS < AG["MAX_RESTARTS"]
                                  and DEADLINE - time.time() > AG["MIN_SOLVE_S"]):
                              RESTARTS += 1
                              print(f"=== harness crashed: relaunching it on {len(ids)} tasks ===", flush=True)
                              hproc, hl = launch_harness(ids); HLOGS.append(hl)
                              continue
                          break
                  except Exception as _tick_ex:
                      print(f"monitor tick error (continuing): {type(_tick_ex).__name__}: {_tick_ex}", flush=True)
          finally:
              stop_harness(hproc, "deadline / cleanup")
              for hl in HLOGS:
                  try:
                      hl.close()
                  except Exception:
                      pass
          print_status(full=True)
          merge_tick()
          if RUN_MODE == "smoke" and AG.get("LEAK_CHECK_S"):
              AGENT_SUMMARY["leak_check"] = leak_check(AG["LEAK_CHECK_S"])
          try:
              print("harness rc", hproc.returncode if hproc else None, "| restarts", RESTARTS, "| out dirs",
                    [d.name for d in OUT_DIRS], "| last log lines:\n"
                    + "".join(Path(HLOGS[-1].name).read_text(errors="replace").splitlines(True)[-12:]), flush=True)
          except Exception as ex:
              print("harness log unreadable:", ex)
          for d in OUT_DIRS:
              try:
                  rep = json.loads((Path(d) / "report.json").read_text())
                  AGENT_SUMMARY[Path(d).name] = {k: v for k, v in rep.items() if k not in ("per_task", "args")}
              except Exception as ex:
                  AGENT_SUMMARY[Path(d).name] = {"report": f"unreadable: {ex}"}
          sc, n_ans = nb_score() if SOLS else (None, 0)
          AGENT_SUMMARY.update(score=sc, answered=n_ans, restarts=RESTARTS, crashes=CRASHES, minutes=round((time.time() - solve_t0) / 60, 1),
                               hic_check=HIC_CHECK,
                               lbprobe=LBPROBE_INFO if LBPROBE else None)
          if SOLS:          # smoke only: per-task result + any model code touching the eval solutions file
              ent = agent_entries(); rows = {}
              rec = {}
              for d in OUT_DIRS:
                  for p in (Path(d) / "tasks").glob("*.json"):
                      try:
                          j = json.loads(p.read_text()); rec[j["id"]] = j
                      except Exception:
                          pass
              for k in RUN_IDS:
                  e, j = ent.get(k), rec.get(k, {})
                  ok = [e is not None and i < len(e) and (e[i]["attempt_1"] == g or e[i]["attempt_2"] == g) for i, g in enumerate(SOLS[k])]
                  rows[k] = {"correct": sum(ok) / len(ok), "accepted": j.get("accepted"), "t_accept": j.get("t_accept"),
                             "gen": j.get("gen_tokens"), "sessions": len(j.get("sessions", [])), "directs": len(j.get("directs", []))}
              AGENT_SUMMARY["tasks"] = rows
              for k, r in rows.items():
                  print(f"  {k} correct {r['correct']:.2f} accepted {r['accepted']} t_accept {r['t_accept']} gen {r['gen']} "
                        f"sessions {r['sessions']} directs {r['directs']}", flush=True)
              hits = [p.name for d in OUT_DIRS for p in (Path(d) / "tasks").glob("*.json")
                      if re.search(r"solutions\.json|evaluation_solutions", p.read_text(errors="replace"))]
              AGENT_SUMMARY["solutions_mentions"] = hits
              print(f"solutions-file mentions in model transcripts: {len(hits)} {hits[:10]}", flush=True)
          print("AGENT_SUMMARY", json.dumps({k: v for k, v in AGENT_SUMMARY.items() if k != "tasks"}), flush=True)
          write_json_atomic(work / "agent_summary.json", AGENT_SUMMARY)
      SOLVE_SUMMARY = AGENT_SUMMARY
  except Exception as _ex:
    stage_failed('agent', _ex, fatal=True)


skipping stage agent: earlier stage failed: ['quick_version']


In [13]:
# [duck2:final]  validate/repair submission.json, summary, stop the server
try:
    try:
        sub = json.loads(SUBMISSION.read_text()); bad = 0
        for k in test_tasks:
            v = sub.get(k)
            if not (isinstance(v, list) and len(v) == len(test_tasks[k]["test"])):
                sub[k] = FALLBACK[k]; bad += 1; continue
            for j, e in enumerate(v):
                if not (isinstance(e, dict) and valid_grid(e.get("attempt_1")) and valid_grid(e.get("attempt_2"))):
                    v[j] = FALLBACK[k][j]; bad += 1
        sub = {k: sub[k] for k in test_tasks}
        write_json_atomic(SUBMISSION, sub)
        print(f"submission.json OK: {len(sub)} tasks, {sum(map(len, sub.values()))} outputs, {bad} repaired", flush=True)
    except Exception as ex:
        print("submission validation error:", repr(ex), "-> fallback", flush=True)
        write_json_atomic(SUBMISSION, FALLBACK)
    SUMMARY = {"run_mode": RUN_MODE, "rung": (SERVER_RUNG or {}).get("NAME"), "failed_stages": FAILED, "kv_tokens": KV_TOKENS,
               "minutes": round((time.time() - T0) / 60, 1), "solve": globals().get("SOLVE_SUMMARY"),
               "bench": globals().get("BENCH_SUMMARY")}
    if SERVER_OK and Path(LOG).exists():
        dec = re.findall(r"accept len: ([\d.]+).*?gen throughput \(token/s\): ([\d.]+)", log_text(log_off))
        if dec:
            thr = sorted(float(b) for _, b in dec); acc = [float(a) for a, _ in dec]
            SUMMARY["decode_log"] = {"lines": len(dec), "median_tok_s_per_group": thr[len(thr) // 2], "accept_len": round(sum(acc) / len(acc), 2)}
    print("SUMMARY", json.dumps(SUMMARY), flush=True)
    (work / "fn_summary.json").write_text(json.dumps(SUMMARY, indent=1))
    if server is not None and RUN_MODE == "full":
        # v111b: a competition rerun ends after this cell (no rung follows it): stop the server group exactly as
        # stop_server does, minus its up-to-180 s GPU-release wait (worst-case tail DEADLINE + ~735 s -> ~555 s)
        for _sig, _wait in ((signal.SIGTERM, 20), (signal.SIGKILL, 5)):
            try:
                os.killpg(server.pid, _sig)
            except Exception:
                pass
            try:
                server.wait(timeout=_wait)
            except Exception:
                pass
        print("server stopped (competition rerun: no GPU-release wait)", flush=True)
    elif server is not None:
        stop_server(server)
    _stop_monitor.set()
    print(f"total notebook time {(time.time()-T0)/60:.1f} min", flush=True)
except Exception as _ex:
    stage_failed('final', _ex, fatal=True)


submission.json OK: 240 tasks, 259 outputs, 0 repaired
SUMMARY {"run_mode": "smoke", "rung": null, "failed_stages": ["quick_version"], "kv_tokens": 0, "minutes": 0.0, "solve": null, "bench": null}
total notebook time 0.0 min


In [14]:
# [duck2:jitexport]  v111 (boot-and-misc L1, producer side only): after an INTERACTIVE GPU run (never a competition rerun,
# never a QUICK run) tar the JIT caches this run built (Triton, TileLang, inductor, flashinfer, CUDA PTX, sglang jit) into
# /kaggle/working/jit_cache/jit_cache.tar + manifest.json (sha256, sizes, wheel, patch md5s, versions, driver, rung,
# GRAPH_BS), so the user can turn this version's output into a dataset for a later consumer cell. Uploads nothing, changes
# nothing the run used (it runs after the final cell), never raises. Inert for scoring.
JIT_EXPORT = {"enabled": bool(CFG.get("JIT_EXPORT")) and RUN_MODE != "full" and "quick_version" not in FAILED
              and "install" not in FAILED and bool(globals().get("PREFIX"))}
if JIT_EXPORT["enabled"]:
    try:
        import tarfile as _jx_tar
        _JX = CFG["JIT_EXPORT"]
        _jx_t = time.time()
        _jx_cache = Path(PREFIX) / "cache"
        _jx_dir = work / _JX["OUT"]
        _jx_sizes = {}
        for _s in _JX["DIRS"]:
            _p = _jx_cache / _s
            if _p.is_dir():
                _jx_sizes[_s] = sum(f.stat().st_size for f in _p.rglob("*") if f.is_file() and not f.is_symlink())
        JIT_EXPORT["dir_bytes"] = _jx_sizes
        if sum(_jx_sizes.values()) > float(_JX["MAX_GB"]) * 1e9:
            raise RuntimeError(f"caches total {sum(_jx_sizes.values())/1e9:.2f} GB > MAX_GB {_JX['MAX_GB']}: not exported")
        _jx_dir.mkdir(parents=True, exist_ok=True)
        _jx_path = _jx_dir / "jit_cache.tar"
        with _jx_tar.open(_jx_path, "w") as _tf:
            for _s in _jx_sizes:
                _tf.add(str(_jx_cache / _s), arcname=_s)
        _jx_h = hashlib.sha256()
        with open(_jx_path, "rb") as _f:
            for _blk in iter(lambda: _f.read(8 << 20), b""):
                _jx_h.update(_blk)
        try:
            _jx_v = subprocess.run([PYTHON, "-c", "import json, torch, triton, sglang\n"
                                    "try:\n import tilelang; tl = tilelang.__version__\nexcept Exception as e:\n tl = repr(e)\n"
                                    "print(json.dumps({'torch': torch.__version__, 'triton': triton.__version__, "
                                    "'sglang': sglang.__version__, 'tilelang': tl}))"],
                                   capture_output=True, text=True, timeout=300, env=env).stdout.strip().splitlines()[-1]
            _jx_versions = json.loads(_jx_v)
        except Exception as _ex:
            _jx_versions = {"error": f"{type(_ex).__name__}: {_ex}"[:200]}
        _jx_manifest = {
            "sha256": _jx_h.hexdigest(), "bytes": _jx_path.stat().st_size, "dirs": _jx_sizes, "cache_root": str(_jx_cache),
            "created_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "run_mode": RUN_MODE,
            "wheel": getattr(globals().get("wheel"), "name", None),
            "patch_md5": {k: hashlib.md5(v.encode()).hexdigest() for k, v in sorted((globals().get("PATCHES") or {}).items())},
            "perf_applied": globals().get("PERF_APPLIED"), "qsa_default": globals().get("QSA_DEFAULT"),
            "versions": _jx_versions, "driver": sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1"),
            "gpu": sh("nvidia-smi --query-gpu=name --format=csv,noheader | head -1"),
            "rung": (SERVER_RUNG or {}).get("NAME"), "graph_bs": (SERVER_RUNG or {}).get("GRAPH_BS"),
            "env": {k: env.get(k) for k in ("TRITON_CACHE_DIR", "TILELANG_CACHE_DIR", "TORCHINDUCTOR_CACHE_DIR",
                                            "FLASHINFER_WORKSPACE_BASE", "CUDA_CACHE_PATH", "SGLANG_JIT_CACHE_DIR")}}
        write_json_atomic(_jx_dir / "manifest.json", _jx_manifest)
        JIT_EXPORT.update(path=str(_jx_path), sha256=_jx_manifest["sha256"], bytes=_jx_manifest["bytes"],
                          seconds=round(time.time() - _jx_t, 1))
    except Exception as _jx_ex:
        JIT_EXPORT["error"] = f"{type(_jx_ex).__name__}: {_jx_ex}"[:400]
    print("JIT_EXPORT", json.dumps(JIT_EXPORT), flush=True)
else:
    print("JIT_EXPORT skipped (competition rerun, QUICK run, or not configured)", flush=True)


JIT_EXPORT skipped (competition rerun, QUICK run, or not configured)


In [15]:
# [duck2:lbprobe]  competition rerun with AG["LBPROBE"]: the LB result names the run's first failure
# real score = no failure | 0.00 = server alive but stuck (stalled / stalled_noanswer / stalled_io / health) |
# Notebook Threw Exception = server process died, never came up, or a stage failed (died / noserver / late / stage) |
# Notebook Timeout = the server died on its strict idle leak check (died_leak) |
# Submission Scoring Error = the harness crashed or hung while the server was fine (harness_crash / harness_hang)
if RUN_MODE == "full" and CFG["AGENT"].get("LBPROBE"):
    _pk = globals().get("LBPROBE_KIND", "") or ("stage:" + ",".join(FAILED) if FAILED else "")
    print("LBPROBE RESULT:", _pk or "none (real score)", json.dumps(globals().get("LBPROBE_INFO", {}))[:4000], flush=True)
    if _pk in ("harness_crash", "harness_hang"):
        SUBMISSION.write_text("lbprobe " + _pk + "\n")
    elif _pk == "died_leak":
        while time.time() < T0 + CFG["FULL_BUDGET_S"] + 1800:
            time.sleep(60)
    elif _pk and _pk not in ("stalled", "stalled_noanswer", "stalled_io", "health"):
        raise RuntimeError("LBPROBE " + _pk)
